# Improve hard clone routing and measure expert learning

Follow-up to the completed T4 4→12 expert study. Five fixed seed-17 arms: hard, 1%-RMS router perturbation, persistent selection-bias balancing, family-aware hard routing with the same balancing, and sampled warmup control. Same trained parent, expert count/top-k, batches, Adam reset, LR and 750-update budget.

**Completed T4 results are consolidated in README.md and the main MoE_Training report.** Synthetic smoke checks remain separate from the GPU evidence. All evaluation is static hard routing. Training-window traffic and static-probe traffic remain separate.

Telemetry includes CE-only expert gradients, updates with weight-decay contribution removed, Adam state/counters, weight drift, sibling output diversity on fixed parent-hidden inputs, and held-out reset-to-parent expert ablations. Positive ablation deltas indicate probe-local benefit, not statistical significance or established semantic specialization.


In [ ]:
import sys, subprocess, torch
assert torch.cuda.is_available(), "Enable a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
subprocess.run([sys.executable,"-m","pip","install","numpy","matplotlib","datasets>=3","tokenizers>=0.20","huggingface_hub"],check=True)


In [ ]:
import os, json, hashlib, shutil
from pathlib import Path
root=Path("/content/MoE_Routing_Followup")
root.mkdir(exist_ok=True)
os.chdir(root)
payload = {'src/gpt.py': '"""Small causal decoder-only Transformer and FFN-upcycled sparse MoE."""\nimport copy\nimport math\nfrom dataclasses import dataclass\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n@dataclass\nclass GPTConfig:\n    vocab_size: int = 65\n    context: int = 128\n    layers: int = 4\n    width: int = 128\n    heads: int = 4\n    intermediate: int = 256\n    experts: int = 4\n    top_k: int = 2\n\nclass SwiGLU(nn.Module):\n    def __init__(self, width, intermediate):\n        super().__init__()\n        self.gate = nn.Linear(width, intermediate, bias=False)\n        self.up = nn.Linear(width, intermediate, bias=False)\n        self.down = nn.Linear(intermediate, width, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\nclass ExpertFFN(nn.Module):\n    def __init__(self, dense_ffn, width, experts, top_k):\n        super().__init__()\n        if not 1 <= top_k <= experts:\n            raise ValueError(\'Require 1 <= top_k <= experts\')\n        self.top_k = top_k\n        self.experts = nn.ModuleList([copy.deepcopy(dense_ffn) for _ in range(experts)])\n        self.router = nn.Linear(width, experts, bias=False)\n        nn.init.normal_(self.router.weight, std=.02)\n\n    def forward(self, hidden):\n        shape = hidden.shape\n        x = hidden.reshape(-1, shape[-1])\n        # FP32 for ordinary training/autocast, FP64 for independent CPU\n        # reference validation. Do not downcast a double reference gate.\n        gate_dtype = torch.float64 if x.dtype == torch.float64 else torch.float32\n        logits = F.linear(x.to(gate_dtype), self.router.weight.to(gate_dtype))\n        probs = logits.softmax(-1)\n        values, ids = logits.topk(self.top_k, dim=-1)\n        weights = values.softmax(-1).to(x.dtype)\n        output = torch.zeros_like(x)\n        for i, expert in enumerate(self.experts):\n            rows, slots = torch.where(ids == i)\n            if rows.numel():\n                contribution = expert(x[rows]) * weights[rows, slots, None]\n                output = output.index_add(0, rows, contribution)\n        counts = torch.bincount(ids.flatten(), minlength=len(self.experts)).float()\n        fraction = counts / ids.numel()\n        balance = len(self.experts) * (fraction.detach() * probs.mean(0)).sum()\n        entropy = -(probs * probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output.reshape(shape), {\'balance\': balance, \'counts\': counts, \'entropy\': entropy}\n\nclass Attention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        if cfg.width % cfg.heads:\n            raise ValueError(\'width must be divisible by heads\')\n        self.heads = cfg.heads\n        self.qkv = nn.Linear(cfg.width, 3*cfg.width, bias=False)\n        self.proj = nn.Linear(cfg.width, cfg.width, bias=False)\n\n    def forward(self, x):\n        b,t,d = x.shape\n        q,k,v = self.qkv(x).chunk(3, dim=-1)\n        q,k,v = [a.reshape(b,t,self.heads,d//self.heads).transpose(1,2) for a in (q,k,v)]\n        y = F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.)\n        return self.proj(y.transpose(1,2).contiguous().reshape(b,t,d))\n\nclass Block(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.norm_attn = nn.LayerNorm(cfg.width)\n        self.attn = Attention(cfg)\n        self.norm_ffn = nn.LayerNorm(cfg.width)\n        self.ffn = SwiGLU(cfg.width, cfg.intermediate)\n\n    def forward(self, x):\n        x = x + self.attn(self.norm_attn(x))\n        if isinstance(self.ffn, ExpertFFN):\n            y, aux = self.ffn(self.norm_ffn(x))\n        else:\n            y, aux = self.ffn(self.norm_ffn(x)), None\n        return x+y, aux\n\nclass TinyGPT(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.token_embedding = nn.Embedding(cfg.vocab_size,cfg.width)\n        self.position_embedding = nn.Embedding(cfg.context,cfg.width)\n        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.layers)])\n        self.norm = nn.LayerNorm(cfg.width)\n        self.head = nn.Linear(cfg.width,cfg.vocab_size,bias=False)\n        self.head.weight = self.token_embedding.weight\n        self.apply(self._initialize)\n        for name,p in self.named_parameters():\n            if name.endswith((\'attn.proj.weight\',\'ffn.down.weight\')):\n                nn.init.normal_(p, std=.02/math.sqrt(2*cfg.layers))\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module,(nn.Linear,nn.Embedding)):\n            nn.init.normal_(module.weight,std=.02)\n            if isinstance(module,nn.Linear) and module.bias is not None:\n                nn.init.zeros_(module.bias)\n\n    def forward(self, tokens):\n        if tokens.shape[1] > self.cfg.context:\n            raise ValueError(\'Sequence exceeds configured context\')\n        x = self.token_embedding(tokens) + self.position_embedding(torch.arange(tokens.shape[1],device=tokens.device))\n        aux=[]\n        for block in self.blocks:\n            x,a = block(x)\n            if a is not None:\n                aux.append(a)\n        return self.head(self.norm(x)), aux\n\n    def upcycle(self):\n        result = copy.deepcopy(self)\n        if any(isinstance(b.ffn,ExpertFFN) for b in result.blocks):\n            raise ValueError(\'Model is already an MoE\')\n        for block in result.blocks:\n            block.ffn = ExpertFFN(block.ffn,self.cfg.width,self.cfg.experts,self.cfg.top_k).to(self.head.weight.device)\n        return result\n\n    def parameter_counts(self):\n        total=sum(p.numel() for p in self.parameters())\n        inactive=0\n        for block in self.blocks:\n            if isinstance(block.ffn,ExpertFFN):\n                each=sum(p.numel() for p in block.ffn.experts[0].parameters())\n                inactive += (len(block.ffn.experts)-block.ffn.top_k)*each\n        return {\'total\':total, \'active_parameter_convention\':total-inactive,\n                \'convention\':\'All nonexpert parameters, including full tied embedding table, plus selected expert parameters. Not a FLOP count.\'}\n', 'src/routing_lab.py': '"""Controlled clone-family routing experiments; discrete sampled routes, not dense soft MoE."""\nimport copy\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom .gpt import ExpertFFN\n\nclass CloneFFN(ExpertFFN):\n    def __init__(self, parent, copies, router_noise=0., family_routing=False):\n        nn.Module.__init__(self)\n        self.top_k=parent.top_k\n        self.parents=len(parent.experts)\n        self.copies=copies\n        self.family_routing=family_routing\n        self.experts=nn.ModuleList([copy.deepcopy(e) for e in parent.experts for _ in range(copies)])\n        device=parent.router.weight.device\n        self.router=nn.Linear(parent.router.in_features,len(self.experts),bias=False).to(device)\n        with torch.no_grad():\n            self.router.weight.copy_(parent.router.weight.repeat_interleave(copies,dim=0))\n            if router_noise:\n                self.router.weight.add_(torch.randn_like(self.router.weight)*router_noise)\n        self.register_buffer(\'routing_bias\',torch.zeros(len(self.experts),device=device))\n        self.exploration=0.\n        self.temperature=1.\n        self.bias_rate=0.\n        self.route_generator=None\n        self.last_counts=None\n        self.last_hidden=None\n\n    def sample_ids(self,scores,k):\n        # Gumbel-top-k samples WITHOUT replacement. The discrete choice is not differentiated.\n        uniform=torch.rand(scores.shape,device=scores.device,generator=self.route_generator).clamp_(1e-7,1-1e-7)\n        gumbel=-torch.log(-torch.log(uniform))\n        return (scores/self.temperature+gumbel).topk(k,dim=-1).indices\n\n    def select(self,scores):\n        if self.family_routing:\n            family_scores=scores.reshape(-1,self.parents,self.copies).mean(-1)\n            families=family_scores.topk(self.top_k,dim=-1).indices\n            child_scores=scores.reshape(-1,self.parents,self.copies)[torch.arange(len(scores),device=scores.device)[:,None],families]\n            children=child_scores.argmax(-1)\n            hard=families*self.copies+children\n            if self.training and self.exploration:\n                sampled_children=self.sample_ids(child_scores.reshape(-1,self.copies),1).reshape(-1,self.top_k)\n                sampled=families*self.copies+sampled_children\n            else:\n                return hard\n        else:\n            hard=scores.topk(self.top_k,dim=-1).indices\n            if not (self.training and self.exploration):\n                return hard\n            sampled=self.sample_ids(scores,self.top_k)\n        choose=torch.rand((len(scores),1),device=scores.device,generator=self.route_generator)<self.exploration\n        return torch.where(choose,sampled,hard)\n\n    def forward(self,hidden):\n        shape=hidden.shape\n        x=hidden.reshape(-1,shape[-1])\n        logits=F.linear(x.float(),self.router.weight.float())\n        scores=logits+self.routing_bias\n        ids=self.select(scores)\n        # Bias/noise changes selection only; mixing weights use unbiased selected logits.\n        weights=logits.gather(1,ids).softmax(-1).to(x.dtype)\n        output=torch.zeros_like(x)\n        for i,expert in enumerate(self.experts):\n            rows,slots=torch.where(ids==i)\n            if rows.numel():\n                output=output.index_add(0,rows,expert(x[rows])*weights[rows,slots,None])\n        counts=torch.bincount(ids.flatten(),minlength=len(self.experts)).float()\n        probs=logits.softmax(-1)\n        balance=len(self.experts)*((counts/counts.sum()).detach()*probs.mean(0)).sum()\n        self.last_counts=counts.detach()\n        self.last_hidden=x[:32].detach()\n        if self.training and self.bias_rate:\n            with torch.no_grad():\n                self.routing_bias.add_(self.bias_rate*torch.sign(counts.mean()-counts))\n                self.routing_bias.sub_(self.routing_bias.mean())\n        entropy=-(probs*probs.clamp_min(1e-9).log()).sum(-1).mean()\n        return output.reshape(shape),{\'counts\':counts,\'balance\':balance,\'entropy\':entropy}\n\n\ndef grow(model,copies,router_noise=0.,family_routing=False):\n    result=copy.deepcopy(model)\n    for b in result.blocks:\n        if not isinstance(b.ffn,ExpertFFN):\n            raise ValueError(\'Grow an already-trained MoE; dense conversion is a separate phase.\')\n        b.ffn=CloneFFN(b.ffn,copies,router_noise,family_routing)\n    return result\n\n\ndef set_policy(model,exploration,temperature=1.,bias_rate=0.):\n    for b in model.blocks:\n        b.ffn.exploration=exploration\n        b.ffn.temperature=temperature\n        b.ffn.bias_rate=bias_rate\n\n\n@torch.no_grad()\ndef family_output_diversity(ffn):\n    """Relative RMS output distance on the SAME hidden probes; not a switching theorem."""\n    x=ffn.last_hidden\n    if x is None or ffn.copies<2:\n        return []\n    distances=[]\n    for parent in range(ffn.parents):\n        indices=torch.linspace(0,ffn.copies-1,min(ffn.copies,8)).round().long().tolist()\n        outputs=torch.stack([ffn.experts[parent*ffn.copies+j](x) for j in indices]).float()\n        flat=outputs.flatten(1)\n        scale=flat.square().mean().sqrt().clamp_min(1e-8)\n        pair=torch.pdist(flat)/flat.shape[-1]**.5/scale\n        distances.append(pair.mean().item())\n    return distances\n', 'src/learning_metrics.py': '"""Expert telemetry: separate exposure, CE gradients, decay and learned behavior."""\nimport copy\nimport torch\nfrom torch.nn import functional as F\nfrom .routing_lab import grow\n\n\ndef build_arm(parent, arm, cfg):\n    torch.manual_seed(cfg[\'seed\']+2)\n    model=grow(parent,cfg[\'copies\'],family_routing=arm==\'family_bias\')\n    if arm==\'noise\':\n        for layer,(source,block) in enumerate(zip(parent.blocks,model.blocks)):\n            weight=block.ffn.router.weight\n            gen=torch.Generator(device=weight.device).manual_seed(cfg[\'seed\']+1000+layer)\n            with torch.no_grad():\n                scale=source.ffn.router.weight.square().mean().sqrt()*cfg[\'noise_fraction\']\n                weight.add_(torch.randn(weight.shape,device=weight.device,generator=gen)*scale)\n    return model\n\n\ndef experts(model):\n    for layer,b in enumerate(model.blocks):\n        for index,e in enumerate(b.ffn.experts):\n            yield layer,index,e\n\n\ndef gradient_norms(model):\n    layers=[]\n    for b in model.blocks:\n        norms=[]\n        for expert in b.ffn.experts:\n            square=sum((p.grad.detach().float().square().sum() for p in expert.parameters() if p.grad is not None),\n                       b.ffn.router.weight.new_zeros(()))\n            norms.append(square.sqrt())\n        layers.append(torch.stack(norms))\n    return torch.stack(layers)\n\n\ndef snapshot_experts(model):\n    return {(layer,index):[p.detach().clone() for p in e.parameters()] for layer,index,e in experts(model)}\n\n\ndef decay_removed_updates(model,before,lr,weight_decay):\n    rows=[]\n    for b in model.blocks:\n        values=[];layer=len(rows)\n        for index,e in enumerate(b.ffn.experts):\n            norm=e.gate.weight.new_zeros(())\n            for previous,p in zip(before[layer,index],e.parameters()):\n                # AdamW skips parameters with grad=None, including decay.\n                decayed=previous*(1-lr*weight_decay) if p.grad is not None else previous\n                norm=norm+(p.detach()-decayed).float().square().sum()\n            values.append(norm.sqrt())\n        rows.append(torch.stack(values))\n    return torch.stack(rows)\n\n\n@torch.no_grad()\ndef fixed_hidden_probes(parent,tokens,device,size=32):\n    probes={};handles=[]\n    for layer,block in enumerate(parent.blocks):\n        def capture(module,args,layer=layer):\n            x=args[0].detach().reshape(-1,args[0].shape[-1])\n            ids=torch.linspace(0,len(x)-1,min(size,len(x)),device=x.device).long()\n            probes[layer]=x[ids].clone().cpu()\n        handles.append(block.ffn.register_forward_pre_hook(capture))\n    parent.eval();parent(tokens.to(device))\n    for handle in handles:handle.remove()\n    return probes\n\n\n@torch.no_grad()\ndef function_diversity(model,hidden):\n    result=[]\n    for layer,b in enumerate(model.blocks):\n        x=hidden[layer].to(next(model.parameters()).device)\n        outputs=torch.stack([e(x) for e in b.ffn.experts])\n        families=[]\n        for family in range(b.ffn.parents):\n            out=outputs[family*b.ffn.copies:(family+1)*b.ffn.copies].flatten(1)\n            denominator=out.square().mean().sqrt().clamp_min(1e-9)\n            rms=torch.pdist(out)/out.shape[-1]**.5/denominator\n            families.append(float(rms.mean().cpu()))\n        result.append(families)\n    return result\n\n\n@torch.no_grad()\ndef weight_drift(model,initial):\n    result=[]\n    for layer,b in enumerate(model.blocks):\n        row=[]\n        for index,e in enumerate(b.ffn.experts):\n            square=sum((p.detach().cpu()-base).square().sum().item()\n                       for p,base in zip(e.parameters(),initial[layer,index]))\n            denominator=sum(base.square().sum().item() for base in initial[layer,index])**.5\n            row.append({\'l2\':square**.5,\'relative_l2\':square**.5/max(denominator,1e-12)})\n        result.append(row)\n    return result\n\n\ndef optimizer_expert_state(model,opt):\n    return [[{\'state_matrices\':sum(bool(opt.state.get(p)) for p in e.parameters()),\n              \'matrix_steps\':[int(opt.state[p][\'step\']) if p in opt.state else 0 for p in e.parameters()]}\n             for e in b.ffn.experts] for b in model.blocks]\n\n\n@torch.no_grad()\ndef ablate_to_initial(model,parent,batch,device):\n    """Reset one expert at a time, keeping router weights fixed; held-out CE delta."""\n    model.eval();x,y=(t.to(device) for t in batch)\n    def ce():\n        logits,_=model(x)\n        return F.cross_entropy(logits.flatten(0,1),y.flatten()).item()\n    baseline=ce();result=[]\n    for layer,b in enumerate(model.blocks):\n        row=[]\n        for index,e in enumerate(b.ffn.experts):\n            source=parent.blocks[layer].ffn.experts[index//b.ffn.copies]\n            replacement=copy.deepcopy(source).to(device)\n            b.ffn.experts[index]=replacement\n            try:delta=ce()-baseline\n            finally:b.ffn.experts[index]=e\n            row.append(delta)\n        result.append(row)\n    return {\'baseline_probe_ce\':baseline,\'targets\':y.numel(),\n            \'expert_reset_ce_delta\':result,\n            \'interpretation\':\'Positive: resetting these learned weights hurts this fixed probe in the current model context. Negative: reset helps. Zero is not evidence of useful learning.\'}\n', 'src/improve_routing.py': '"""Controlled hard-routing repair study, with expert learning telemetry."""\nimport argparse,copy,hashlib,json,platform,time\nfrom collections import deque\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom .gpt import TinyGPT,GPTConfig\nfrom .routing_lab import set_policy\nfrom .learning_metrics import (build_arm,experts,gradient_norms,snapshot_experts,decay_removed_updates,\n    fixed_hidden_probes,function_diversity,weight_drift,optimizer_expert_state,ablate_to_initial)\n\n\ndef sample(data,gen,n,context):\n    start=torch.randint(len(data)-context,(n,),generator=gen)\n    seq=data[start[:,None]+torch.arange(context+1)]\n    return seq[:,:-1],seq[:,1:]\n\n\n@torch.no_grad()\ndef evaluate(model,batch,device):\n    model.eval();loss=0.;counts=None\n    for x,y in zip(batch[0].split(4),batch[1].split(4)):\n        logits,aux=model(x.to(device))\n        loss+=F.cross_entropy(logits.flatten(0,1),y.to(device).flatten(),reduction=\'sum\').item()\n        current=torch.stack([a[\'counts\'].cpu() for a in aux])\n        counts=current if counts is None else counts+current\n    return {\'ce\':loss/batch[1].numel(),\'counts\':counts}\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--config\',default=\'configs/t4.json\');p.add_argument(\'--device\',default=\'cuda\')\n    p.add_argument(\'--data\',default=\'data/tinystories_lab.npz\');p.add_argument(\'--parent\',default=\'artifacts/parent.pt\')\n    p.add_argument(\'--output\',required=True);p.add_argument(\'--toy\',action=\'store_true\')\n    a=p.parse_args();torch.set_num_threads(2);cfg=json.loads(Path(a.config).read_text())\n    out=Path(a.output);out.mkdir(parents=True,exist_ok=False)\n    torch.manual_seed(cfg[\'seed\'])\n    if a.toy:\n        parent=TinyGPT(GPTConfig(vocab_size=13,context=8,layers=1,width=16,heads=4,intermediate=32)).upcycle()\n        with torch.no_grad():parent.blocks[0].ffn.router.weight.zero_()\n        tokens=torch.arange(13).repeat(400);data={k:tokens.clone() for k in [\'train\',\'validation\',\'test\']}\n        provenance={\'toy\':True,\'meaning\':\'Execution/telemetry smoke only; not language-generalization evidence.\'}\n    else:\n        manifest=json.loads(Path(\'data_specs/parent_manifest.json\').read_text())\n        digest=hashlib.sha256(Path(a.parent).read_bytes()).hexdigest()\n        if digest!=manifest[\'parent_sha256\']:raise ValueError(\'Parent checkpoint checksum mismatch\')\n        saved=torch.load(a.parent,map_location=\'cpu\',weights_only=True)\n        parent=TinyGPT(GPTConfig(**saved[\'model_config\'])).upcycle();parent.load_state_dict(saved[\'model\'])\n        expected=json.loads(Path(\'data_specs/manifest.json\').read_text())\n        actual=json.loads(Path(a.data).with_suffix(\'\').joinpath(\'manifest.json\').read_text())\n        for key in [\'revision\',\'tokenizer_sha256\',\'vocab_size\',\'token_counts\']:\n            if actual[key]!=expected[key]:raise ValueError(\'Dataset mismatch: \'+key)\n        with np.load(a.data) as arrays:\n            data={k:torch.from_numpy(arrays[k].astype(\'int64\')) for k in [\'train\',\'validation\',\'test\']}\n            if int(arrays[\'vocab_size\'])!=parent.cfg.vocab_size:raise ValueError(\'Vocabulary mismatch\')\n        provenance={\'parent_sha256\':digest,\'parent_source\':manifest,\'data_manifest\':actual,\n                    \'npz_sha256\':hashlib.sha256(Path(a.data).read_bytes()).hexdigest()}\n    cfg[\'model\']=vars(parent.cfg)\n    (out/\'config.json\').write_text(json.dumps(cfg,indent=2));(out/\'provenance.json\').write_text(json.dumps(provenance,indent=2))\n    env={\'device\':a.device,\'gpu\':torch.cuda.get_device_name() if a.device.startswith(\'cuda\') else None,\n         \'torch\':str(torch.__version__),\'python\':platform.python_version(),\'precision\':\'FP32\',\n         \'source_sha256\':{str(f):hashlib.sha256(f.read_bytes()).hexdigest() for f in Path(\'src\').glob(\'*.py\')}}\n    (out/\'environment.json\').write_text(json.dumps(env,indent=2))\n    evals={k:sample(v,torch.Generator().manual_seed(9000+i),cfg[\'eval_sequences\'],parent.cfg.context)\n           for i,(k,v) in enumerate(data.items())}\n    calibration=sample(data[\'train\'],torch.Generator().manual_seed(9999),cfg[\'calibration_sequences\'],parent.cfg.context)\n    ablation=(evals[\'validation\'][0][:cfg[\'ablation_sequences\']],evals[\'validation\'][1][:cfg[\'ablation_sequences\']])\n    parent.to(a.device).eval();hidden=fixed_hidden_probes(parent,calibration[0],a.device)\n    with torch.no_grad():parent_logits=parent(calibration[0][:4].to(a.device))[0].cpu()\n    parent.cpu();summary=[]\n    checkpoints=Path(\'checkpoints\')/out.name;checkpoints.mkdir(parents=True,exist_ok=False)\n    for arm in cfg[\'arms\']:\n        model=build_arm(parent,arm,cfg).to(a.device)\n        model.eval()\n        with torch.no_grad():growth_drift=(model(calibration[0][:4].to(a.device))[0].cpu()-parent_logits).abs().max().item()\n        if a.device.startswith(\'cuda\'):torch.cuda.reset_peak_memory_stats()\n        for b in model.blocks:b.ffn.route_generator=torch.Generator(device=a.device).manual_seed(cfg[\'seed\']+100)\n        opt=torch.optim.AdamW(model.parameters(),lr=cfg[\'lr\'],betas=tuple(cfg[\'betas\']),weight_decay=cfg[\'weight_decay\'])\n        initial={(l,e):[p.detach().cpu().clone() for p in expert.parameters()] for l,e,expert in experts(model)}\n        layers=len(model.blocks);E=len(model.blocks[0].ffn.experts)\n        exposure=torch.zeros(layers,E);positive=torch.zeros(layers,E);gradient_sum=torch.zeros(layers,E)\n        update_sum=torch.zeros(layers,E);update_samples=0;last_update=torch.zeros(layers,E);last_update_step=0;ring=deque(maxlen=cfg[\'window_steps\'])\n        gradient_ring=deque(maxlen=cfg[\'window_steps\']);last_token_step=torch.full((layers,E),-1.)\n        gen=torch.Generator().manual_seed(cfg[\'seed\']+200);started=time.perf_counter();records=[]\n        for i in range(cfg[\'steps\']+1):\n            exploration=float(arm==\'warmup\' and i<cfg[\'warmup_steps\'])\n            set_policy(model,exploration,bias_rate=cfg[\'bias_rate\'] if arm in [\'bias\',\'family_bias\'] else 0.)\n            if i%cfg[\'eval_every\']==0 or i==cfg[\'steps\']:\n                val=evaluate(model,evals[\'validation\'],a.device);cal=evaluate(model,calibration,a.device)\n                load=torch.stack(list(ring)).sum(0) if ring else torch.zeros_like(exposure)\n                fractions=load/load.sum(-1,keepdim=True).clamp_min(1)\n                entropy=-(fractions*fractions.clamp_min(1e-12).log()).sum(-1)\n                r={\'arm\':arm,\'step\':i,\'validation_ce\':val[\'ce\'],\'exploration\':exploration,\n                   \'training_window_tokens\':min(i,cfg[\'window_steps\'])*cfg[\'batch_size\']*parent.cfg.context,\n                   \'window_route_fractions\':fractions.tolist(),\'window_zero_experts\':(load==0).sum(-1).tolist(),\n                   \'hard_probe_zero_experts\':(cal[\'counts\']==0).sum(-1).tolist(),\n                   \'window_load_cv\':(fractions.std(-1,unbiased=False)/fractions.mean(-1).clamp_min(1e-12)).tolist(),\n                   \'effective_experts\':entropy.exp().tolist() if i else [None]*layers,\'cumulative_tokens\':exposure.tolist(),\n                   \'window_positive_task_gradient_steps\':torch.stack(list(gradient_ring)).sum(0).tolist() if gradient_ring else torch.zeros_like(positive).tolist(),\n                   \'last_token_step\':last_token_step.tolist(),\n                   \'positive_task_gradient_steps\':positive.tolist(),\'mean_task_gradient_norm\':(gradient_sum/max(i,1)).tolist(),\n                   \'mean_task_gradient_norm_when_nonzero\':(gradient_sum/positive.clamp_min(1)).tolist(),\n                   \'mean_decay_removed_update_norm\':(update_sum/max(update_samples,1)).tolist(),\n                   \'update_samples\':update_samples,\'last_decay_removed_update_norm\':last_update.tolist(),\n                   \'last_update_sample_step\':last_update_step,\'weight_drift\':weight_drift(model,initial),\n                   \'family_output_relative_rms\':function_diversity(model,hidden),\n                   \'expert_optimizer_state\':optimizer_expert_state(model,opt),\n                   \'elapsed_seconds_including_diagnostics\':time.perf_counter()-started}\n                records.append(r)\n                with (out/\'metrics.jsonl\').open(\'a\') as f:f.write(json.dumps(r)+\'\\n\')\n                print(json.dumps({k:r[k] for k in [\'arm\',\'step\',\'validation_ce\',\'window_zero_experts\',\'hard_probe_zero_experts\',\'effective_experts\']}),flush=True)\n            if i==cfg[\'steps\']:break\n            model.train();opt.zero_grad(set_to_none=True)\n            x,y=(t.to(a.device) for t in sample(data[\'train\'],gen,cfg[\'batch_size\'],parent.cfg.context))\n            logits,aux=model(x);ce=F.cross_entropy(logits.flatten(0,1),y.flatten())\n            ce.backward()\n            task_norms=gradient_norms(model).cpu();positive+=(task_norms>0);gradient_sum+=task_norms\n            gradient_ring.append((task_norms>0).float())\n            norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg[\'gradient_clip\'])\n            if not torch.isfinite(norm) or not torch.isfinite(ce):raise RuntimeError(\'Nonfinite objective/gradient\')\n            capture=(i+1)%cfg[\'eval_every\']==0\n            before=snapshot_experts(model) if capture else None\n            opt.step()\n            if capture:\n                last_update=decay_removed_updates(model,before,cfg[\'lr\'],cfg[\'weight_decay\']).cpu()\n                update_sum+=last_update;update_samples+=1;last_update_step=i+1\n            routed=torch.stack([item[\'counts\'].detach().cpu() for item in aux]);exposure+=routed;ring.append(routed)\n            last_token_step[routed>0]=i+1\n        result={\'arm\':arm,\'parameters\':model.parameter_counts(),\'growth_probe_max_logit_drift\':growth_drift,\n                \'noise_std_by_layer\':[float(cfg[\'noise_fraction\']*source.ffn.router.weight.square().mean().sqrt()) if arm==\'noise\' else 0. for source,b in zip(parent.blocks,model.blocks)],\n                \'initial\':records[0],\'final\':records[-1],\n                \'test_ce\':evaluate(model,evals[\'test\'],a.device)[\'ce\'],\n                \'reset_ablation\':ablate_to_initial(model,parent,ablation,a.device) if cfg[\'ablation\'] else None}\n        result[\'peak_cuda_allocated_bytes\']=torch.cuda.max_memory_allocated() if a.device.startswith(\'cuda\') else None\n        summary.append(result);(out/\'summary.json\').write_text(json.dumps(summary,indent=2))\n        torch.save({\'model\':model.state_dict(),\'optimizer\':opt.state_dict(),\'sampler_rng\':gen.get_state(),\n                    \'routing_rng\':[b.ffn.route_generator.get_state() for b in model.blocks],\n                    \'torch_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all() if a.device.startswith(\'cuda\') else [],\n                    \'config\':cfg,\'arm\':arm,\'step\':cfg[\'steps\'],\'cumulative_tokens\':exposure,\n                    \'positive_task_gradient_steps\':positive,\'gradient_sum\':gradient_sum,\n                    \'update_sum\':update_sum,\'update_samples\':update_samples,\'last_update\':last_update,\n                    \'last_update_step\':last_update_step,\'rolling_counts\':list(ring),\n                    \'gradient_ring\':list(gradient_ring),\'last_token_step\':last_token_step},checkpoints/f\'{arm}.pt\')\n        del model,opt\n        if a.device.startswith(\'cuda\'):torch.cuda.empty_cache()\n    print(\'Completed:\',[r[\'arm\'] for r in summary],flush=True)\n\nif __name__==\'__main__\':main()\n', 'scripts/prepare_tinystories_lab.py': '"""Prepare a frozen bounded TinyStories subset and train a BPE on TRAIN stories only."""\nimport argparse,hashlib,json\nfrom pathlib import Path\nimport numpy as np\nfrom datasets import load_dataset\nfrom huggingface_hub import HfApi\nfrom tokenizers import Tokenizer,models,trainers,pre_tokenizers,decoders\n\np=argparse.ArgumentParser()\np.add_argument(\'--output\',default=\'data/tinystories_lab\')\np.add_argument(\'--train-stories\',type=int,default=12000)\np.add_argument(\'--heldout-stories\',type=int,default=1200)\np.add_argument(\'--vocab-size\',type=int,default=4096)\np.add_argument(\'--revision\',default=\'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64\')\np.add_argument(\'--tokenizer\',help=\'Reuse the archived tokenizer for exact token IDs instead of fitting a new one\')\na=p.parse_args()\nroot=Path(a.output);root.mkdir(parents=True,exist_ok=False)\nrevision=a.revision\nsplits={}\nfor key,n in [(\'train\',a.train_stories),(\'validation\',a.heldout_stories*2)]:\n ds=load_dataset(\'roneneldan/TinyStories\',split=key,revision=revision,streaming=True)\n # Bounded shuffled sample; order and revision are recorded, not a globally uniform dataset sample.\n ds=ds.shuffle(seed=17,buffer_size=2000)\n splits[key]=[r[\'text\'] for r in ds.take(n)]\ntrain=splits[\'train\']; held=splits[\'validation\']\n# Remove exact document overlaps across splits; near duplicates are not addressed here.\nseen=set(hashlib.sha256(t.encode()).hexdigest() for t in train)\nvalidation=[];test=[]\nfor i,text in enumerate(held):\n digest=hashlib.sha256(text.encode()).hexdigest()\n if digest in seen:continue\n seen.add(digest)\n (validation if i%2==0 else test).append(text)\nif a.tokenizer:\n tokenizer=Tokenizer.from_file(a.tokenizer)\n if tokenizer.get_vocab_size()!=a.vocab_size or tokenizer.token_to_id(\'[EOS]\') is None:\n  raise ValueError(\'Archived tokenizer vocabulary/special tokens do not match configuration\')\nelse:\n tokenizer=Tokenizer(models.BPE(unk_token=\'[UNK]\'))\n tokenizer.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)\n tokenizer.decoder=decoders.ByteLevel()\n trainer=trainers.BpeTrainer(vocab_size=a.vocab_size,special_tokens=[\'[UNK]\',\'[EOS]\'],initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\n tokenizer.train_from_iterator(train,trainer=trainer)\nif a.tokenizer:\n import shutil\n shutil.copyfile(a.tokenizer,root/\'tokenizer.json\')\nelse:\n tokenizer.save(str(root/\'tokenizer.json\'))\neos=tokenizer.token_to_id(\'[EOS]\')\nencoded={}\nfor key,stories in [(\'train\',train),(\'validation\',validation),(\'test\',test)]:\n ids=[]\n for text in stories:ids.extend(tokenizer.encode(text).ids+[eos])\n encoded[key]=np.asarray(ids,dtype=np.int32)\nnp.savez(root.with_suffix(\'.npz\'),**encoded,vocab_size=tokenizer.get_vocab_size())\nmanifest={\'dataset\':\'roneneldan/TinyStories\',\'revision\':revision,\'shuffle_seed\':17,\'shuffle_buffer\':2000,\n          \'train_stories\':len(train),\'validation_stories\':len(validation),\'test_stories\':len(test),\n          \'token_counts\':{k:len(v) for k,v in encoded.items()},\'vocab_size\':tokenizer.get_vocab_size(),\n          \'tokenizer_sha256\':hashlib.sha256((root/\'tokenizer.json\').read_bytes()).hexdigest(),\n          \'packed_policy\':\'concatenate documents with EOS; causal attention may cross EOS\',\n          \'deduplication\':\'exact cross-split document hashing only; not near-duplicate decontamination\'}\n(root/\'manifest.json\').write_text(json.dumps(manifest,indent=2))\nprint(json.dumps(manifest,indent=2))\n', 'scripts/run_logged.py': '"""Stream child output to notebook and disk, preserving its actual failure."""\nfrom collections import deque\nfrom pathlib import Path\nimport subprocess\n\n\ndef run_logged(command, log_path, cwd=None):\n    """Run a command with combined stdout/stderr; refuse to replace its log."""\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    tail = deque(maxlen=40)\n    with log_path.open(\'x\', encoding=\'utf-8\') as log:\n        heading = f\'Command: {command!r}\\n\'\n        print(heading, end=\'\', flush=True)\n        log.write(heading)\n        log.flush()\n        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,\n                              stderr=subprocess.STDOUT, text=True,\n                              encoding=\'utf-8\', errors=\'replace\', bufsize=1) as process:\n            for line in process.stdout:\n                print(line, end=\'\', flush=True)\n                log.write(line)\n                log.flush()\n                tail.append(line)\n            code = process.wait()\n        log.write(f\'\\nExit status: {code}\\n\')\n        log.flush()\n    if code:\n        details = \'\'.join(tail)[-20000:]\n        raise RuntimeError(\n            f\'Command failed with exit status {code}. Full output: {log_path.resolve()}\\n\'\n            f\'Last output lines (the underlying error is here):\\n{details}\'\n        )\n    return log_path\n', 'scripts/report_learning.py': '"""Expert-learning and routing report from measured records only."""\nimport argparse,json\nfrom pathlib import Path\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.colors import TwoSlopeNorm\n\np=argparse.ArgumentParser();p.add_argument(\'--results\',required=True);a=p.parse_args()\nroot=Path(a.results);summary=json.loads((root/\'summary.json\').read_text())\nrows=[json.loads(line) for line in (root/\'metrics.jsonl\').read_text().splitlines()]\nprovenance=json.loads((root/\'provenance.json\').read_text())\nfig,axes=plt.subplots(1,3,figsize=(15,4.5))\nfor arm in summary:\n    name=arm[\'arm\'];records=[r for r in rows if r[\'arm\']==name];trained=[r for r in records if r[\'step\']>0]\n    axes[0].plot([r[\'step\'] for r in records],[r[\'validation_ce\'] for r in records],label=name)\n    axes[1].plot([r[\'step\'] for r in trained],[max(r[\'window_zero_experts\']) for r in trained],label=name)\n    axes[2].plot([r[\'step\'] for r in trained],[max(r[\'hard_probe_zero_experts\']) for r in trained],label=name)\nfor ax,title in zip(axes,[\'Hard-evaluation validation CE\',\'Max layer zero traffic (training window)\',\'Max layer zero traffic (static hard probe)\']):\n    ax.set(title=title,xlabel=\'Continuation update\');ax.grid(alpha=.2);ax.legend(fontsize=8)\nfig.tight_layout();fig.savefig(root/\'routing_curves.png\',dpi=160);plt.close(fig)\nlines=[\'# Hard-routing repair and expert-learning measurements\',\'\',\n       \'**CPU synthetic execution smoke; not language-training evidence.**\' if provenance.get(\'toy\') else \'Measured continuation from the pinned trained parent; all arms use matched data order and reset AdamW.\',\n       \'\', \'| Arm | Start / final validation CE | Lifetime zero experts | Final window zero experts | Static probe zero experts | Zero CE-gradient experts | No Adam state | Positive reset deltas (>1e-6) |\',\n       \'|---|---|---|---|---|---|---|---|\']\nfor arm in summary:\n    f=arm[\'final\'];name=arm[\'arm\']\n    lifetime=np.asarray(f[\'cumulative_tokens\']);grads=np.asarray(f[\'positive_task_gradient_steps\'])\n    no_state=sum(s[\'state_matrices\']==0 for row in f[\'expert_optimizer_state\'] for s in row)\n    deltas=np.asarray(arm[\'reset_ablation\'][\'expert_reset_ce_delta\']) if arm[\'reset_ablation\'] else np.array([])\n    positives=int((deltas>1e-6).sum()) if deltas.size else \'not measured\'\n    lines.append(f"| {name} | {arm[\'initial\'][\'validation_ce\']:.6f} / {f[\'validation_ce\']:.6f} | {int((lifetime==0).sum())} | {sum(f[\'window_zero_experts\'])} | {sum(f[\'hard_probe_zero_experts\'])} | {int((grads==0).sum())} | {no_state} | {positives} |")\n    arrays=[np.log10(lifetime.flatten()+1),np.log10(np.asarray(f[\'mean_task_gradient_norm_when_nonzero\']).flatten()+1e-12),\n            np.log10(np.asarray(f[\'last_decay_removed_update_norm\']).flatten()+1e-12),deltas.flatten() if deltas.size else np.zeros(lifetime.size)]\n    labels=[\'Lifetime token routes + 1\\n(log10)\',\'CE gradient norm when nonzero\\n(log10)\',\n            f"Decay-removed update, step {f[\'last_update_sample_step\']}\\n(log10)",\'Reset-to-parent CE change\']\n    fig,axs=plt.subplots(1,4,figsize=(14,max(4.5,lifetime.size*.13)),layout=\'constrained\')\n    for idx,(ax,data,label) in enumerate(zip(axs,arrays,labels)):\n        kwargs={}\n        if idx==3:\n            limit=max(abs(data).max(),1e-8);kwargs={\'norm\':TwoSlopeNorm(vmin=-limit,vcenter=0,vmax=limit),\'cmap\':\'RdBu_r\'}\n        im=ax.imshow(data[:,None],aspect=\'auto\',**kwargs);ax.set(xticks=[]);ax.set_title(label,fontsize=10)\n        ax.set_yticks(range(lifetime.size),[f\'L{l+1}/E{e}\' for l in range(lifetime.shape[0]) for e in range(lifetime.shape[1])],fontsize=6)\n        fig.colorbar(im,ax=ax,shrink=.7)\n    fig.suptitle(name);fig.savefig(root/f\'experts-{name}.png\',dpi=150);plt.close(fig)\nlines+=[\'\',\'![Routing/loss comparison](routing_curves.png)\',\'\',\'## Interpretation\',\n        \'\', \'Token traffic is exposure, not proof of learning. CE gradients are measured before clipping and contain no auxiliary-loss term in this comparison. AdamW parameter drift includes decay; sampled update norms subtract the decay-only prediction (including the grad=None skip rule). Those updates include current and retained Adam moments, not solely this step’s gradient.\',\n        \'\', \'Output diversity compares sibling functions on the same fixed parent-hidden inputs across arms. Diversity alone does not prove useful specialization. Reset ablation measures held-out CE after replacing one expert’s current weights with its original parent weights while retaining router weights/bias. Positive deltas suggest those learned weights help this specific probe/current context; negative deltas favor resetting. Subsequent hidden states and routes may change as part of that causal effect.\',\n        \'\', \'The positive-ablation threshold 1e-6 is a numerical display convention, not statistical significance. The small fixed ablation probe cannot establish generalization per expert. State/coverage and task loss must be read together. Training-window coverage under continually updated selection biases can coexist with starvation on a static hard probe; the two columns deliberately remain separate.\',\n        \'\', \'Full per-layer/per-expert telemetry and optimizer counters are in summary.json and metrics.jsonl. Checkpoints retain weights, moments, biases and RNG/counter state. No exact stochastic-resume equivalence claim is made.\']\nfor arm in summary:\n    lines+=[\'\',f"## {arm[\'arm\']} expert telemetry",\'\',f"![Expert metrics](experts-{arm[\'arm\']}.png)"]\n(root/\'REPORT.md\').write_text(\'\\n\'.join(lines)+\'\\n\')\nprint(\'\\n\'.join(lines[:13]))\n', 'configs/t4.json': '{\n  "seed": 17,\n  "copies": 3,\n  "arms": [\n    "hard",\n    "noise",\n    "bias",\n    "family_bias",\n    "warmup"\n  ],\n  "steps": 750,\n  "batch_size": 4,\n  "eval_sequences": 32,\n  "calibration_sequences": 64,\n  "ablation_sequences": 4,\n  "eval_every": 25,\n  "window_steps": 50,\n  "warmup_steps": 125,\n  "noise_fraction": 0.01,\n  "bias_rate": 0.001,\n  "lr": 0.0003,\n  "betas": [\n    0.9,\n    0.95\n  ],\n  "weight_decay": 0.01,\n  "gradient_clip": 1.0,\n  "ablation": true\n}', 'data_specs/parent_manifest.json': '{\n  "parent_sha256": "8e15403572a5a629b0b6ef7060275bbe5efb68cd9b07bda597bcc143dced911d",\n  "source": "MoE_Clone_Lab-evidence/checkpoints/clone-first-20261012-133612/parent.pt",\n  "architecture": {\n    "vocab_size": 4096,\n    "context": 128,\n    "layers": 4,\n    "width": 192,\n    "heads": 6,\n    "intermediate": 384,\n    "experts": 4,\n    "top_k": 2\n  },\n  "parent_training_steps": 250,\n  "dense_training_steps": 500,\n  "parent_validation_ce": 4.123746156692505\n}', 'data_specs/manifest.json': '{\n  "dataset": "roneneldan/TinyStories",\n  "revision": "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64",\n  "shuffle_seed": 17,\n  "shuffle_buffer": 2000,\n  "train_stories": 12000,\n  "validation_stories": 1200,\n  "test_stories": 1200,\n  "token_counts": {\n    "train": 2690332,\n    "validation": 250020,\n    "test": 250173\n  },\n  "vocab_size": 4096,\n  "tokenizer_sha256": "deb6208dc72ae00908b6449c0c24d06a875452552381e83cfce141f6d0623093",\n  "packed_policy": "concatenate documents with EOS; causal attention may cross EOS",\n  "deduplication": "exact cross-split document hashing only; not near-duplicate decontamination"\n}', 'data_specs/tokenizer.json': '{\n  "version": "1.0",\n  "truncation": null,\n  "padding": null,\n  "added_tokens": [\n    {\n      "id": 0,\n      "content": "[UNK]",\n      "single_word": false,\n      "lstrip": false,\n      "rstrip": false,\n      "normalized": false,\n      "special": true\n    },\n    {\n      "id": 1,\n      "content": "[EOS]",\n      "single_word": false,\n      "lstrip": false,\n      "rstrip": false,\n      "normalized": false,\n      "special": true\n    }\n  ],\n  "normalizer": null,\n  "pre_tokenizer": {\n    "type": "ByteLevel",\n    "add_prefix_space": false,\n    "trim_offsets": true,\n    "use_regex": true\n  },\n  "post_processor": null,\n  "decoder": {\n    "type": "ByteLevel",\n    "add_prefix_space": true,\n    "trim_offsets": true,\n    "use_regex": true\n  },\n  "model": {\n    "type": "BPE",\n    "dropout": null,\n    "unk_token": "[UNK]",\n    "continuing_subword_prefix": null,\n    "end_of_word_suffix": null,\n    "fuse_unk": false,\n    "byte_fallback": false,\n    "ignore_merges": false,\n    "vocab": {\n      "[UNK]": 0,\n      "[EOS]": 1,\n      "!": 2,\n      "\\"": 3,\n      "#": 4,\n      "$": 5,\n      "%": 6,\n      "&": 7,\n      "\'": 8,\n      "(": 9,\n      ")": 10,\n      "*": 11,\n      "+": 12,\n      ",": 13,\n      "-": 14,\n      ".": 15,\n      "/": 16,\n      "0": 17,\n      "1": 18,\n      "2": 19,\n      "3": 20,\n      "4": 21,\n      "5": 22,\n      "6": 23,\n      "7": 24,\n      "8": 25,\n      "9": 26,\n      ":": 27,\n      ";": 28,\n      "<": 29,\n      "=": 30,\n      ">": 31,\n      "?": 32,\n      "@": 33,\n      "A": 34,\n      "B": 35,\n      "C": 36,\n      "D": 37,\n      "E": 38,\n      "F": 39,\n      "G": 40,\n      "H": 41,\n      "I": 42,\n      "J": 43,\n      "K": 44,\n      "L": 45,\n      "M": 46,\n      "N": 47,\n      "O": 48,\n      "P": 49,\n      "Q": 50,\n      "R": 51,\n      "S": 52,\n      "T": 53,\n      "U": 54,\n      "V": 55,\n      "W": 56,\n      "X": 57,\n      "Y": 58,\n      "Z": 59,\n      "[": 60,\n      "\\\\": 61,\n      "]": 62,\n      "^": 63,\n      "_": 64,\n      "`": 65,\n      "a": 66,\n      "b": 67,\n      "c": 68,\n      "d": 69,\n      "e": 70,\n      "f": 71,\n      "g": 72,\n      "h": 73,\n      "i": 74,\n      "j": 75,\n      "k": 76,\n      "l": 77,\n      "m": 78,\n      "n": 79,\n      "o": 80,\n      "p": 81,\n      "q": 82,\n      "r": 83,\n      "s": 84,\n      "t": 85,\n      "u": 86,\n      "v": 87,\n      "w": 88,\n      "x": 89,\n      "y": 90,\n      "z": 91,\n      "{": 92,\n      "|": 93,\n      "}": 94,\n      "~": 95,\n      "¡": 96,\n      "¢": 97,\n      "£": 98,\n      "¤": 99,\n      "¥": 100,\n      "¦": 101,\n      "§": 102,\n      "¨": 103,\n      "©": 104,\n      "ª": 105,\n      "«": 106,\n      "¬": 107,\n      "®": 108,\n      "¯": 109,\n      "°": 110,\n      "±": 111,\n      "²": 112,\n      "³": 113,\n      "´": 114,\n      "µ": 115,\n      "¶": 116,\n      "·": 117,\n      "¸": 118,\n      "¹": 119,\n      "º": 120,\n      "»": 121,\n      "¼": 122,\n      "½": 123,\n      "¾": 124,\n      "¿": 125,\n      "À": 126,\n      "Á": 127,\n      "Â": 128,\n      "Ã": 129,\n      "Ä": 130,\n      "Å": 131,\n      "Æ": 132,\n      "Ç": 133,\n      "È": 134,\n      "É": 135,\n      "Ê": 136,\n      "Ë": 137,\n      "Ì": 138,\n      "Í": 139,\n      "Î": 140,\n      "Ï": 141,\n      "Ð": 142,\n      "Ñ": 143,\n      "Ò": 144,\n      "Ó": 145,\n      "Ô": 146,\n      "Õ": 147,\n      "Ö": 148,\n      "×": 149,\n      "Ø": 150,\n      "Ù": 151,\n      "Ú": 152,\n      "Û": 153,\n      "Ü": 154,\n      "Ý": 155,\n      "Þ": 156,\n      "ß": 157,\n      "à": 158,\n      "á": 159,\n      "â": 160,\n      "ã": 161,\n      "ä": 162,\n      "å": 163,\n      "æ": 164,\n      "ç": 165,\n      "è": 166,\n      "é": 167,\n      "ê": 168,\n      "ë": 169,\n      "ì": 170,\n      "í": 171,\n      "î": 172,\n      "ï": 173,\n      "ð": 174,\n      "ñ": 175,\n      "ò": 176,\n      "ó": 177,\n      "ô": 178,\n      "õ": 179,\n      "ö": 180,\n      "÷": 181,\n      "ø": 182,\n      "ù": 183,\n      "ú": 184,\n      "û": 185,\n      "ü": 186,\n      "ý": 187,\n      "þ": 188,\n      "ÿ": 189,\n      "Ā": 190,\n      "ā": 191,\n      "Ă": 192,\n      "ă": 193,\n      "Ą": 194,\n      "ą": 195,\n      "Ć": 196,\n      "ć": 197,\n      "Ĉ": 198,\n      "ĉ": 199,\n      "Ċ": 200,\n      "ċ": 201,\n      "Č": 202,\n      "č": 203,\n      "Ď": 204,\n      "ď": 205,\n      "Đ": 206,\n      "đ": 207,\n      "Ē": 208,\n      "ē": 209,\n      "Ĕ": 210,\n      "ĕ": 211,\n      "Ė": 212,\n      "ė": 213,\n      "Ę": 214,\n      "ę": 215,\n      "Ě": 216,\n      "ě": 217,\n      "Ĝ": 218,\n      "ĝ": 219,\n      "Ğ": 220,\n      "ğ": 221,\n      "Ġ": 222,\n      "ġ": 223,\n      "Ģ": 224,\n      "ģ": 225,\n      "Ĥ": 226,\n      "ĥ": 227,\n      "Ħ": 228,\n      "ħ": 229,\n      "Ĩ": 230,\n      "ĩ": 231,\n      "Ī": 232,\n      "ī": 233,\n      "Ĭ": 234,\n      "ĭ": 235,\n      "Į": 236,\n      "į": 237,\n      "İ": 238,\n      "ı": 239,\n      "Ĳ": 240,\n      "ĳ": 241,\n      "Ĵ": 242,\n      "ĵ": 243,\n      "Ķ": 244,\n      "ķ": 245,\n      "ĸ": 246,\n      "Ĺ": 247,\n      "ĺ": 248,\n      "Ļ": 249,\n      "ļ": 250,\n      "Ľ": 251,\n      "ľ": 252,\n      "Ŀ": 253,\n      "ŀ": 254,\n      "Ł": 255,\n      "ł": 256,\n      "Ń": 257,\n      "he": 258,\n      "Ġt": 259,\n      "Ġa": 260,\n      "Ġs": 261,\n      "Ġw": 262,\n      "nd": 263,\n      "Ġthe": 264,\n      "ed": 265,\n      "Ġand": 266,\n      "Ġto": 267,\n      "in": 268,\n      "Ġb": 269,\n      "Ġh": 270,\n      "Ġwa": 271,\n      "re": 272,\n      "ou": 273,\n      "Ġf": 274,\n      "it": 275,\n      "Ġl": 276,\n      "Ġc": 277,\n      "Ġhe": 278,\n      "Ġd": 279,\n      "er": 280,\n      "Ġm": 281,\n      "Ġwas": 282,\n      "Ġp": 283,\n      "om": 284,\n      "Ġo": 285,\n      "ĠT": 286,\n      "ay": 287,\n      "ing": 288,\n      "ar": 289,\n      "is": 290,\n      "il": 291,\n      "Ġg": 292,\n      "id": 293,\n      "at": 294,\n      "Ġsa": 295,\n      "en": 296,\n      "Ġn": 297,\n      "Ġha": 298,\n      "ĠS": 299,\n      "an": 300,\n      "im": 301,\n      "ĠThe": 302,\n      "or": 303,\n      "on": 304,\n      "Ġit": 305,\n      "Ġth": 306,\n      "ll": 307,\n      "le": 308,\n      "ĠH": 309,\n      "et": 310,\n      "Ġher": 311,\n      "ir": 312,\n      "ot": 313,\n      "ĠShe": 314,\n      "ĠHe": 315,\n      "Ġin": 316,\n      "es": 317,\n      "ver": 318,\n      "ow": 319,\n      "ut": 320,\n      "ck": 321,\n      "Ġsaid": 322,\n      "Ġe": 323,\n      "Ġu": 324,\n      "ld": 325,\n      "Ġy": 326,\n      "Ġ\\"": 327,\n      "oo": 328,\n      "ig": 329,\n      "ĠThey": 330,\n      "Ġbe": 331,\n      "am": 332,\n      "Ġr": 333,\n      "Ġst": 334,\n      "ily": 335,\n      "ce": 336,\n      "Ġshe": 337,\n      "pp": 338,\n      "ĠI": 339,\n      "ith": 340,\n      "ke": 341,\n      "On": 342,\n      "Ġwith": 343,\n      "ve": 344,\n      "ked": 345,\n      "Ġof": 346,\n      "Ġon": 347,\n      "Lily": 348,\n      "ri": 349,\n      "Ġso": 350,\n      "Ġyou": 351,\n      "Ġhis": 352,\n      "Ġpl": 353,\n      "Ġday": 354,\n      "ad": 355,\n      "nt": 356,\n      "very": 357,\n      "Ġthat": 358,\n      "Ġup": 359,\n      "Ġplay": 360,\n      "st": 361,\n      "Ġhad": 362,\n      "Ġwe": 363,\n      "Ġmom": 364,\n      "Ġthey": 365,\n      "ĠLily": 366,\n      "el": 367,\n      "Ġfor": 368,\n      "my": 369,\n      "\'s": 370,\n      "ould": 371,\n      "un": 372,\n      "ĠB": 373,\n      "The": 374,\n      "itt": 375,\n      "ent": 376,\n      "Ġli": 377,\n      "out": 378,\n      "her": 379,\n      "Ġhapp": 380,\n      "ch": 381,\n      "Ġwant": 382,\n      "Ġsh": 383,\n      "ly": 384,\n      "ound": 385,\n      "ittle": 386,\n      "ime": 387,\n      "all": 388,\n      "Ġdo": 389,\n      "Ġlittle": 390,\n      "end": 391,\n      "Ġvery": 392,\n      "ome": 393,\n      "Ġtime": 394,\n      "se": 395,\n      "Ġwh": 396,\n      "Ġthere": 397,\n      "ĠĊ": 398,\n      "ht": 399,\n      "Ġsm": 400,\n      "Ġk": 401,\n      "Ġre": 402,\n      "Ġne": 403,\n      "al": 404,\n      "Ġnot": 405,\n      "Ġbig": 406,\n      "Ġhappy": 407,\n      "Ġas": 408,\n      "ack": 409,\n      "riend": 410,\n      "Ġfriend": 411,\n      "Ġbut": 412,\n      "Ġsaw": 413,\n      "Ġan": 414,\n      "ĠM": 415,\n      "ra": 416,\n      ".\\"": 417,\n      "Ġis": 418,\n      "One": 419,\n      "ry": 420,\n      "ĠIt": 421,\n      "\'t": 422,\n      "ved": 423,\n      "!\\"": 424,\n      "ide": 425,\n      "Once": 426,\n      "Ġloo": 427,\n      "ake": 428,\n      "ug": 429,\n      "Ġwere": 430,\n      "ter": 431,\n      "Ġlo": 432,\n      "ore": 433,\n      "ec": 434,\n      "ĠTim": 435,\n      "Ġbo": 436,\n      "Ġhim": 437,\n      "irl": 438,\n      "Ġupon": 439,\n      "Ġtoo": 440,\n      "Ġgirl": 441,\n      "Ġgo": 442,\n      "Ġse": 443,\n      "ard": 444,\n      "Ġj": 445,\n      "Ġout": 446,\n      "Ġwanted": 447,\n      "ill": 448,\n      "Ġsmil": 449,\n      "fu": 450,\n      "way": 451,\n      "hen": 452,\n      "ind": 453,\n      "Ġat": 454,\n      "Ġsp": 455,\n      "Ġcould": 456,\n      "ur": 457,\n      "Ġthem": 458,\n      "ĠJ": 459,\n      "Ġex": 460,\n      "ĠA": 461,\n      "ain": 462,\n      "Ġdid": 463,\n      "Ġare": 464,\n      "Ġtheir": 465,\n      "round": 466,\n      "art": 467,\n      "Ġwent": 468,\n      "hed": 469,\n      "ic": 470,\n      "Ġnam": 471,\n      "lp": 472,\n      "rom": 473,\n      "Ġhave": 474,\n      "Ġcan": 475,\n      "hing": 476,\n      "?\\"": 477,\n      "Ġhelp": 478,\n      "ful": 479,\n      "Ġfriends": 480,\n      "um": 481,\n      "ight": 482,\n      "Ġall": 483,\n      "Ġkn": 484,\n      "ood": 485,\n      "one": 486,\n      "Ġback": 487,\n      "ark": 488,\n      "Ġsmiled": 489,\n      "Ã¢": 490,\n      "âĤ": 491,\n      "âĤ¬": 492,\n      "ĠBen": 493,\n      "Ġcl": 494,\n      "Ġnamed": 495,\n      "are": 496,\n      "Ġfun": 497,\n      "Ġasked": 498,\n      "oug": 499,\n      "Ġno": 500,\n      "elt": 501,\n      "Ġle": 502,\n      "Ġloved": 503,\n      "op": 504,\n      "ĠOne": 505,\n      "You": 506,\n      "Ġfelt": 507,\n      "Ġsc": 508,\n      "Ġlooked": 509,\n      "ĠTimmy": 510,\n      "ick": 511,\n      "Ġal": 512,\n      "Ġaround": 513,\n      "side": 514,\n      "ame": 515,\n      "get": 516,\n      "omet": 517,\n      "ĠTom": 518,\n      "Ġfe": 519,\n      "Ġboy": 520,\n      "Ġtoy": 521,\n      "ure": 522,\n      "Ġman": 523,\n      "Ġme": 524,\n      "Ġsee": 525,\n      "ird": 526,\n      "Ġro": 527,\n      "Ġlike": 528,\n      "Ġstart": 529,\n      "Ġbr": 530,\n      "Ġsome": 531,\n      "dd": 532,\n      "Ġwould": 533,\n      "as": 534,\n      "Ġsay": 535,\n      "ice": 536,\n      "ong": 537,\n      "ĠW": 538,\n      "Ġsomet": 539,\n      "Ġbird": 540,\n      "Ġwor": 541,\n      "gether": 542,\n      "king": 543,\n      "ree": 544,\n      "Ġag": 545,\n      ",\\"": 546,\n      "Ġwhat": 547,\n      "ade": 548,\n      "ag": 549,\n      "ie": 550,\n      "Ġfa": 551,\n      "ings": 552,\n      "Ġstarted": 553,\n      "own": 554,\n      "Ġtogether": 555,\n      "Ġmake": 556,\n      "Ġaway": 557,\n      "ared": 558,\n      "Ġsomething": 559,\n      "ited": 560,\n      "Ġran": 561,\n      "if": 562,\n      "ĠBut": 563,\n      "oud": 564,\n      "Ġexc": 565,\n      "Ġpark": 566,\n      "Ġco": 567,\n      "Ġmade": 568,\n      "Ġsad": 569,\n      "Ġot": 570,\n      "ble": 571,\n      "hat": 572,\n      "ell": 573,\n      "Ġwhen": 574,\n      "Ġnew": 575,\n      "Ġfrom": 576,\n      "pped": 577,\n      "ĠYou": 578,\n      "Ġother": 579,\n      "Ġpr": 580,\n      "Ġmu": 581,\n      "ĠL": 582,\n      "Ġcar": 583,\n      "Ġla": 584,\n      "Ġput": 585,\n      "Ġagain": 586,\n      "Ġhome": 587,\n      "Ġthen": 588,\n      "pl": 589,\n      "Ġfound": 590,\n      "est": 591,\n      "ought": 592,\n      "ss": 593,\n      "Ġwho": 594,\n      "qu": 595,\n      "Ġwal": 596,\n      "ep": 597,\n      "Ġhug": 598,\n      "Ġplaying": 599,\n      "Ġfl": 600,\n      "ally": 601,\n      "ook": 602,\n      "ried": 603,\n      "Ġgood": 604,\n      "Ġget": 605,\n      "Ġdec": 606,\n      "Ġexcited": 607,\n      "Ġch": 608,\n      "ach": 609,\n      "ous": 610,\n      "Ġsw": 611,\n      "ĠE": 612,\n      "ro": 613,\n      "Ġgot": 614,\n      "ny": 615,\n      "Ġsays": 616,\n      "Ġthings": 617,\n      "uck": 618,\n      "ave": 619,\n      "Ġcame": 620,\n      "Ġbl": 621,\n      "Ġv": 622,\n      "ĠD": 623,\n      "ided": 624,\n      "Ġevery": 625,\n      "Ġdecided": 626,\n      "ust": 627,\n      "udd": 628,\n      "Ġliked": 629,\n      "Ġdog": 630,\n      "Ġbec": 631,\n      "Ġone": 632,\n      "Ġdown": 633,\n      "us": 634,\n      "Ġthan": 635,\n      "Ġyour": 636,\n      "Ġscared": 637,\n      "Ġdad": 638,\n      "Ġcare": 639,\n      "Ġfind": 640,\n      "ouse": 641,\n      "Ġab": 642,\n      "Ġfeel": 643,\n      "ways": 644,\n      "ara": 645,\n      "Ġtree": 646,\n      "ĠF": 647,\n      "ue": 648,\n      "Ġta": 649,\n      "Ġpo": 650,\n      "Ġalways": 651,\n      "ist": 652,\n      "ax": 653,\n      "ant": 654,\n      "Ġdidn": 655,\n      "Ġabout": 656,\n      "Ġmy": 657,\n      "our": 658,\n      "Ġthought": 659,\n      "bb": 660,\n      "nder": 661,\n      "Ġke": 662,\n      "Ġtook": 663,\n      "Ġkne": 664,\n      "Ġgra": 665,\n      "Ġlook": 666,\n      "Ġold": 667,\n      "Ġlot": 668,\n      "ers": 669,\n      "arn": 670,\n      "ite": 671,\n      "ĠMom": 672,\n      "na": 673,\n      "ess": 674,\n      "Ġint": 675,\n      "fe": 676,\n      "ise": 677,\n      "Ġoutside": 678,\n      "eci": 679,\n      "ched": 680,\n      "udden": 681,\n      "Ġho": 682,\n      "Ġlaug": 683,\n      "Ġtake": 684,\n      "ged": 685,\n      "Ġmommy": 686,\n      "Ġwill": 687,\n      "ened": 688,\n      "They": 689,\n      "Ġtoys": 690,\n      "Ġknow": 691,\n      "ap": 692,\n      "Ġball": 693,\n      "Ġmore": 694,\n      "uddenly": 695,\n      "ret": 696,\n      "Ġpe": 697,\n      "ecial": 698,\n      "Ġspecial": 699,\n      "Ġknew": 700,\n      "Ġany": 701,\n      "reat": 702,\n      "Ġlearn": 703,\n      "fter": 704,\n      "Ġtr": 705,\n      "ive": 706,\n      "nna": 707,\n      "Ġhow": 708,\n      "ma": 709,\n      "Ġcat": 710,\n      "Ġsun": 711,\n      "Ġtw": 712,\n      "ĠSam": 713,\n      "ven": 714,\n      "ob": 715,\n      "ia": 716,\n      "au": 717,\n      "Ġshow": 718,\n      "ish": 719,\n      "os": 720,\n      "Ġmuch": 721,\n      "ĠMax": 722,\n      "Ġsor": 723,\n      "Ġsk": 724,\n      "Ġinto": 725,\n      "Ġqu": 726,\n      "Ġen": 727,\n      "and": 728,\n      "But": 729,\n      "Ġif": 730,\n      "ank": 731,\n      "Ġcouldn": 732,\n      "Ġexpl": 733,\n      "Ġsl": 734,\n      "Ġtold": 735,\n      "Ġhand": 736,\n      "Ġmo": 737,\n      "Ġhouse": 738,\n      "Ġop": 739,\n      "other": 740,\n      "Ġnever": 741,\n      "ge": 742,\n      "Ġproud": 743,\n      "Ġinside": 744,\n      "Ġover": 745,\n      "Ġheard": 746,\n      "Ġgave": 747,\n      "Ġthis": 748,\n      "Ġroom": 749,\n      "Åĵ": 750,\n      "ink": 751,\n      "Ġrun": 752,\n      "Ġor": 753,\n      "ty": 754,\n      "Ġpick": 755,\n      "Ġtried": 756,\n      "Ġstr": 757,\n      "Ġsorry": 758,\n      "Ġplayed": 759,\n      "ause": 760,\n      "ate": 761,\n      "ion": 762,\n      "ace": 763,\n      "uc": 764,\n      "Ġjust": 765,\n      "ump": 766,\n      "Ġwater": 767,\n      "Ġwat": 768,\n      "Ġeat": 769,\n      "ĠEvery": 770,\n      "dy": 771,\n      "Ġfo": 772,\n      "etter": 773,\n      "here": 774,\n      "Ġbu": 775,\n      "Ġsmall": 776,\n      "Ġhugged": 777,\n      "Ġneed": 778,\n      "\'m": 779,\n      "ough": 780,\n      "ĠÃ¢": 781,\n      "Ġcom": 782,\n      "oth": 783,\n      "ĠSara": 784,\n      "ĠJack": 785,\n      "ile": 786,\n      "Ġbecause": 787,\n      "by": 788,\n      "ort": 789,\n      "Ġlong": 790,\n      "Tom": 791,\n      "Mom": 792,\n      "Ġcall": 793,\n      "ving": 794,\n      "ĠJo": 795,\n      "iz": 796,\n      "Ġtry": 797,\n      "Ġoff": 798,\n      "Ġpret": 799,\n      "Ġbox": 800,\n      "ild": 801,\n      "Ġeach": 802,\n      "Ġnice": 803,\n      "Ġmany": 804,\n      "ak": 805,\n      "lly": 806,\n      "Ġbear": 807,\n      "Yes": 808,\n      "elf": 809,\n      "ĠWe": 810,\n      "hn": 811,\n      "ft": 812,\n      "Ġunt": 813,\n      "ear": 814,\n      "ine": 815,\n      "imal": 816,\n      "ady": 817,\n      "Ġanimal": 818,\n      "Ġthanked": 819,\n      "Ġdon": 820,\n      "aut": 821,\n      "Ġbest": 822,\n      "Ġbetter": 823,\n      "Ġad": 824,\n      "Ġflow": 825,\n      "Ġuntil": 826,\n      "em": 827,\n      "gry": 828,\n      "urt": 829,\n      "Th": 830,\n      "Ġim": 831,\n      "Ġlaughed": 832,\n      "Ġlearned": 833,\n      "Ġend": 834,\n      "vent": 835,\n      "oy": 836,\n      "Ġbeaut": 837,\n      "It": 838,\n      "Ġkind": 839,\n      "ĠAnna": 840,\n      "ber": 841,\n      "Ġeven": 842,\n      "Ġby": 843,\n      "Ġdre": 844,\n      "ning": 845,\n      "ĠHer": 846,\n      "ĠSo": 847,\n      "Ġgr": 848,\n      "pt": 849,\n      "ĠC": 850,\n      "Ġgard": 851,\n      "Ġsn": 852,\n      "Ġgarden": 853,\n      "Ġfi": 854,\n      "Ġpu": 855,\n      "Ġus": 856,\n      "Ġcle": 857,\n      "He": 858,\n      "urp": 859,\n      "iny": 860,\n      "self": 861,\n      "ase": 862,\n      "Ġsky": 863,\n      "Ġra": 864,\n      "ĠAnd": 865,\n      "Ġlove": 866,\n      "Ġclos": 867,\n      "Ġbeauti": 868,\n      "Ġway": 869,\n      "âĤ¬â": 870,\n      "Ġbeautiful": 871,\n      "lew": 872,\n      "Ġbra": 873,\n      "Ġfast": 874,\n      "Ġlots": 875,\n      "Ġcareful": 876,\n      "ct": 877,\n      "Ġlived": 878,\n      "ĠHis": 879,\n      "Ġstill": 880,\n      "Ġche": 881,\n      "Ġcome": 882,\n      "Ġtwo": 883,\n      "Ġjump": 884,\n      "Ġgl": 885,\n      "ĠMia": 886,\n      "urn": 887,\n      "ves": 888,\n      "ucy": 889,\n      "ĠFrom": 890,\n      "Ġwo": 891,\n      "Ġcalled": 892,\n      "ane": 893,\n      "ect": 894,\n      "Ġhard": 895,\n      "Ġboth": 896,\n      "Ħ¢": 897,\n      "âĤ¬âĦ¢": 898,\n      "wn": 899,\n      "Ġar": 900,\n      "Ġloud": 901,\n      "ock": 902,\n      "Ġlet": 903,\n      "Ġfam": 904,\n      "Tim": 905,\n      "Ġcu": 906,\n      "Ġstay": 907,\n      "Ġlist": 908,\n      "Ġrem": 909,\n      "Ġcol": 910,\n      "igh": 911,\n      "Let": 912,\n      "Ġunder": 913,\n      "Ġhurt": 914,\n      "hes": 915,\n      "When": 916,\n      "ast": 917,\n      "ool": 918,\n      "eet": 919,\n      "Ġbrave": 920,\n      "Ġits": 921,\n      "Ġdan": 922,\n      "Ġsafe": 923,\n      "ĠThen": 924,\n      "ied": 925,\n      "Ġimp": 926,\n      "ies": 927,\n      "Ġwalked": 928,\n      "Ġanimals": 929,\n      "What": 930,\n      "Ġflew": 931,\n      "Ġfin": 932,\n      "urpr": 933,\n      "ĠP": 934,\n      "dded": 935,\n      "Ġgoing": 936,\n      "Ġsurpr": 937,\n      "kay": 938,\n      "ished": 939,\n      "ople": 940,\n      "Ġbad": 941,\n      "Ġfly": 942,\n      "Ġstor": 943,\n      "ĠWhen": 944,\n      "Ġsoon": 945,\n      "Ġdra": 946,\n      "Ġpic": 947,\n      "ip": 948,\n      "Ġexplore": 949,\n      "fore": 950,\n      "ummy": 951,\n      "Ġdoll": 952,\n      "Ġsto": 953,\n      "Ġred": 954,\n      "Ġshould": 955,\n      "led": 956,\n      "Ġcon": 957,\n      "That": 958,\n      "Ġkept": 959,\n      "Ġpeople": 960,\n      "Ġfamily": 961,\n      "be": 962,\n      "No": 963,\n      "xt": 964,\n      "ised": 965,\n      "age": 966,\n      "illy": 967,\n      "Ġground": 968,\n      "Ġnow": 969,\n      "Ġbook": 970,\n      "Ġcry": 971,\n      "Ġshiny": 972,\n      "Ġangry": 973,\n      "Ġnodded": 974,\n      "Ġbefore": 975,\n      "Ġwind": 976,\n      "Ġfore": 977,\n      "So": 978,\n      "iced": 979,\n      "Ġey": 980,\n      "Ġkeep": 981,\n      "Ġdoor": 982,\n      "ary": 983,\n      "Ġwhile": 984,\n      "ĠLucy": 985,\n      "Ġadvent": 986,\n      "Ġshare": 987,\n      "Ġrepl": 988,\n      "Ġclean": 989,\n      "Ġopened": 990,\n      "Ġfeeling": 991,\n      "Ġwalking": 992,\n      "ĠBob": 993,\n      "Ġfar": 994,\n      "thing": 995,\n      "Ġrock": 996,\n      "ĠG": 997,\n      "Ġpretty": 998,\n      "hy": 999,\n      "Ġwar": 1000,\n      "ture": 1001,\n      "Ġfr": 1002,\n      "ff": 1003,\n      "ting": 1004,\n      "Ġpicked": 1005,\n      "Ġide": 1006,\n      "ger": 1007,\n      "Ġnoise": 1008,\n      "Ġstopped": 1009,\n      "Ġreplied": 1010,\n      "Ġreal": 1011,\n      "Ġcolor": 1012,\n      "joy": 1013,\n      "Ġnoticed": 1014,\n      "Ġwr": 1015,\n      "Ġthr": 1016,\n      "Ġnext": 1017,\n      "Ġbed": 1018,\n      "Ġtruck": 1019,\n      "ted": 1020,\n      "ĠSuddenly": 1021,\n      "Ġjo": 1022,\n      "Ġun": 1023,\n      "Ġidea": 1024,\n      "imb": 1025,\n      "Ġtre": 1026,\n      "Ġvo": 1027,\n      "ious": 1028,\n      "Thank": 1029,\n      "Ġte": 1030,\n      "Ġclimb": 1031,\n      "Ġpicture": 1032,\n      "Ġsmile": 1033,\n      "Ġadventure": 1034,\n      "ember": 1035,\n      "Ġeyes": 1036,\n      "Ġlooking": 1037,\n      "Ġquick": 1038,\n      "Ġdis": 1039,\n      "Ġgreat": 1040,\n      "Ġenjoy": 1041,\n      "Suddenly": 1042,\n      "bbit": 1043,\n      "Ġdel": 1044,\n      "uch": 1045,\n      "ught": 1046,\n      "Ġwait": 1047,\n      "les": 1048,\n      "Ġtow": 1049,\n      "Ġturn": 1050,\n      "Ġmor": 1051,\n      "Ben": 1052,\n      "ĠDad": 1053,\n      "Ġwonder": 1054,\n      "Ġremember": 1055,\n      "Ġhands": 1056,\n      "able": 1057,\n      "Ġma": 1058,\n      "maz": 1059,\n      "Ġamaz": 1060,\n      "Ġfish": 1061,\n      "Ġwalk": 1062,\n      "Ġwatch": 1063,\n      "oon": 1064,\n      "ĠR": 1065,\n      "Ġyear": 1066,\n      "Ġrabbit": 1067,\n      "Ġafter": 1068,\n      "irst": 1069,\n      "Ġever": 1070,\n      "iss": 1071,\n      "Ġgive": 1072,\n      "gan": 1073,\n      "so": 1074,\n      "Ġapp": 1075,\n      "Ġbeing": 1076,\n      "ac": 1077,\n      "ange": 1078,\n      "Ġright": 1079,\n      "Ġdif": 1080,\n      "Ġfood": 1081,\n      "Ġshout": 1082,\n      "Ġquickly": 1083,\n      "ĠJohn": 1084,\n      "aybe": 1085,\n      "Ġalso": 1086,\n      "Ġvoice": 1087,\n      "Ġblue": 1088,\n      "Ġwork": 1089,\n      "ĠN": 1090,\n      "Ġpain": 1091,\n      "Jo": 1092,\n      "kes": 1093,\n      "Ġimport": 1094,\n      "Ġcloser": 1095,\n      "Ġac": 1096,\n      "isy": 1097,\n      "Ġyell": 1098,\n      "Ġlisten": 1099,\n      "llow": 1100,\n      "Ġforest": 1101,\n      "Ġbeen": 1102,\n      "Ġbro": 1103,\n      "Ġgre": 1104,\n      "Ġthink": 1105,\n      "Ġnear": 1106,\n      "Ġstrong": 1107,\n      "She": 1108,\n      "Ġimportant": 1109,\n      "Ġrain": 1110,\n      "Ġtal": 1111,\n      "ream": 1112,\n      "lease": 1113,\n      "Ġflowers": 1114,\n      "ng": 1115,\n      "Ġcuri": 1116,\n      "Ġhas": 1117,\n      "ized": 1118,\n      "Ġsqu": 1119,\n      "ĠSue": 1120,\n      "Ġde": 1121,\n      "Ġuse": 1122,\n      "day": 1123,\n      "Ġokay": 1124,\n      "Ġthree": 1125,\n      "bbed": 1126,\n      "Ġhead": 1127,\n      "Ġbecame": 1128,\n      "bye": 1129,\n      "Ġsure": 1130,\n      "Her": 1131,\n      "Ġshowed": 1132,\n      "Ġstick": 1133,\n      "Ġplace": 1134,\n      "ila": 1135,\n      "Ġown": 1136,\n      "Ġface": 1137,\n      "Ġwatched": 1138,\n      "Ġeveryone": 1139,\n      "Ġcurious": 1140,\n      "ose": 1141,\n      "Ġwhere": 1142,\n      "Ġbright": 1143,\n      "Ġfollow": 1144,\n      "Ġtra": 1145,\n      "Ġhigh": 1146,\n      "aught": 1147,\n      "Ġslide": 1148,\n      "Look": 1149,\n      "Ġsound": 1150,\n      "ered": 1151,\n      "Ġmag": 1152,\n      "ĠSp": 1153,\n      "We": 1154,\n      "fere": 1155,\n      "Ġbus": 1156,\n      "Ġmum": 1157,\n      "Ġhere": 1158,\n      "Ġbel": 1159,\n      "Ġcr": 1160,\n      "Ġask": 1161,\n      "Ġdiffere": 1162,\n      "olly": 1163,\n      "Ġam": 1164,\n      "Ġfirst": 1165,\n      "Ġblock": 1166,\n      "ĠJane": 1167,\n      "Ġmean": 1168,\n      "Ġchild": 1169,\n      "Ġstory": 1170,\n      "Ġdress": 1171,\n      "leep": 1172,\n      "Can": 1173,\n      "ass": 1174,\n      "Ġgoodbye": 1175,\n      "ired": 1176,\n      "Sara": 1177,\n      "Ġwarm": 1178,\n      "aisy": 1179,\n      "Ġbit": 1180,\n      "Ġcont": 1181,\n      "ĠMommy": 1182,\n      "itc": 1183,\n      "Ġper": 1184,\n      "more": 1185,\n      "Ġanymore": 1186,\n      "Ġstore": 1187,\n      "Ġboat": 1188,\n      "Ġnight": 1189,\n      "Ġforg": 1190,\n      "Ġhop": 1191,\n      "Ġbeh": 1192,\n      "ached": 1193,\n      "co": 1194,\n      "Ġdifferent": 1195,\n      "Ġbutter": 1196,\n      "ĠAs": 1197,\n      "Ġfire": 1198,\n      "Ġsweet": 1199,\n      "ull": 1200,\n      "Ġgame": 1201,\n      "Ġour": 1202,\n      "ars": 1203,\n      "uff": 1204,\n      "Ġopen": 1205,\n      "ndma": 1206,\n      "ppy": 1207,\n      "Ġdraw": 1208,\n      "imes": 1209,\n      "Ġworld": 1210,\n      "Ġtell": 1211,\n      "Ġhelped": 1212,\n      "ount": 1213,\n      "Ġpat": 1214,\n      "Ġbegan": 1215,\n      "Ġfav": 1216,\n      "ĠTh": 1217,\n      "Ġonly": 1218,\n      "Ġmon": 1219,\n      "Ġthrough": 1220,\n      "Ġjumped": 1221,\n      "ner": 1222,\n      "Ġsand": 1223,\n      "Ġrest": 1224,\n      "Ġlea": 1225,\n      "ely": 1226,\n      "arent": 1227,\n      "Ġreally": 1228,\n      "Ġcake": 1229,\n      "Ġbrother": 1230,\n      "Ġpull": 1231,\n      "Ġz": 1232,\n      "Ġname": 1233,\n      "ĠLet": 1234,\n      "ĠSpot": 1235,\n      "zy": 1236,\n      "Ġlady": 1237,\n      "Ġfell": 1238,\n      "Ġhappened": 1239,\n      "Ġgrabbed": 1240,\n      "Ġru": 1241,\n      "\'re": 1242,\n      "After": 1243,\n      "Ġshouted": 1244,\n      "Ġcook": 1245,\n      "Ġkitc": 1246,\n      "Ġkitchen": 1247,\n      "unny": 1248,\n      "Ġready": 1249,\n      "Ġparent": 1250,\n      "John": 1251,\n      "Ġsoft": 1252,\n      "ater": 1253,\n      "Ġdoes": 1254,\n      "Ġflower": 1255,\n      "Ġflo": 1256,\n      "Ġglad": 1257,\n      "Ġthing": 1258,\n      "Ġgrass": 1259,\n      "ster": 1260,\n      "ĠSarah": 1261,\n      "Ġdin": 1262,\n      "Ġbirds": 1263,\n      "Mommy": 1264,\n      "ĠJoe": 1265,\n      "As": 1266,\n      "Ġreached": 1267,\n      "Ġhaving": 1268,\n      "Ġdr": 1269,\n      "Ġfro": 1270,\n      "Ġhopped": 1271,\n      "ins": 1272,\n      "Ġworry": 1273,\n      "Ġwhy": 1274,\n      "Ġmagic": 1275,\n      "Ġtired": 1276,\n      "th": 1277,\n      "Anna": 1278,\n      "fully": 1279,\n      "Ġsees": 1280,\n      "Ġtast": 1281,\n      "Ġpond": 1282,\n      "Ġcoming": 1283,\n      "Ġstop": 1284,\n      "Ġrealized": 1285,\n      "Ġfill": 1286,\n      "ĠDaisy": 1287,\n      "ather": 1288,\n      "ient": 1289,\n      "Ġdone": 1290,\n      "Ġseen": 1291,\n      "Ġyears": 1292,\n      "where": 1293,\n      "Ġkid": 1294,\n      "Ġbutterf": 1295,\n      "Ġless": 1296,\n      "Ġmouse": 1297,\n      "ul": 1298,\n      "Ġsing": 1299,\n      "Ġhimself": 1300,\n      "Ġwaved": 1301,\n      "Ġblocks": 1302,\n      "Ġsmell": 1303,\n      "Ġprin": 1304,\n      "Ġwood": 1305,\n      "\\".": 1306,\n      "Ġbehind": 1307,\n      "Don": 1308,\n      "Ġyummy": 1309,\n      "ued": 1310,\n      "Ġparents": 1311,\n      "Ġsurprise": 1312,\n      "Ġbre": 1313,\n      "Ġcut": 1314,\n      "air": 1315,\n      "Ġfunny": 1316,\n      "Timmy": 1317,\n      "Ġunderst": 1318,\n      "mer": 1319,\n      "Ġlooks": 1320,\n      "ĠEveryone": 1321,\n      "Wow": 1322,\n      "ards": 1323,\n      "Ġfavor": 1324,\n      "Ġate": 1325,\n      "Ġcreat": 1326,\n      "ign": 1327,\n      "Ġworked": 1328,\n      "Ġrunning": 1329,\n      "ĠThat": 1330,\n      "Ġfavorite": 1331,\n      "ken": 1332,\n      "ĠDo": 1333,\n      "Ġyellow": 1334,\n      "Ġset": 1335,\n      "Ġfix": 1336,\n      "Ġbunny": 1337,\n      "Ġhat": 1338,\n      "Ġgreen": 1339,\n      "Ġpart": 1340,\n      "Ġ-": 1341,\n      "Ġpus": 1342,\n      "Ġpap": 1343,\n      "Ġmess": 1344,\n      "Ġfrog": 1345,\n      "Ġmaking": 1346,\n      "Jack": 1347,\n      "Ġlight": 1348,\n      "ĠCan": 1349,\n      "Ġdelic": 1350,\n      "Ġperf": 1351,\n      "Ġhappily": 1352,\n      "Ġba": 1353,\n      "Ġprom": 1354,\n      "Ġdark": 1355,\n      "Ġswim": 1356,\n      "Ġcatch": 1357,\n      "ike": 1358,\n      "Ġbag": 1359,\n      "Ġanother": 1360,\n      "Ġtri": 1361,\n      "Ġbat": 1362,\n      "Ġused": 1363,\n      "Ġdro": 1364,\n      "Ġtrees": 1365,\n      "Ġpre": 1366,\n      "Ġste": 1367,\n      "Ġcried": 1368,\n      "Ġremembered": 1369,\n      "eddy": 1370,\n      "Ġtop": 1371,\n      "Ġcold": 1372,\n      "ĠSally": 1373,\n      "Ġenjoyed": 1374,\n      "Ġhole": 1375,\n      "Ġmove": 1376,\n      "Ġsleep": 1377,\n      "Ġslow": 1378,\n      "akes": 1379,\n      "andy": 1380,\n      "Hell": 1381,\n      "Hello": 1382,\n      "Ġtouch": 1383,\n      "Sam": 1384,\n      "Ġbug": 1385,\n      "Ġdoing": 1386,\n      "rel": 1387,\n      "bo": 1388,\n      "Ġwished": 1389,\n      "ĠAfter": 1390,\n      "sh": 1391,\n      "Ġcarefully": 1392,\n      "Ġdri": 1393,\n      "iver": 1394,\n      "oney": 1395,\n      "Ġcontin": 1396,\n      "Ġbutterfly": 1397,\n      "Ġsat": 1398,\n      "Ġfull": 1399,\n      "ation": 1400,\n      "Ġbuy": 1401,\n      "Ġbuild": 1402,\n      "reed": 1403,\n      "Ġleft": 1404,\n      "Ġsnow": 1405,\n      "Ġkids": 1406,\n      "Ġwear": 1407,\n      "Ġspot": 1408,\n      "Ġarri": 1409,\n      "Ġanything": 1410,\n      "Ġpretend": 1411,\n      "\'ll": 1412,\n      "Ġhair": 1413,\n      "Ġread": 1414,\n      "Ġtall": 1415,\n      "Ġduck": 1416,\n      "ons": 1417,\n      "Ġhid": 1418,\n      "Ġwrong": 1419,\n      "ren": 1420,\n      "Ġsho": 1421,\n      "Ġlost": 1422,\n      "Ġcool": 1423,\n      "Ġherself": 1424,\n      "Ġswing": 1425,\n      "Ġbow": 1426,\n      "ĠIn": 1427,\n      "ield": 1428,\n      "Ġdream": 1429,\n      "Ġfinished": 1430,\n      "Ġlesson": 1431,\n      "Ġneeded": 1432,\n      "Ġsurprised": 1433,\n      "Ġhill": 1434,\n      "Ġfriendly": 1435,\n      "Ġwindow": 1436,\n      "Ġcra": 1437,\n      "Ġheld": 1438,\n      "ash": 1439,\n      "ĠK": 1440,\n      "ĠBenny": 1441,\n      "Ġfair": 1442,\n      "Ġlaugh": 1443,\n      "Ġarrived": 1444,\n      "room": 1445,\n      "Ġcontinued": 1446,\n      "Ġwonderful": 1447,\n      "uffy": 1448,\n      "Ġpaper": 1449,\n      "Ġrel": 1450,\n      "Ġhold": 1451,\n      "av": 1452,\n      "Ġair": 1453,\n      "inally": 1454,\n      "Ġmoment": 1455,\n      "Ġfollowed": 1456,\n      "Ġfloor": 1457,\n      "ĠEm": 1458,\n      "Ġtable": 1459,\n      "astle": 1460,\n      "ey": 1461,\n      "Ġplease": 1462,\n      "Ġthank": 1463,\n      "cess": 1464,\n      "asure": 1465,\n      "Ġchildren": 1466,\n      "Ġgi": 1467,\n      "ens": 1468,\n      "Ġcars": 1469,\n      "Ġpoin": 1470,\n      "His": 1471,\n      "Ġfilled": 1472,\n      "Ġhun": 1473,\n      "ined": 1474,\n      "Ġclose": 1475,\n      "Ġmother": 1476,\n      "Ġmat": 1477,\n      "ĠLila": 1478,\n      "Ġable": 1479,\n      "ask": 1480,\n      "ĠNow": 1481,\n      "Ġgent": 1482,\n      "Ġenough": 1483,\n      "ross": 1484,\n      "Ġmorning": 1485,\n      "Ġagreed": 1486,\n      "Ġice": 1487,\n      "Ġtight": 1488,\n      "Ġtrying": 1489,\n      "Ġdelicious": 1490,\n      "Ġsomeone": 1491,\n      "This": 1492,\n      "Ġlion": 1493,\n      "Ġforgot": 1494,\n      "Ġdinner": 1495,\n      "sp": 1496,\n      "Ġgetting": 1497,\n      "ĠJen": 1498,\n      "ama": 1499,\n      "ds": 1500,\n      "Ġyes": 1501,\n      "Ġhear": 1502,\n      "ĠThere": 1503,\n      "Ġvis": 1504,\n      "Ġarm": 1505,\n      "Ġel": 1506,\n      "Ġdir": 1507,\n      "Then": 1508,\n      "From": 1509,\n      "gon": 1510,\n      "Ġlikes": 1511,\n      "Ġfox": 1512,\n      "Ġothers": 1513,\n      "Ġgone": 1514,\n      "Ġunderstand": 1515,\n      "ee": 1516,\n      "Ġwall": 1517,\n      "Ġcolors": 1518,\n      "ich": 1519,\n      "Ġstrange": 1520,\n      "Ġtail": 1521,\n      "Ġsong": 1522,\n      "irrel": 1523,\n      "ced": 1524,\n      "Ġhot": 1525,\n      "icy": 1526,\n      "Ġmar": 1527,\n      "Ġmight": 1528,\n      "Ġscary": 1529,\n      "Ġmus": 1530,\n      "Ġwoods": 1531,\n      "oring": 1532,\n      "Ġleaves": 1533,\n      "orn": 1534,\n      "Ġbrought": 1535,\n      "Ġbooks": 1536,\n      "Ġsquirrel": 1537,\n      "oup": 1538,\n      "Ġelse": 1539,\n      "app": 1540,\n      "Ġbroken": 1541,\n      "ĠWhat": 1542,\n      "uit": 1543,\n      "Ġamazing": 1544,\n      "Ġperfect": 1545,\n      "At": 1546,\n      "Ġfew": 1547,\n      "Ġballoon": 1548,\n      "Ġpuppy": 1549,\n      "Ġclot": 1550,\n      "Ġcastle": 1551,\n      "ĠFl": 1552,\n      "Ġmet": 1553,\n      "Ġfinally": 1554,\n      "ĠThis": 1555,\n      "Ġbeach": 1556,\n      "Ġclothes": 1557,\n      "ance": 1558,\n      "Ġroll": 1559,\n      "Ġdeep": 1560,\n      "Ġpink": 1561,\n      "ph": 1562,\n      "Ġpictures": 1563,\n      "ctor": 1564,\n      "eter": 1565,\n      "ĠBilly": 1566,\n      "key": 1567,\n      "Ġpaint": 1568,\n      "ĠGra": 1569,\n      "Ġpet": 1570,\n      "Ġturned": 1571,\n      "Ġsec": 1572,\n      "Ġmoral": 1573,\n      "Ġcount": 1574,\n      "Ġwhite": 1575,\n      "Ġstayed": 1576,\n      "Oh": 1577,\n      "Ġride": 1578,\n      "Ġtalk": 1579,\n      "Ġeverything": 1580,\n      "Ġfight": 1581,\n      "Ġwet": 1582,\n      "Ġamazed": 1583,\n      "der": 1584,\n      "Ġwants": 1585,\n      "Ġgrandma": 1586,\n      "lc": 1587,\n      "wl": 1588,\n      "Ġdaddy": 1589,\n      "red": 1590,\n      "Ġsail": 1591,\n      "Ġcomp": 1592,\n      "Ġclimbed": 1593,\n      "Ġjoy": 1594,\n      "Ġflying": 1595,\n      "Ġhungry": 1596,\n      "lf": 1597,\n      "Ġpen": 1598,\n      "cket": 1599,\n      "ower": 1600,\n      "Ġbreak": 1601,\n      "man": 1602,\n      "ated": 1603,\n      "ries": 1604,\n      "Ġalone": 1605,\n      "Ġseem": 1606,\n      "Ġsometimes": 1607,\n      "Ġbaby": 1608,\n      "Ġvisit": 1609,\n      "ze": 1610,\n      "Ġgrow": 1611,\n      "row": 1612,\n      "Ġwasn": 1613,\n      "amp": 1614,\n      "Ġspl": 1615,\n      "ĠTommy": 1616,\n      "Ġmusic": 1617,\n      "Ġsear": 1618,\n      "Ġlistened": 1619,\n      "Ġspark": 1620,\n      "Ġcloud": 1621,\n      "Ġwaited": 1622,\n      "Ġve": 1623,\n      "Ġdanger": 1624,\n      "ĠMolly": 1625,\n      "Max": 1626,\n      "Ġhide": 1627,\n      "Ġmakes": 1628,\n      "immy": 1629,\n      "Ġtowards": 1630,\n      "Ġcookies": 1631,\n      "ever": 1632,\n      "Ġfall": 1633,\n      "Ġwon": 1634,\n      "Ġpi": 1635,\n      "Ġbigger": 1636,\n      "Ġsilly": 1637,\n      "Maybe": 1638,\n      "Ġfield": 1639,\n      "leph": 1640,\n      "Ġjob": 1641,\n      "OK": 1642,\n      "anc": 1643,\n      "apped": 1644,\n      "orm": 1645,\n      "igg": 1646,\n      "ope": 1647,\n      "ucky": 1648,\n      "Ġbelie": 1649,\n      "Ġshared": 1650,\n      "Ġju": 1651,\n      "avy": 1652,\n      "ix": 1653,\n      "Ġpol": 1654,\n      "ssed": 1655,\n      "Ġpromised": 1656,\n      "ually": 1657,\n      "Ġside": 1658,\n      "ĠAmy": 1659,\n      "ence": 1660,\n      "oom": 1661,\n      "Ġbike": 1662,\n      "ister": 1663,\n      "Ġliving": 1664,\n      "gg": 1665,\n      "ire": 1666,\n      "sw": 1667,\n      "Ġmiss": 1668,\n      "Ġpa": 1669,\n      "Ġfaster": 1670,\n      "Ġslowly": 1671,\n      "rew": 1672,\n      "ĠTo": 1673,\n      "Ġgu": 1674,\n      "Ġpulled": 1675,\n      "Ġansw": 1676,\n      "Ġmost": 1677,\n      "Ġmust": 1678,\n      "Ġsmiles": 1679,\n      "Ġworried": 1680,\n      "Ġparty": 1681,\n      "set": 1682,\n      "Ġfarmer": 1683,\n      "Ġem": 1684,\n      "Why": 1685,\n      "me": 1686,\n      "Ġteddy": 1687,\n      "ained": 1688,\n      "thy": 1689,\n      "Ġstepped": 1690,\n      "ipp": 1691,\n      "ott": 1692,\n      "ling": 1693,\n      "Ġcrying": 1694,\n      "Hi": 1695,\n      "ipped": 1696,\n      "ol": 1697,\n      "ames": 1698,\n      "ĠAll": 1699,\n      "Ġjoin": 1700,\n      "Ġlake": 1701,\n      "Ġplan": 1702,\n      "ĠJill": 1703,\n      "Ġcream": 1704,\n      "Okay": 1705,\n      "ang": 1706,\n      "Ġhelping": 1707,\n      "Ġblack": 1708,\n      "Ġdisapp": 1709,\n      "Ġleave": 1710,\n      "Ġcup": 1711,\n      "Ġdragon": 1712,\n      "Ġsit": 1713,\n      "Ġgrate": 1714,\n      "Ġpass": 1715,\n      "Ġatt": 1716,\n      "Ġmagical": 1717,\n      "ĠMaybe": 1718,\n      "Ġgrateful": 1719,\n      "ugg": 1720,\n      "Ġcandy": 1721,\n      "eared": 1722,\n      "fort": 1723,\n      "Ġneigh": 1724,\n      "Ġdance": 1725,\n      "rot": 1726,\n      "Ġsu": 1727,\n      "Ġfur": 1728,\n      "less": 1729,\n      "Ġtreasure": 1730,\n      "unch": 1731,\n      "Ġneighb": 1732,\n      "Ġwelc": 1733,\n      "Ġacross": 1734,\n      "Ġshar": 1735,\n      "Ġdolls": 1736,\n      "ming": 1737,\n      "Ġtreat": 1738,\n      "Ġheavy": 1739,\n      "ises": 1740,\n      "Ġblank": 1741,\n      "ĠFluffy": 1742,\n      "bble": 1743,\n      "Ġreach": 1744,\n      "Ġasks": 1745,\n      "Ġbath": 1746,\n      "Ġsister": 1747,\n      "Ġbee": 1748,\n      "iet": 1749,\n      "zz": 1750,\n      "Ġdangerous": 1751,\n      "Ġmix": 1752,\n      "Ġcolorful": 1753,\n      "umb": 1754,\n      "Ġpro": 1755,\n      "Ġsecret": 1756,\n      "Ġblanket": 1757,\n      "eces": 1758,\n      "ased": 1759,\n      "Ġmonster": 1760,\n      "Ġdoctor": 1761,\n      "Ġshap": 1762,\n      "Ġbott": 1763,\n      "Ġmoney": 1764,\n      "Ġplant": 1765,\n      "Ġtrain": 1766,\n      "Ġpieces": 1767,\n      "outh": 1768,\n      "Ġbelieve": 1769,\n      "br": 1770,\n      "Ġriver": 1771,\n      "Ġeverywhere": 1772,\n      "Ġacc": 1773,\n      "ment": 1774,\n      "Ġtown": 1775,\n      "Ġmonkey": 1776,\n      "weet": 1777,\n      "Ġsign": 1778,\n      "Ġlon": 1779,\n      "Ġwords": 1780,\n      "Ġland": 1781,\n      "de": 1782,\n      "Ġmouth": 1783,\n      "Ġgif": 1784,\n      "Ġsuch": 1785,\n      "Ġrace": 1786,\n      "Ġquiet": 1787,\n      "ventually": 1788,\n      "Sure": 1789,\n      "Ġstuck": 1790,\n      "Ġbroke": 1791,\n      "Ġcray": 1792,\n      "Ġca": 1793,\n      "Ġhig": 1794,\n      "ident": 1795,\n      "Ġwithout": 1796,\n      "sed": 1797,\n      "Ġpocket": 1798,\n      "Ġlonely": 1799,\n      "Ġclosed": 1800,\n      "lephant": 1801,\n      "Ġbuck": 1802,\n      "Ġtower": 1803,\n      "Ġelephant": 1804,\n      "Ġstand": 1805,\n      "bby": 1806,\n      "Ġje": 1807,\n      "iting": 1808,\n      "Ġcour": 1809,\n      "ract": 1810,\n      "Ġthinks": 1811,\n      "Ġce": 1812,\n      "Ġstu": 1813,\n      "Ġgentle": 1814,\n      "Ġheart": 1815,\n      "chool": 1816,\n      "althy": 1817,\n      "Ġclapped": 1818,\n      "Ġtwins": 1819,\n      "Bob": 1820,\n      "Ġsame": 1821,\n      "Ġexploring": 1822,\n      "Ġorange": 1823,\n      "Ġtoday": 1824,\n      "Ġbite": 1825,\n      "Ġpot": 1826,\n      "ored": 1827,\n      "Ġlonger": 1828,\n      "Ġsnack": 1829,\n      "Ġnap": 1830,\n      "Ġyoung": 1831,\n      "Ġbowl": 1832,\n      "Ġeating": 1833,\n      "Ġpers": 1834,\n      "Ġupset": 1835,\n      "Ġteac": 1836,\n      "Ġdirty": 1837,\n      "Ġowl": 1838,\n      "Ġprincess": 1839,\n      "Ġcourse": 1840,\n      "Ġscream": 1841,\n      "ney": 1842,\n      "Ġsuddenly": 1843,\n      "Ġschool": 1844,\n      "Ġhigher": 1845,\n      "Ġkiss": 1846,\n      "Ġshop": 1847,\n      "cle": 1848,\n      "Ġbar": 1849,\n      "oun": 1850,\n      "Ġbring": 1851,\n      "Ġhoped": 1852,\n      "Finally": 1853,\n      "ek": 1854,\n      "Ġsmo": 1855,\n      "Ġalong": 1856,\n      "Ġshoes": 1857,\n      "Mia": 1858,\n      "Ġsmart": 1859,\n      "Ġowner": 1860,\n      "Ġfairy": 1861,\n      "Ġseemed": 1862,\n      "Lila": 1863,\n      "Ġfruit": 1864,\n      "Ġapple": 1865,\n      "Ġcave": 1866,\n      "ĠMr": 1867,\n      "ĠDaddy": 1868,\n      "Ġthrew": 1869,\n      "Ġpushed": 1870,\n      "Do": 1871,\n      "hone": 1872,\n      "itty": 1873,\n      "fused": 1874,\n      "Ġwouldn": 1875,\n      "Ġcarrot": 1876,\n      "Ġswam": 1877,\n      "Ġdrove": 1878,\n      "ts": 1879,\n      "Ġpie": 1880,\n      "Ġhappen": 1881,\n      "Ġbrown": 1882,\n      "asket": 1883,\n      "Ġbasket": 1884,\n      "Ġwelcome": 1885,\n      "ier": 1886,\n      "Ġpig": 1887,\n      "Ġkite": 1888,\n      "Ġsea": 1889,\n      "ĠEmma": 1890,\n      "Ġdes": 1891,\n      "Ġaccident": 1892,\n      "Ġwin": 1893,\n      "Ġfree": 1894,\n      "Ġteacher": 1895,\n      "Ġob": 1896,\n      "Ġruns": 1897,\n      "ible": 1898,\n      "Ġinst": 1899,\n      "Ġcheered": 1900,\n      "Ġexplained": 1901,\n      "cked": 1902,\n      "Ġmin": 1903,\n      "Ġpack": 1904,\n      "Ġcoll": 1905,\n      "ead": 1906,\n      "sy": 1907,\n      "Ġdry": 1908,\n      "ors": 1909,\n      "Ġbucket": 1910,\n      "Ġadventures": 1911,\n      "Ġnearby": 1912,\n      "Ġtiny": 1913,\n      "Ġthough": 1914,\n      "ieved": 1915,\n      "Ġdropped": 1916,\n      "ung": 1917,\n      "Ġplac": 1918,\n      "Ġspr": 1919,\n      "Ġsoup": 1920,\n      "Ġwings": 1921,\n      "Ġlunch": 1922,\n      "ĠTweet": 1923,\n      "Ġforget": 1924,\n      "Ġstre": 1925,\n      "Ġ3": 1926,\n      "Ġlate": 1927,\n      "Ġcaught": 1928,\n      "Ġnaught": 1929,\n      "orrow": 1930,\n      "Ġlaughing": 1931,\n      "Ġbelong": 1932,\n      "Ġpres": 1933,\n      "Ġes": 1934,\n      "Ġtaking": 1935,\n      "yard": 1936,\n      "Ġshook": 1937,\n      "Ġjour": 1938,\n      "Ġmoved": 1939,\n      "ions": 1940,\n      "ward": 1941,\n      "And": 1942,\n      "bow": 1943,\n      "Ġspoon": 1944,\n      "Ġbackyard": 1945,\n      "ho": 1946,\n      "Ġsang": 1947,\n      "Ġexp": 1948,\n      "Ġnaughty": 1949,\n      "Lucy": 1950,\n      "ail": 1951,\n      "Ġboun": 1952,\n      "Ġwell": 1953,\n      "Ġwish": 1954,\n      "Ġpast": 1955,\n      "Ġgames": 1956,\n      "Ġresp": 1957,\n      "aughter": 1958,\n      "Ġcolour": 1959,\n      "Ġyard": 1960,\n      "itting": 1961,\n      "Ġnoises": 1962,\n      "Ġtrou": 1963,\n      "Ġsunny": 1964,\n      "Ġwoke": 1965,\n      "Ġrainbow": 1966,\n      "vered": 1967,\n      "vel": 1968,\n      "Ġmine": 1969,\n      "Ġneck": 1970,\n      "Ġswings": 1971,\n      "aur": 1972,\n      "Ġcor": 1973,\n      "Ġcarry": 1974,\n      "ocked": 1975,\n      "Ġrocks": 1976,\n      "Ġwondered": 1977,\n      "Ġpresent": 1978,\n      "Ġdist": 1979,\n      "cing": 1980,\n      "Ġwise": 1981,\n      "Ġpower": 1982,\n      "pping": 1983,\n      "Ġrelieved": 1984,\n      "Ġcou": 1985,\n      "ush": 1986,\n      "Ġmark": 1987,\n      "ountain": 1988,\n      "Ġstars": 1989,\n      "Ġsinging": 1990,\n      "cer": 1991,\n      "een": 1992,\n      "gged": 1993,\n      "Ġlife": 1994,\n      "Ġhugs": 1995,\n      "Ġbusy": 1996,\n      "ining": 1997,\n      "ĠMama": 1998,\n      "Ġfeels": 1999,\n      "Ġwatching": 2000,\n      "Ġcreature": 2001,\n      "Ġdriver": 2002,\n      "Later": 2003,\n      "uched": 2004,\n      "Ġtouched": 2005,\n      "Ġbreat": 2006,\n      "rest": 2007,\n      "ĠMary": 2008,\n      "Ġvill": 2009,\n      "Ġtom": 2010,\n      "Ġdaughter": 2011,\n      "adow": 2012,\n      "Ġnose": 2013,\n      "Ġthrow": 2014,\n      "Ġbush": 2015,\n      "Ġgigg": 2016,\n      "Ġwhist": 2017,\n      "Ġnothing": 2018,\n      "ÅĵI": 2019,\n      "Ġpath": 2020,\n      "Ġhit": 2021,\n      "llo": 2022,\n      "Ġinc": 2023,\n      "Ġgrand": 2024,\n      "Ġarms": 2025,\n      "Ġfence": 2026,\n      "Ġchair": 2027,\n      "Ġdrink": 2028,\n      "Of": 2029,\n      "pe": 2030,\n      "Ġsick": 2031,\n      "Ġwhe": 2032,\n      "Ġpeace": 2033,\n      "Ġjuice": 2034,\n      "erest": 2035,\n      "Ġben": 2036,\n      "Ġinterest": 2037,\n      "Ġswimming": 2038,\n      "Ġpointed": 2039,\n      "Ġshapes": 2040,\n      "ped": 2041,\n      "Ġdays": 2042,\n      "Ġpatient": 2043,\n      "Ġsnowman": 2044,\n      "Ġbutt": 2045,\n      "Ġjar": 2046,\n      "Ġmyster": 2047,\n      "Ġholding": 2048,\n      "oose": 2049,\n      "Ġret": 2050,\n      "ĠJimmy": 2051,\n      "Ġkey": 2052,\n      "pa": 2053,\n      "Ġbought": 2054,\n      "Ġeas": 2055,\n      "itten": 2056,\n      "Ġaf": 2057,\n      "Ġsitting": 2058,\n      "anch": 2059,\n      "Ġleg": 2060,\n      "Ġbranch": 2061,\n      "Ġcozy": 2062,\n      "ĠSammy": 2063,\n      "Ġinv": 2064,\n      "Ġsmiling": 2065,\n      "Ġexciting": 2066,\n      "ĠGrandma": 2067,\n      "la": 2068,\n      "to": 2069,\n      "Ġcir": 2070,\n      "Ġpudd": 2071,\n      "irth": 2072,\n      "alm": 2073,\n      "Ġdisco": 2074,\n      "Ġturns": 2075,\n      "irthday": 2076,\n      "Ġdanced": 2077,\n      "ĠInside": 2078,\n      "Molly": 2079,\n      "ness": 2080,\n      "Ġhor": 2081,\n      "Ġnest": 2082,\n      "Ġround": 2083,\n      "Hey": 2084,\n      "Ġcute": 2085,\n      "Ġcoin": 2086,\n      "ord": 2087,\n      "Ġcho": 2088,\n      "Tommy": 2089,\n      "ĠThank": 2090,\n      "Mum": 2091,\n      "idge": 2092,\n      "Ġmail": 2093,\n      "Ġgift": 2094,\n      "Ġmist": 2095,\n      "Ġspo": 2096,\n      "ize": 2097,\n      "Ġadm": 2098,\n      "shine": 2099,\n      "Ġjourney": 2100,\n      "ball": 2101,\n      "Ġact": 2102,\n      "Ġstar": 2103,\n      "og": 2104,\n      "Ġfit": 2105,\n      "Ġletter": 2106,\n      "Ġpract": 2107,\n      "Ġsack": 2108,\n      "Ġthin": 2109,\n      "Ġlift": 2110,\n      "raid": 2111,\n      "ĠCome": 2112,\n      "Ġbirthday": 2113,\n      "Ġeag": 2114,\n      "Ġsunshine": 2115,\n      "Come": 2116,\n      "Ġmad": 2117,\n      "Ġear": 2118,\n      "Ġcomes": 2119,\n      "col": 2120,\n      "Ġcrayons": 2121,\n      "Ġafraid": 2122,\n      "irt": 2123,\n      "Ġdelight": 2124,\n      "board": 2125,\n      "Ġaccidentally": 2126,\n      "Ġreturn": 2127,\n      "Ġwra": 2128,\n      "read": 2129,\n      "Ġanyone": 2130,\n      "ĠSoon": 2131,\n      "Ġhello": 2132,\n      "ĠJenny": 2133,\n      "ces": 2134,\n      "ĠO": 2135,\n      "Ġsuc": 2136,\n      "Ġlovely": 2137,\n      "Ġtasted": 2138,\n      "ove": 2139,\n      "Ġtear": 2140,\n      "Ġpurp": 2141,\n      "let": 2142,\n      "ont": 2143,\n      "amed": 2144,\n      "Ġhuge": 2145,\n      "Ġinteresting": 2146,\n      "ten": 2147,\n      "Ġking": 2148,\n      "Ġres": 2149,\n      "blem": 2150,\n      "ĠFinally": 2151,\n      "Ġonto": 2152,\n      "Ġfront": 2153,\n      "Ġproblem": 2154,\n      "Ġinstead": 2155,\n      "Ġsur": 2156,\n      "Ġlucky": 2157,\n      "Ġstreet": 2158,\n      "lice": 2159,\n      "Ġband": 2160,\n      "epend": 2161,\n      "Ġpiece": 2162,\n      "izz": 2163,\n      "Ġimag": 2164,\n      "Ġsounds": 2165,\n      "Ġrolled": 2166,\n      "Ġperson": 2167,\n      "unt": 2168,\n      "Ġtun": 2169,\n      "Ġcomfort": 2170,\n      "Ġtasty": 2171,\n      "ert": 2172,\n      "Ġsandw": 2173,\n      "Ġsmelled": 2174,\n      "itch": 2175,\n      "Ġwhich": 2176,\n      "Ġkick": 2177,\n      "Ġstories": 2178,\n      "Ġemb": 2179,\n      "ity": 2180,\n      "Ġgrab": 2181,\n      "Ġfarm": 2182,\n      "Ġsearched": 2183,\n      "Ġpurple": 2184,\n      "Ġ\'": 2185,\n      "Ġap": 2186,\n      "Ġwhole": 2187,\n      "Ġvillage": 2188,\n      "Ġhorse": 2189,\n      "My": 2190,\n      "Ġent": 2191,\n      "Ġplane": 2192,\n      "ittens": 2193,\n      "Ġgather": 2194,\n      "Ġshr": 2195,\n      "Ġshining": 2196,\n      "Ġtakes": 2197,\n      "Ġclouds": 2198,\n      "Ġcollect": 2199,\n      "Ġappeared": 2200,\n      "Ġbuilding": 2201,\n      "Ġbench": 2202,\n      "Ġhoney": 2203,\n      "Ġcard": 2204,\n      "arra": 2205,\n      "Ġspicy": 2206,\n      "Ġlem": 2207,\n      "Ġforever": 2208,\n      "Ġembarra": 2209,\n      "Ġhelpful": 2210,\n      "Ġprot": 2211,\n      "Ġquest": 2212,\n      "Just": 2213,\n      "Ġbal": 2214,\n      "Ġpool": 2215,\n      "cean": 2216,\n      "Ġash": 2217,\n      "Ġspid": 2218,\n      "Ġrob": 2219,\n      "Ġwrite": 2220,\n      "Ġocean": 2221,\n      "Ġeventually": 2222,\n      "Ġspotted": 2223,\n      "Ġrespect": 2224,\n      "lebr": 2225,\n      "ometimes": 2226,\n      "Ġhope": 2227,\n      "Ġneighbor": 2228,\n      "Ġbottle": 2229,\n      "Good": 2230,\n      "Ġlater": 2231,\n      "Ġmach": 2232,\n      "arge": 2233,\n      "Ġasleep": 2234,\n      "Ġfold": 2235,\n      "Ġcelebr": 2236,\n      "Ġminut": 2237,\n      "Ġplaces": 2238,\n      "Ġrobot": 2239,\n      "Ġwild": 2240,\n      "Ġphone": 2241,\n      "Ġsaf": 2242,\n      "Ġspin": 2243,\n      "usie": 2244,\n      "Ġsearch": 2245,\n      "Ġsharp": 2246,\n      "Ġprotect": 2247,\n      "Ġlad": 2248,\n      "Ġcow": 2249,\n      "Ġmeet": 2250,\n      "Ġconfused": 2251,\n      "Ġteach": 2252,\n      "Ġwearing": 2253,\n      "Ġoven": 2254,\n      "Ġspent": 2255,\n      "Ġremind": 2256,\n      "Ġanswer": 2257,\n      "nn": 2258,\n      "up": 2259,\n      "Ġhealthy": 2260,\n      "Ġorgan": 2261,\n      "Ġtalked": 2262,\n      "hile": 2263,\n      "Ġant": 2264,\n      "isa": 2265,\n      "Ġshy": 2266,\n      "Ġkitten": 2267,\n      "Ġfoot": 2268,\n      "In": 2269,\n      "Now": 2270,\n      "Ġser": 2271,\n      "Ġfing": 2272,\n      "Ġmil": 2273,\n      "Ġclever": 2274,\n      "Ġfinish": 2275,\n      "Ġrocket": 2276,\n      "ular": 2277,\n      "Ġpromise": 2278,\n      "Ġcorner": 2279,\n      "ned": 2280,\n      "Ġegg": 2281,\n      "Ġtalking": 2282,\n      "ĠEmily": 2283,\n      "Ġwave": 2284,\n      "erly": 2285,\n      "ĠWith": 2286,\n      "Ġstorm": 2287,\n      "Ġwaiting": 2288,\n      "Ġpile": 2289,\n      "els": 2290,\n      "ĠCh": 2291,\n      "Ġembarrassed": 2292,\n      "Ġladder": 2293,\n      "Ġlay": 2294,\n      "Ġlarge": 2295,\n      "Ġfeather": 2296,\n      "ife": 2297,\n      "ours": 2298,\n      "ĠGo": 2299,\n      "Ġtrip": 2300,\n      "Ġsplas": 2301,\n      "Every": 2302,\n      "ĠJake": 2303,\n      "Ġmud": 2304,\n      "Ġtrouble": 2305,\n      "Ġblew": 2306,\n      "Ġmind": 2307,\n      "Ġrec": 2308,\n      "Ġscar": 2309,\n      "Ġval": 2310,\n      "Ġpuddle": 2311,\n      "nic": 2312,\n      "ĠJust": 2313,\n      "Ġpicnic": 2314,\n      "Ġbugs": 2315,\n      "ete": 2316,\n      "Ġforward": 2317,\n      "Ġfurry": 2318,\n      "ese": 2319,\n      "Ġroad": 2320,\n      "Ġsol": 2321,\n      "Ġpush": 2322,\n      "ancy": 2323,\n      "Ġscreamed": 2324,\n      "den": 2325,\n      "Ġmilk": 2326,\n      "St": 2327,\n      "ond": 2328,\n      "Ġcheer": 2329,\n      "Ġsmooth": 2330,\n      "Ġwide": 2331,\n      "inted": 2332,\n      "Ġhar": 2333,\n      "Ġfeet": 2334,\n      "Ġzoom": 2335,\n      "ĠKitty": 2336,\n      "Ġmatter": 2337,\n      "Ġmysterious": 2338,\n      "eb": 2339,\n      "Ġign": 2340,\n      "Ġrope": 2341,\n      "Ġwhenever": 2342,\n      "Ġbread": 2343,\n      "Ġgently": 2344,\n      "ungle": 2345,\n      "Ġpar": 2346,\n      "Ġjungle": 2347,\n      "old": 2348,\n      "Ġpill": 2349,\n      "Ġsal": 2350,\n      "Ġloves": 2351,\n      "Ġseek": 2352,\n      "Ġimpre": 2353,\n      "Ġapples": 2354,\n      "Ġcouch": 2355,\n      "Ġcalm": 2356,\n      "cks": 2357,\n      "Ġcoat": 2358,\n      "osaur": 2359,\n      "Ġsleepy": 2360,\n      "Ġsupp": 2361,\n      "Ġbreath": 2362,\n      "owed": 2363,\n      "Ġdisappo": 2364,\n      "Ġmed": 2365,\n      "ĠTV": 2366,\n      "lex": 2367,\n      "Ġstret": 2368,\n      "pper": 2369,\n      "Ġdrive": 2370,\n      "ĠTogether": 2371,\n      "hin": 2372,\n      "obby": 2373,\n      "Ġcoo": 2374,\n      "Ġgas": 2375,\n      "getable": 2376,\n      "Ġlast": 2377,\n      "Ġhidden": 2378,\n      "Ġstanding": 2379,\n      "Ġhon": 2380,\n      "Ġstood": 2381,\n      "Ġseat": 2382,\n      "Ġcarried": 2383,\n      "Ġslid": 2384,\n      "Ġhiding": 2385,\n      "Ġvegetable": 2386,\n      "ym": 2387,\n      "lla": 2388,\n      "Ġknee": 2389,\n      "Ġfixed": 2390,\n      "Ġpretended": 2391,\n      "Ġgives": 2392,\n      "Ġdirt": 2393,\n      "Ġlemon": 2394,\n      "Ġspider": 2395,\n      "ground": 2396,\n      "Ġban": 2397,\n      "Ġbark": 2398,\n      "ĠSome": 2399,\n      "Ġspe": 2400,\n      "Ġqueen": 2401,\n      "Ġchest": 2402,\n      "ĠRe": 2403,\n      "Ġforgive": 2404,\n      "Ġtaste": 2405,\n      "Ġsongs": 2406,\n      "Ġnut": 2407,\n      "ĠBe": 2408,\n      "Ġshirt": 2409,\n      "ult": 2410,\n      "ale": 2411,\n      "mo": 2412,\n      "Ġstra": 2413,\n      "Ġstone": 2414,\n      "Ġstep": 2415,\n      "Ġyelled": 2416,\n      "Ġducks": 2417,\n      "Ġmot": 2418,\n      "ĠSometimes": 2419,\n      "Ġyarn": 2420,\n      "Ġgroup": 2421,\n      "gly": 2422,\n      "Ġfancy": 2423,\n      "Ġwash": 2424,\n      "Ġugly": 2425,\n      "Ġdinosaur": 2426,\n      "Jane": 2427,\n      "Mama": 2428,\n      "ato": 2429,\n      "Ġtightly": 2430,\n      "Ġroar": 2431,\n      "Ġfaces": 2432,\n      "Ġgrew": 2433,\n      "Ġrude": 2434,\n      "airs": 2435,\n      "Ġsuccess": 2436,\n      "Ġdisappointed": 2437,\n      "per": 2438,\n      "ery": 2439,\n      "ighed": 2440,\n      "urse": 2441,\n      "Ġbecome": 2442,\n      "Ġcheese": 2443,\n      "eterm": 2444,\n      "cap": 2445,\n      "Ġtest": 2446,\n      "Ġple": 2447,\n      "Ġstir": 2448,\n      "Ġmeadow": 2449,\n      "Ġblow": 2450,\n      "Ġpuzz": 2451,\n      "Soon": 2452,\n      "Ġtomorrow": 2453,\n      "izza": 2454,\n      "Mummy": 2455,\n      "Ġcart": 2456,\n      "Ġdeterm": 2457,\n      "Ġship": 2458,\n      "Ġspread": 2459,\n      "Ġvase": 2460,\n      "Ġbubble": 2461,\n      "Ġwhistle": 2462,\n      "Where": 2463,\n      "dle": 2464,\n      "Ġtid": 2465,\n      "Ġwip": 2466,\n      "Ġfig": 2467,\n      "Ġgar": 2468,\n      "Ġgen": 2469,\n      "Ġsave": 2470,\n      "Ġchased": 2471,\n      "Ġsnake": 2472,\n      "Ġclear": 2473,\n      "ĠPeter": 2474,\n      "Ġcookie": 2475,\n      "Ġmachine": 2476,\n      "vous": 2477,\n      "Ġsour": 2478,\n      "ĠHow": 2479,\n      "Ġflut": 2480,\n      "Ġstring": 2481,\n      "ĠEverywhere": 2482,\n      "Ġjumping": 2483,\n      "Ġgiant": 2484,\n      "aking": 2485,\n      "Ġself": 2486,\n      "Ġpour": 2487,\n      "Ġnet": 2488,\n      "ĠTheir": 2489,\n      "ĠJohnny": 2490,\n      "Ġsticks": 2491,\n      "Sarah": 2492,\n      "Ġgun": 2493,\n      "Ġshell": 2494,\n      "umber": 2495,\n      "ĠEven": 2496,\n      "Ġmatch": 2497,\n      "Ġjew": 2498,\n      "\'d": 2499,\n      "Ġber": 2500,\n      "Ġhel": 2501,\n      "Ġreg": 2502,\n      "Ġchick": 2503,\n      "ĠPlease": 2504,\n      "Ġcreative": 2505,\n      "Ġfighting": 2506,\n      "Ġsparkly": 2507,\n      "ual": 2508,\n      "als": 2509,\n      "?\\".": 2510,\n      "Ġemp": 2511,\n      "Ġpowerful": 2512,\n      "Ġchocol": 2513,\n      "Ġscarf": 2514,\n      "Jimmy": 2515,\n      "Ġteet": 2516,\n      "Ġhours": 2517,\n      "Ġmountain": 2518,\n      "ĠSusie": 2519,\n      "ĠMittens": 2520,\n      "Ġexam": 2521,\n      "Ġknocked": 2522,\n      "Ġdancing": 2523,\n      "Ġmotor": 2524,\n      "Ġteeth": 2525,\n      "Ġwore": 2526,\n      "Ġpizza": 2527,\n      "Ġner": 2528,\n      "Ġbell": 2529,\n      "Ġmeant": 2530,\n      "Ġdecor": 2531,\n      "Ġneckl": 2532,\n      "Ġbanan": 2533,\n      "Ġempty": 2534,\n      "Dad": 2535,\n      "Ġhears": 2536,\n      "era": 2537,\n      "Ġhang": 2538,\n      "Ġtrust": 2539,\n      "Ġbarked": 2540,\n      "Ġthese": 2541,\n      "Ġdrew": 2542,\n      "Ġcrab": 2543,\n      "Ġbathroom": 2544,\n      "Ġplants": 2545,\n      "Ġpeaceful": 2546,\n      "Ġgarage": 2547,\n      "Ġchocolate": 2548,\n      "Ġnervous": 2549,\n      "Please": 2550,\n      "light": 2551,\n      "ĠOn": 2552,\n      "Ġpop": 2553,\n      "itter": 2554,\n      "Ġgoes": 2555,\n      "Ġsplash": 2556,\n      "Ġashamed": 2557,\n      "Ġselfish": 2558,\n      "not": 2559,\n      "ying": 2560,\n      "Ġmic": 2561,\n      "Ġrid": 2562,\n      "cept": 2563,\n      "ĠJim": 2564,\n      "Ġanswered": 2565,\n      "Ġgiggled": 2566,\n      "Ġpri": 2567,\n      "oman": 2568,\n      "anut": 2569,\n      "ĠBr": 2570,\n      "There": 2571,\n      "Ġknight": 2572,\n      "izzy": 2573,\n      "Ġpolite": 2574,\n      "Ġsharing": 2575,\n      "To": 2576,\n      "Ġwoman": 2577,\n      "Ġdi": 2578,\n      "Ġcannot": 2579,\n      "Ġswan": 2580,\n      "Ġchew": 2581,\n      "ĠPete": 2582,\n      "Ġzoo": 2583,\n      "Ġpaw": 2584,\n      "Ġeasy": 2585,\n      "most": 2586,\n      "Ġcap": 2587,\n      "Ġmaybe": 2588,\n      "ĠSt": 2589,\n      "Ġshelf": 2590,\n      "Ġlive": 2591,\n      "Ġalmost": 2592,\n      "Ġgener": 2593,\n      "Jill": 2594,\n      "aser": 2595,\n      "Ġhopping": 2596,\n      "box": 2597,\n      "Ġhur": 2598,\n      "ital": 2599,\n      "ĠLisa": 2600,\n      "Ġhosp": 2601,\n      "Ġknowing": 2602,\n      "Ġskipped": 2603,\n      "Ġgray": 2604,\n      "Ġlanded": 2605,\n      "Ġcarrots": 2606,\n      "Ġspray": 2607,\n      "Ġtag": 2608,\n      "Ġpin": 2609,\n      "Ġplayground": 2610,\n      "Ġwhis": 2611,\n      "Ġlock": 2612,\n      "Ġputs": 2613,\n      "Ġcirc": 2614,\n      "Ġnecklace": 2615,\n      "Ġhospital": 2616,\n      "Ġcand": 2617,\n      "Ġpray": 2618,\n      "Ġflag": 2619,\n      "ĠDon": 2620,\n      "Ġbone": 2621,\n      "Ġmummy": 2622,\n      "Ġtea": 2623,\n      "Ġmissed": 2624,\n      "Ġbutton": 2625,\n      "Ġgenerous": 2626,\n      "Be": 2627,\n      "Ġcage": 2628,\n      "Ġoften": 2629,\n      "ĠMum": 2630,\n      "Ġlead": 2631,\n      "ĠWhy": 2632,\n      "Ġexcite": 2633,\n      "Ġpoor": 2634,\n      "Ġbedroom": 2635,\n      "Ġunderstood": 2636,\n      "Ġdisappeared": 2637,\n      "Ġtreats": 2638,\n      "Ġminutes": 2639,\n      "Stop": 2640,\n      "Ġfil": 2641,\n      "Ġplate": 2642,\n      "Ġreturned": 2643,\n      "Ġtears": 2644,\n      "Ġsandwich": 2645,\n      "rated": 2646,\n      "Ġhum": 2647,\n      "Ġlit": 2648,\n      "Ġline": 2649,\n      "Ġasking": 2650,\n      "Ġtool": 2651,\n      "Ġfear": 2652,\n      "Ġworm": 2653,\n      "Ġmarket": 2654,\n      "Ġexcitement": 2655,\n      "Ġsil": 2656,\n      "Ġwander": 2657,\n      "ones": 2658,\n      "Ġgold": 2659,\n      "usual": 2660,\n      "ecially": 2661,\n      "Ġunusual": 2662,\n      "Joe": 2663,\n      "Ġsnacks": 2664,\n      "Ġspoil": 2665,\n      "Ġbalance": 2666,\n      "Ġtidy": 2667,\n      "pecially": 2668,\n      "ub": 2669,\n      "Ġmid": 2670,\n      "Ġsaved": 2671,\n      "orry": 2672,\n      "Ġboss": 2673,\n      "illi": 2674,\n      "Ġtrue": 2675,\n      "Ġglow": 2676,\n      "Ġespecially": 2677,\n      "Ġgathered": 2678,\n      "Ġdetermined": 2679,\n      "Gra": 2680,\n      "cy": 2681,\n      "fl": 2682,\n      "iger": 2683,\n      "Ġmeasure": 2684,\n      "Ġfrust": 2685,\n      "Ġjoke": 2686,\n      "Ġdrum": 2687,\n      "Ġvegetables": 2688,\n      "Ġfather": 2689,\n      "Ġcross": 2690,\n      "Ġfeed": 2691,\n      "Ġpeanut": 2692,\n      "Ġdrawing": 2693,\n      "Ġsmelly": 2694,\n      "Ġperform": 2695,\n      "Ġpassed": 2696,\n      "Ġed": 2697,\n      "ating": 2698,\n      "ork": 2699,\n      "Ġweak": 2700,\n      "Ġweek": 2701,\n      "Ġyourself": 2702,\n      "Ġtaught": 2703,\n      "Ġtells": 2704,\n      "ensive": 2705,\n      "Ġexpensive": 2706,\n      "Ġcolourful": 2707,\n      "Ġmiddle": 2708,\n      "port": 2709,\n      "unk": 2710,\n      "umpy": 2711,\n      "Ġleaf": 2712,\n      "Ġthankful": 2713,\n      "Ġescap": 2714,\n      "Sue": 2715,\n      "pack": 2716,\n      "Ġav": 2717,\n      "Ġpir": 2718,\n      "ĠMummy": 2719,\n      "Ġclap": 2720,\n      "elly": 2721,\n      "Ġpolice": 2722,\n      "Ġfrustrated": 2723,\n      "Ġadd": 2724,\n      "Ġwaves": 2725,\n      "Ġwagged": 2726,\n      "ano": 2727,\n      "ets": 2728,\n      "Ġshore": 2729,\n      "Ġscr": 2730,\n      "ĠLe": 2731,\n      "Ġlazy": 2732,\n      "bin": 2733,\n      "mb": 2734,\n      "ps": 2735,\n      "Ġtur": 2736,\n      "Ġboard": 2737,\n      "ĠAnn": 2738,\n      "Ġtwirl": 2739,\n      "Ġmoon": 2740,\n      "Ġmoving": 2741,\n      "ÅĵLet": 2742,\n      "gy": 2743,\n      "Ġmir": 2744,\n      "Ġstuff": 2745,\n      "Ġcovered": 2746,\n      "Ġcalling": 2747,\n      "Ġwalks": 2748,\n      "Sally": 2749,\n      "uce": 2750,\n      "work": 2751,\n      "Ġboring": 2752,\n      "Ġdepend": 2753,\n      "erp": 2754,\n      "que": 2755,\n      "Ġtaken": 2756,\n      "Ġdeer": 2757,\n      "Ġmarble": 2758,\n      "Ġsailed": 2759,\n      "Who": 2760,\n      "ror": 2761,\n      "Ġhorn": 2762,\n      "ĠTeddy": 2763,\n      "arl": 2764,\n      "Ġcloth": 2765,\n      "Ġdogs": 2766,\n      "Ġcheck": 2767,\n      "Ġcontent": 2768,\n      "Ġpointing": 2769,\n      "ĠTweety": 2770,\n      "Em": 2771,\n      "ches": 2772,\n      "ian": 2773,\n      "Ġtick": 2774,\n      "Ġpil": 2775,\n      "Ġok": 2776,\n      "Ġthick": 2777,\n      "ĠAt": 2778,\n      "issors": 2779,\n      "Ġruined": 2780,\n      "Ġpillow": 2781,\n      "Ġjewel": 2782,\n      "How": 2783,\n      "Ġter": 2784,\n      "Ġburn": 2785,\n      "Ġbored": 2786,\n      "ots": 2787,\n      "Ġang": 2788,\n      "illed": 2789,\n      "Ġflash": 2790,\n      "Ġsweetie": 2791,\n      "Ġtunn": 2792,\n      "Ġcop": 2793,\n      "orge": 2794,\n      "ony": 2795,\n      "Ġweird": 2796,\n      "Ġjam": 2797,\n      "Ġclown": 2798,\n      "Ġnods": 2799,\n      "Ġscissors": 2800,\n      "iff": 2801,\n      "Ġfluffy": 2802,\n      "ushed": 2803,\n      "Ġpainting": 2804,\n      "Ġpan": 2805,\n      "alous": 2806,\n      "ured": 2807,\n      "Ġbull": 2808,\n      "Ġcrack": 2809,\n      "Ġtries": 2810,\n      "Ġjealous": 2811,\n      "Ġmistake": 2812,\n      "Ġhonest": 2813,\n      "Ġpleased": 2814,\n      "Ġfigure": 2815,\n      "Are": 2816,\n      "Billy": 2817,\n      "Ġtask": 2818,\n      "Ġsighed": 2819,\n      "Ġcri": 2820,\n      "imed": 2821,\n      "Ġdolp": 2822,\n      "Ġmanag": 2823,\n      "asing": 2824,\n      "Ġcloset": 2825,\n      "Ġmarch": 2826,\n      "Ġpenny": 2827,\n      "Ġcabin": 2828,\n      "Ġhappening": 2829,\n      "Ġdistance": 2830,\n      "Ġincred": 2831,\n      "Ġcomfortable": 2832,\n      "Ġsplashed": 2833,\n      "sel": 2834,\n      "iness": 2835,\n      "Ġsoap": 2836,\n      "Ġlog": 2837,\n      "Ġspend": 2838,\n      "Ġclass": 2839,\n      "Ġbridge": 2840,\n      "Ġjog": 2841,\n      "Ġdrank": 2842,\n      "Ġdependable": 2843,\n      "selves": 2844,\n      "ĠZ": 2845,\n      "reen": 2846,\n      "Ġcher": 2847,\n      "ilt": 2848,\n      "ator": 2849,\n      "Ġring": 2850,\n      "Ġworking": 2851,\n      "ÅĵThat": 2852,\n      "Ġglue": 2853,\n      "Ġstronger": 2854,\n      "Ġmessy": 2855,\n      "ĠMrs": 2856,\n      "Ġlegs": 2857,\n      "illiant": 2858,\n      "dge": 2859,\n      "eorge": 2860,\n      "ti": 2861,\n      "Ġwis": 2862,\n      "ination": 2863,\n      "Ġlying": 2864,\n      "Ġdest": 2865,\n      "Ġrough": 2866,\n      "Ġknife": 2867,\n      "Ġbrilliant": 2868,\n      "ested": 2869,\n      "Ġfrown": 2870,\n      "Ġrealised": 2871,\n      "Ġtowel": 2872,\n      "ĠRed": 2873,\n      "Ġaccept": 2874,\n      "Ġmighty": 2875,\n      "Ġeager": 2876,\n      "Ġmirror": 2877,\n      "Ġterri": 2878,\n      "Well": 2879,\n      "ject": 2880,\n      "Ġcries": 2881,\n      "oof": 2882,\n      "keep": 2883,\n      "Ġcaterp": 2884,\n      "Ġcrown": 2885,\n      "Ġfinger": 2886,\n      "Ġedge": 2887,\n      "Ġcaterpill": 2888,\n      "Sorry": 2889,\n      "land": 2890,\n      "Ġsight": 2891,\n      "Ġwagon": 2892,\n      "Ġpay": 2893,\n      "Ġwealthy": 2894,\n      "Ġgets": 2895,\n      "Ġchoose": 2896,\n      "Ġsquare": 2897,\n      "Ġcraw": 2898,\n      "Ġsafely": 2899,\n      "Me": 2900,\n      "Ġfake": 2901,\n      "Ġbody": 2902,\n      "ief": 2903,\n      "ÅĵIt": 2904,\n      "Ġdiscovered": 2905,\n      "Ġcaterpillar": 2906,\n      "Ġcrawled": 2907,\n      "ina": 2908,\n      "Ġbitter": 2909,\n      "Ġwand": 2910,\n      "Ġgate": 2911,\n      "Ġrare": 2912,\n      "Ġones": 2913,\n      "ĠAlice": 2914,\n      "Ġbrush": 2915,\n      "ĠEnd": 2916,\n      "aul": 2917,\n      "Ġsmaller": 2918,\n      "umbers": 2919,\n      "Ġkissed": 2920,\n      "Ġsuccessful": 2921,\n      "keeper": 2922,\n      "ĠIf": 2923,\n      "Ġnod": 2924,\n      "oph": 2925,\n      "Ġexcitedly": 2926,\n      "Ġchange": 2927,\n      "Ġpee": 2928,\n      "Ġpicking": 2929,\n      "Ġneeds": 2930,\n      "Ġcleaned": 2931,\n      "Ġprep": 2932,\n      "eez": 2933,\n      "ĠBrown": 2934,\n      "Ġincredible": 2935,\n      "Amy": 2936,\n      "Jen": 2937,\n      "aw": 2938,\n      "Ġer": 2939,\n      "ndpa": 2940,\n      "ately": 2941,\n      "Ġanc": 2942,\n      "Ġtape": 2943,\n      "ctus": 2944,\n      "Ġfamous": 2945,\n      "Ġhurts": 2946,\n      "Ġmixed": 2947,\n      "Ġharm": 2948,\n      "Ġpuzzle": 2949,\n      "aby": 2950,\n      "hip": 2951,\n      "kin": 2952,\n      "Ġsugg": 2953,\n      "Ġheal": 2954,\n      "Ġstones": 2955,\n      "Ġsold": 2956,\n      "Ġfork": 2957,\n      "Ġexcla": 2958,\n      "ÅĵWhat": 2959,\n      "Ġord": 2960,\n      "Ġwolf": 2961,\n      "Ġeye": 2962,\n      "Ġzipp": 2963,\n      "Ġprince": 2964,\n      "Ġdesk": 2965,\n      "Ġpacked": 2966,\n      "ĠTweetie": 2967,\n      "Ġbelonged": 2968,\n      "ĠLeo": 2969,\n      "Ġancient": 2970,\n      "Ġbur": 2971,\n      "Ġnumbers": 2972,\n      "imp": 2973,\n      "Ġears": 2974,\n      "Ġonce": 2975,\n      "Ġseed": 2976,\n      "Ġbranc": 2977,\n      "Ġmod": 2978,\n      "Ġteam": 2979,\n      "Ġprint": 2980,\n      "umbled": 2981,\n      "Ġcactus": 2982,\n      "Ġwheel": 2983,\n      "Ġexclaimed": 2984,\n      "Ġbranches": 2985,\n      "ique": 2986,\n      "ĠOK": 2987,\n      "Ġtie": 2988,\n      "Ġtank": 2989,\n      "ready": 2990,\n      "ats": 2991,\n      "ory": 2992,\n      "Ġsoar": 2993,\n      "!\\".": 2994,\n      "Ġalready": 2995,\n      "Ġrode": 2996,\n      "Ġword": 2997,\n      "Ġlaw": 2998,\n      "Ġpebble": 2999,\n      "ÅĵYes": 3000,\n      "Ġglass": 3001,\n      "Ġunique": 3002,\n      "Ġuseful": 3003,\n      "Ġdrop": 3004,\n      "Ġwalls": 3005,\n      "Ġpractice": 3006,\n      "Ġdelighted": 3007,\n      "Ġimpressed": 3008,\n      "ÅĵLetÃ¢": 3009,\n      "Ġmanaged": 3010,\n      "fic": 3011,\n      "Ġtiger": 3012,\n      "Ġadded": 3013,\n      "Ġfool": 3014,\n      "Ġdish": 3015,\n      "Ġnumber": 3016,\n      "adi": 3017,\n      "ĠBl": 3018,\n      "umsy": 3019,\n      "ĠDucky": 3020,\n      "Ġballs": 3021,\n      "oby": 3022,\n      "berry": 3023,\n      "ĠBobo": 3024,\n      "Ġdolphin": 3025,\n      "cil": 3026,\n      "ha": 3027,\n      "Ġaw": 3028,\n      "edient": 3029,\n      "Ġtoug": 3030,\n      "Ġbake": 3031,\n      "Ġcone": 3032,\n      "anger": 3033,\n      "ĠBet": 3034,\n      "ĠBobby": 3035,\n      "arth": 3036,\n      "Ġcover": 3037,\n      "Ġkinds": 3038,\n      "Ġkindness": 3039,\n      "Ġbushes": 3040,\n      "Ġpencil": 3041,\n      "Ġbottom": 3042,\n      "Ġfruits": 3043,\n      "Ġgrandpa": 3044,\n      "Ġwrapped": 3045,\n      "Ġmedic": 3046,\n      "Ġstraw": 3047,\n      "Ġbossy": 3048,\n      "vice": 3049,\n      "Ġsug": 3050,\n      "Ġhunt": 3051,\n      "Ġboys": 3052,\n      "ages": 3053,\n      "Ġdecide": 3054,\n      "Ġabove": 3055,\n      "Ġexplored": 3056,\n      "Ġadventur": 3057,\n      "Ġenjoying": 3058,\n      "Ġdiffic": 3059,\n      "Ġpainted": 3060,\n      "Ġdead": 3061,\n      "Ġgiving": 3062,\n      "Ġtunnel": 3063,\n      "Ġmedicine": 3064,\n      "uable": 3065,\n      "Ġtent": 3066,\n      "Ġgoose": 3067,\n      "ates": 3068,\n      "Ġeld": 3069,\n      "elp": 3070,\n      "ĠBu": 3071,\n      "Ġgrace": 3072,\n      "Ġenvel": 3073,\n      "ipe": 3074,\n      "Ġobedient": 3075,\n      "Ġreminded": 3076,\n      "Ġflashlight": 3077,\n      "Ġfoolish": 3078,\n      "Ġadventurous": 3079,\n      "gu": 3080,\n      "oe": 3081,\n      "ote": 3082,\n      "Ġsell": 3083,\n      "Ġpost": 3084,\n      "Ġfier": 3085,\n      "eeze": 3086,\n      "Ġguess": 3087,\n      "olog": 3088,\n      "Ġsmoke": 3089,\n      "Ġbounce": 3090,\n      "Ġquestions": 3091,\n      "Ġvaluable": 3092,\n      "Ġterrible": 3093,\n      "Ġdifficult": 3094,\n      "wards": 3095,\n      "Ġicy": 3096,\n      "Ġborrow": 3097,\n      "Ġfis": 3098,\n      "Ġcam": 3099,\n      "ature": 3100,\n      "Ġchar": 3101,\n      "Ġtrump": 3102,\n      "iable": 3103,\n      "Ġgrown": 3104,\n      "ĠCl": 3105,\n      "Ġtrucks": 3106,\n      "Ġbrightly": 3107,\n      "Ġsounded": 3108,\n      "Ġcrawl": 3109,\n      "Ġrelax": 3110,\n      "Ġreliable": 3111,\n      "ĠToday": 3112,\n      "ola": 3113,\n      "Ġapolog": 3114,\n      "Ġangel": 3115,\n      "Ġtough": 3116,\n      "Ġsugar": 3117,\n      "Ġfierce": 3118,\n      "und": 3119,\n      "yal": 3120,\n      "Ġdizzy": 3121,\n      "anged": 3122,\n      "Ġmeal": 3123,\n      "Ġcoins": 3124,\n      "Ġskip": 3125,\n      "Ġfridge": 3126,\n      "Ġrecord": 3127,\n      "Ġbanana": 3128,\n      "Ġcircles": 3129,\n      "Help": 3130,\n      "Ġnorm": 3131,\n      "Ġthose": 3132,\n      "rag": 3133,\n      "Ġext": 3134,\n      "Ġnoisy": 3135,\n      "Ġscare": 3136,\n      "ĠLittle": 3137,\n      "Ġstranger": 3138,\n      "Ġoffice": 3139,\n      "Ġwrote": 3140,\n      "Ġfavour": 3141,\n      "Ġleaving": 3142,\n      "Ġlights": 3143,\n      "Ġcheerful": 3144,\n      "Re": 3145,\n      "ĠNo": 3146,\n      "head": 3147,\n      "aroo": 3148,\n      "Ġsheep": 3149,\n      "Ġsock": 3150,\n      "rage": 3151,\n      "Ġotter": 3152,\n      "apa": 3153,\n      "Ġmov": 3154,\n      "Ġgrumpy": 3155,\n      "Ġdreaming": 3156,\n      "angaroo": 3157,\n      "Ġcrayon": 3158,\n      "Ġenvelope": 3159,\n      "Ġtrumpet": 3160,\n      "af": 3161,\n      "loo": 3162,\n      "Ġfu": 3163,\n      "Ġmis": 3164,\n      "Their": 3165,\n      "Ġkangaroo": 3166,\n      "Ġlose": 3167,\n      "Ġclumsy": 3168,\n      "ella": 3169,\n      "Ġcarp": 3170,\n      "Ġchase": 3171,\n      "Ġboxes": 3172,\n      "body": 3173,\n      "Ġflute": 3174,\n      "Ġescape": 3175,\n      "Ġturkey": 3176,\n      "Ġcamera": 3177,\n      "Ġnormal": 3178,\n      "Ġfavourite": 3179,\n      "rop": 3180,\n      "Ġind": 3181,\n      "Ġham": 3182,\n      "etit": 3183,\n      "Ġbeet": 3184,\n      "Ġrat": 3185,\n      "Ġshake": 3186,\n      "Ġloyal": 3187,\n      "Ġgoat": 3188,\n      "Ġclay": 3189,\n      "Ġcoco": 3190,\n      "ĠFr": 3191,\n      "Ġslipped": 3192,\n      "emo": 3193,\n      "ippo": 3194,\n      "Ġstudy": 3195,\n      "Ġorganized": 3196,\n      "Ġgasped": 3197,\n      "Ġcocoa": 3198,\n      "Alice": 3199,\n      "fa": 3200,\n      "iv": 3201,\n      "oppy": 3202,\n      "Ġfountain": 3203,\n      "ilty": 3204,\n      "Ġtwig": 3205,\n      "Ġpicks": 3206,\n      "Ġadvice": 3207,\n      "berries": 3208,\n      "Ġbags": 3209,\n      "Ġcompetit": 3210,\n      "Ġguilty": 3211,\n      "Ġfeathers": 3212,\n      "Ġeraser": 3213,\n      "Ġsna": 3214,\n      "inal": 3215,\n      "Ġdull": 3216,\n      "Ġstairs": 3217,\n      "Ġspun": 3218,\n      "ĠWhile": 3219,\n      "Ġyours": 3220,\n      "Ġbuilt": 3221,\n      "Ġusing": 3222,\n      "Ġloudly": 3223,\n      "ache": 3224,\n      "Ġzip": 3225,\n      "Ġobject": 3226,\n      "Ġbuttons": 3227,\n      "Ġstretch": 3228,\n      "Ġberries": 3229,\n      "Ġsuggested": 3230,\n      "See": 3231,\n      "Ġfre": 3232,\n      "Ġow": 3233,\n      "llie": 3234,\n      "iginal": 3235,\n      "ention": 3236,\n      "ĠMar": 3237,\n      "Ġsunfl": 3238,\n      "Ġmole": 3239,\n      "Ġstrugg": 3240,\n      "Ġoffered": 3241,\n      "Ġwarned": 3242,\n      "Ġsett": 3243,\n      "Ġdelicate": 3244,\n      "Ġshoot": 3245,\n      "Ġpale": 3246,\n      "Ġthinking": 3247,\n      "Ġresc": 3248,\n      "Ġcooking": 3249,\n      "Ġelderly": 3250,\n      "Ġextra": 3251,\n      "Ġmay": 3252,\n      "Ġgem": 3253,\n      "anny": 3254,\n      "Ġstri": 3255,\n      "Ġhappiness": 3256,\n      "ĠMy": 3257,\n      "Ġisland": 3258,\n      "opus": 3259,\n      "Ġchain": 3260,\n      "Ġlearning": 3261,\n      "Ġtrop": 3262,\n      "Ġoriginal": 3263,\n      "Ġpup": 3264,\n      "âĤ¬âĢ": 3265,\n      "ctopus": 3266,\n      "Ġpatch": 3267,\n      "Ġdoesn": 3268,\n      "Ġjuicy": 3269,\n      "Ġdesign": 3270,\n      "Ġregular": 3271,\n      "fast": 3272,\n      "kn": 3273,\n      "Ġoctopus": 3274,\n      "ĠSm": 3275,\n      "Ġyog": 3276,\n      "oop": 3277,\n      "uring": 3278,\n      "Ġsaying": 3279,\n      "Ġcareless": 3280,\n      "Ġsolve": 3281,\n      "Ġtummy": 3282,\n      "Ġtimes": 3283,\n      "Ġcorn": 3284,\n      "Ġmelt": 3285,\n      "Ġpant": 3286,\n      "Ġbees": 3287,\n      "Ġlicked": 3288,\n      "Ġjack": 3289,\n      "Ġnosy": 3290,\n      "ellig": 3291,\n      "Ġchir": 3292,\n      "Ġvan": 3293,\n      "Ġthoughtful": 3294,\n      "Ġintellig": 3295,\n      "Ġshowing": 3296,\n      "osie": 3297,\n      "Ġmoder": 3298,\n      "Ġfright": 3299,\n      "ĠGrace": 3300,\n      "Ġbreakfast": 3301,\n      "Ġquestion": 3302,\n      "Ġpirate": 3303,\n      "Ġpeeked": 3304,\n      "Ġcarpet": 3305,\n      "Ġtrophy": 3306,\n      "com": 3307,\n      "lies": 3308,\n      "Ġhippo": 3309,\n      "ither": 3310,\n      "Ġstumbled": 3311,\n      "Ġmoms": 3312,\n      "Ġdove": 3313,\n      "Ġjelly": 3314,\n      "Ġhelps": 3315,\n      "Ġolder": 3316,\n      "Ġknows": 3317,\n      "Ġskin": 3318,\n      "Ġenc": 3319,\n      "Ġcleaning": 3320,\n      "Ġmagn": 3321,\n      "Ġhats": 3322,\n      "Ġattic": 3323,\n      "Ġcelebrate": 3324,\n      "Ġserious": 3325,\n      "ĠRex": 3326,\n      "known": 3327,\n      "Ġmodern": 3328,\n      "ress": 3329,\n      "Ġfine": 3330,\n      "omed": 3331,\n      "Ġhero": 3332,\n      "Ġshone": 3333,\n      "Ġrefused": 3334,\n      "Ġalert": 3335,\n      "ouses": 3336,\n      "isted": 3337,\n      "Ġproudly": 3338,\n      "ĠPapa": 3339,\n      "oomy": 3340,\n      "Ġneighbour": 3341,\n      "Ġsalad": 3342,\n      "Ġbubbles": 3343,\n      "Ġchicken": 3344,\n      "ĠBlue": 3345,\n      "Ġintelligent": 3346,\n      "time": 3347,\n      "het": 3348,\n      "ĠToby": 3349,\n      "orant": 3350,\n      "Ġrich": 3351,\n      "ugged": 3352,\n      "Ġleop": 3353,\n      "Ġleash": 3354,\n      "Ġending": 3355,\n      "Here": 3356,\n      "ĠPolly": 3357,\n      "arsh": 3358,\n      "Ġinvited": 3359,\n      "Ġcircle": 3360,\n      "Ġadmired": 3361,\n      "Ġeggs": 3362,\n      "Ġzipper": 3363,\n      "az": 3364,\n      "ines": 3365,\n      "Ġhouses": 3366,\n      "Ġmap": 3367,\n      "Ġpow": 3368,\n      "Ġoy": 3369,\n      "Ġstream": 3370,\n      "Ġsofa": 3371,\n      "Ġlibr": 3372,\n      "Ġlocked": 3373,\n      "ĠAl": 3374,\n      "Ġlean": 3375,\n      "Ġsword": 3376,\n      "Ġharder": 3377,\n      "ĠNemo": 3378,\n      "Ġperm": 3379,\n      "Ġdreamed": 3380,\n      "ĠGrandpa": 3381,\n      "Ġcounted": 3382,\n      "Ġpengu": 3383,\n      "ophie": 3384,\n      "Sp": 3385,\n      "Ġing": 3386,\n      "Ġturt": 3387,\n      "Ġph": 3388,\n      "Ġgum": 3389,\n      "Ġgym": 3390,\n      "Ġfriendship": 3391,\n      "aghet": 3392,\n      "istent": 3393,\n      "Ġtrack": 3394,\n      "Ġhandle": 3395,\n      "Ġraced": 3396,\n      "ĠGeorge": 3397,\n      "Ġmaze": 3398,\n      "redient": 3399,\n      "Ġpige": 3400,\n      "Ġlifted": 3401,\n      "Ġparade": 3402,\n      "Ġhurry": 3403,\n      "Ġcandle": 3404,\n      "Ġbeetle": 3405,\n      "Ġfresh": 3406,\n      "ĠSmith": 3407,\n      "Ġfrightened": 3408,\n      "Ġingredient": 3409,\n      "aghetti": 3410,\n      "Mary": 3411,\n      "ext": 3412,\n      "gust": 3413,\n      "ĠU": 3414,\n      "Ġdig": 3415,\n      "Ġwashed": 3416,\n      "Ġrushed": 3417,\n      "Ġroof": 3418,\n      "Ġplayful": 3419,\n      "Ġforth": 3420,\n      "Ġspaghetti": 3421,\n      "Ġchanged": 3422,\n      "Ġshows": 3423,\n      "Ġsorts": 3424,\n      "ĠWhenever": 3425,\n      "Ġstove": 3426,\n      "Ġunl": 3427,\n      "Ġrubbed": 3428,\n      "Ġignorant": 3429,\n      "Ġpilot": 3430,\n      "Ġcricket": 3431,\n      "Ġsunflower": 3432,\n      "now": 3433,\n      "ud": 3434,\n      "year": 3435,\n      "Ġsudden": 3436,\n      "Ġhook": 3437,\n      "erable": 3438,\n      "Ġped": 3439,\n      "Ġearth": 3440,\n      "Ġstamp": 3441,\n      "Ġliz": 3442,\n      "Ġbirdie": 3443,\n      "ĠFred": 3444,\n      "Ġpepper": 3445,\n      "Ġsniff": 3446,\n      "Ġarrow": 3447,\n      "Ġlouder": 3448,\n      "Ġunknown": 3449,\n      "Ġdisgust": 3450,\n      "Ġbald": 3451,\n      "Ġdreams": 3452,\n      "Ġshape": 3453,\n      "Ġspoke": 3454,\n      "Ġsearching": 3455,\n      "Ġspeed": 3456,\n      "Ġwiped": 3457,\n      "Ġharmless": 3458,\n      "Ġmiserable": 3459,\n      "Ġingredients": 3460,\n      "gest": 3461,\n      "Ġson": 3462,\n      "Ġtoast": 3463,\n      "Ġbin": 3464,\n      "Ġbase": 3465,\n      "Ġbiggest": 3466,\n      "Ġswe": 3467,\n      "ĠFin": 3468,\n      "Ġcomb": 3469,\n      "Ġsnugg": 3470,\n      "ĠJoey": 3471,\n      "Ġfalls": 3472,\n      "Ġwhisp": 3473,\n      "Ġspoiled": 3474,\n      "Ġcharming": 3475,\n      "Ġmovie": 3476,\n      "Ġencou": 3477,\n      "Ġphot": 3478,\n      "Ġfour": 3479,\n      "Ġlamp": 3480,\n      "Ġcudd": 3481,\n      "atch": 3482,\n      "Ġshield": 3483,\n      "Ġspace": 3484,\n      "Ġscale": 3485,\n      "Ġcost": 3486,\n      "ĠcouldnÃ¢": 3487,\n      "Ġways": 3488,\n      "apping": 3489,\n      "Ġshopkeeper": 3490,\n      "Ġtomato": 3491,\n      "Ġdiscover": 3492,\n      "Ġhelm": 3493,\n      "comfort": 3494,\n      "Ġtire": 3495,\n      "Ġaunt": 3496,\n      "Ġbump": 3497,\n      "Ġcush": 3498,\n      "Ġmask": 3499,\n      "Ġumb": 3500,\n      "seum": 3501,\n      "Ġlaund": 3502,\n      "bborn": 3503,\n      "Ġenorm": 3504,\n      "ĠdonÃ¢": 3505,\n      "Ġmeans": 3506,\n      "Ġtelling": 3507,\n      "Ġbatter": 3508,\n      "Ġguard": 3509,\n      "Ġstubborn": 3510,\n      "Ġpersistent": 3511,\n      "host": 3512,\n      "Ġearly": 3513,\n      "Ġmicrop": 3514,\n      "eezed": 3515,\n      "Ġpants": 3516,\n      "Ġturtle": 3517,\n      "Ġwhispered": 3518,\n      "Ġumbre": 3519,\n      "Ġenormous": 3520,\n      "Ġmicrophone": 3521,\n      "Ġcell": 3522,\n      "Ġnurse": 3523,\n      "ckey": 3524,\n      "Ġyet": 3525,\n      "Ġload": 3526,\n      "illa": 3527,\n      "Ġthemselves": 3528,\n      "Ġroared": 3529,\n      "ĠWould": 3530,\n      "ĠYour": 3531,\n      "Ġmuseum": 3532,\n      "Ġagainst": 3533,\n      "Ġhockey": 3534,\n      "ĠPe": 3535,\n      "Ġjoined": 3536,\n      "Ġuncomfort": 3537,\n      "Ġsandc": 3538,\n      "Ġcounting": 3539,\n      "Ġpackage": 3540,\n      "Ġkicked": 3541,\n      "Ġnuts": 3542,\n      "Ġtools": 3543,\n      "Ġlizard": 3544,\n      "Ġuncomfortable": 3545,\n      "While": 3546,\n      "bled": 3547,\n      "cream": 3548,\n      "xe": 3549,\n      "Ġtied": 3550,\n      "Ġaxe": 3551,\n      "Ġflex": 3552,\n      "Ġcamp": 3553,\n      "Ġthirst": 3554,\n      "Ġthief": 3555,\n      "irit": 3556,\n      "Ġanx": 3557,\n      "Ġloll": 3558,\n      "Ġsadly": 3559,\n      "Ġenv": 3560,\n      "Ġtrains": 3561,\n      "Ġprize": 3562,\n      "Ġfilm": 3563,\n      "Ġleopard": 3564,\n      "Ġlaundry": 3565,\n      "cope": 3566,\n      "pid": 3567,\n      "Ġsuit": 3568,\n      "Ġwitch": 3569,\n      "inary": 3570,\n      "erry": 3571,\n      "ĠSandy": 3572,\n      "ĠLook": 3573,\n      "Ġchasing": 3574,\n      "Ġvi": 3575,\n      "ĠDave": 3576,\n      "ipop": 3577,\n      "Ġrestaur": 3578,\n      "Ġrules": 3579,\n      "Ġballoons": 3580,\n      "Ġgrowing": 3581,\n      "gging": 3582,\n      "Ġstupid": 3583,\n      "Ġbarber": 3584,\n      "Ġcarrying": 3585,\n      "Ġimagined": 3586,\n      "Ġpigeon": 3587,\n      "Ġthirsty": 3588,\n      "Ġlollipop": 3589,\n      "Ġrestaurant": 3590,\n      "Yay": 3591,\n      "ience": 3592,\n      "heart": 3593,\n      "Ġahead": 3594,\n      "Ġwake": 3595,\n      "Ġfrag": 3596,\n      "Ġled": 3597,\n      "Ġlow": 3598,\n      "Ġghost": 3599,\n      "lev": 3600,\n      "Ġkitty": 3601,\n      "Ġrep": 3602,\n      "Ġnotice": 3603,\n      "Ġspilled": 3604,\n      "Ġscreen": 3605,\n      "Ġchose": 3606,\n      "Ġfinding": 3607,\n      "Ġcompl": 3608,\n      "Ġrub": 3609,\n      "Ġhung": 3610,\n      "Ġcompass": 3611,\n      "Ġsparkled": 3612,\n      "Ġsilver": 3613,\n      "Ġflexible": 3614,\n      "eer": 3615,\n      "Ġhive": 3616,\n      "Ġcity": 3617,\n      "ath": 3618,\n      "Ġshocked": 3619,\n      "Ġreind": 3620,\n      "Ġspirit": 3621,\n      "ict": 3622,\n      "Ġmem": 3623,\n      "Ġhose": 3624,\n      "Ġanywhere": 3625,\n      "Ġquite": 3626,\n      "Ġbeauty": 3627,\n      "ĠRose": 3628,\n      "Ġstops": 3629,\n      "Ġbutterflies": 3630,\n      "Ġcreate": 3631,\n      "Ġlaughs": 3632,\n      "Ġcomplet": 3633,\n      "Ġwrap": 3634,\n      "Ġindepend": 3635,\n      "Ġdisgusting": 3636,\n      "Ġanxious": 3637,\n      "ikes": 3638,\n      "pes": 3639,\n      "Ġbent": 3640,\n      "atter": 3641,\n      "Ġya": 3642,\n      "Ġstat": 3643,\n      "Ġshiver": 3644,\n      "ummer": 3645,\n      "iew": 3646,\n      "ĠEventually": 3647,\n      "Ġbloom": 3648,\n      "Ġview": 3649,\n      "Ġhanded": 3650,\n      "Ġnicely": 3651,\n      "Ġunpack": 3652,\n      "Ġlistening": 3653,\n      "Ġtrash": 3654,\n      "ĠKate": 3655,\n      "Ġmetal": 3656,\n      "ggy": 3657,\n      "Ġfolder": 3658,\n      "Ġtwirled": 3659,\n      "Ġgraceful": 3660,\n      "Ġumbrella": 3661,\n      "Ġreindeer": 3662,\n      "Ġtheater": 3663,\n      "Ġhay": 3664,\n      "Ġpump": 3665,\n      "arnt": 3666,\n      "ĠSnow": 3667,\n      "imney": 3668,\n      "Ġur": 3669,\n      "Ġbeak": 3670,\n      "Ġlearnt": 3671,\n      "Ġchimney": 3672,\n      "uct": 3673,\n      "Ġarg": 3674,\n      "Ġjokes": 3675,\n      "Ġdeliver": 3676,\n      "Ġbehave": 3677,\n      "Ġsweetheart": 3678,\n      "Ġsleeping": 3679,\n      "ations": 3680,\n      "Ġreading": 3681,\n      "Ġwindows": 3682,\n      "Ġadmire": 3683,\n      "Ġpoured": 3684,\n      "Ġdestro": 3685,\n      "Ġfragile": 3686,\n      "gn": 3687,\n      "noon": 3688,\n      "ubby": 3689,\n      "Ġwire": 3690,\n      "ince": 3691,\n      "Ġhoop": 3692,\n      "render": 3693,\n      "Ġnature": 3694,\n      "Ġref": 3695,\n      "Ġrece": 3696,\n      "ĠMike": 3697,\n      "Ġfactor": 3698,\n      "Ġmush": 3699,\n      "Ġchubby": 3700,\n      "Ġjolly": 3701,\n      "Ġafternoon": 3702,\n      "Ġsticker": 3703,\n      "Ġattract": 3704,\n      "Ġcourage": 3705,\n      "Ġmailbox": 3706,\n      "Ġriding": 3707,\n      "Ġjacket": 3708,\n      "Spot": 3709,\n      "Ġhelmet": 3710,\n      "Ġyawn": 3711,\n      "Ġfactory": 3712,\n      "Ġmushroom": 3713,\n      "med": 3714,\n      "ser": 3715,\n      "zzy": 3716,\n      "Ġshort": 3717,\n      "Ġwhale": 3718,\n      "Ġreco": 3719,\n      "ĠMark": 3720,\n      "tern": 3721,\n      "ĠWh": 3722,\n      "Ġagree": 3723,\n      "robe": 3724,\n      "ucked": 3725,\n      "Ġeng": 3726,\n      "ÅĵYou": 3727,\n      "ĠAndy": 3728,\n      "Ġtowers": 3729,\n      "Ġrubb": 3730,\n      "Ġpotato": 3731,\n      "Ġtroubled": 3732,\n      "Ġcreatures": 3733,\n      "Ġsurrender": 3734,\n      "Ġfootball": 3735,\n      "Ġstatue": 3736,\n      "Little": 3737,\n      "aid": 3738,\n      "hood": 3739,\n      "oot": 3740,\n      "Ġtem": 3741,\n      "Ġsym": 3742,\n      "Ġharsh": 3743,\n      "Ġcomet": 3744,\n      "Ġcards": 3745,\n      "Ġdess": 3746,\n      "isk": 3747,\n      "iron": 3748,\n      "Ġrose": 3749,\n      "Ġmule": 3750,\n      "Ġtrunk": 3751,\n      "Ġraven": 3752,\n      "Ġclosely": 3753,\n      "Ġhairy": 3754,\n      "lcano": 3755,\n      "Ġplaced": 3756,\n      "Ġsurf": 3757,\n      "Ġfingers": 3758,\n      "Ġenvious": 3759,\n      "Ġcompassion": 3760,\n      "Ġindependent": 3761,\n      "cu": 3762,\n      "Ġsend": 3763,\n      "Ġbet": 3764,\n      "anic": 3765,\n      "ĠitÃ¢": 3766,\n      "Ġitself": 3767,\n      "Ġstared": 3768,\n      "Ġstation": 3769,\n      "une": 3770,\n      "Ġshaking": 3771,\n      "ĠAre": 3772,\n      "iches": 3773,\n      "Ġstarts": 3774,\n      "Ġmuff": 3775,\n      "Ġflour": 3776,\n      "ĠElla": 3777,\n      "uddy": 3778,\n      "Ġpeek": 3779,\n      "Ġstickers": 3780,\n      "Ġcrane": 3781,\n      "Ġpoint": 3782,\n      "Ġmissing": 3783,\n      "Ġmixer": 3784,\n      "Ġdistant": 3785,\n      "Ġwheat": 3786,\n      "Ġpracticed": 3787,\n      "Ġshells": 3788,\n      "Ġdecorate": 3789,\n      "Ġfearful": 3790,\n      "Ġviol": 3791,\n      "Ġdessert": 3792,\n      "par": 3793,\n      "Ġdough": 3794,\n      "Ġoak": 3795,\n      "Ġstack": 3796,\n      "chanic": 3797,\n      "Ġrepe": 3798,\n      "aled": 3799,\n      "ecause": 3800,\n      "Ġboots": 3801,\n      "Ġsepar": 3802,\n      "Ġallowed": 3803,\n      "Ġscatter": 3804,\n      "Ġmechanic": 3805,\n      "used": 3806,\n      "Ġgrin": 3807,\n      "Ġpainter": 3808,\n      "Ġsquash": 3809,\n      "Ġtravel": 3810,\n      "Ġdrawer": 3811,\n      "Ġtrick": 3812,\n      "bol": 3813,\n      "Ġsecond": 3814,\n      "ipping": 3815,\n      "Grandma": 3816,\n      "Ġdestroy": 3817,\n      "Ġrecogn": 3818,\n      "Ġsummer": 3819,\n      "Ġfat": 3820,\n      "Ġfurt": 3821,\n      "Ġgor": 3822,\n      "Ġsauce": 3823,\n      "Ġherb": 3824,\n      "ĠIÃ¢": 3825,\n      "ĠBaby": 3826,\n      "Ġshark": 3827,\n      "Ġshadow": 3828,\n      "icop": 3829,\n      "ĠWhere": 3830,\n      "ĠLola": 3831,\n      "Ġsne": 3832,\n      "Ġgloomy": 3833,\n      "Ġart": 3834,\n      "Ġframe": 3835,\n      "ĠRo": 3836,\n      "Ġpattern": 3837,\n      "Ġsandwiches": 3838,\n      "Ġhelicop": 3839,\n      "Ġordinary": 3840,\n      "Ġmodest": 3841,\n      "Ġcompetitive": 3842,\n      "Ġsymbol": 3843,\n      "Ġviolin": 3844,\n      "Ġhelicopter": 3845,\n      "Ow": 3846,\n      "Ġmill": 3847,\n      "Ġmama": 3848,\n      "Ġpit": 3849,\n      "Ġpurse": 3850,\n      "Ġol": 3851,\n      "orable": 3852,\n      "ĠBella": 3853,\n      "Ġshine": 3854,\n      "Ġimpat": 3855,\n      "Ġimpro": 3856,\n      "Ġteleph": 3857,\n      "ĠRosie": 3858,\n      "Ġsailor": 3859,\n      "Ġpassport": 3860,\n      "ĠRemy": 3861,\n      "Ġhumble": 3862,\n      "Ġoyster": 3863,\n      "Ġlibrary": 3864,\n      "Ġattractive": 3865,\n      "Ġtelephone": 3866,\n      "Go": 3867,\n      "ci": 3868,\n      "ffe": 3869,\n      "iest": 3870,\n      "loe": 3871,\n      "yer": 3872,\n      "Ġcur": 3873,\n      "Ġnail": 3874,\n      "ric": 3875,\n      "elery": 3876,\n      "Ġkay": 3877,\n      "illie": 3878,\n      "ĠAlex": 3879,\n      "Ġknock": 3880,\n      "Ġclock": 3881,\n      "Ġstarting": 3882,\n      "play": 3883,\n      "bbie": 3884,\n      "Ġcats": 3885,\n      "osh": 3886,\n      "Ġadorable": 3887,\n      "Ġcheap": 3888,\n      "Ġglove": 3889,\n      "Ġbrighter": 3890,\n      "Ġbrothers": 3891,\n      "Ġgiven": 3892,\n      "Ġvisited": 3893,\n      "Ġrolling": 3894,\n      "Ġincl": 3895,\n      "Ġorganize": 3896,\n      "Ġlemonade": 3897,\n      "Ġseeds": 3898,\n      "achel": 3899,\n      "âĤ¬âĢľ": 3900,\n      "Ġcushion": 3901,\n      "Ġfurther": 3902,\n      "Ġgorilla": 3903,\n      "Ġimpatient": 3904,\n      "ffee": 3905,\n      "night": 3906,\n      "shed": 3907,\n      "Ġsize": 3908,\n      "ision": 3909,\n      "ans": 3910,\n      "uth": 3911,\n      "amond": 3912,\n      "kelet": 3913,\n      "Ġhadn": 3914,\n      "Ġspell": 3915,\n      "ĠJul": 3916,\n      "Ġscrew": 3917,\n      "Ġpole": 3918,\n      "Ġener": 3919,\n      "Ġnowhere": 3920,\n      "Ġvolcano": 3921,\n      "Ġdisag": 3922,\n      "Ġdisplay": 3923,\n      "Ġforgave": 3924,\n      "Ġzeb": 3925,\n      "Ġpushing": 3926,\n      "Ġsteal": 3927,\n      "ĠKim": 3928,\n      "Ġvend": 3929,\n      "Ġtreasures": 3930,\n      "Ġenth": 3931,\n      "Ġspinning": 3932,\n      "Ġdiamond": 3933,\n      "Ġavo": 3934,\n      "Ġlawyer": 3935,\n      "Ġfuel": 3936,\n      "Ġzebra": 3937,\n      "Daisy": 3938,\n      "reci": 3939,\n      "ison": 3940,\n      "Ġsaus": 3941,\n      "ĠSophie": 3942,\n      "Ġrice": 3943,\n      "alk": 3944,\n      "ĠWill": 3945,\n      "Ġvest": 3946,\n      "Ġtruth": 3947,\n      "Ġclimbing": 3948,\n      "Ġappreci": 3949,\n      "Ġboats": 3950,\n      "Ġfirework": 3951,\n      "Ġdriving": 3952,\n      "Ġbounced": 3953,\n      "Ġexamine": 3954,\n      "Ġpopular": 3955,\n      "Ġfrowned": 3956,\n      "ÅĵItÃ¢": 3957,\n      "Ġsnuggled": 3958,\n      "ĠSnowy": 3959,\n      "Ġenthus": 3960,\n      "come": 3961,\n      "eath": 3962,\n      "ray": 3963,\n      "Ġiron": 3964,\n      "Ġtor": 3965,\n      "Ġcree": 3966,\n      "Ġcro": 3967,\n      "ĠTr": 3968,\n      "arian": 3969,\n      "Ġhalf": 3970,\n      "ĠHere": 3971,\n      "Ġshel": 3972,\n      "ĠBuddy": 3973,\n      "Ġmeas": 3974,\n      "Ġtrap": 3975,\n      "Ġobser": 3976,\n      "Ġspring": 3977,\n      "Ġcolours": 3978,\n      "Ġcardboard": 3979,\n      "Ġimpress": 3980,\n      "Ġstretched": 3981,\n      "cycle": 3982,\n      "Ġrescue": 3983,\n      "Give": 3984,\n      "ron": 3985,\n      "Ġskelet": 3986,\n      "inking": 3987,\n      "Ġbikes": 3988,\n      "Ġlend": 3989,\n      "Ġluck": 3990,\n      "Ġdive": 3991,\n      "Ġmen": 3992,\n      "Ġpage": 3993,\n      "Ġins": 3994,\n      "Ġrug": 3995,\n      "Ġshut": 3996,\n      "Ġreve": 3997,\n      "Ġisn": 3998,\n      "Ġmeat": 3999,\n      "Ġraft": 4000,\n      "Ġbadly": 4001,\n      "Ġcupboard": 4002,\n      "Ġcere": 4003,\n      "Ġslides": 4004,\n      "Ġhunter": 4005,\n      "Ġmagnet": 4006,\n      "Ġpowder": 4007,\n      "ef": 4008,\n      "fish": 4009,\n      "zor": 4010,\n      "erc": 4011,\n      "Ġpad": 4012,\n      "Ġpair": 4013,\n      "Ġgro": 4014,\n      "Ġyach": 4015,\n      "Ġgoal": 4016,\n      "urch": 4017,\n      "Ġsomewhere": 4018,\n      "Ġcoffee": 4019,\n      "Ġsunset": 4020,\n      "Ġrazor": 4021,\n      "Ġdeaf": 4022,\n      "Ġsub": 4023,\n      "Ġcollar": 4024,\n      "Ġwishes": 4025,\n      "Ġhammer": 4026,\n      "ĠMaria": 4027,\n      "Ġbaseball": 4028,\n      "Ġcostum": 4029,\n      "Ġengine": 4030,\n      "Ġvendor": 4031,\n      "Ġyacht": 4032,\n      "Because": 4033,\n      "cc": 4034,\n      "dom": 4035,\n      "lb": 4036,\n      "ocer": 4037,\n      "Ġdug": 4038,\n      "any": 4039,\n      "Ġrhin": 4040,\n      "ado": 4041,\n      "Ġweep": 4042,\n      "Ġreef": 4043,\n      "indy": 4044,\n      "Ġnames": 4045,\n      "Ġhelpless": 4046,\n      "Ġscold": 4047,\n      "Ġlab": 4048,\n      "Ġputting": 4049,\n      "Ġanyway": 4050,\n      "Ġbulb": 4051,\n      "ected": 4052,\n      "astles": 4053,\n      "Ġshouldn": 4054,\n      "Ġopens": 4055,\n      "Ġexper": 4056,\n      "ailable": 4057,\n      "Ġrespectful": 4058,\n      "Ġsupport": 4059,\n      "arlie": 4060,\n      "Ġprepare": 4061,\n      "Jenny": 4062,\n      "Ġcreek": 4063,\n      "oceros": 4064,\n      "Ġrhinoceros": 4065,\n      "corn": 4066,\n      "ccol": 4067,\n      "gs": 4068,\n      "gen": 4069,\n      "rise": 4070,\n      "Ġten": 4071,\n      "Ġtut": 4072,\n      "Ġwife": 4073,\n      "Ġpun": 4074,\n      "Ġbeep": 4075,\n      "stairs": 4076,\n      "Ġweather": 4077,\n      "Ġreward": 4078,\n      "Ġjet": 4079,\n      "ills": 4080,\n      "Ġexerc": 4081,\n      "âĤ¬.": 4082,\n      "Ġclapping": 4083,\n      "Ġswamp": 4084,\n      "Ġvide": 4085,\n      "Ġvine": 4086,\n      "ĠdidnÃ¢": 4087,\n      "ĠEverything": 4088,\n      "Ġneedle": 4089,\n      "ĠGive": 4090,\n      "Ġhoping": 4091,\n      "Ġpapers": 4092,\n      "Ġmarched": 4093,\n      "Ġresist": 4094,\n      "Ġbandage": 4095\n    },\n    "merges": [\n      [\n        "h",\n        "e"\n      ],\n      [\n        "Ġ",\n        "t"\n      ],\n      [\n        "Ġ",\n        "a"\n      ],\n      [\n        "Ġ",\n        "s"\n      ],\n      [\n        "Ġ",\n        "w"\n      ],\n      [\n        "n",\n        "d"\n      ],\n      [\n        "Ġt",\n        "he"\n      ],\n      [\n        "e",\n        "d"\n      ],\n      [\n        "Ġa",\n        "nd"\n      ],\n      [\n        "Ġt",\n        "o"\n      ],\n      [\n        "i",\n        "n"\n      ],\n      [\n        "Ġ",\n        "b"\n      ],\n      [\n        "Ġ",\n        "h"\n      ],\n      [\n        "Ġw",\n        "a"\n      ],\n      [\n        "r",\n        "e"\n      ],\n      [\n        "o",\n        "u"\n      ],\n      [\n        "Ġ",\n        "f"\n      ],\n      [\n        "i",\n        "t"\n      ],\n      [\n        "Ġ",\n        "l"\n      ],\n      [\n        "Ġ",\n        "c"\n      ],\n      [\n        "Ġ",\n        "he"\n      ],\n      [\n        "Ġ",\n        "d"\n      ],\n      [\n        "e",\n        "r"\n      ],\n      [\n        "Ġ",\n        "m"\n      ],\n      [\n        "Ġwa",\n        "s"\n      ],\n      [\n        "Ġ",\n        "p"\n      ],\n      [\n        "o",\n        "m"\n      ],\n      [\n        "Ġ",\n        "o"\n      ],\n      [\n        "Ġ",\n        "T"\n      ],\n      [\n        "a",\n        "y"\n      ],\n      [\n        "in",\n        "g"\n      ],\n      [\n        "a",\n        "r"\n      ],\n      [\n        "i",\n        "s"\n      ],\n      [\n        "i",\n        "l"\n      ],\n      [\n        "Ġ",\n        "g"\n      ],\n      [\n        "i",\n        "d"\n      ],\n      [\n        "a",\n        "t"\n      ],\n      [\n        "Ġs",\n        "a"\n      ],\n      [\n        "e",\n        "n"\n      ],\n      [\n        "Ġ",\n        "n"\n      ],\n      [\n        "Ġh",\n        "a"\n      ],\n      [\n        "Ġ",\n        "S"\n      ],\n      [\n        "a",\n        "n"\n      ],\n      [\n        "i",\n        "m"\n      ],\n      [\n        "ĠT",\n        "he"\n      ],\n      [\n        "o",\n        "r"\n      ],\n      [\n        "o",\n        "n"\n      ],\n      [\n        "Ġ",\n        "it"\n      ],\n      [\n        "Ġt",\n        "h"\n      ],\n      [\n        "l",\n        "l"\n      ],\n      [\n        "l",\n        "e"\n      ],\n      [\n        "Ġ",\n        "H"\n      ],\n      [\n        "e",\n        "t"\n      ],\n      [\n        "Ġhe",\n        "r"\n      ],\n      [\n        "i",\n        "r"\n      ],\n      [\n        "o",\n        "t"\n      ],\n      [\n        "ĠS",\n        "he"\n      ],\n      [\n        "ĠH",\n        "e"\n      ],\n      [\n        "Ġ",\n        "in"\n      ],\n      [\n        "e",\n        "s"\n      ],\n      [\n        "v",\n        "er"\n      ],\n      [\n        "o",\n        "w"\n      ],\n      [\n        "u",\n        "t"\n      ],\n      [\n        "c",\n        "k"\n      ],\n      [\n        "Ġsa",\n        "id"\n      ],\n      [\n        "Ġ",\n        "e"\n      ],\n      [\n        "Ġ",\n        "u"\n      ],\n      [\n        "l",\n        "d"\n      ],\n      [\n        "Ġ",\n        "y"\n      ],\n      [\n        "Ġ",\n        "\\""\n      ],\n      [\n        "o",\n        "o"\n      ],\n      [\n        "i",\n        "g"\n      ],\n      [\n        "ĠThe",\n        "y"\n      ],\n      [\n        "Ġb",\n        "e"\n      ],\n      [\n        "a",\n        "m"\n      ],\n      [\n        "Ġ",\n        "r"\n      ],\n      [\n        "Ġs",\n        "t"\n      ],\n      [\n        "il",\n        "y"\n      ],\n      [\n        "c",\n        "e"\n      ],\n      [\n        "Ġs",\n        "he"\n      ],\n      [\n        "p",\n        "p"\n      ],\n      [\n        "Ġ",\n        "I"\n      ],\n      [\n        "it",\n        "h"\n      ],\n      [\n        "k",\n        "e"\n      ],\n      [\n        "O",\n        "n"\n      ],\n      [\n        "Ġw",\n        "ith"\n      ],\n      [\n        "v",\n        "e"\n      ],\n      [\n        "k",\n        "ed"\n      ],\n      [\n        "Ġo",\n        "f"\n      ],\n      [\n        "Ġo",\n        "n"\n      ],\n      [\n        "L",\n        "ily"\n      ],\n      [\n        "r",\n        "i"\n      ],\n      [\n        "Ġs",\n        "o"\n      ],\n      [\n        "Ġy",\n        "ou"\n      ],\n      [\n        "Ġh",\n        "is"\n      ],\n      [\n        "Ġp",\n        "l"\n      ],\n      [\n        "Ġd",\n        "ay"\n      ],\n      [\n        "a",\n        "d"\n      ],\n      [\n        "n",\n        "t"\n      ],\n      [\n        "ver",\n        "y"\n      ],\n      [\n        "Ġth",\n        "at"\n      ],\n      [\n        "Ġu",\n        "p"\n      ],\n      [\n        "Ġpl",\n        "ay"\n      ],\n      [\n        "s",\n        "t"\n      ],\n      [\n        "Ġha",\n        "d"\n      ],\n      [\n        "Ġw",\n        "e"\n      ],\n      [\n        "Ġm",\n        "om"\n      ],\n      [\n        "Ġthe",\n        "y"\n      ],\n      [\n        "Ġ",\n        "Lily"\n      ],\n      [\n        "e",\n        "l"\n      ],\n      [\n        "Ġf",\n        "or"\n      ],\n      [\n        "m",\n        "y"\n      ],\n      [\n        "\'",\n        "s"\n      ],\n      [\n        "ou",\n        "ld"\n      ],\n      [\n        "u",\n        "n"\n      ],\n      [\n        "Ġ",\n        "B"\n      ],\n      [\n        "T",\n        "he"\n      ],\n      [\n        "it",\n        "t"\n      ],\n      [\n        "en",\n        "t"\n      ],\n      [\n        "Ġl",\n        "i"\n      ],\n      [\n        "ou",\n        "t"\n      ],\n      [\n        "he",\n        "r"\n      ],\n      [\n        "Ġha",\n        "pp"\n      ],\n      [\n        "c",\n        "h"\n      ],\n      [\n        "Ġwa",\n        "nt"\n      ],\n      [\n        "Ġs",\n        "h"\n      ],\n      [\n        "l",\n        "y"\n      ],\n      [\n        "ou",\n        "nd"\n      ],\n      [\n        "itt",\n        "le"\n      ],\n      [\n        "im",\n        "e"\n      ],\n      [\n        "a",\n        "ll"\n      ],\n      [\n        "Ġd",\n        "o"\n      ],\n      [\n        "Ġl",\n        "ittle"\n      ],\n      [\n        "e",\n        "nd"\n      ],\n      [\n        "Ġ",\n        "very"\n      ],\n      [\n        "om",\n        "e"\n      ],\n      [\n        "Ġt",\n        "ime"\n      ],\n      [\n        "s",\n        "e"\n      ],\n      [\n        "Ġw",\n        "h"\n      ],\n      [\n        "Ġthe",\n        "re"\n      ],\n      [\n        "Ġ",\n        "Ċ"\n      ],\n      [\n        "h",\n        "t"\n      ],\n      [\n        "Ġs",\n        "m"\n      ],\n      [\n        "Ġ",\n        "k"\n      ],\n      [\n        "Ġ",\n        "re"\n      ],\n      [\n        "Ġn",\n        "e"\n      ],\n      [\n        "a",\n        "l"\n      ],\n      [\n        "Ġn",\n        "ot"\n      ],\n      [\n        "Ġb",\n        "ig"\n      ],\n      [\n        "Ġhapp",\n        "y"\n      ],\n      [\n        "Ġa",\n        "s"\n      ],\n      [\n        "a",\n        "ck"\n      ],\n      [\n        "ri",\n        "end"\n      ],\n      [\n        "Ġf",\n        "riend"\n      ],\n      [\n        "Ġb",\n        "ut"\n      ],\n      [\n        "Ġsa",\n        "w"\n      ],\n      [\n        "Ġa",\n        "n"\n      ],\n      [\n        "Ġ",\n        "M"\n      ],\n      [\n        "r",\n        "a"\n      ],\n      [\n        ".",\n        "\\""\n      ],\n      [\n        "Ġ",\n        "is"\n      ],\n      [\n        "On",\n        "e"\n      ],\n      [\n        "r",\n        "y"\n      ],\n      [\n        "ĠI",\n        "t"\n      ],\n      [\n        "\'",\n        "t"\n      ],\n      [\n        "v",\n        "ed"\n      ],\n      [\n        "!",\n        "\\""\n      ],\n      [\n        "id",\n        "e"\n      ],\n      [\n        "On",\n        "ce"\n      ],\n      [\n        "Ġl",\n        "oo"\n      ],\n      [\n        "a",\n        "ke"\n      ],\n      [\n        "u",\n        "g"\n      ],\n      [\n        "Ġwe",\n        "re"\n      ],\n      [\n        "t",\n        "er"\n      ],\n      [\n        "Ġl",\n        "o"\n      ],\n      [\n        "o",\n        "re"\n      ],\n      [\n        "e",\n        "c"\n      ],\n      [\n        "ĠT",\n        "im"\n      ],\n      [\n        "Ġb",\n        "o"\n      ],\n      [\n        "Ġh",\n        "im"\n      ],\n      [\n        "ir",\n        "l"\n      ],\n      [\n        "Ġup",\n        "on"\n      ],\n      [\n        "Ġto",\n        "o"\n      ],\n      [\n        "Ġg",\n        "irl"\n      ],\n      [\n        "Ġg",\n        "o"\n      ],\n      [\n        "Ġs",\n        "e"\n      ],\n      [\n        "ar",\n        "d"\n      ],\n      [\n        "Ġ",\n        "j"\n      ],\n      [\n        "Ġ",\n        "out"\n      ],\n      [\n        "Ġwant",\n        "ed"\n      ],\n      [\n        "il",\n        "l"\n      ],\n      [\n        "Ġsm",\n        "il"\n      ],\n      [\n        "f",\n        "u"\n      ],\n      [\n        "w",\n        "ay"\n      ],\n      [\n        "he",\n        "n"\n      ],\n      [\n        "i",\n        "nd"\n      ],\n      [\n        "Ġa",\n        "t"\n      ],\n      [\n        "Ġs",\n        "p"\n      ],\n      [\n        "Ġc",\n        "ould"\n      ],\n      [\n        "u",\n        "r"\n      ],\n      [\n        "Ġthe",\n        "m"\n      ],\n      [\n        "Ġ",\n        "J"\n      ],\n      [\n        "Ġe",\n        "x"\n      ],\n      [\n        "Ġ",\n        "A"\n      ],\n      [\n        "a",\n        "in"\n      ],\n      [\n        "Ġd",\n        "id"\n      ],\n      [\n        "Ġa",\n        "re"\n      ],\n      [\n        "Ġthe",\n        "ir"\n      ],\n      [\n        "r",\n        "ound"\n      ],\n      [\n        "ar",\n        "t"\n      ],\n      [\n        "Ġw",\n        "ent"\n      ],\n      [\n        "he",\n        "d"\n      ],\n      [\n        "i",\n        "c"\n      ],\n      [\n        "Ġn",\n        "am"\n      ],\n      [\n        "l",\n        "p"\n      ],\n      [\n        "r",\n        "om"\n      ],\n      [\n        "Ġha",\n        "ve"\n      ],\n      [\n        "Ġc",\n        "an"\n      ],\n      [\n        "h",\n        "ing"\n      ],\n      [\n        "?",\n        "\\""\n      ],\n      [\n        "Ġhe",\n        "lp"\n      ],\n      [\n        "fu",\n        "l"\n      ],\n      [\n        "Ġfriend",\n        "s"\n      ],\n      [\n        "u",\n        "m"\n      ],\n      [\n        "ig",\n        "ht"\n      ],\n      [\n        "Ġa",\n        "ll"\n      ],\n      [\n        "Ġk",\n        "n"\n      ],\n      [\n        "oo",\n        "d"\n      ],\n      [\n        "on",\n        "e"\n      ],\n      [\n        "Ġb",\n        "ack"\n      ],\n      [\n        "ar",\n        "k"\n      ],\n      [\n        "Ġsmil",\n        "ed"\n      ],\n      [\n        "Ã",\n        "¢"\n      ],\n      [\n        "â",\n        "Ĥ"\n      ],\n      [\n        "âĤ",\n        "¬"\n      ],\n      [\n        "ĠB",\n        "en"\n      ],\n      [\n        "Ġc",\n        "l"\n      ],\n      [\n        "Ġnam",\n        "ed"\n      ],\n      [\n        "a",\n        "re"\n      ],\n      [\n        "Ġf",\n        "un"\n      ],\n      [\n        "Ġas",\n        "ked"\n      ],\n      [\n        "ou",\n        "g"\n      ],\n      [\n        "Ġn",\n        "o"\n      ],\n      [\n        "el",\n        "t"\n      ],\n      [\n        "Ġl",\n        "e"\n      ],\n      [\n        "Ġlo",\n        "ved"\n      ],\n      [\n        "o",\n        "p"\n      ],\n      [\n        "Ġ",\n        "One"\n      ],\n      [\n        "Y",\n        "ou"\n      ],\n      [\n        "Ġf",\n        "elt"\n      ],\n      [\n        "Ġs",\n        "c"\n      ],\n      [\n        "Ġloo",\n        "ked"\n      ],\n      [\n        "ĠTim",\n        "my"\n      ],\n      [\n        "i",\n        "ck"\n      ],\n      [\n        "Ġa",\n        "l"\n      ],\n      [\n        "Ġa",\n        "round"\n      ],\n      [\n        "s",\n        "ide"\n      ],\n      [\n        "am",\n        "e"\n      ],\n      [\n        "g",\n        "et"\n      ],\n      [\n        "om",\n        "et"\n      ],\n      [\n        "ĠT",\n        "om"\n      ],\n      [\n        "Ġf",\n        "e"\n      ],\n      [\n        "Ġbo",\n        "y"\n      ],\n      [\n        "Ġto",\n        "y"\n      ],\n      [\n        "u",\n        "re"\n      ],\n      [\n        "Ġm",\n        "an"\n      ],\n      [\n        "Ġm",\n        "e"\n      ],\n      [\n        "Ġse",\n        "e"\n      ],\n      [\n        "ir",\n        "d"\n      ],\n      [\n        "Ġr",\n        "o"\n      ],\n      [\n        "Ġli",\n        "ke"\n      ],\n      [\n        "Ġst",\n        "art"\n      ],\n      [\n        "Ġb",\n        "r"\n      ],\n      [\n        "Ġs",\n        "ome"\n      ],\n      [\n        "d",\n        "d"\n      ],\n      [\n        "Ġw",\n        "ould"\n      ],\n      [\n        "a",\n        "s"\n      ],\n      [\n        "Ġs",\n        "ay"\n      ],\n      [\n        "i",\n        "ce"\n      ],\n      [\n        "on",\n        "g"\n      ],\n      [\n        "Ġ",\n        "W"\n      ],\n      [\n        "Ġs",\n        "omet"\n      ],\n      [\n        "Ġb",\n        "ird"\n      ],\n      [\n        "Ġw",\n        "or"\n      ],\n      [\n        "get",\n        "her"\n      ],\n      [\n        "k",\n        "ing"\n      ],\n      [\n        "re",\n        "e"\n      ],\n      [\n        "Ġa",\n        "g"\n      ],\n      [\n        ",",\n        "\\""\n      ],\n      [\n        "Ġwh",\n        "at"\n      ],\n      [\n        "ad",\n        "e"\n      ],\n      [\n        "a",\n        "g"\n      ],\n      [\n        "i",\n        "e"\n      ],\n      [\n        "Ġf",\n        "a"\n      ],\n      [\n        "ing",\n        "s"\n      ],\n      [\n        "Ġstart",\n        "ed"\n      ],\n      [\n        "ow",\n        "n"\n      ],\n      [\n        "Ġto",\n        "gether"\n      ],\n      [\n        "Ġm",\n        "ake"\n      ],\n      [\n        "Ġa",\n        "way"\n      ],\n      [\n        "ar",\n        "ed"\n      ],\n      [\n        "Ġsomet",\n        "hing"\n      ],\n      [\n        "it",\n        "ed"\n      ],\n      [\n        "Ġr",\n        "an"\n      ],\n      [\n        "i",\n        "f"\n      ],\n      [\n        "ĠB",\n        "ut"\n      ],\n      [\n        "ou",\n        "d"\n      ],\n      [\n        "Ġex",\n        "c"\n      ],\n      [\n        "Ġp",\n        "ark"\n      ],\n      [\n        "Ġc",\n        "o"\n      ],\n      [\n        "Ġm",\n        "ade"\n      ],\n      [\n        "Ġsa",\n        "d"\n      ],\n      [\n        "Ġo",\n        "t"\n      ],\n      [\n        "b",\n        "le"\n      ],\n      [\n        "h",\n        "at"\n      ],\n      [\n        "e",\n        "ll"\n      ],\n      [\n        "Ġw",\n        "hen"\n      ],\n      [\n        "Ġne",\n        "w"\n      ],\n      [\n        "Ġf",\n        "rom"\n      ],\n      [\n        "pp",\n        "ed"\n      ],\n      [\n        "Ġ",\n        "You"\n      ],\n      [\n        "Ġot",\n        "her"\n      ],\n      [\n        "Ġp",\n        "r"\n      ],\n      [\n        "Ġm",\n        "u"\n      ],\n      [\n        "Ġ",\n        "L"\n      ],\n      [\n        "Ġc",\n        "ar"\n      ],\n      [\n        "Ġl",\n        "a"\n      ],\n      [\n        "Ġp",\n        "ut"\n      ],\n      [\n        "Ġag",\n        "ain"\n      ],\n      [\n        "Ġh",\n        "ome"\n      ],\n      [\n        "Ġthe",\n        "n"\n      ],\n      [\n        "p",\n        "l"\n      ],\n      [\n        "Ġf",\n        "ound"\n      ],\n      [\n        "es",\n        "t"\n      ],\n      [\n        "oug",\n        "ht"\n      ],\n      [\n        "s",\n        "s"\n      ],\n      [\n        "Ġwh",\n        "o"\n      ],\n      [\n        "q",\n        "u"\n      ],\n      [\n        "Ġwa",\n        "l"\n      ],\n      [\n        "e",\n        "p"\n      ],\n      [\n        "Ġh",\n        "ug"\n      ],\n      [\n        "Ġplay",\n        "ing"\n      ],\n      [\n        "Ġf",\n        "l"\n      ],\n      [\n        "all",\n        "y"\n      ],\n      [\n        "oo",\n        "k"\n      ],\n      [\n        "ri",\n        "ed"\n      ],\n      [\n        "Ġg",\n        "ood"\n      ],\n      [\n        "Ġg",\n        "et"\n      ],\n      [\n        "Ġd",\n        "ec"\n      ],\n      [\n        "Ġexc",\n        "ited"\n      ],\n      [\n        "Ġc",\n        "h"\n      ],\n      [\n        "a",\n        "ch"\n      ],\n      [\n        "ou",\n        "s"\n      ],\n      [\n        "Ġs",\n        "w"\n      ],\n      [\n        "Ġ",\n        "E"\n      ],\n      [\n        "r",\n        "o"\n      ],\n      [\n        "Ġg",\n        "ot"\n      ],\n      [\n        "n",\n        "y"\n      ],\n      [\n        "Ġsay",\n        "s"\n      ],\n      [\n        "Ġth",\n        "ings"\n      ],\n      [\n        "u",\n        "ck"\n      ],\n      [\n        "a",\n        "ve"\n      ],\n      [\n        "Ġc",\n        "ame"\n      ],\n      [\n        "Ġb",\n        "l"\n      ],\n      [\n        "Ġ",\n        "v"\n      ],\n      [\n        "Ġ",\n        "D"\n      ],\n      [\n        "id",\n        "ed"\n      ],\n      [\n        "Ġe",\n        "very"\n      ],\n      [\n        "Ġdec",\n        "ided"\n      ],\n      [\n        "u",\n        "st"\n      ],\n      [\n        "u",\n        "dd"\n      ],\n      [\n        "Ġli",\n        "ked"\n      ],\n      [\n        "Ġdo",\n        "g"\n      ],\n      [\n        "Ġbe",\n        "c"\n      ],\n      [\n        "Ġon",\n        "e"\n      ],\n      [\n        "Ġd",\n        "own"\n      ],\n      [\n        "u",\n        "s"\n      ],\n      [\n        "Ġth",\n        "an"\n      ],\n      [\n        "Ġyou",\n        "r"\n      ],\n      [\n        "Ġsc",\n        "ared"\n      ],\n      [\n        "Ġd",\n        "ad"\n      ],\n      [\n        "Ġc",\n        "are"\n      ],\n      [\n        "Ġf",\n        "ind"\n      ],\n      [\n        "ou",\n        "se"\n      ],\n      [\n        "Ġa",\n        "b"\n      ],\n      [\n        "Ġfe",\n        "el"\n      ],\n      [\n        "way",\n        "s"\n      ],\n      [\n        "ar",\n        "a"\n      ],\n      [\n        "Ġt",\n        "ree"\n      ],\n      [\n        "Ġ",\n        "F"\n      ],\n      [\n        "u",\n        "e"\n      ],\n      [\n        "Ġt",\n        "a"\n      ],\n      [\n        "Ġp",\n        "o"\n      ],\n      [\n        "Ġal",\n        "ways"\n      ],\n      [\n        "is",\n        "t"\n      ],\n      [\n        "a",\n        "x"\n      ],\n      [\n        "an",\n        "t"\n      ],\n      [\n        "Ġdid",\n        "n"\n      ],\n      [\n        "Ġab",\n        "out"\n      ],\n      [\n        "Ġm",\n        "y"\n      ],\n      [\n        "ou",\n        "r"\n      ],\n      [\n        "Ġth",\n        "ought"\n      ],\n      [\n        "b",\n        "b"\n      ],\n      [\n        "nd",\n        "er"\n      ],\n      [\n        "Ġ",\n        "ke"\n      ],\n      [\n        "Ġtoo",\n        "k"\n      ],\n      [\n        "Ġkn",\n        "e"\n      ],\n      [\n        "Ġg",\n        "ra"\n      ],\n      [\n        "Ġloo",\n        "k"\n      ],\n      [\n        "Ġo",\n        "ld"\n      ],\n      [\n        "Ġl",\n        "ot"\n      ],\n      [\n        "er",\n        "s"\n      ],\n      [\n        "ar",\n        "n"\n      ],\n      [\n        "it",\n        "e"\n      ],\n      [\n        "ĠM",\n        "om"\n      ],\n      [\n        "n",\n        "a"\n      ],\n      [\n        "es",\n        "s"\n      ],\n      [\n        "Ġin",\n        "t"\n      ],\n      [\n        "f",\n        "e"\n      ],\n      [\n        "is",\n        "e"\n      ],\n      [\n        "Ġout",\n        "side"\n      ],\n      [\n        "ec",\n        "i"\n      ],\n      [\n        "c",\n        "hed"\n      ],\n      [\n        "udd",\n        "en"\n      ],\n      [\n        "Ġh",\n        "o"\n      ],\n      [\n        "Ġla",\n        "ug"\n      ],\n      [\n        "Ġt",\n        "ake"\n      ],\n      [\n        "g",\n        "ed"\n      ],\n      [\n        "Ġmom",\n        "my"\n      ],\n      [\n        "Ġw",\n        "ill"\n      ],\n      [\n        "en",\n        "ed"\n      ],\n      [\n        "The",\n        "y"\n      ],\n      [\n        "Ġtoy",\n        "s"\n      ],\n      [\n        "Ġkn",\n        "ow"\n      ],\n      [\n        "a",\n        "p"\n      ],\n      [\n        "Ġb",\n        "all"\n      ],\n      [\n        "Ġm",\n        "ore"\n      ],\n      [\n        "udden",\n        "ly"\n      ],\n      [\n        "re",\n        "t"\n      ],\n      [\n        "Ġp",\n        "e"\n      ],\n      [\n        "eci",\n        "al"\n      ],\n      [\n        "Ġsp",\n        "ecial"\n      ],\n      [\n        "Ġkne",\n        "w"\n      ],\n      [\n        "Ġan",\n        "y"\n      ],\n      [\n        "re",\n        "at"\n      ],\n      [\n        "Ġle",\n        "arn"\n      ],\n      [\n        "f",\n        "ter"\n      ],\n      [\n        "Ġt",\n        "r"\n      ],\n      [\n        "i",\n        "ve"\n      ],\n      [\n        "n",\n        "na"\n      ],\n      [\n        "Ġh",\n        "ow"\n      ],\n      [\n        "m",\n        "a"\n      ],\n      [\n        "Ġc",\n        "at"\n      ],\n      [\n        "Ġs",\n        "un"\n      ],\n      [\n        "Ġt",\n        "w"\n      ],\n      [\n        "ĠS",\n        "am"\n      ],\n      [\n        "v",\n        "en"\n      ],\n      [\n        "o",\n        "b"\n      ],\n      [\n        "i",\n        "a"\n      ],\n      [\n        "a",\n        "u"\n      ],\n      [\n        "Ġsh",\n        "ow"\n      ],\n      [\n        "is",\n        "h"\n      ],\n      [\n        "o",\n        "s"\n      ],\n      [\n        "Ġmu",\n        "ch"\n      ],\n      [\n        "ĠM",\n        "ax"\n      ],\n      [\n        "Ġs",\n        "or"\n      ],\n      [\n        "Ġs",\n        "k"\n      ],\n      [\n        "Ġint",\n        "o"\n      ],\n      [\n        "Ġ",\n        "qu"\n      ],\n      [\n        "Ġ",\n        "en"\n      ],\n      [\n        "a",\n        "nd"\n      ],\n      [\n        "B",\n        "ut"\n      ],\n      [\n        "Ġ",\n        "if"\n      ],\n      [\n        "an",\n        "k"\n      ],\n      [\n        "Ġcould",\n        "n"\n      ],\n      [\n        "Ġex",\n        "pl"\n      ],\n      [\n        "Ġs",\n        "l"\n      ],\n      [\n        "Ġto",\n        "ld"\n      ],\n      [\n        "Ġha",\n        "nd"\n      ],\n      [\n        "Ġm",\n        "o"\n      ],\n      [\n        "Ġh",\n        "ouse"\n      ],\n      [\n        "Ġo",\n        "p"\n      ],\n      [\n        "ot",\n        "her"\n      ],\n      [\n        "Ġne",\n        "ver"\n      ],\n      [\n        "g",\n        "e"\n      ],\n      [\n        "Ġpr",\n        "oud"\n      ],\n      [\n        "Ġin",\n        "side"\n      ],\n      [\n        "Ġo",\n        "ver"\n      ],\n      [\n        "Ġhe",\n        "ard"\n      ],\n      [\n        "Ġg",\n        "ave"\n      ],\n      [\n        "Ġth",\n        "is"\n      ],\n      [\n        "Ġro",\n        "om"\n      ],\n      [\n        "Å",\n        "ĵ"\n      ],\n      [\n        "in",\n        "k"\n      ],\n      [\n        "Ġr",\n        "un"\n      ],\n      [\n        "Ġo",\n        "r"\n      ],\n      [\n        "t",\n        "y"\n      ],\n      [\n        "Ġp",\n        "ick"\n      ],\n      [\n        "Ġt",\n        "ried"\n      ],\n      [\n        "Ġst",\n        "r"\n      ],\n      [\n        "Ġsor",\n        "ry"\n      ],\n      [\n        "Ġplay",\n        "ed"\n      ],\n      [\n        "au",\n        "se"\n      ],\n      [\n        "at",\n        "e"\n      ],\n      [\n        "i",\n        "on"\n      ],\n      [\n        "a",\n        "ce"\n      ],\n      [\n        "u",\n        "c"\n      ],\n      [\n        "Ġj",\n        "ust"\n      ],\n      [\n        "um",\n        "p"\n      ],\n      [\n        "Ġwa",\n        "ter"\n      ],\n      [\n        "Ġwa",\n        "t"\n      ],\n      [\n        "Ġe",\n        "at"\n      ],\n      [\n        "ĠE",\n        "very"\n      ],\n      [\n        "d",\n        "y"\n      ],\n      [\n        "Ġf",\n        "o"\n      ],\n      [\n        "et",\n        "ter"\n      ],\n      [\n        "he",\n        "re"\n      ],\n      [\n        "Ġb",\n        "u"\n      ],\n      [\n        "Ġsm",\n        "all"\n      ],\n      [\n        "Ġhug",\n        "ged"\n      ],\n      [\n        "Ġne",\n        "ed"\n      ],\n      [\n        "\'",\n        "m"\n      ],\n      [\n        "oug",\n        "h"\n      ],\n      [\n        "Ġ",\n        "Ã¢"\n      ],\n      [\n        "Ġc",\n        "om"\n      ],\n      [\n        "ot",\n        "h"\n      ],\n      [\n        "ĠS",\n        "ara"\n      ],\n      [\n        "ĠJ",\n        "ack"\n      ],\n      [\n        "il",\n        "e"\n      ],\n      [\n        "Ġbec",\n        "ause"\n      ],\n      [\n        "b",\n        "y"\n      ],\n      [\n        "or",\n        "t"\n      ],\n      [\n        "Ġl",\n        "ong"\n      ],\n      [\n        "T",\n        "om"\n      ],\n      [\n        "M",\n        "om"\n      ],\n      [\n        "Ġc",\n        "all"\n      ],\n      [\n        "v",\n        "ing"\n      ],\n      [\n        "ĠJ",\n        "o"\n      ],\n      [\n        "i",\n        "z"\n      ],\n      [\n        "Ġt",\n        "ry"\n      ],\n      [\n        "Ġof",\n        "f"\n      ],\n      [\n        "Ġp",\n        "ret"\n      ],\n      [\n        "Ġbo",\n        "x"\n      ],\n      [\n        "il",\n        "d"\n      ],\n      [\n        "Ġe",\n        "ach"\n      ],\n      [\n        "Ġn",\n        "ice"\n      ],\n      [\n        "Ġman",\n        "y"\n      ],\n      [\n        "a",\n        "k"\n      ],\n      [\n        "ll",\n        "y"\n      ],\n      [\n        "Ġbe",\n        "ar"\n      ],\n      [\n        "Y",\n        "es"\n      ],\n      [\n        "el",\n        "f"\n      ],\n      [\n        "ĠW",\n        "e"\n      ],\n      [\n        "h",\n        "n"\n      ],\n      [\n        "f",\n        "t"\n      ],\n      [\n        "Ġu",\n        "nt"\n      ],\n      [\n        "e",\n        "ar"\n      ],\n      [\n        "in",\n        "e"\n      ],\n      [\n        "im",\n        "al"\n      ],\n      [\n        "ad",\n        "y"\n      ],\n      [\n        "Ġan",\n        "imal"\n      ],\n      [\n        "Ġthan",\n        "ked"\n      ],\n      [\n        "Ġd",\n        "on"\n      ],\n      [\n        "a",\n        "ut"\n      ],\n      [\n        "Ġb",\n        "est"\n      ],\n      [\n        "Ġb",\n        "etter"\n      ],\n      [\n        "Ġa",\n        "d"\n      ],\n      [\n        "Ġfl",\n        "ow"\n      ],\n      [\n        "Ġunt",\n        "il"\n      ],\n      [\n        "e",\n        "m"\n      ],\n      [\n        "g",\n        "ry"\n      ],\n      [\n        "ur",\n        "t"\n      ],\n      [\n        "T",\n        "h"\n      ],\n      [\n        "Ġ",\n        "im"\n      ],\n      [\n        "Ġlaug",\n        "hed"\n      ],\n      [\n        "Ġlearn",\n        "ed"\n      ],\n      [\n        "Ġe",\n        "nd"\n      ],\n      [\n        "v",\n        "ent"\n      ],\n      [\n        "o",\n        "y"\n      ],\n      [\n        "Ġbe",\n        "aut"\n      ],\n      [\n        "I",\n        "t"\n      ],\n      [\n        "Ġk",\n        "ind"\n      ],\n      [\n        "ĠA",\n        "nna"\n      ],\n      [\n        "b",\n        "er"\n      ],\n      [\n        "Ġe",\n        "ven"\n      ],\n      [\n        "Ġb",\n        "y"\n      ],\n      [\n        "Ġd",\n        "re"\n      ],\n      [\n        "n",\n        "ing"\n      ],\n      [\n        "ĠH",\n        "er"\n      ],\n      [\n        "ĠS",\n        "o"\n      ],\n      [\n        "Ġg",\n        "r"\n      ],\n      [\n        "p",\n        "t"\n      ],\n      [\n        "Ġ",\n        "C"\n      ],\n      [\n        "Ġg",\n        "ard"\n      ],\n      [\n        "Ġs",\n        "n"\n      ],\n      [\n        "Ġgard",\n        "en"\n      ],\n      [\n        "Ġf",\n        "i"\n      ],\n      [\n        "Ġp",\n        "u"\n      ],\n      [\n        "Ġu",\n        "s"\n      ],\n      [\n        "Ġc",\n        "le"\n      ],\n      [\n        "H",\n        "e"\n      ],\n      [\n        "ur",\n        "p"\n      ],\n      [\n        "in",\n        "y"\n      ],\n      [\n        "s",\n        "elf"\n      ],\n      [\n        "a",\n        "se"\n      ],\n      [\n        "Ġsk",\n        "y"\n      ],\n      [\n        "Ġr",\n        "a"\n      ],\n      [\n        "ĠA",\n        "nd"\n      ],\n      [\n        "Ġlo",\n        "ve"\n      ],\n      [\n        "Ġcl",\n        "os"\n      ],\n      [\n        "Ġbeaut",\n        "i"\n      ],\n      [\n        "Ġwa",\n        "y"\n      ],\n      [\n        "âĤ¬",\n        "â"\n      ],\n      [\n        "Ġbeauti",\n        "ful"\n      ],\n      [\n        "le",\n        "w"\n      ],\n      [\n        "Ġb",\n        "ra"\n      ],\n      [\n        "Ġfa",\n        "st"\n      ],\n      [\n        "Ġlot",\n        "s"\n      ],\n      [\n        "Ġcare",\n        "ful"\n      ],\n      [\n        "c",\n        "t"\n      ],\n      [\n        "Ġli",\n        "ved"\n      ],\n      [\n        "ĠH",\n        "is"\n      ],\n      [\n        "Ġst",\n        "ill"\n      ],\n      [\n        "Ġc",\n        "he"\n      ],\n      [\n        "Ġc",\n        "ome"\n      ],\n      [\n        "Ġtw",\n        "o"\n      ],\n      [\n        "Ġj",\n        "ump"\n      ],\n      [\n        "Ġg",\n        "l"\n      ],\n      [\n        "ĠM",\n        "ia"\n      ],\n      [\n        "ur",\n        "n"\n      ],\n      [\n        "v",\n        "es"\n      ],\n      [\n        "uc",\n        "y"\n      ],\n      [\n        "ĠF",\n        "rom"\n      ],\n      [\n        "Ġw",\n        "o"\n      ],\n      [\n        "Ġcall",\n        "ed"\n      ],\n      [\n        "an",\n        "e"\n      ],\n      [\n        "ec",\n        "t"\n      ],\n      [\n        "Ġh",\n        "ard"\n      ],\n      [\n        "Ġb",\n        "oth"\n      ],\n      [\n        "Ħ",\n        "¢"\n      ],\n      [\n        "âĤ¬â",\n        "Ħ¢"\n      ],\n      [\n        "w",\n        "n"\n      ],\n      [\n        "Ġa",\n        "r"\n      ],\n      [\n        "Ġl",\n        "oud"\n      ],\n      [\n        "o",\n        "ck"\n      ],\n      [\n        "Ġl",\n        "et"\n      ],\n      [\n        "Ġf",\n        "am"\n      ],\n      [\n        "T",\n        "im"\n      ],\n      [\n        "Ġc",\n        "u"\n      ],\n      [\n        "Ġst",\n        "ay"\n      ],\n      [\n        "Ġl",\n        "ist"\n      ],\n      [\n        "Ġre",\n        "m"\n      ],\n      [\n        "Ġco",\n        "l"\n      ],\n      [\n        "ig",\n        "h"\n      ],\n      [\n        "L",\n        "et"\n      ],\n      [\n        "Ġu",\n        "nder"\n      ],\n      [\n        "Ġh",\n        "urt"\n      ],\n      [\n        "he",\n        "s"\n      ],\n      [\n        "W",\n        "hen"\n      ],\n      [\n        "a",\n        "st"\n      ],\n      [\n        "oo",\n        "l"\n      ],\n      [\n        "e",\n        "et"\n      ],\n      [\n        "Ġbra",\n        "ve"\n      ],\n      [\n        "Ġit",\n        "s"\n      ],\n      [\n        "Ġd",\n        "an"\n      ],\n      [\n        "Ġsa",\n        "fe"\n      ],\n      [\n        "ĠThe",\n        "n"\n      ],\n      [\n        "i",\n        "ed"\n      ],\n      [\n        "Ġim",\n        "p"\n      ],\n      [\n        "i",\n        "es"\n      ],\n      [\n        "Ġwal",\n        "ked"\n      ],\n      [\n        "Ġanimal",\n        "s"\n      ],\n      [\n        "W",\n        "hat"\n      ],\n      [\n        "Ġf",\n        "lew"\n      ],\n      [\n        "Ġf",\n        "in"\n      ],\n      [\n        "urp",\n        "r"\n      ],\n      [\n        "Ġ",\n        "P"\n      ],\n      [\n        "dd",\n        "ed"\n      ],\n      [\n        "Ġgo",\n        "ing"\n      ],\n      [\n        "Ġs",\n        "urpr"\n      ],\n      [\n        "k",\n        "ay"\n      ],\n      [\n        "is",\n        "hed"\n      ],\n      [\n        "op",\n        "le"\n      ],\n      [\n        "Ġb",\n        "ad"\n      ],\n      [\n        "Ġf",\n        "ly"\n      ],\n      [\n        "Ġst",\n        "or"\n      ],\n      [\n        "ĠW",\n        "hen"\n      ],\n      [\n        "Ġso",\n        "on"\n      ],\n      [\n        "Ġd",\n        "ra"\n      ],\n      [\n        "Ġp",\n        "ic"\n      ],\n      [\n        "i",\n        "p"\n      ],\n      [\n        "Ġexpl",\n        "ore"\n      ],\n      [\n        "f",\n        "ore"\n      ],\n      [\n        "um",\n        "my"\n      ],\n      [\n        "Ġdo",\n        "ll"\n      ],\n      [\n        "Ġst",\n        "o"\n      ],\n      [\n        "Ġr",\n        "ed"\n      ],\n      [\n        "Ġsh",\n        "ould"\n      ],\n      [\n        "l",\n        "ed"\n      ],\n      [\n        "Ġc",\n        "on"\n      ],\n      [\n        "T",\n        "hat"\n      ],\n      [\n        "Ġke",\n        "pt"\n      ],\n      [\n        "Ġpe",\n        "ople"\n      ],\n      [\n        "Ġfam",\n        "ily"\n      ],\n      [\n        "b",\n        "e"\n      ],\n      [\n        "N",\n        "o"\n      ],\n      [\n        "x",\n        "t"\n      ],\n      [\n        "is",\n        "ed"\n      ],\n      [\n        "ag",\n        "e"\n      ],\n      [\n        "il",\n        "ly"\n      ],\n      [\n        "Ġg",\n        "round"\n      ],\n      [\n        "Ġn",\n        "ow"\n      ],\n      [\n        "Ġb",\n        "ook"\n      ],\n      [\n        "Ġc",\n        "ry"\n      ],\n      [\n        "Ġsh",\n        "iny"\n      ],\n      [\n        "Ġan",\n        "gry"\n      ],\n      [\n        "Ġno",\n        "dded"\n      ],\n      [\n        "Ġbe",\n        "fore"\n      ],\n      [\n        "Ġw",\n        "ind"\n      ],\n      [\n        "Ġf",\n        "ore"\n      ],\n      [\n        "S",\n        "o"\n      ],\n      [\n        "ic",\n        "ed"\n      ],\n      [\n        "Ġe",\n        "y"\n      ],\n      [\n        "Ġke",\n        "ep"\n      ],\n      [\n        "Ġdo",\n        "or"\n      ],\n      [\n        "ar",\n        "y"\n      ],\n      [\n        "Ġwh",\n        "ile"\n      ],\n      [\n        "ĠL",\n        "ucy"\n      ],\n      [\n        "Ġad",\n        "vent"\n      ],\n      [\n        "Ġsh",\n        "are"\n      ],\n      [\n        "Ġre",\n        "pl"\n      ],\n      [\n        "Ġcle",\n        "an"\n      ],\n      [\n        "Ġop",\n        "ened"\n      ],\n      [\n        "Ġfeel",\n        "ing"\n      ],\n      [\n        "Ġwal",\n        "king"\n      ],\n      [\n        "ĠB",\n        "ob"\n      ],\n      [\n        "Ġf",\n        "ar"\n      ],\n      [\n        "t",\n        "hing"\n      ],\n      [\n        "Ġro",\n        "ck"\n      ],\n      [\n        "Ġ",\n        "G"\n      ],\n      [\n        "Ġpret",\n        "ty"\n      ],\n      [\n        "h",\n        "y"\n      ],\n      [\n        "Ġwa",\n        "r"\n      ],\n      [\n        "t",\n        "ure"\n      ],\n      [\n        "Ġf",\n        "r"\n      ],\n      [\n        "f",\n        "f"\n      ],\n      [\n        "t",\n        "ing"\n      ],\n      [\n        "Ġpick",\n        "ed"\n      ],\n      [\n        "Ġ",\n        "ide"\n      ],\n      [\n        "g",\n        "er"\n      ],\n      [\n        "Ġno",\n        "ise"\n      ],\n      [\n        "Ġsto",\n        "pped"\n      ],\n      [\n        "Ġrepl",\n        "ied"\n      ],\n      [\n        "Ġre",\n        "al"\n      ],\n      [\n        "Ġcol",\n        "or"\n      ],\n      [\n        "j",\n        "oy"\n      ],\n      [\n        "Ġnot",\n        "iced"\n      ],\n      [\n        "Ġw",\n        "r"\n      ],\n      [\n        "Ġth",\n        "r"\n      ],\n      [\n        "Ġne",\n        "xt"\n      ],\n      [\n        "Ġb",\n        "ed"\n      ],\n      [\n        "Ġtr",\n        "uck"\n      ],\n      [\n        "t",\n        "ed"\n      ],\n      [\n        "ĠS",\n        "uddenly"\n      ],\n      [\n        "Ġj",\n        "o"\n      ],\n      [\n        "Ġu",\n        "n"\n      ],\n      [\n        "Ġide",\n        "a"\n      ],\n      [\n        "im",\n        "b"\n      ],\n      [\n        "Ġt",\n        "re"\n      ],\n      [\n        "Ġv",\n        "o"\n      ],\n      [\n        "i",\n        "ous"\n      ],\n      [\n        "Th",\n        "ank"\n      ],\n      [\n        "Ġt",\n        "e"\n      ],\n      [\n        "Ġcl",\n        "imb"\n      ],\n      [\n        "Ġpic",\n        "ture"\n      ],\n      [\n        "Ġsmil",\n        "e"\n      ],\n      [\n        "Ġadvent",\n        "ure"\n      ],\n      [\n        "em",\n        "ber"\n      ],\n      [\n        "Ġey",\n        "es"\n      ],\n      [\n        "Ġloo",\n        "king"\n      ],\n      [\n        "Ġqu",\n        "ick"\n      ],\n      [\n        "Ġd",\n        "is"\n      ],\n      [\n        "Ġg",\n        "reat"\n      ],\n      [\n        "Ġen",\n        "joy"\n      ],\n      [\n        "S",\n        "uddenly"\n      ],\n      [\n        "bb",\n        "it"\n      ],\n      [\n        "Ġd",\n        "el"\n      ],\n      [\n        "u",\n        "ch"\n      ],\n      [\n        "ug",\n        "ht"\n      ],\n      [\n        "Ġwa",\n        "it"\n      ],\n      [\n        "le",\n        "s"\n      ],\n      [\n        "Ġto",\n        "w"\n      ],\n      [\n        "Ġt",\n        "urn"\n      ],\n      [\n        "Ġm",\n        "or"\n      ],\n      [\n        "B",\n        "en"\n      ],\n      [\n        "ĠD",\n        "ad"\n      ],\n      [\n        "Ġwo",\n        "nder"\n      ],\n      [\n        "Ġrem",\n        "ember"\n      ],\n      [\n        "Ġhand",\n        "s"\n      ],\n      [\n        "a",\n        "ble"\n      ],\n      [\n        "Ġm",\n        "a"\n      ],\n      [\n        "ma",\n        "z"\n      ],\n      [\n        "Ġa",\n        "maz"\n      ],\n      [\n        "Ġf",\n        "ish"\n      ],\n      [\n        "Ġwal",\n        "k"\n      ],\n      [\n        "Ġwat",\n        "ch"\n      ],\n      [\n        "o",\n        "on"\n      ],\n      [\n        "Ġ",\n        "R"\n      ],\n      [\n        "Ġy",\n        "ear"\n      ],\n      [\n        "Ġra",\n        "bbit"\n      ],\n      [\n        "Ġa",\n        "fter"\n      ],\n      [\n        "ir",\n        "st"\n      ],\n      [\n        "Ġe",\n        "ver"\n      ],\n      [\n        "is",\n        "s"\n      ],\n      [\n        "Ġg",\n        "ive"\n      ],\n      [\n        "g",\n        "an"\n      ],\n      [\n        "s",\n        "o"\n      ],\n      [\n        "Ġa",\n        "pp"\n      ],\n      [\n        "Ġbe",\n        "ing"\n      ],\n      [\n        "a",\n        "c"\n      ],\n      [\n        "an",\n        "ge"\n      ],\n      [\n        "Ġr",\n        "ight"\n      ],\n      [\n        "Ġd",\n        "if"\n      ],\n      [\n        "Ġf",\n        "ood"\n      ],\n      [\n        "Ġsh",\n        "out"\n      ],\n      [\n        "Ġquick",\n        "ly"\n      ],\n      [\n        "ĠJo",\n        "hn"\n      ],\n      [\n        "ay",\n        "be"\n      ],\n      [\n        "Ġal",\n        "so"\n      ],\n      [\n        "Ġvo",\n        "ice"\n      ],\n      [\n        "Ġbl",\n        "ue"\n      ],\n      [\n        "Ġwor",\n        "k"\n      ],\n      [\n        "Ġ",\n        "N"\n      ],\n      [\n        "Ġp",\n        "ain"\n      ],\n      [\n        "J",\n        "o"\n      ],\n      [\n        "k",\n        "es"\n      ],\n      [\n        "Ġimp",\n        "ort"\n      ],\n      [\n        "Ġclos",\n        "er"\n      ],\n      [\n        "Ġa",\n        "c"\n      ],\n      [\n        "is",\n        "y"\n      ],\n      [\n        "Ġy",\n        "ell"\n      ],\n      [\n        "Ġlist",\n        "en"\n      ],\n      [\n        "ll",\n        "ow"\n      ],\n      [\n        "Ġfore",\n        "st"\n      ],\n      [\n        "Ġbe",\n        "en"\n      ],\n      [\n        "Ġbr",\n        "o"\n      ],\n      [\n        "Ġg",\n        "re"\n      ],\n      [\n        "Ġth",\n        "ink"\n      ],\n      [\n        "Ġne",\n        "ar"\n      ],\n      [\n        "Ġstr",\n        "ong"\n      ],\n      [\n        "S",\n        "he"\n      ],\n      [\n        "Ġimport",\n        "ant"\n      ],\n      [\n        "Ġr",\n        "ain"\n      ],\n      [\n        "Ġt",\n        "al"\n      ],\n      [\n        "re",\n        "am"\n      ],\n      [\n        "le",\n        "ase"\n      ],\n      [\n        "Ġflow",\n        "ers"\n      ],\n      [\n        "n",\n        "g"\n      ],\n      [\n        "Ġcu",\n        "ri"\n      ],\n      [\n        "Ġha",\n        "s"\n      ],\n      [\n        "iz",\n        "ed"\n      ],\n      [\n        "Ġs",\n        "qu"\n      ],\n      [\n        "ĠS",\n        "ue"\n      ],\n      [\n        "Ġd",\n        "e"\n      ],\n      [\n        "Ġu",\n        "se"\n      ],\n      [\n        "d",\n        "ay"\n      ],\n      [\n        "Ġo",\n        "kay"\n      ],\n      [\n        "Ġth",\n        "ree"\n      ],\n      [\n        "bb",\n        "ed"\n      ],\n      [\n        "Ġhe",\n        "ad"\n      ],\n      [\n        "Ġbec",\n        "ame"\n      ],\n      [\n        "by",\n        "e"\n      ],\n      [\n        "Ġs",\n        "ure"\n      ],\n      [\n        "H",\n        "er"\n      ],\n      [\n        "Ġshow",\n        "ed"\n      ],\n      [\n        "Ġst",\n        "ick"\n      ],\n      [\n        "Ġpl",\n        "ace"\n      ],\n      [\n        "il",\n        "a"\n      ],\n      [\n        "Ġo",\n        "wn"\n      ],\n      [\n        "Ġfa",\n        "ce"\n      ],\n      [\n        "Ġwat",\n        "ched"\n      ],\n      [\n        "Ġevery",\n        "one"\n      ],\n      [\n        "Ġcuri",\n        "ous"\n      ],\n      [\n        "o",\n        "se"\n      ],\n      [\n        "Ġw",\n        "here"\n      ],\n      [\n        "Ġbr",\n        "ight"\n      ],\n      [\n        "Ġfo",\n        "llow"\n      ],\n      [\n        "Ġt",\n        "ra"\n      ],\n      [\n        "Ġh",\n        "igh"\n      ],\n      [\n        "a",\n        "ught"\n      ],\n      [\n        "Ġsl",\n        "ide"\n      ],\n      [\n        "L",\n        "ook"\n      ],\n      [\n        "Ġs",\n        "ound"\n      ],\n      [\n        "er",\n        "ed"\n      ],\n      [\n        "Ġm",\n        "ag"\n      ],\n      [\n        "ĠS",\n        "p"\n      ],\n      [\n        "W",\n        "e"\n      ],\n      [\n        "fe",\n        "re"\n      ],\n      [\n        "Ġb",\n        "us"\n      ],\n      [\n        "Ġm",\n        "um"\n      ],\n      [\n        "Ġhe",\n        "re"\n      ],\n      [\n        "Ġbe",\n        "l"\n      ],\n      [\n        "Ġc",\n        "r"\n      ],\n      [\n        "Ġas",\n        "k"\n      ],\n      [\n        "Ġdif",\n        "fere"\n      ],\n      [\n        "o",\n        "lly"\n      ],\n      [\n        "Ġa",\n        "m"\n      ],\n      [\n        "Ġf",\n        "irst"\n      ],\n      [\n        "Ġbl",\n        "ock"\n      ],\n      [\n        "ĠJ",\n        "ane"\n      ],\n      [\n        "Ġme",\n        "an"\n      ],\n      [\n        "Ġch",\n        "ild"\n      ],\n      [\n        "Ġstor",\n        "y"\n      ],\n      [\n        "Ġdre",\n        "ss"\n      ],\n      [\n        "le",\n        "ep"\n      ],\n      [\n        "C",\n        "an"\n      ],\n      [\n        "as",\n        "s"\n      ],\n      [\n        "Ġgood",\n        "bye"\n      ],\n      [\n        "ir",\n        "ed"\n      ],\n      [\n        "S",\n        "ara"\n      ],\n      [\n        "Ġwar",\n        "m"\n      ],\n      [\n        "a",\n        "isy"\n      ],\n      [\n        "Ġb",\n        "it"\n      ],\n      [\n        "Ġcon",\n        "t"\n      ],\n      [\n        "ĠMom",\n        "my"\n      ],\n      [\n        "it",\n        "c"\n      ],\n      [\n        "Ġp",\n        "er"\n      ],\n      [\n        "m",\n        "ore"\n      ],\n      [\n        "Ġany",\n        "more"\n      ],\n      [\n        "Ġst",\n        "ore"\n      ],\n      [\n        "Ġbo",\n        "at"\n      ],\n      [\n        "Ġn",\n        "ight"\n      ],\n      [\n        "Ġfor",\n        "g"\n      ],\n      [\n        "Ġh",\n        "op"\n      ],\n      [\n        "Ġbe",\n        "h"\n      ],\n      [\n        "a",\n        "ched"\n      ],\n      [\n        "c",\n        "o"\n      ],\n      [\n        "Ġdiffere",\n        "nt"\n      ],\n      [\n        "Ġbut",\n        "ter"\n      ],\n      [\n        "ĠA",\n        "s"\n      ],\n      [\n        "Ġfi",\n        "re"\n      ],\n      [\n        "Ġsw",\n        "eet"\n      ],\n      [\n        "u",\n        "ll"\n      ],\n      [\n        "Ġg",\n        "ame"\n      ],\n      [\n        "Ġ",\n        "our"\n      ],\n      [\n        "ar",\n        "s"\n      ],\n      [\n        "u",\n        "ff"\n      ],\n      [\n        "Ġop",\n        "en"\n      ],\n      [\n        "nd",\n        "ma"\n      ],\n      [\n        "pp",\n        "y"\n      ],\n      [\n        "Ġdra",\n        "w"\n      ],\n      [\n        "im",\n        "es"\n      ],\n      [\n        "Ġwor",\n        "ld"\n      ],\n      [\n        "Ġt",\n        "ell"\n      ],\n      [\n        "Ġhelp",\n        "ed"\n      ],\n      [\n        "ou",\n        "nt"\n      ],\n      [\n        "Ġp",\n        "at"\n      ],\n      [\n        "Ġbe",\n        "gan"\n      ],\n      [\n        "Ġfa",\n        "v"\n      ],\n      [\n        "ĠT",\n        "h"\n      ],\n      [\n        "Ġon",\n        "ly"\n      ],\n      [\n        "Ġm",\n        "on"\n      ],\n      [\n        "Ġthr",\n        "ough"\n      ],\n      [\n        "Ġjump",\n        "ed"\n      ],\n      [\n        "n",\n        "er"\n      ],\n      [\n        "Ġsa",\n        "nd"\n      ],\n      [\n        "Ġre",\n        "st"\n      ],\n      [\n        "Ġle",\n        "a"\n      ],\n      [\n        "el",\n        "y"\n      ],\n      [\n        "are",\n        "nt"\n      ],\n      [\n        "Ġre",\n        "ally"\n      ],\n      [\n        "Ġc",\n        "ake"\n      ],\n      [\n        "Ġbr",\n        "other"\n      ],\n      [\n        "Ġpu",\n        "ll"\n      ],\n      [\n        "Ġ",\n        "z"\n      ],\n      [\n        "Ġnam",\n        "e"\n      ],\n      [\n        "ĠL",\n        "et"\n      ],\n      [\n        "ĠSp",\n        "ot"\n      ],\n      [\n        "z",\n        "y"\n      ],\n      [\n        "Ġl",\n        "ady"\n      ],\n      [\n        "Ġfe",\n        "ll"\n      ],\n      [\n        "Ġhapp",\n        "ened"\n      ],\n      [\n        "Ġgra",\n        "bbed"\n      ],\n      [\n        "Ġr",\n        "u"\n      ],\n      [\n        "\'",\n        "re"\n      ],\n      [\n        "A",\n        "fter"\n      ],\n      [\n        "Ġshout",\n        "ed"\n      ],\n      [\n        "Ġc",\n        "ook"\n      ],\n      [\n        "Ġk",\n        "itc"\n      ],\n      [\n        "Ġkitc",\n        "hen"\n      ],\n      [\n        "un",\n        "ny"\n      ],\n      [\n        "Ġre",\n        "ady"\n      ],\n      [\n        "Ġp",\n        "arent"\n      ],\n      [\n        "Jo",\n        "hn"\n      ],\n      [\n        "Ġso",\n        "ft"\n      ],\n      [\n        "at",\n        "er"\n      ],\n      [\n        "Ġdo",\n        "es"\n      ],\n      [\n        "Ġflow",\n        "er"\n      ],\n      [\n        "Ġfl",\n        "o"\n      ],\n      [\n        "Ġgl",\n        "ad"\n      ],\n      [\n        "Ġth",\n        "ing"\n      ],\n      [\n        "Ġgra",\n        "ss"\n      ],\n      [\n        "st",\n        "er"\n      ],\n      [\n        "ĠSara",\n        "h"\n      ],\n      [\n        "Ġd",\n        "in"\n      ],\n      [\n        "Ġbird",\n        "s"\n      ],\n      [\n        "Mom",\n        "my"\n      ],\n      [\n        "ĠJo",\n        "e"\n      ],\n      [\n        "A",\n        "s"\n      ],\n      [\n        "Ġre",\n        "ached"\n      ],\n      [\n        "Ġha",\n        "ving"\n      ],\n      [\n        "Ġd",\n        "r"\n      ],\n      [\n        "Ġf",\n        "ro"\n      ],\n      [\n        "Ġho",\n        "pped"\n      ],\n      [\n        "in",\n        "s"\n      ],\n      [\n        "Ġwor",\n        "ry"\n      ],\n      [\n        "Ġwh",\n        "y"\n      ],\n      [\n        "Ġmag",\n        "ic"\n      ],\n      [\n        "Ġt",\n        "ired"\n      ],\n      [\n        "t",\n        "h"\n      ],\n      [\n        "A",\n        "nna"\n      ],\n      [\n        "fu",\n        "lly"\n      ],\n      [\n        "Ġse",\n        "es"\n      ],\n      [\n        "Ġta",\n        "st"\n      ],\n      [\n        "Ġpo",\n        "nd"\n      ],\n      [\n        "Ġcom",\n        "ing"\n      ],\n      [\n        "Ġst",\n        "op"\n      ],\n      [\n        "Ġreal",\n        "ized"\n      ],\n      [\n        "Ġf",\n        "ill"\n      ],\n      [\n        "ĠD",\n        "aisy"\n      ],\n      [\n        "at",\n        "her"\n      ],\n      [\n        "i",\n        "ent"\n      ],\n      [\n        "Ġd",\n        "one"\n      ],\n      [\n        "Ġse",\n        "en"\n      ],\n      [\n        "Ġyear",\n        "s"\n      ],\n      [\n        "w",\n        "here"\n      ],\n      [\n        "Ġk",\n        "id"\n      ],\n      [\n        "Ġbutter",\n        "f"\n      ],\n      [\n        "Ġl",\n        "ess"\n      ],\n      [\n        "Ġm",\n        "ouse"\n      ],\n      [\n        "u",\n        "l"\n      ],\n      [\n        "Ġs",\n        "ing"\n      ],\n      [\n        "Ġhim",\n        "self"\n      ],\n      [\n        "Ġwa",\n        "ved"\n      ],\n      [\n        "Ġblock",\n        "s"\n      ],\n      [\n        "Ġsm",\n        "ell"\n      ],\n      [\n        "Ġpr",\n        "in"\n      ],\n      [\n        "Ġw",\n        "ood"\n      ],\n      [\n        "\\"",\n        "."\n      ],\n      [\n        "Ġbeh",\n        "ind"\n      ],\n      [\n        "D",\n        "on"\n      ],\n      [\n        "Ġy",\n        "ummy"\n      ],\n      [\n        "u",\n        "ed"\n      ],\n      [\n        "Ġparent",\n        "s"\n      ],\n      [\n        "Ġsurpr",\n        "ise"\n      ],\n      [\n        "Ġb",\n        "re"\n      ],\n      [\n        "Ġc",\n        "ut"\n      ],\n      [\n        "a",\n        "ir"\n      ],\n      [\n        "Ġfun",\n        "ny"\n      ],\n      [\n        "Tim",\n        "my"\n      ],\n      [\n        "Ġunder",\n        "st"\n      ],\n      [\n        "m",\n        "er"\n      ],\n      [\n        "Ġlook",\n        "s"\n      ],\n      [\n        "ĠEvery",\n        "one"\n      ],\n      [\n        "W",\n        "ow"\n      ],\n      [\n        "ard",\n        "s"\n      ],\n      [\n        "Ġfav",\n        "or"\n      ],\n      [\n        "Ġat",\n        "e"\n      ],\n      [\n        "Ġc",\n        "reat"\n      ],\n      [\n        "ig",\n        "n"\n      ],\n      [\n        "Ġwor",\n        "ked"\n      ],\n      [\n        "Ġrun",\n        "ning"\n      ],\n      [\n        "ĠT",\n        "hat"\n      ],\n      [\n        "Ġfavor",\n        "ite"\n      ],\n      [\n        "k",\n        "en"\n      ],\n      [\n        "ĠD",\n        "o"\n      ],\n      [\n        "Ġyell",\n        "ow"\n      ],\n      [\n        "Ġs",\n        "et"\n      ],\n      [\n        "Ġfi",\n        "x"\n      ],\n      [\n        "Ġb",\n        "unny"\n      ],\n      [\n        "Ġh",\n        "at"\n      ],\n      [\n        "Ġgre",\n        "en"\n      ],\n      [\n        "Ġp",\n        "art"\n      ],\n      [\n        "Ġ",\n        "-"\n      ],\n      [\n        "Ġp",\n        "us"\n      ],\n      [\n        "Ġp",\n        "ap"\n      ],\n      [\n        "Ġm",\n        "ess"\n      ],\n      [\n        "Ġfro",\n        "g"\n      ],\n      [\n        "Ġma",\n        "king"\n      ],\n      [\n        "J",\n        "ack"\n      ],\n      [\n        "Ġl",\n        "ight"\n      ],\n      [\n        "ĠC",\n        "an"\n      ],\n      [\n        "Ġdel",\n        "ic"\n      ],\n      [\n        "Ġper",\n        "f"\n      ],\n      [\n        "Ġhapp",\n        "ily"\n      ],\n      [\n        "Ġb",\n        "a"\n      ],\n      [\n        "Ġp",\n        "rom"\n      ],\n      [\n        "Ġd",\n        "ark"\n      ],\n      [\n        "Ġsw",\n        "im"\n      ],\n      [\n        "Ġcat",\n        "ch"\n      ],\n      [\n        "i",\n        "ke"\n      ],\n      [\n        "Ġb",\n        "ag"\n      ],\n      [\n        "Ġan",\n        "other"\n      ],\n      [\n        "Ġt",\n        "ri"\n      ],\n      [\n        "Ġb",\n        "at"\n      ],\n      [\n        "Ġus",\n        "ed"\n      ],\n      [\n        "Ġd",\n        "ro"\n      ],\n      [\n        "Ġtre",\n        "es"\n      ],\n      [\n        "Ġp",\n        "re"\n      ],\n      [\n        "Ġst",\n        "e"\n      ],\n      [\n        "Ġc",\n        "ried"\n      ],\n      [\n        "Ġremember",\n        "ed"\n      ],\n      [\n        "ed",\n        "dy"\n      ],\n      [\n        "Ġto",\n        "p"\n      ],\n      [\n        "Ġco",\n        "ld"\n      ],\n      [\n        "ĠS",\n        "ally"\n      ],\n      [\n        "Ġenjoy",\n        "ed"\n      ],\n      [\n        "Ġho",\n        "le"\n      ],\n      [\n        "Ġmo",\n        "ve"\n      ],\n      [\n        "Ġs",\n        "leep"\n      ],\n      [\n        "Ġsl",\n        "ow"\n      ],\n      [\n        "ak",\n        "es"\n      ],\n      [\n        "and",\n        "y"\n      ],\n      [\n        "H",\n        "ell"\n      ],\n      [\n        "Hell",\n        "o"\n      ],\n      [\n        "Ġto",\n        "uch"\n      ],\n      [\n        "S",\n        "am"\n      ],\n      [\n        "Ġb",\n        "ug"\n      ],\n      [\n        "Ġdo",\n        "ing"\n      ],\n      [\n        "re",\n        "l"\n      ],\n      [\n        "b",\n        "o"\n      ],\n      [\n        "Ġw",\n        "ished"\n      ],\n      [\n        "ĠA",\n        "fter"\n      ],\n      [\n        "s",\n        "h"\n      ],\n      [\n        "Ġcare",\n        "fully"\n      ],\n      [\n        "Ġd",\n        "ri"\n      ],\n      [\n        "i",\n        "ver"\n      ],\n      [\n        "one",\n        "y"\n      ],\n      [\n        "Ġcont",\n        "in"\n      ],\n      [\n        "Ġbutterf",\n        "ly"\n      ],\n      [\n        "Ġs",\n        "at"\n      ],\n      [\n        "Ġf",\n        "ull"\n      ],\n      [\n        "at",\n        "ion"\n      ],\n      [\n        "Ġbu",\n        "y"\n      ],\n      [\n        "Ġbu",\n        "ild"\n      ],\n      [\n        "re",\n        "ed"\n      ],\n      [\n        "Ġle",\n        "ft"\n      ],\n      [\n        "Ġsn",\n        "ow"\n      ],\n      [\n        "Ġkid",\n        "s"\n      ],\n      [\n        "Ġwe",\n        "ar"\n      ],\n      [\n        "Ġsp",\n        "ot"\n      ],\n      [\n        "Ġar",\n        "ri"\n      ],\n      [\n        "Ġany",\n        "thing"\n      ],\n      [\n        "Ġpret",\n        "end"\n      ],\n      [\n        "\'",\n        "ll"\n      ],\n      [\n        "Ġha",\n        "ir"\n      ],\n      [\n        "Ġre",\n        "ad"\n      ],\n      [\n        "Ġt",\n        "all"\n      ],\n      [\n        "Ġd",\n        "uck"\n      ],\n      [\n        "on",\n        "s"\n      ],\n      [\n        "Ġh",\n        "id"\n      ],\n      [\n        "Ġwr",\n        "ong"\n      ],\n      [\n        "re",\n        "n"\n      ],\n      [\n        "Ġsh",\n        "o"\n      ],\n      [\n        "Ġlo",\n        "st"\n      ],\n      [\n        "Ġc",\n        "ool"\n      ],\n      [\n        "Ġher",\n        "self"\n      ],\n      [\n        "Ġsw",\n        "ing"\n      ],\n      [\n        "Ġb",\n        "ow"\n      ],\n      [\n        "ĠI",\n        "n"\n      ],\n      [\n        "ie",\n        "ld"\n      ],\n      [\n        "Ġdre",\n        "am"\n      ],\n      [\n        "Ġfin",\n        "ished"\n      ],\n      [\n        "Ġless",\n        "on"\n      ],\n      [\n        "Ġneed",\n        "ed"\n      ],\n      [\n        "Ġsurpr",\n        "ised"\n      ],\n      [\n        "Ġh",\n        "ill"\n      ],\n      [\n        "Ġfriend",\n        "ly"\n      ],\n      [\n        "Ġwind",\n        "ow"\n      ],\n      [\n        "Ġc",\n        "ra"\n      ],\n      [\n        "Ġhe",\n        "ld"\n      ],\n      [\n        "as",\n        "h"\n      ],\n      [\n        "Ġ",\n        "K"\n      ],\n      [\n        "ĠBen",\n        "ny"\n      ],\n      [\n        "Ġfa",\n        "ir"\n      ],\n      [\n        "Ġlaug",\n        "h"\n      ],\n      [\n        "Ġarri",\n        "ved"\n      ],\n      [\n        "ro",\n        "om"\n      ],\n      [\n        "Ġcontin",\n        "ued"\n      ],\n      [\n        "Ġwonder",\n        "ful"\n      ],\n      [\n        "uff",\n        "y"\n      ],\n      [\n        "Ġpap",\n        "er"\n      ],\n      [\n        "Ġre",\n        "l"\n      ],\n      [\n        "Ġho",\n        "ld"\n      ],\n      [\n        "a",\n        "v"\n      ],\n      [\n        "Ġa",\n        "ir"\n      ],\n      [\n        "in",\n        "ally"\n      ],\n      [\n        "Ġmom",\n        "ent"\n      ],\n      [\n        "Ġfollow",\n        "ed"\n      ],\n      [\n        "Ġflo",\n        "or"\n      ],\n      [\n        "ĠE",\n        "m"\n      ],\n      [\n        "Ġta",\n        "ble"\n      ],\n      [\n        "ast",\n        "le"\n      ],\n      [\n        "e",\n        "y"\n      ],\n      [\n        "Ġp",\n        "lease"\n      ],\n      [\n        "Ġthan",\n        "k"\n      ],\n      [\n        "c",\n        "ess"\n      ],\n      [\n        "as",\n        "ure"\n      ],\n      [\n        "Ġchild",\n        "ren"\n      ],\n      [\n        "Ġg",\n        "i"\n      ],\n      [\n        "en",\n        "s"\n      ],\n      [\n        "Ġcar",\n        "s"\n      ],\n      [\n        "Ġpo",\n        "in"\n      ],\n      [\n        "H",\n        "is"\n      ],\n      [\n        "Ġfill",\n        "ed"\n      ],\n      [\n        "Ġh",\n        "un"\n      ],\n      [\n        "in",\n        "ed"\n      ],\n      [\n        "Ġcl",\n        "ose"\n      ],\n      [\n        "Ġm",\n        "other"\n      ],\n      [\n        "Ġm",\n        "at"\n      ],\n      [\n        "ĠL",\n        "ila"\n      ],\n      [\n        "Ġa",\n        "ble"\n      ],\n      [\n        "as",\n        "k"\n      ],\n      [\n        "ĠN",\n        "ow"\n      ],\n      [\n        "Ġg",\n        "ent"\n      ],\n      [\n        "Ġen",\n        "ough"\n      ],\n      [\n        "ro",\n        "ss"\n      ],\n      [\n        "Ġmor",\n        "ning"\n      ],\n      [\n        "Ġag",\n        "reed"\n      ],\n      [\n        "Ġ",\n        "ice"\n      ],\n      [\n        "Ġt",\n        "ight"\n      ],\n      [\n        "Ġtry",\n        "ing"\n      ],\n      [\n        "Ġdelic",\n        "ious"\n      ],\n      [\n        "Ġsome",\n        "one"\n      ],\n      [\n        "Th",\n        "is"\n      ],\n      [\n        "Ġli",\n        "on"\n      ],\n      [\n        "Ġforg",\n        "ot"\n      ],\n      [\n        "Ġdin",\n        "ner"\n      ],\n      [\n        "s",\n        "p"\n      ],\n      [\n        "Ġget",\n        "ting"\n      ],\n      [\n        "ĠJ",\n        "en"\n      ],\n      [\n        "am",\n        "a"\n      ],\n      [\n        "d",\n        "s"\n      ],\n      [\n        "Ġy",\n        "es"\n      ],\n      [\n        "Ġhe",\n        "ar"\n      ],\n      [\n        "ĠThe",\n        "re"\n      ],\n      [\n        "Ġv",\n        "is"\n      ],\n      [\n        "Ġar",\n        "m"\n      ],\n      [\n        "Ġe",\n        "l"\n      ],\n      [\n        "Ġd",\n        "ir"\n      ],\n      [\n        "The",\n        "n"\n      ],\n      [\n        "F",\n        "rom"\n      ],\n      [\n        "g",\n        "on"\n      ],\n      [\n        "Ġli",\n        "kes"\n      ],\n      [\n        "Ġfo",\n        "x"\n      ],\n      [\n        "Ġother",\n        "s"\n      ],\n      [\n        "Ġg",\n        "one"\n      ],\n      [\n        "Ġunderst",\n        "and"\n      ],\n      [\n        "e",\n        "e"\n      ],\n      [\n        "Ġwa",\n        "ll"\n      ],\n      [\n        "Ġcolor",\n        "s"\n      ],\n      [\n        "i",\n        "ch"\n      ],\n      [\n        "Ġstr",\n        "ange"\n      ],\n      [\n        "Ġta",\n        "il"\n      ],\n      [\n        "Ġs",\n        "ong"\n      ],\n      [\n        "ir",\n        "rel"\n      ],\n      [\n        "c",\n        "ed"\n      ],\n      [\n        "Ġh",\n        "ot"\n      ],\n      [\n        "ic",\n        "y"\n      ],\n      [\n        "Ġm",\n        "ar"\n      ],\n      [\n        "Ġm",\n        "ight"\n      ],\n      [\n        "Ġsc",\n        "ary"\n      ],\n      [\n        "Ġmu",\n        "s"\n      ],\n      [\n        "Ġwood",\n        "s"\n      ],\n      [\n        "or",\n        "ing"\n      ],\n      [\n        "Ġlea",\n        "ves"\n      ],\n      [\n        "or",\n        "n"\n      ],\n      [\n        "Ġbr",\n        "ought"\n      ],\n      [\n        "Ġbook",\n        "s"\n      ],\n      [\n        "Ġsqu",\n        "irrel"\n      ],\n      [\n        "ou",\n        "p"\n      ],\n      [\n        "Ġel",\n        "se"\n      ],\n      [\n        "a",\n        "pp"\n      ],\n      [\n        "Ġbro",\n        "ken"\n      ],\n      [\n        "ĠW",\n        "hat"\n      ],\n      [\n        "u",\n        "it"\n      ],\n      [\n        "Ġamaz",\n        "ing"\n      ],\n      [\n        "Ġperf",\n        "ect"\n      ],\n      [\n        "A",\n        "t"\n      ],\n      [\n        "Ġfe",\n        "w"\n      ],\n      [\n        "Ġball",\n        "oon"\n      ],\n      [\n        "Ġpu",\n        "ppy"\n      ],\n      [\n        "Ġcl",\n        "ot"\n      ],\n      [\n        "Ġc",\n        "astle"\n      ],\n      [\n        "ĠF",\n        "l"\n      ],\n      [\n        "Ġm",\n        "et"\n      ],\n      [\n        "Ġfin",\n        "ally"\n      ],\n      [\n        "ĠTh",\n        "is"\n      ],\n      [\n        "Ġbe",\n        "ach"\n      ],\n      [\n        "Ġclot",\n        "hes"\n      ],\n      [\n        "an",\n        "ce"\n      ],\n      [\n        "Ġro",\n        "ll"\n      ],\n      [\n        "Ġde",\n        "ep"\n      ],\n      [\n        "Ġp",\n        "ink"\n      ],\n      [\n        "p",\n        "h"\n      ],\n      [\n        "Ġpicture",\n        "s"\n      ],\n      [\n        "ct",\n        "or"\n      ],\n      [\n        "et",\n        "er"\n      ],\n      [\n        "ĠB",\n        "illy"\n      ],\n      [\n        "ke",\n        "y"\n      ],\n      [\n        "Ġpain",\n        "t"\n      ],\n      [\n        "ĠG",\n        "ra"\n      ],\n      [\n        "Ġp",\n        "et"\n      ],\n      [\n        "Ġturn",\n        "ed"\n      ],\n      [\n        "Ġs",\n        "ec"\n      ],\n      [\n        "Ġmor",\n        "al"\n      ],\n      [\n        "Ġc",\n        "ount"\n      ],\n      [\n        "Ġwh",\n        "ite"\n      ],\n      [\n        "Ġstay",\n        "ed"\n      ],\n      [\n        "O",\n        "h"\n      ],\n      [\n        "Ġr",\n        "ide"\n      ],\n      [\n        "Ġtal",\n        "k"\n      ],\n      [\n        "Ġevery",\n        "thing"\n      ],\n      [\n        "Ġf",\n        "ight"\n      ],\n      [\n        "Ġw",\n        "et"\n      ],\n      [\n        "Ġamaz",\n        "ed"\n      ],\n      [\n        "d",\n        "er"\n      ],\n      [\n        "Ġwant",\n        "s"\n      ],\n      [\n        "Ġgra",\n        "ndma"\n      ],\n      [\n        "l",\n        "c"\n      ],\n      [\n        "w",\n        "l"\n      ],\n      [\n        "Ġdad",\n        "dy"\n      ],\n      [\n        "r",\n        "ed"\n      ],\n      [\n        "Ġsa",\n        "il"\n      ],\n      [\n        "Ġcom",\n        "p"\n      ],\n      [\n        "Ġclimb",\n        "ed"\n      ],\n      [\n        "Ġj",\n        "oy"\n      ],\n      [\n        "Ġfly",\n        "ing"\n      ],\n      [\n        "Ġhun",\n        "gry"\n      ],\n      [\n        "l",\n        "f"\n      ],\n      [\n        "Ġp",\n        "en"\n      ],\n      [\n        "ck",\n        "et"\n      ],\n      [\n        "ow",\n        "er"\n      ],\n      [\n        "Ġbre",\n        "ak"\n      ],\n      [\n        "m",\n        "an"\n      ],\n      [\n        "at",\n        "ed"\n      ],\n      [\n        "ri",\n        "es"\n      ],\n      [\n        "Ġal",\n        "one"\n      ],\n      [\n        "Ġsee",\n        "m"\n      ],\n      [\n        "Ġsomet",\n        "imes"\n      ],\n      [\n        "Ġba",\n        "by"\n      ],\n      [\n        "Ġvis",\n        "it"\n      ],\n      [\n        "z",\n        "e"\n      ],\n      [\n        "Ġgr",\n        "ow"\n      ],\n      [\n        "r",\n        "ow"\n      ],\n      [\n        "Ġwas",\n        "n"\n      ],\n      [\n        "am",\n        "p"\n      ],\n      [\n        "Ġsp",\n        "l"\n      ],\n      [\n        "ĠTom",\n        "my"\n      ],\n      [\n        "Ġmus",\n        "ic"\n      ],\n      [\n        "Ġse",\n        "ar"\n      ],\n      [\n        "Ġlist",\n        "ened"\n      ],\n      [\n        "Ġsp",\n        "ark"\n      ],\n      [\n        "Ġcl",\n        "oud"\n      ],\n      [\n        "Ġwa",\n        "ited"\n      ],\n      [\n        "Ġ",\n        "ve"\n      ],\n      [\n        "Ġdan",\n        "ger"\n      ],\n      [\n        "ĠM",\n        "olly"\n      ],\n      [\n        "M",\n        "ax"\n      ],\n      [\n        "Ġh",\n        "ide"\n      ],\n      [\n        "Ġm",\n        "akes"\n      ],\n      [\n        "im",\n        "my"\n      ],\n      [\n        "Ġtow",\n        "ards"\n      ],\n      [\n        "Ġcook",\n        "ies"\n      ],\n      [\n        "e",\n        "ver"\n      ],\n      [\n        "Ġf",\n        "all"\n      ],\n      [\n        "Ġw",\n        "on"\n      ],\n      [\n        "Ġp",\n        "i"\n      ],\n      [\n        "Ġbig",\n        "ger"\n      ],\n      [\n        "Ġs",\n        "illy"\n      ],\n      [\n        "M",\n        "aybe"\n      ],\n      [\n        "Ġf",\n        "ield"\n      ],\n      [\n        "le",\n        "ph"\n      ],\n      [\n        "Ġj",\n        "ob"\n      ],\n      [\n        "O",\n        "K"\n      ],\n      [\n        "an",\n        "c"\n      ],\n      [\n        "a",\n        "pped"\n      ],\n      [\n        "or",\n        "m"\n      ],\n      [\n        "ig",\n        "g"\n      ],\n      [\n        "op",\n        "e"\n      ],\n      [\n        "uck",\n        "y"\n      ],\n      [\n        "Ġbel",\n        "ie"\n      ],\n      [\n        "Ġsh",\n        "ared"\n      ],\n      [\n        "Ġj",\n        "u"\n      ],\n      [\n        "av",\n        "y"\n      ],\n      [\n        "i",\n        "x"\n      ],\n      [\n        "Ġpo",\n        "l"\n      ],\n      [\n        "ss",\n        "ed"\n      ],\n      [\n        "Ġprom",\n        "ised"\n      ],\n      [\n        "u",\n        "ally"\n      ],\n      [\n        "Ġs",\n        "ide"\n      ],\n      [\n        "ĠA",\n        "my"\n      ],\n      [\n        "en",\n        "ce"\n      ],\n      [\n        "o",\n        "om"\n      ],\n      [\n        "Ġb",\n        "ike"\n      ],\n      [\n        "is",\n        "ter"\n      ],\n      [\n        "Ġli",\n        "ving"\n      ],\n      [\n        "g",\n        "g"\n      ],\n      [\n        "i",\n        "re"\n      ],\n      [\n        "s",\n        "w"\n      ],\n      [\n        "Ġm",\n        "iss"\n      ],\n      [\n        "Ġp",\n        "a"\n      ],\n      [\n        "Ġfast",\n        "er"\n      ],\n      [\n        "Ġslow",\n        "ly"\n      ],\n      [\n        "re",\n        "w"\n      ],\n      [\n        "ĠT",\n        "o"\n      ],\n      [\n        "Ġg",\n        "u"\n      ],\n      [\n        "Ġpull",\n        "ed"\n      ],\n      [\n        "Ġan",\n        "sw"\n      ],\n      [\n        "Ġmo",\n        "st"\n      ],\n      [\n        "Ġmu",\n        "st"\n      ],\n      [\n        "Ġsmil",\n        "es"\n      ],\n      [\n        "Ġwor",\n        "ried"\n      ],\n      [\n        "Ġpart",\n        "y"\n      ],\n      [\n        "s",\n        "et"\n      ],\n      [\n        "Ġfar",\n        "mer"\n      ],\n      [\n        "Ġe",\n        "m"\n      ],\n      [\n        "W",\n        "hy"\n      ],\n      [\n        "m",\n        "e"\n      ],\n      [\n        "Ġt",\n        "eddy"\n      ],\n      [\n        "ain",\n        "ed"\n      ],\n      [\n        "t",\n        "hy"\n      ],\n      [\n        "Ġste",\n        "pped"\n      ],\n      [\n        "i",\n        "pp"\n      ],\n      [\n        "ot",\n        "t"\n      ],\n      [\n        "l",\n        "ing"\n      ],\n      [\n        "Ġcry",\n        "ing"\n      ],\n      [\n        "H",\n        "i"\n      ],\n      [\n        "i",\n        "pped"\n      ],\n      [\n        "o",\n        "l"\n      ],\n      [\n        "am",\n        "es"\n      ],\n      [\n        "ĠA",\n        "ll"\n      ],\n      [\n        "Ġjo",\n        "in"\n      ],\n      [\n        "Ġl",\n        "ake"\n      ],\n      [\n        "Ġpl",\n        "an"\n      ],\n      [\n        "ĠJ",\n        "ill"\n      ],\n      [\n        "Ġc",\n        "ream"\n      ],\n      [\n        "O",\n        "kay"\n      ],\n      [\n        "an",\n        "g"\n      ],\n      [\n        "Ġhelp",\n        "ing"\n      ],\n      [\n        "Ġbl",\n        "ack"\n      ],\n      [\n        "Ġdis",\n        "app"\n      ],\n      [\n        "Ġle",\n        "ave"\n      ],\n      [\n        "Ġcu",\n        "p"\n      ],\n      [\n        "Ġdra",\n        "gon"\n      ],\n      [\n        "Ġs",\n        "it"\n      ],\n      [\n        "Ġgr",\n        "ate"\n      ],\n      [\n        "Ġp",\n        "ass"\n      ],\n      [\n        "Ġat",\n        "t"\n      ],\n      [\n        "Ġmagic",\n        "al"\n      ],\n      [\n        "ĠM",\n        "aybe"\n      ],\n      [\n        "Ġgrate",\n        "ful"\n      ],\n      [\n        "ug",\n        "g"\n      ],\n      [\n        "Ġc",\n        "andy"\n      ],\n      [\n        "e",\n        "ared"\n      ],\n      [\n        "f",\n        "ort"\n      ],\n      [\n        "Ġne",\n        "igh"\n      ],\n      [\n        "Ġdan",\n        "ce"\n      ],\n      [\n        "r",\n        "ot"\n      ],\n      [\n        "Ġs",\n        "u"\n      ],\n      [\n        "Ġf",\n        "ur"\n      ],\n      [\n        "le",\n        "ss"\n      ],\n      [\n        "Ġtre",\n        "asure"\n      ],\n      [\n        "un",\n        "ch"\n      ],\n      [\n        "Ġneigh",\n        "b"\n      ],\n      [\n        "Ġwe",\n        "lc"\n      ],\n      [\n        "Ġac",\n        "ross"\n      ],\n      [\n        "Ġsh",\n        "ar"\n      ],\n      [\n        "Ġdoll",\n        "s"\n      ],\n      [\n        "m",\n        "ing"\n      ],\n      [\n        "Ġt",\n        "reat"\n      ],\n      [\n        "Ġhe",\n        "avy"\n      ],\n      [\n        "is",\n        "es"\n      ],\n      [\n        "Ġbl",\n        "ank"\n      ],\n      [\n        "ĠFl",\n        "uffy"\n      ],\n      [\n        "b",\n        "ble"\n      ],\n      [\n        "Ġre",\n        "ach"\n      ],\n      [\n        "Ġask",\n        "s"\n      ],\n      [\n        "Ġbat",\n        "h"\n      ],\n      [\n        "Ġs",\n        "ister"\n      ],\n      [\n        "Ġbe",\n        "e"\n      ],\n      [\n        "i",\n        "et"\n      ],\n      [\n        "z",\n        "z"\n      ],\n      [\n        "Ġdanger",\n        "ous"\n      ],\n      [\n        "Ġm",\n        "ix"\n      ],\n      [\n        "Ġcolor",\n        "ful"\n      ],\n      [\n        "um",\n        "b"\n      ],\n      [\n        "Ġpr",\n        "o"\n      ],\n      [\n        "Ġsec",\n        "ret"\n      ],\n      [\n        "Ġblank",\n        "et"\n      ],\n      [\n        "ec",\n        "es"\n      ],\n      [\n        "as",\n        "ed"\n      ],\n      [\n        "Ġmon",\n        "ster"\n      ],\n      [\n        "Ġdo",\n        "ctor"\n      ],\n      [\n        "Ġsh",\n        "ap"\n      ],\n      [\n        "Ġb",\n        "ott"\n      ],\n      [\n        "Ġm",\n        "oney"\n      ],\n      [\n        "Ġpl",\n        "ant"\n      ],\n      [\n        "Ġtra",\n        "in"\n      ],\n      [\n        "Ġpi",\n        "eces"\n      ],\n      [\n        "out",\n        "h"\n      ],\n      [\n        "Ġbelie",\n        "ve"\n      ],\n      [\n        "b",\n        "r"\n      ],\n      [\n        "Ġr",\n        "iver"\n      ],\n      [\n        "Ġevery",\n        "where"\n      ],\n      [\n        "Ġac",\n        "c"\n      ],\n      [\n        "m",\n        "ent"\n      ],\n      [\n        "Ġto",\n        "wn"\n      ],\n      [\n        "Ġmon",\n        "key"\n      ],\n      [\n        "w",\n        "eet"\n      ],\n      [\n        "Ġs",\n        "ign"\n      ],\n      [\n        "Ġl",\n        "on"\n      ],\n      [\n        "Ġwor",\n        "ds"\n      ],\n      [\n        "Ġla",\n        "nd"\n      ],\n      [\n        "d",\n        "e"\n      ],\n      [\n        "Ġm",\n        "outh"\n      ],\n      [\n        "Ġg",\n        "if"\n      ],\n      [\n        "Ġs",\n        "uch"\n      ],\n      [\n        "Ġr",\n        "ace"\n      ],\n      [\n        "Ġqu",\n        "iet"\n      ],\n      [\n        "vent",\n        "ually"\n      ],\n      [\n        "S",\n        "ure"\n      ],\n      [\n        "Ġst",\n        "uck"\n      ],\n      [\n        "Ġbro",\n        "ke"\n      ],\n      [\n        "Ġcr",\n        "ay"\n      ],\n      [\n        "Ġc",\n        "a"\n      ],\n      [\n        "Ġh",\n        "ig"\n      ],\n      [\n        "id",\n        "ent"\n      ],\n      [\n        "Ġwith",\n        "out"\n      ],\n      [\n        "s",\n        "ed"\n      ],\n      [\n        "Ġpo",\n        "cket"\n      ],\n      [\n        "Ġlon",\n        "ely"\n      ],\n      [\n        "Ġclos",\n        "ed"\n      ],\n      [\n        "leph",\n        "ant"\n      ],\n      [\n        "Ġb",\n        "uck"\n      ],\n      [\n        "Ġtow",\n        "er"\n      ],\n      [\n        "Ġe",\n        "lephant"\n      ],\n      [\n        "Ġst",\n        "and"\n      ],\n      [\n        "bb",\n        "y"\n      ],\n      [\n        "Ġj",\n        "e"\n      ],\n      [\n        "it",\n        "ing"\n      ],\n      [\n        "Ġc",\n        "our"\n      ],\n      [\n        "ra",\n        "ct"\n      ],\n      [\n        "Ġthink",\n        "s"\n      ],\n      [\n        "Ġc",\n        "e"\n      ],\n      [\n        "Ġst",\n        "u"\n      ],\n      [\n        "Ġgent",\n        "le"\n      ],\n      [\n        "Ġhe",\n        "art"\n      ],\n      [\n        "ch",\n        "ool"\n      ],\n      [\n        "al",\n        "thy"\n      ],\n      [\n        "Ġcl",\n        "apped"\n      ],\n      [\n        "Ġtw",\n        "ins"\n      ],\n      [\n        "B",\n        "ob"\n      ],\n      [\n        "Ġsa",\n        "me"\n      ],\n      [\n        "Ġexpl",\n        "oring"\n      ],\n      [\n        "Ġor",\n        "ange"\n      ],\n      [\n        "Ġto",\n        "day"\n      ],\n      [\n        "Ġb",\n        "ite"\n      ],\n      [\n        "Ġp",\n        "ot"\n      ],\n      [\n        "or",\n        "ed"\n      ],\n      [\n        "Ġlong",\n        "er"\n      ],\n      [\n        "Ġsn",\n        "ack"\n      ],\n      [\n        "Ġn",\n        "ap"\n      ],\n      [\n        "Ġyou",\n        "ng"\n      ],\n      [\n        "Ġbow",\n        "l"\n      ],\n      [\n        "Ġeat",\n        "ing"\n      ],\n      [\n        "Ġp",\n        "ers"\n      ],\n      [\n        "Ġup",\n        "set"\n      ],\n      [\n        "Ġte",\n        "ac"\n      ],\n      [\n        "Ġdir",\n        "ty"\n      ],\n      [\n        "Ġo",\n        "wl"\n      ],\n      [\n        "Ġprin",\n        "cess"\n      ],\n      [\n        "Ġcour",\n        "se"\n      ],\n      [\n        "Ġsc",\n        "ream"\n      ],\n      [\n        "n",\n        "ey"\n      ],\n      [\n        "Ġs",\n        "uddenly"\n      ],\n      [\n        "Ġs",\n        "chool"\n      ],\n      [\n        "Ġhig",\n        "her"\n      ],\n      [\n        "Ġk",\n        "iss"\n      ],\n      [\n        "Ġsh",\n        "op"\n      ],\n      [\n        "c",\n        "le"\n      ],\n      [\n        "Ġb",\n        "ar"\n      ],\n      [\n        "ou",\n        "n"\n      ],\n      [\n        "Ġbr",\n        "ing"\n      ],\n      [\n        "Ġhop",\n        "ed"\n      ],\n      [\n        "F",\n        "inally"\n      ],\n      [\n        "e",\n        "k"\n      ],\n      [\n        "Ġsm",\n        "o"\n      ],\n      [\n        "Ġal",\n        "ong"\n      ],\n      [\n        "Ġsho",\n        "es"\n      ],\n      [\n        "M",\n        "ia"\n      ],\n      [\n        "Ġsm",\n        "art"\n      ],\n      [\n        "Ġown",\n        "er"\n      ],\n      [\n        "Ġfair",\n        "y"\n      ],\n      [\n        "Ġseem",\n        "ed"\n      ],\n      [\n        "L",\n        "ila"\n      ],\n      [\n        "Ġfr",\n        "uit"\n      ],\n      [\n        "Ġapp",\n        "le"\n      ],\n      [\n        "Ġc",\n        "ave"\n      ],\n      [\n        "ĠM",\n        "r"\n      ],\n      [\n        "ĠDad",\n        "dy"\n      ],\n      [\n        "Ġth",\n        "rew"\n      ],\n      [\n        "Ġpus",\n        "hed"\n      ],\n      [\n        "D",\n        "o"\n      ],\n      [\n        "h",\n        "one"\n      ],\n      [\n        "itt",\n        "y"\n      ],\n      [\n        "fu",\n        "sed"\n      ],\n      [\n        "Ġwould",\n        "n"\n      ],\n      [\n        "Ġcar",\n        "rot"\n      ],\n      [\n        "Ġsw",\n        "am"\n      ],\n      [\n        "Ġdro",\n        "ve"\n      ],\n      [\n        "t",\n        "s"\n      ],\n      [\n        "Ġp",\n        "ie"\n      ],\n      [\n        "Ġhapp",\n        "en"\n      ],\n      [\n        "Ġbr",\n        "own"\n      ],\n      [\n        "ask",\n        "et"\n      ],\n      [\n        "Ġb",\n        "asket"\n      ],\n      [\n        "Ġwelc",\n        "ome"\n      ],\n      [\n        "i",\n        "er"\n      ],\n      [\n        "Ġp",\n        "ig"\n      ],\n      [\n        "Ġk",\n        "ite"\n      ],\n      [\n        "Ġse",\n        "a"\n      ],\n      [\n        "ĠEm",\n        "ma"\n      ],\n      [\n        "Ġd",\n        "es"\n      ],\n      [\n        "Ġacc",\n        "ident"\n      ],\n      [\n        "Ġw",\n        "in"\n      ],\n      [\n        "Ġf",\n        "ree"\n      ],\n      [\n        "Ġteac",\n        "her"\n      ],\n      [\n        "Ġo",\n        "b"\n      ],\n      [\n        "Ġrun",\n        "s"\n      ],\n      [\n        "i",\n        "ble"\n      ],\n      [\n        "Ġin",\n        "st"\n      ],\n      [\n        "Ġche",\n        "ered"\n      ],\n      [\n        "Ġexpl",\n        "ained"\n      ],\n      [\n        "ck",\n        "ed"\n      ],\n      [\n        "Ġm",\n        "in"\n      ],\n      [\n        "Ġp",\n        "ack"\n      ],\n      [\n        "Ġco",\n        "ll"\n      ],\n      [\n        "e",\n        "ad"\n      ],\n      [\n        "s",\n        "y"\n      ],\n      [\n        "Ġd",\n        "ry"\n      ],\n      [\n        "or",\n        "s"\n      ],\n      [\n        "Ġbuck",\n        "et"\n      ],\n      [\n        "Ġadventure",\n        "s"\n      ],\n      [\n        "Ġnear",\n        "by"\n      ],\n      [\n        "Ġt",\n        "iny"\n      ],\n      [\n        "Ġth",\n        "ough"\n      ],\n      [\n        "ie",\n        "ved"\n      ],\n      [\n        "Ġdro",\n        "pped"\n      ],\n      [\n        "un",\n        "g"\n      ],\n      [\n        "Ġpl",\n        "ac"\n      ],\n      [\n        "Ġsp",\n        "r"\n      ],\n      [\n        "Ġs",\n        "oup"\n      ],\n      [\n        "Ġw",\n        "ings"\n      ],\n      [\n        "Ġl",\n        "unch"\n      ],\n      [\n        "ĠT",\n        "weet"\n      ],\n      [\n        "Ġfor",\n        "get"\n      ],\n      [\n        "Ġst",\n        "re"\n      ],\n      [\n        "Ġ",\n        "3"\n      ],\n      [\n        "Ġl",\n        "ate"\n      ],\n      [\n        "Ġc",\n        "aught"\n      ],\n      [\n        "Ġn",\n        "aught"\n      ],\n      [\n        "or",\n        "row"\n      ],\n      [\n        "Ġlaug",\n        "hing"\n      ],\n      [\n        "Ġbel",\n        "ong"\n      ],\n      [\n        "Ġpre",\n        "s"\n      ],\n      [\n        "Ġ",\n        "es"\n      ],\n      [\n        "Ġta",\n        "king"\n      ],\n      [\n        "y",\n        "ard"\n      ],\n      [\n        "Ġsh",\n        "ook"\n      ],\n      [\n        "Ġj",\n        "our"\n      ],\n      [\n        "Ġmo",\n        "ved"\n      ],\n      [\n        "ion",\n        "s"\n      ],\n      [\n        "w",\n        "ard"\n      ],\n      [\n        "A",\n        "nd"\n      ],\n      [\n        "b",\n        "ow"\n      ],\n      [\n        "Ġsp",\n        "oon"\n      ],\n      [\n        "Ġback",\n        "yard"\n      ],\n      [\n        "h",\n        "o"\n      ],\n      [\n        "Ġsa",\n        "ng"\n      ],\n      [\n        "Ġex",\n        "p"\n      ],\n      [\n        "Ġnaught",\n        "y"\n      ],\n      [\n        "L",\n        "ucy"\n      ],\n      [\n        "a",\n        "il"\n      ],\n      [\n        "Ġb",\n        "oun"\n      ],\n      [\n        "Ġwe",\n        "ll"\n      ],\n      [\n        "Ġw",\n        "ish"\n      ],\n      [\n        "Ġp",\n        "ast"\n      ],\n      [\n        "Ġg",\n        "ames"\n      ],\n      [\n        "Ġre",\n        "sp"\n      ],\n      [\n        "aught",\n        "er"\n      ],\n      [\n        "Ġcol",\n        "our"\n      ],\n      [\n        "Ġy",\n        "ard"\n      ],\n      [\n        "itt",\n        "ing"\n      ],\n      [\n        "Ġno",\n        "ises"\n      ],\n      [\n        "Ġtr",\n        "ou"\n      ],\n      [\n        "Ġsun",\n        "ny"\n      ],\n      [\n        "Ġwo",\n        "ke"\n      ],\n      [\n        "Ġrain",\n        "bow"\n      ],\n      [\n        "ver",\n        "ed"\n      ],\n      [\n        "ve",\n        "l"\n      ],\n      [\n        "Ġm",\n        "ine"\n      ],\n      [\n        "Ġne",\n        "ck"\n      ],\n      [\n        "Ġsw",\n        "ings"\n      ],\n      [\n        "a",\n        "ur"\n      ],\n      [\n        "Ġc",\n        "or"\n      ],\n      [\n        "Ġcar",\n        "ry"\n      ],\n      [\n        "ock",\n        "ed"\n      ],\n      [\n        "Ġrock",\n        "s"\n      ],\n      [\n        "Ġwonder",\n        "ed"\n      ],\n      [\n        "Ġpres",\n        "ent"\n      ],\n      [\n        "Ġd",\n        "ist"\n      ],\n      [\n        "c",\n        "ing"\n      ],\n      [\n        "Ġw",\n        "ise"\n      ],\n      [\n        "Ġp",\n        "ower"\n      ],\n      [\n        "pp",\n        "ing"\n      ],\n      [\n        "Ġrel",\n        "ieved"\n      ],\n      [\n        "Ġc",\n        "ou"\n      ],\n      [\n        "us",\n        "h"\n      ],\n      [\n        "Ġm",\n        "ark"\n      ],\n      [\n        "ount",\n        "ain"\n      ],\n      [\n        "Ġst",\n        "ars"\n      ],\n      [\n        "Ġsing",\n        "ing"\n      ],\n      [\n        "c",\n        "er"\n      ],\n      [\n        "e",\n        "en"\n      ],\n      [\n        "g",\n        "ged"\n      ],\n      [\n        "Ġli",\n        "fe"\n      ],\n      [\n        "Ġhug",\n        "s"\n      ],\n      [\n        "Ġbus",\n        "y"\n      ],\n      [\n        "in",\n        "ing"\n      ],\n      [\n        "ĠM",\n        "ama"\n      ],\n      [\n        "Ġfeel",\n        "s"\n      ],\n      [\n        "Ġwatch",\n        "ing"\n      ],\n      [\n        "Ġcreat",\n        "ure"\n      ],\n      [\n        "Ġdri",\n        "ver"\n      ],\n      [\n        "L",\n        "ater"\n      ],\n      [\n        "u",\n        "ched"\n      ],\n      [\n        "Ġto",\n        "uched"\n      ],\n      [\n        "Ġb",\n        "reat"\n      ],\n      [\n        "re",\n        "st"\n      ],\n      [\n        "ĠM",\n        "ary"\n      ],\n      [\n        "Ġv",\n        "ill"\n      ],\n      [\n        "Ġto",\n        "m"\n      ],\n      [\n        "Ġd",\n        "aughter"\n      ],\n      [\n        "ad",\n        "ow"\n      ],\n      [\n        "Ġno",\n        "se"\n      ],\n      [\n        "Ġthr",\n        "ow"\n      ],\n      [\n        "Ġbus",\n        "h"\n      ],\n      [\n        "Ġg",\n        "igg"\n      ],\n      [\n        "Ġwh",\n        "ist"\n      ],\n      [\n        "Ġnot",\n        "hing"\n      ],\n      [\n        "Åĵ",\n        "I"\n      ],\n      [\n        "Ġpat",\n        "h"\n      ],\n      [\n        "Ġh",\n        "it"\n      ],\n      [\n        "ll",\n        "o"\n      ],\n      [\n        "Ġin",\n        "c"\n      ],\n      [\n        "Ġgra",\n        "nd"\n      ],\n      [\n        "Ġarm",\n        "s"\n      ],\n      [\n        "Ġf",\n        "ence"\n      ],\n      [\n        "Ġch",\n        "air"\n      ],\n      [\n        "Ġdr",\n        "ink"\n      ],\n      [\n        "O",\n        "f"\n      ],\n      [\n        "p",\n        "e"\n      ],\n      [\n        "Ġs",\n        "ick"\n      ],\n      [\n        "Ġw",\n        "he"\n      ],\n      [\n        "Ġpe",\n        "ace"\n      ],\n      [\n        "Ġju",\n        "ice"\n      ],\n      [\n        "e",\n        "rest"\n      ],\n      [\n        "Ġb",\n        "en"\n      ],\n      [\n        "Ġint",\n        "erest"\n      ],\n      [\n        "Ġswim",\n        "ming"\n      ],\n      [\n        "Ġpoin",\n        "ted"\n      ],\n      [\n        "Ġshap",\n        "es"\n      ],\n      [\n        "p",\n        "ed"\n      ],\n      [\n        "Ġday",\n        "s"\n      ],\n      [\n        "Ġpat",\n        "ient"\n      ],\n      [\n        "Ġsnow",\n        "man"\n      ],\n      [\n        "Ġbut",\n        "t"\n      ],\n      [\n        "Ġj",\n        "ar"\n      ],\n      [\n        "Ġmy",\n        "ster"\n      ],\n      [\n        "Ġhold",\n        "ing"\n      ],\n      [\n        "oo",\n        "se"\n      ],\n      [\n        "Ġre",\n        "t"\n      ],\n      [\n        "ĠJ",\n        "immy"\n      ],\n      [\n        "Ġke",\n        "y"\n      ],\n      [\n        "p",\n        "a"\n      ],\n      [\n        "Ġb",\n        "ought"\n      ],\n      [\n        "Ġe",\n        "as"\n      ],\n      [\n        "itt",\n        "en"\n      ],\n      [\n        "Ġa",\n        "f"\n      ],\n      [\n        "Ġs",\n        "itting"\n      ],\n      [\n        "an",\n        "ch"\n      ],\n      [\n        "Ġle",\n        "g"\n      ],\n      [\n        "Ġbr",\n        "anch"\n      ],\n      [\n        "Ġco",\n        "zy"\n      ],\n      [\n        "ĠSam",\n        "my"\n      ],\n      [\n        "Ġin",\n        "v"\n      ],\n      [\n        "Ġsmil",\n        "ing"\n      ],\n      [\n        "Ġexc",\n        "iting"\n      ],\n      [\n        "ĠGra",\n        "ndma"\n      ],\n      [\n        "l",\n        "a"\n      ],\n      [\n        "t",\n        "o"\n      ],\n      [\n        "Ġc",\n        "ir"\n      ],\n      [\n        "Ġp",\n        "udd"\n      ],\n      [\n        "ir",\n        "th"\n      ],\n      [\n        "al",\n        "m"\n      ],\n      [\n        "Ġdis",\n        "co"\n      ],\n      [\n        "Ġturn",\n        "s"\n      ],\n      [\n        "irth",\n        "day"\n      ],\n      [\n        "Ġdan",\n        "ced"\n      ],\n      [\n        "ĠIn",\n        "side"\n      ],\n      [\n        "M",\n        "olly"\n      ],\n      [\n        "n",\n        "ess"\n      ],\n      [\n        "Ġh",\n        "or"\n      ],\n      [\n        "Ġn",\n        "est"\n      ],\n      [\n        "Ġr",\n        "ound"\n      ],\n      [\n        "He",\n        "y"\n      ],\n      [\n        "Ġcut",\n        "e"\n      ],\n      [\n        "Ġco",\n        "in"\n      ],\n      [\n        "or",\n        "d"\n      ],\n      [\n        "Ġch",\n        "o"\n      ],\n      [\n        "Tom",\n        "my"\n      ],\n      [\n        "ĠTh",\n        "ank"\n      ],\n      [\n        "M",\n        "um"\n      ],\n      [\n        "id",\n        "ge"\n      ],\n      [\n        "Ġma",\n        "il"\n      ],\n      [\n        "Ġgif",\n        "t"\n      ],\n      [\n        "Ġm",\n        "ist"\n      ],\n      [\n        "Ġsp",\n        "o"\n      ],\n      [\n        "iz",\n        "e"\n      ],\n      [\n        "Ġad",\n        "m"\n      ],\n      [\n        "sh",\n        "ine"\n      ],\n      [\n        "Ġjour",\n        "ney"\n      ],\n      [\n        "b",\n        "all"\n      ],\n      [\n        "Ġa",\n        "ct"\n      ],\n      [\n        "Ġst",\n        "ar"\n      ],\n      [\n        "o",\n        "g"\n      ],\n      [\n        "Ġf",\n        "it"\n      ],\n      [\n        "Ġl",\n        "etter"\n      ],\n      [\n        "Ġp",\n        "ract"\n      ],\n      [\n        "Ġsa",\n        "ck"\n      ],\n      [\n        "Ġth",\n        "in"\n      ],\n      [\n        "Ġli",\n        "ft"\n      ],\n      [\n        "ra",\n        "id"\n      ],\n      [\n        "ĠC",\n        "ome"\n      ],\n      [\n        "Ġb",\n        "irthday"\n      ],\n      [\n        "Ġe",\n        "ag"\n      ],\n      [\n        "Ġsun",\n        "shine"\n      ],\n      [\n        "C",\n        "ome"\n      ],\n      [\n        "Ġm",\n        "ad"\n      ],\n      [\n        "Ġe",\n        "ar"\n      ],\n      [\n        "Ġcom",\n        "es"\n      ],\n      [\n        "co",\n        "l"\n      ],\n      [\n        "Ġcray",\n        "ons"\n      ],\n      [\n        "Ġaf",\n        "raid"\n      ],\n      [\n        "ir",\n        "t"\n      ],\n      [\n        "Ġdel",\n        "ight"\n      ],\n      [\n        "bo",\n        "ard"\n      ],\n      [\n        "Ġaccident",\n        "ally"\n      ],\n      [\n        "Ġret",\n        "urn"\n      ],\n      [\n        "Ġw",\n        "ra"\n      ],\n      [\n        "re",\n        "ad"\n      ],\n      [\n        "Ġany",\n        "one"\n      ],\n      [\n        "ĠSo",\n        "on"\n      ],\n      [\n        "Ġhe",\n        "llo"\n      ],\n      [\n        "ĠJen",\n        "ny"\n      ],\n      [\n        "c",\n        "es"\n      ],\n      [\n        "Ġ",\n        "O"\n      ],\n      [\n        "Ġs",\n        "uc"\n      ],\n      [\n        "Ġlove",\n        "ly"\n      ],\n      [\n        "Ġtast",\n        "ed"\n      ],\n      [\n        "o",\n        "ve"\n      ],\n      [\n        "Ġt",\n        "ear"\n      ],\n      [\n        "Ġp",\n        "urp"\n      ],\n      [\n        "le",\n        "t"\n      ],\n      [\n        "on",\n        "t"\n      ],\n      [\n        "am",\n        "ed"\n      ],\n      [\n        "Ġhug",\n        "e"\n      ],\n      [\n        "Ġinterest",\n        "ing"\n      ],\n      [\n        "t",\n        "en"\n      ],\n      [\n        "Ġk",\n        "ing"\n      ],\n      [\n        "Ġre",\n        "s"\n      ],\n      [\n        "ble",\n        "m"\n      ],\n      [\n        "ĠF",\n        "inally"\n      ],\n      [\n        "Ġon",\n        "to"\n      ],\n      [\n        "Ġfr",\n        "ont"\n      ],\n      [\n        "Ġpro",\n        "blem"\n      ],\n      [\n        "Ġinst",\n        "ead"\n      ],\n      [\n        "Ġs",\n        "ur"\n      ],\n      [\n        "Ġl",\n        "ucky"\n      ],\n      [\n        "Ġstre",\n        "et"\n      ],\n      [\n        "l",\n        "ice"\n      ],\n      [\n        "Ġb",\n        "and"\n      ],\n      [\n        "ep",\n        "end"\n      ],\n      [\n        "Ġpie",\n        "ce"\n      ],\n      [\n        "iz",\n        "z"\n      ],\n      [\n        "Ġim",\n        "ag"\n      ],\n      [\n        "Ġsound",\n        "s"\n      ],\n      [\n        "Ġroll",\n        "ed"\n      ],\n      [\n        "Ġpers",\n        "on"\n      ],\n      [\n        "u",\n        "nt"\n      ],\n      [\n        "Ġt",\n        "un"\n      ],\n      [\n        "Ġcom",\n        "fort"\n      ],\n      [\n        "Ġtast",\n        "y"\n      ],\n      [\n        "er",\n        "t"\n      ],\n      [\n        "Ġsand",\n        "w"\n      ],\n      [\n        "Ġsmell",\n        "ed"\n      ],\n      [\n        "it",\n        "ch"\n      ],\n      [\n        "Ġwh",\n        "ich"\n      ],\n      [\n        "Ġk",\n        "ick"\n      ],\n      [\n        "Ġstor",\n        "ies"\n      ],\n      [\n        "Ġem",\n        "b"\n      ],\n      [\n        "it",\n        "y"\n      ],\n      [\n        "Ġgra",\n        "b"\n      ],\n      [\n        "Ġfar",\n        "m"\n      ],\n      [\n        "Ġsear",\n        "ched"\n      ],\n      [\n        "Ġpurp",\n        "le"\n      ],\n      [\n        "Ġ",\n        "\'"\n      ],\n      [\n        "Ġa",\n        "p"\n      ],\n      [\n        "Ġwho",\n        "le"\n      ],\n      [\n        "Ġvill",\n        "age"\n      ],\n      [\n        "Ġhor",\n        "se"\n      ],\n      [\n        "M",\n        "y"\n      ],\n      [\n        "Ġ",\n        "ent"\n      ],\n      [\n        "Ġpl",\n        "ane"\n      ],\n      [\n        "itt",\n        "ens"\n      ],\n      [\n        "Ġg",\n        "ather"\n      ],\n      [\n        "Ġsh",\n        "r"\n      ],\n      [\n        "Ġsh",\n        "ining"\n      ],\n      [\n        "Ġta",\n        "kes"\n      ],\n      [\n        "Ġcloud",\n        "s"\n      ],\n      [\n        "Ġcoll",\n        "ect"\n      ],\n      [\n        "Ġapp",\n        "eared"\n      ],\n      [\n        "Ġbuild",\n        "ing"\n      ],\n      [\n        "Ġben",\n        "ch"\n      ],\n      [\n        "Ġh",\n        "oney"\n      ],\n      [\n        "Ġc",\n        "ard"\n      ],\n      [\n        "ar",\n        "ra"\n      ],\n      [\n        "Ġsp",\n        "icy"\n      ],\n      [\n        "Ġle",\n        "m"\n      ],\n      [\n        "Ġfore",\n        "ver"\n      ],\n      [\n        "Ġemb",\n        "arra"\n      ],\n      [\n        "Ġhelp",\n        "ful"\n      ],\n      [\n        "Ġpr",\n        "ot"\n      ],\n      [\n        "Ġqu",\n        "est"\n      ],\n      [\n        "J",\n        "ust"\n      ],\n      [\n        "Ġb",\n        "al"\n      ],\n      [\n        "Ġp",\n        "ool"\n      ],\n      [\n        "ce",\n        "an"\n      ],\n      [\n        "Ġas",\n        "h"\n      ],\n      [\n        "Ġsp",\n        "id"\n      ],\n      [\n        "Ġro",\n        "b"\n      ],\n      [\n        "Ġwr",\n        "ite"\n      ],\n      [\n        "Ġo",\n        "cean"\n      ],\n      [\n        "Ġe",\n        "ventually"\n      ],\n      [\n        "Ġspot",\n        "ted"\n      ],\n      [\n        "Ġresp",\n        "ect"\n      ],\n      [\n        "le",\n        "br"\n      ],\n      [\n        "omet",\n        "imes"\n      ],\n      [\n        "Ġhop",\n        "e"\n      ],\n      [\n        "Ġneighb",\n        "or"\n      ],\n      [\n        "Ġbott",\n        "le"\n      ],\n      [\n        "G",\n        "ood"\n      ],\n      [\n        "Ġl",\n        "ater"\n      ],\n      [\n        "Ġm",\n        "ach"\n      ],\n      [\n        "ar",\n        "ge"\n      ],\n      [\n        "Ġas",\n        "leep"\n      ],\n      [\n        "Ġfo",\n        "ld"\n      ],\n      [\n        "Ġce",\n        "lebr"\n      ],\n      [\n        "Ġmin",\n        "ut"\n      ],\n      [\n        "Ġplac",\n        "es"\n      ],\n      [\n        "Ġrob",\n        "ot"\n      ],\n      [\n        "Ġw",\n        "ild"\n      ],\n      [\n        "Ġp",\n        "hone"\n      ],\n      [\n        "Ġsa",\n        "f"\n      ],\n      [\n        "Ġsp",\n        "in"\n      ],\n      [\n        "us",\n        "ie"\n      ],\n      [\n        "Ġsear",\n        "ch"\n      ],\n      [\n        "Ġshar",\n        "p"\n      ],\n      [\n        "Ġprot",\n        "ect"\n      ],\n      [\n        "Ġl",\n        "ad"\n      ],\n      [\n        "Ġc",\n        "ow"\n      ],\n      [\n        "Ġme",\n        "et"\n      ],\n      [\n        "Ġcon",\n        "fused"\n      ],\n      [\n        "Ġte",\n        "ach"\n      ],\n      [\n        "Ġwear",\n        "ing"\n      ],\n      [\n        "Ġo",\n        "ven"\n      ],\n      [\n        "Ġsp",\n        "ent"\n      ],\n      [\n        "Ġrem",\n        "ind"\n      ],\n      [\n        "Ġansw",\n        "er"\n      ],\n      [\n        "n",\n        "n"\n      ],\n      [\n        "u",\n        "p"\n      ],\n      [\n        "Ġhe",\n        "althy"\n      ],\n      [\n        "Ġor",\n        "gan"\n      ],\n      [\n        "Ġtal",\n        "ked"\n      ],\n      [\n        "h",\n        "ile"\n      ],\n      [\n        "Ġa",\n        "nt"\n      ],\n      [\n        "is",\n        "a"\n      ],\n      [\n        "Ġsh",\n        "y"\n      ],\n      [\n        "Ġk",\n        "itten"\n      ],\n      [\n        "Ġfo",\n        "ot"\n      ],\n      [\n        "I",\n        "n"\n      ],\n      [\n        "N",\n        "ow"\n      ],\n      [\n        "Ġs",\n        "er"\n      ],\n      [\n        "Ġf",\n        "ing"\n      ],\n      [\n        "Ġm",\n        "il"\n      ],\n      [\n        "Ġcle",\n        "ver"\n      ],\n      [\n        "Ġfin",\n        "ish"\n      ],\n      [\n        "Ġrock",\n        "et"\n      ],\n      [\n        "ul",\n        "ar"\n      ],\n      [\n        "Ġprom",\n        "ise"\n      ],\n      [\n        "Ġcor",\n        "ner"\n      ],\n      [\n        "n",\n        "ed"\n      ],\n      [\n        "Ġe",\n        "gg"\n      ],\n      [\n        "Ġtal",\n        "king"\n      ],\n      [\n        "ĠEm",\n        "ily"\n      ],\n      [\n        "Ġwa",\n        "ve"\n      ],\n      [\n        "er",\n        "ly"\n      ],\n      [\n        "ĠW",\n        "ith"\n      ],\n      [\n        "Ġstor",\n        "m"\n      ],\n      [\n        "Ġwait",\n        "ing"\n      ],\n      [\n        "Ġp",\n        "ile"\n      ],\n      [\n        "el",\n        "s"\n      ],\n      [\n        "ĠC",\n        "h"\n      ],\n      [\n        "Ġembarra",\n        "ssed"\n      ],\n      [\n        "Ġlad",\n        "der"\n      ],\n      [\n        "Ġl",\n        "ay"\n      ],\n      [\n        "Ġl",\n        "arge"\n      ],\n      [\n        "Ġfe",\n        "ather"\n      ],\n      [\n        "if",\n        "e"\n      ],\n      [\n        "our",\n        "s"\n      ],\n      [\n        "ĠG",\n        "o"\n      ],\n      [\n        "Ġtri",\n        "p"\n      ],\n      [\n        "Ġspl",\n        "as"\n      ],\n      [\n        "E",\n        "very"\n      ],\n      [\n        "ĠJ",\n        "ake"\n      ],\n      [\n        "Ġmu",\n        "d"\n      ],\n      [\n        "Ġtrou",\n        "ble"\n      ],\n      [\n        "Ġb",\n        "lew"\n      ],\n      [\n        "Ġm",\n        "ind"\n      ],\n      [\n        "Ġre",\n        "c"\n      ],\n      [\n        "Ġsc",\n        "ar"\n      ],\n      [\n        "Ġv",\n        "al"\n      ],\n      [\n        "Ġpudd",\n        "le"\n      ],\n      [\n        "n",\n        "ic"\n      ],\n      [\n        "ĠJ",\n        "ust"\n      ],\n      [\n        "Ġpic",\n        "nic"\n      ],\n      [\n        "Ġbug",\n        "s"\n      ],\n      [\n        "et",\n        "e"\n      ],\n      [\n        "Ġfor",\n        "ward"\n      ],\n      [\n        "Ġfur",\n        "ry"\n      ],\n      [\n        "es",\n        "e"\n      ],\n      [\n        "Ġro",\n        "ad"\n      ],\n      [\n        "Ġso",\n        "l"\n      ],\n      [\n        "Ġpus",\n        "h"\n      ],\n      [\n        "anc",\n        "y"\n      ],\n      [\n        "Ġscream",\n        "ed"\n      ],\n      [\n        "d",\n        "en"\n      ],\n      [\n        "Ġmil",\n        "k"\n      ],\n      [\n        "S",\n        "t"\n      ],\n      [\n        "o",\n        "nd"\n      ],\n      [\n        "Ġche",\n        "er"\n      ],\n      [\n        "Ġsmo",\n        "oth"\n      ],\n      [\n        "Ġw",\n        "ide"\n      ],\n      [\n        "in",\n        "ted"\n      ],\n      [\n        "Ġh",\n        "ar"\n      ],\n      [\n        "Ġfe",\n        "et"\n      ],\n      [\n        "Ġz",\n        "oom"\n      ],\n      [\n        "ĠK",\n        "itty"\n      ],\n      [\n        "Ġmat",\n        "ter"\n      ],\n      [\n        "Ġmyster",\n        "ious"\n      ],\n      [\n        "e",\n        "b"\n      ],\n      [\n        "Ġ",\n        "ign"\n      ],\n      [\n        "Ġr",\n        "ope"\n      ],\n      [\n        "Ġwhen",\n        "ever"\n      ],\n      [\n        "Ġbre",\n        "ad"\n      ],\n      [\n        "Ġgent",\n        "ly"\n      ],\n      [\n        "ung",\n        "le"\n      ],\n      [\n        "Ġp",\n        "ar"\n      ],\n      [\n        "Ġj",\n        "ungle"\n      ],\n      [\n        "o",\n        "ld"\n      ],\n      [\n        "Ġp",\n        "ill"\n      ],\n      [\n        "Ġsa",\n        "l"\n      ],\n      [\n        "Ġlo",\n        "ves"\n      ],\n      [\n        "Ġsee",\n        "k"\n      ],\n      [\n        "Ġimp",\n        "re"\n      ],\n      [\n        "Ġapp",\n        "les"\n      ],\n      [\n        "Ġcou",\n        "ch"\n      ],\n      [\n        "Ġc",\n        "alm"\n      ],\n      [\n        "ck",\n        "s"\n      ],\n      [\n        "Ġco",\n        "at"\n      ],\n      [\n        "os",\n        "aur"\n      ],\n      [\n        "Ġsleep",\n        "y"\n      ],\n      [\n        "Ġsu",\n        "pp"\n      ],\n      [\n        "Ġbreat",\n        "h"\n      ],\n      [\n        "ow",\n        "ed"\n      ],\n      [\n        "Ġdisapp",\n        "o"\n      ],\n      [\n        "Ġm",\n        "ed"\n      ],\n      [\n        "ĠT",\n        "V"\n      ],\n      [\n        "le",\n        "x"\n      ],\n      [\n        "Ġst",\n        "ret"\n      ],\n      [\n        "pp",\n        "er"\n      ],\n      [\n        "Ġdri",\n        "ve"\n      ],\n      [\n        "ĠTo",\n        "gether"\n      ],\n      [\n        "h",\n        "in"\n      ],\n      [\n        "o",\n        "bby"\n      ],\n      [\n        "Ġc",\n        "oo"\n      ],\n      [\n        "Ġg",\n        "as"\n      ],\n      [\n        "get",\n        "able"\n      ],\n      [\n        "Ġla",\n        "st"\n      ],\n      [\n        "Ġhid",\n        "den"\n      ],\n      [\n        "Ġstand",\n        "ing"\n      ],\n      [\n        "Ġh",\n        "on"\n      ],\n      [\n        "Ġst",\n        "ood"\n      ],\n      [\n        "Ġse",\n        "at"\n      ],\n      [\n        "Ġcar",\n        "ried"\n      ],\n      [\n        "Ġsl",\n        "id"\n      ],\n      [\n        "Ġhid",\n        "ing"\n      ],\n      [\n        "Ġve",\n        "getable"\n      ],\n      [\n        "y",\n        "m"\n      ],\n      [\n        "ll",\n        "a"\n      ],\n      [\n        "Ġkne",\n        "e"\n      ],\n      [\n        "Ġfix",\n        "ed"\n      ],\n      [\n        "Ġpretend",\n        "ed"\n      ],\n      [\n        "Ġgi",\n        "ves"\n      ],\n      [\n        "Ġdir",\n        "t"\n      ],\n      [\n        "Ġlem",\n        "on"\n      ],\n      [\n        "Ġspid",\n        "er"\n      ],\n      [\n        "g",\n        "round"\n      ],\n      [\n        "Ġb",\n        "an"\n      ],\n      [\n        "Ġb",\n        "ark"\n      ],\n      [\n        "ĠS",\n        "ome"\n      ],\n      [\n        "Ġsp",\n        "e"\n      ],\n      [\n        "Ġqu",\n        "een"\n      ],\n      [\n        "Ġche",\n        "st"\n      ],\n      [\n        "ĠR",\n        "e"\n      ],\n      [\n        "Ġforg",\n        "ive"\n      ],\n      [\n        "Ġtast",\n        "e"\n      ],\n      [\n        "Ġsong",\n        "s"\n      ],\n      [\n        "Ġn",\n        "ut"\n      ],\n      [\n        "ĠB",\n        "e"\n      ],\n      [\n        "Ġsh",\n        "irt"\n      ],\n      [\n        "ul",\n        "t"\n      ],\n      [\n        "a",\n        "le"\n      ],\n      [\n        "m",\n        "o"\n      ],\n      [\n        "Ġst",\n        "ra"\n      ],\n      [\n        "Ġst",\n        "one"\n      ],\n      [\n        "Ġst",\n        "ep"\n      ],\n      [\n        "Ġyell",\n        "ed"\n      ],\n      [\n        "Ġduck",\n        "s"\n      ],\n      [\n        "Ġm",\n        "ot"\n      ],\n      [\n        "ĠS",\n        "ometimes"\n      ],\n      [\n        "Ġy",\n        "arn"\n      ],\n      [\n        "Ġgr",\n        "oup"\n      ],\n      [\n        "g",\n        "ly"\n      ],\n      [\n        "Ġf",\n        "ancy"\n      ],\n      [\n        "Ġwas",\n        "h"\n      ],\n      [\n        "Ġu",\n        "gly"\n      ],\n      [\n        "Ġdin",\n        "osaur"\n      ],\n      [\n        "J",\n        "ane"\n      ],\n      [\n        "M",\n        "ama"\n      ],\n      [\n        "at",\n        "o"\n      ],\n      [\n        "Ġtight",\n        "ly"\n      ],\n      [\n        "Ġro",\n        "ar"\n      ],\n      [\n        "Ġfa",\n        "ces"\n      ],\n      [\n        "Ġgre",\n        "w"\n      ],\n      [\n        "Ġru",\n        "de"\n      ],\n      [\n        "air",\n        "s"\n      ],\n      [\n        "Ġsuc",\n        "cess"\n      ],\n      [\n        "Ġdisappo",\n        "inted"\n      ],\n      [\n        "p",\n        "er"\n      ],\n      [\n        "er",\n        "y"\n      ],\n      [\n        "ig",\n        "hed"\n      ],\n      [\n        "ur",\n        "se"\n      ],\n      [\n        "Ġbec",\n        "ome"\n      ],\n      [\n        "Ġche",\n        "ese"\n      ],\n      [\n        "eter",\n        "m"\n      ],\n      [\n        "c",\n        "ap"\n      ],\n      [\n        "Ġt",\n        "est"\n      ],\n      [\n        "Ġp",\n        "le"\n      ],\n      [\n        "Ġst",\n        "ir"\n      ],\n      [\n        "Ġme",\n        "adow"\n      ],\n      [\n        "Ġbl",\n        "ow"\n      ],\n      [\n        "Ġpu",\n        "zz"\n      ],\n      [\n        "So",\n        "on"\n      ],\n      [\n        "Ġtom",\n        "orrow"\n      ],\n      [\n        "izz",\n        "a"\n      ],\n      [\n        "M",\n        "ummy"\n      ],\n      [\n        "Ġc",\n        "art"\n      ],\n      [\n        "Ġd",\n        "eterm"\n      ],\n      [\n        "Ġsh",\n        "ip"\n      ],\n      [\n        "Ġsp",\n        "read"\n      ],\n      [\n        "Ġv",\n        "ase"\n      ],\n      [\n        "Ġbu",\n        "bble"\n      ],\n      [\n        "Ġwhist",\n        "le"\n      ],\n      [\n        "W",\n        "here"\n      ],\n      [\n        "d",\n        "le"\n      ],\n      [\n        "Ġt",\n        "id"\n      ],\n      [\n        "Ġw",\n        "ip"\n      ],\n      [\n        "Ġf",\n        "ig"\n      ],\n      [\n        "Ġg",\n        "ar"\n      ],\n      [\n        "Ġg",\n        "en"\n      ],\n      [\n        "Ġsa",\n        "ve"\n      ],\n      [\n        "Ġch",\n        "ased"\n      ],\n      [\n        "Ġsn",\n        "ake"\n      ],\n      [\n        "Ġcle",\n        "ar"\n      ],\n      [\n        "ĠP",\n        "eter"\n      ],\n      [\n        "Ġcook",\n        "ie"\n      ],\n      [\n        "Ġmach",\n        "ine"\n      ],\n      [\n        "v",\n        "ous"\n      ],\n      [\n        "Ġs",\n        "our"\n      ],\n      [\n        "ĠH",\n        "ow"\n      ],\n      [\n        "Ġfl",\n        "ut"\n      ],\n      [\n        "Ġstr",\n        "ing"\n      ],\n      [\n        "ĠEvery",\n        "where"\n      ],\n      [\n        "Ġjump",\n        "ing"\n      ],\n      [\n        "Ġgi",\n        "ant"\n      ],\n      [\n        "a",\n        "king"\n      ],\n      [\n        "Ġs",\n        "elf"\n      ],\n      [\n        "Ġp",\n        "our"\n      ],\n      [\n        "Ġn",\n        "et"\n      ],\n      [\n        "ĠThe",\n        "ir"\n      ],\n      [\n        "ĠJohn",\n        "ny"\n      ],\n      [\n        "Ġstick",\n        "s"\n      ],\n      [\n        "Sara",\n        "h"\n      ],\n      [\n        "Ġg",\n        "un"\n      ],\n      [\n        "Ġshe",\n        "ll"\n      ],\n      [\n        "um",\n        "ber"\n      ],\n      [\n        "ĠE",\n        "ven"\n      ],\n      [\n        "Ġmat",\n        "ch"\n      ],\n      [\n        "Ġje",\n        "w"\n      ],\n      [\n        "\'",\n        "d"\n      ],\n      [\n        "Ġb",\n        "er"\n      ],\n      [\n        "Ġhe",\n        "l"\n      ],\n      [\n        "Ġre",\n        "g"\n      ],\n      [\n        "Ġch",\n        "ick"\n      ],\n      [\n        "ĠP",\n        "lease"\n      ],\n      [\n        "Ġcreat",\n        "ive"\n      ],\n      [\n        "Ġfight",\n        "ing"\n      ],\n      [\n        "Ġspark",\n        "ly"\n      ],\n      [\n        "u",\n        "al"\n      ],\n      [\n        "al",\n        "s"\n      ],\n      [\n        "?\\"",\n        "."\n      ],\n      [\n        "Ġem",\n        "p"\n      ],\n      [\n        "Ġpower",\n        "ful"\n      ],\n      [\n        "Ġcho",\n        "col"\n      ],\n      [\n        "Ġscar",\n        "f"\n      ],\n      [\n        "J",\n        "immy"\n      ],\n      [\n        "Ġt",\n        "eet"\n      ],\n      [\n        "Ġh",\n        "ours"\n      ],\n      [\n        "Ġm",\n        "ountain"\n      ],\n      [\n        "ĠS",\n        "usie"\n      ],\n      [\n        "ĠM",\n        "ittens"\n      ],\n      [\n        "Ġex",\n        "am"\n      ],\n      [\n        "Ġkn",\n        "ocked"\n      ],\n      [\n        "Ġdan",\n        "cing"\n      ],\n      [\n        "Ġmot",\n        "or"\n      ],\n      [\n        "Ġteet",\n        "h"\n      ],\n      [\n        "Ġw",\n        "ore"\n      ],\n      [\n        "Ġp",\n        "izza"\n      ],\n      [\n        "Ġn",\n        "er"\n      ],\n      [\n        "Ġbe",\n        "ll"\n      ],\n      [\n        "Ġme",\n        "ant"\n      ],\n      [\n        "Ġdec",\n        "or"\n      ],\n      [\n        "Ġneck",\n        "l"\n      ],\n      [\n        "Ġban",\n        "an"\n      ],\n      [\n        "Ġemp",\n        "ty"\n      ],\n      [\n        "D",\n        "ad"\n      ],\n      [\n        "Ġhe",\n        "ars"\n      ],\n      [\n        "er",\n        "a"\n      ],\n      [\n        "Ġha",\n        "ng"\n      ],\n      [\n        "Ġtr",\n        "ust"\n      ],\n      [\n        "Ġbar",\n        "ked"\n      ],\n      [\n        "Ġthe",\n        "se"\n      ],\n      [\n        "Ġdre",\n        "w"\n      ],\n      [\n        "Ġcra",\n        "b"\n      ],\n      [\n        "Ġbath",\n        "room"\n      ],\n      [\n        "Ġplant",\n        "s"\n      ],\n      [\n        "Ġpeace",\n        "ful"\n      ],\n      [\n        "Ġgar",\n        "age"\n      ],\n      [\n        "Ġchocol",\n        "ate"\n      ],\n      [\n        "Ġner",\n        "vous"\n      ],\n      [\n        "P",\n        "lease"\n      ],\n      [\n        "l",\n        "ight"\n      ],\n      [\n        "Ġ",\n        "On"\n      ],\n      [\n        "Ġp",\n        "op"\n      ],\n      [\n        "itt",\n        "er"\n      ],\n      [\n        "Ġgo",\n        "es"\n      ],\n      [\n        "Ġspl",\n        "ash"\n      ],\n      [\n        "Ġash",\n        "amed"\n      ],\n      [\n        "Ġself",\n        "ish"\n      ],\n      [\n        "n",\n        "ot"\n      ],\n      [\n        "y",\n        "ing"\n      ],\n      [\n        "Ġm",\n        "ic"\n      ],\n      [\n        "Ġr",\n        "id"\n      ],\n      [\n        "ce",\n        "pt"\n      ],\n      [\n        "ĠJ",\n        "im"\n      ],\n      [\n        "Ġansw",\n        "ered"\n      ],\n      [\n        "Ġgigg",\n        "led"\n      ],\n      [\n        "Ġp",\n        "ri"\n      ],\n      [\n        "om",\n        "an"\n      ],\n      [\n        "an",\n        "ut"\n      ],\n      [\n        "ĠB",\n        "r"\n      ],\n      [\n        "The",\n        "re"\n      ],\n      [\n        "Ġkn",\n        "ight"\n      ],\n      [\n        "iz",\n        "zy"\n      ],\n      [\n        "Ġpol",\n        "ite"\n      ],\n      [\n        "Ġshar",\n        "ing"\n      ],\n      [\n        "T",\n        "o"\n      ],\n      [\n        "Ġw",\n        "oman"\n      ],\n      [\n        "Ġd",\n        "i"\n      ],\n      [\n        "Ġcan",\n        "not"\n      ],\n      [\n        "Ġsw",\n        "an"\n      ],\n      [\n        "Ġche",\n        "w"\n      ],\n      [\n        "ĠP",\n        "ete"\n      ],\n      [\n        "Ġz",\n        "oo"\n      ],\n      [\n        "Ġpa",\n        "w"\n      ],\n      [\n        "Ġeas",\n        "y"\n      ],\n      [\n        "mo",\n        "st"\n      ],\n      [\n        "Ġc",\n        "ap"\n      ],\n      [\n        "Ġm",\n        "aybe"\n      ],\n      [\n        "ĠS",\n        "t"\n      ],\n      [\n        "Ġshe",\n        "lf"\n      ],\n      [\n        "Ġli",\n        "ve"\n      ],\n      [\n        "Ġal",\n        "most"\n      ],\n      [\n        "Ġgen",\n        "er"\n      ],\n      [\n        "J",\n        "ill"\n      ],\n      [\n        "as",\n        "er"\n      ],\n      [\n        "Ġho",\n        "pping"\n      ],\n      [\n        "bo",\n        "x"\n      ],\n      [\n        "Ġh",\n        "ur"\n      ],\n      [\n        "it",\n        "al"\n      ],\n      [\n        "ĠL",\n        "isa"\n      ],\n      [\n        "Ġho",\n        "sp"\n      ],\n      [\n        "Ġknow",\n        "ing"\n      ],\n      [\n        "Ġsk",\n        "ipped"\n      ],\n      [\n        "Ġgr",\n        "ay"\n      ],\n      [\n        "Ġland",\n        "ed"\n      ],\n      [\n        "Ġcarrot",\n        "s"\n      ],\n      [\n        "Ġspr",\n        "ay"\n      ],\n      [\n        "Ġt",\n        "ag"\n      ],\n      [\n        "Ġp",\n        "in"\n      ],\n      [\n        "Ġplay",\n        "ground"\n      ],\n      [\n        "Ġwh",\n        "is"\n      ],\n      [\n        "Ġlo",\n        "ck"\n      ],\n      [\n        "Ġput",\n        "s"\n      ],\n      [\n        "Ġcir",\n        "c"\n      ],\n      [\n        "Ġneckl",\n        "ace"\n      ],\n      [\n        "Ġhosp",\n        "ital"\n      ],\n      [\n        "Ġc",\n        "and"\n      ],\n      [\n        "Ġpr",\n        "ay"\n      ],\n      [\n        "Ġfl",\n        "ag"\n      ],\n      [\n        "ĠD",\n        "on"\n      ],\n      [\n        "Ġb",\n        "one"\n      ],\n      [\n        "Ġm",\n        "ummy"\n      ],\n      [\n        "Ġte",\n        "a"\n      ],\n      [\n        "Ġmiss",\n        "ed"\n      ],\n      [\n        "Ġbutt",\n        "on"\n      ],\n      [\n        "Ġgener",\n        "ous"\n      ],\n      [\n        "B",\n        "e"\n      ],\n      [\n        "Ġc",\n        "age"\n      ],\n      [\n        "Ġof",\n        "ten"\n      ],\n      [\n        "ĠM",\n        "um"\n      ],\n      [\n        "Ġle",\n        "ad"\n      ],\n      [\n        "ĠW",\n        "hy"\n      ],\n      [\n        "Ġexc",\n        "ite"\n      ],\n      [\n        "Ġpo",\n        "or"\n      ],\n      [\n        "Ġbed",\n        "room"\n      ],\n      [\n        "Ġunderst",\n        "ood"\n      ],\n      [\n        "Ġdisapp",\n        "eared"\n      ],\n      [\n        "Ġtreat",\n        "s"\n      ],\n      [\n        "Ġminut",\n        "es"\n      ],\n      [\n        "St",\n        "op"\n      ],\n      [\n        "Ġf",\n        "il"\n      ],\n      [\n        "Ġpl",\n        "ate"\n      ],\n      [\n        "Ġreturn",\n        "ed"\n      ],\n      [\n        "Ġtear",\n        "s"\n      ],\n      [\n        "Ġsandw",\n        "ich"\n      ],\n      [\n        "r",\n        "ated"\n      ],\n      [\n        "Ġh",\n        "um"\n      ],\n      [\n        "Ġl",\n        "it"\n      ],\n      [\n        "Ġl",\n        "ine"\n      ],\n      [\n        "Ġas",\n        "king"\n      ],\n      [\n        "Ġtoo",\n        "l"\n      ],\n      [\n        "Ġfe",\n        "ar"\n      ],\n      [\n        "Ġwor",\n        "m"\n      ],\n      [\n        "Ġmark",\n        "et"\n      ],\n      [\n        "Ġexcite",\n        "ment"\n      ],\n      [\n        "Ġs",\n        "il"\n      ],\n      [\n        "Ġwa",\n        "nder"\n      ],\n      [\n        "on",\n        "es"\n      ],\n      [\n        "Ġgo",\n        "ld"\n      ],\n      [\n        "us",\n        "ual"\n      ],\n      [\n        "eci",\n        "ally"\n      ],\n      [\n        "Ġun",\n        "usual"\n      ],\n      [\n        "Jo",\n        "e"\n      ],\n      [\n        "Ġsnack",\n        "s"\n      ],\n      [\n        "Ġspo",\n        "il"\n      ],\n      [\n        "Ġbal",\n        "ance"\n      ],\n      [\n        "Ġtid",\n        "y"\n      ],\n      [\n        "p",\n        "ecially"\n      ],\n      [\n        "u",\n        "b"\n      ],\n      [\n        "Ġm",\n        "id"\n      ],\n      [\n        "Ġsa",\n        "ved"\n      ],\n      [\n        "or",\n        "ry"\n      ],\n      [\n        "Ġbo",\n        "ss"\n      ],\n      [\n        "ill",\n        "i"\n      ],\n      [\n        "Ġtr",\n        "ue"\n      ],\n      [\n        "Ġgl",\n        "ow"\n      ],\n      [\n        "Ġes",\n        "pecially"\n      ],\n      [\n        "Ġgather",\n        "ed"\n      ],\n      [\n        "Ġdeterm",\n        "ined"\n      ],\n      [\n        "G",\n        "ra"\n      ],\n      [\n        "c",\n        "y"\n      ],\n      [\n        "f",\n        "l"\n      ],\n      [\n        "ig",\n        "er"\n      ],\n      [\n        "Ġme",\n        "asure"\n      ],\n      [\n        "Ġfr",\n        "ust"\n      ],\n      [\n        "Ġjo",\n        "ke"\n      ],\n      [\n        "Ġdr",\n        "um"\n      ],\n      [\n        "Ġvegetable",\n        "s"\n      ],\n      [\n        "Ġf",\n        "ather"\n      ],\n      [\n        "Ġc",\n        "ross"\n      ],\n      [\n        "Ġfe",\n        "ed"\n      ],\n      [\n        "Ġpe",\n        "anut"\n      ],\n      [\n        "Ġdraw",\n        "ing"\n      ],\n      [\n        "Ġsmell",\n        "y"\n      ],\n      [\n        "Ġperf",\n        "orm"\n      ],\n      [\n        "Ġpass",\n        "ed"\n      ],\n      [\n        "Ġ",\n        "ed"\n      ],\n      [\n        "at",\n        "ing"\n      ],\n      [\n        "or",\n        "k"\n      ],\n      [\n        "Ġwe",\n        "ak"\n      ],\n      [\n        "Ġwe",\n        "ek"\n      ],\n      [\n        "Ġyour",\n        "self"\n      ],\n      [\n        "Ġta",\n        "ught"\n      ],\n      [\n        "Ġtell",\n        "s"\n      ],\n      [\n        "ens",\n        "ive"\n      ],\n      [\n        "Ġexp",\n        "ensive"\n      ],\n      [\n        "Ġcolour",\n        "ful"\n      ],\n      [\n        "Ġmid",\n        "dle"\n      ],\n      [\n        "p",\n        "ort"\n      ],\n      [\n        "un",\n        "k"\n      ],\n      [\n        "ump",\n        "y"\n      ],\n      [\n        "Ġlea",\n        "f"\n      ],\n      [\n        "Ġthank",\n        "ful"\n      ],\n      [\n        "Ġes",\n        "cap"\n      ],\n      [\n        "S",\n        "ue"\n      ],\n      [\n        "p",\n        "ack"\n      ],\n      [\n        "Ġa",\n        "v"\n      ],\n      [\n        "Ġp",\n        "ir"\n      ],\n      [\n        "ĠM",\n        "ummy"\n      ],\n      [\n        "Ġcl",\n        "ap"\n      ],\n      [\n        "ell",\n        "y"\n      ],\n      [\n        "Ġpol",\n        "ice"\n      ],\n      [\n        "Ġfrust",\n        "rated"\n      ],\n      [\n        "Ġa",\n        "dd"\n      ],\n      [\n        "Ġwa",\n        "ves"\n      ],\n      [\n        "Ġwa",\n        "gged"\n      ],\n      [\n        "an",\n        "o"\n      ],\n      [\n        "et",\n        "s"\n      ],\n      [\n        "Ġsh",\n        "ore"\n      ],\n      [\n        "Ġsc",\n        "r"\n      ],\n      [\n        "ĠL",\n        "e"\n      ],\n      [\n        "Ġla",\n        "zy"\n      ],\n      [\n        "b",\n        "in"\n      ],\n      [\n        "m",\n        "b"\n      ],\n      [\n        "p",\n        "s"\n      ],\n      [\n        "Ġt",\n        "ur"\n      ],\n      [\n        "Ġbo",\n        "ard"\n      ],\n      [\n        "ĠA",\n        "nn"\n      ],\n      [\n        "Ġtw",\n        "irl"\n      ],\n      [\n        "Ġmo",\n        "on"\n      ],\n      [\n        "Ġmo",\n        "ving"\n      ],\n      [\n        "Åĵ",\n        "Let"\n      ],\n      [\n        "g",\n        "y"\n      ],\n      [\n        "Ġm",\n        "ir"\n      ],\n      [\n        "Ġst",\n        "uff"\n      ],\n      [\n        "Ġco",\n        "vered"\n      ],\n      [\n        "Ġcall",\n        "ing"\n      ],\n      [\n        "Ġwalk",\n        "s"\n      ],\n      [\n        "S",\n        "ally"\n      ],\n      [\n        "u",\n        "ce"\n      ],\n      [\n        "w",\n        "ork"\n      ],\n      [\n        "Ġb",\n        "oring"\n      ],\n      [\n        "Ġd",\n        "epend"\n      ],\n      [\n        "er",\n        "p"\n      ],\n      [\n        "qu",\n        "e"\n      ],\n      [\n        "Ġta",\n        "ken"\n      ],\n      [\n        "Ġde",\n        "er"\n      ],\n      [\n        "Ġmar",\n        "ble"\n      ],\n      [\n        "Ġsail",\n        "ed"\n      ],\n      [\n        "W",\n        "ho"\n      ],\n      [\n        "r",\n        "or"\n      ],\n      [\n        "Ġh",\n        "orn"\n      ],\n      [\n        "ĠT",\n        "eddy"\n      ],\n      [\n        "ar",\n        "l"\n      ],\n      [\n        "Ġcl",\n        "oth"\n      ],\n      [\n        "Ġdog",\n        "s"\n      ],\n      [\n        "Ġche",\n        "ck"\n      ],\n      [\n        "Ġcont",\n        "ent"\n      ],\n      [\n        "Ġpoin",\n        "ting"\n      ],\n      [\n        "ĠTweet",\n        "y"\n      ],\n      [\n        "E",\n        "m"\n      ],\n      [\n        "c",\n        "hes"\n      ],\n      [\n        "i",\n        "an"\n      ],\n      [\n        "Ġt",\n        "ick"\n      ],\n      [\n        "Ġp",\n        "il"\n      ],\n      [\n        "Ġo",\n        "k"\n      ],\n      [\n        "Ġth",\n        "ick"\n      ],\n      [\n        "ĠA",\n        "t"\n      ],\n      [\n        "iss",\n        "ors"\n      ],\n      [\n        "Ġru",\n        "ined"\n      ],\n      [\n        "Ġpill",\n        "ow"\n      ],\n      [\n        "Ġjew",\n        "el"\n      ],\n      [\n        "H",\n        "ow"\n      ],\n      [\n        "Ġt",\n        "er"\n      ],\n      [\n        "Ġb",\n        "urn"\n      ],\n      [\n        "Ġb",\n        "ored"\n      ],\n      [\n        "ot",\n        "s"\n      ],\n      [\n        "Ġan",\n        "g"\n      ],\n      [\n        "ill",\n        "ed"\n      ],\n      [\n        "Ġfl",\n        "ash"\n      ],\n      [\n        "Ġsweet",\n        "ie"\n      ],\n      [\n        "Ġtun",\n        "n"\n      ],\n      [\n        "Ġc",\n        "op"\n      ],\n      [\n        "or",\n        "ge"\n      ],\n      [\n        "on",\n        "y"\n      ],\n      [\n        "Ġwe",\n        "ird"\n      ],\n      [\n        "Ġj",\n        "am"\n      ],\n      [\n        "Ġcl",\n        "own"\n      ],\n      [\n        "Ġno",\n        "ds"\n      ],\n      [\n        "Ġsc",\n        "issors"\n      ],\n      [\n        "if",\n        "f"\n      ],\n      [\n        "Ġfl",\n        "uffy"\n      ],\n      [\n        "us",\n        "hed"\n      ],\n      [\n        "Ġpain",\n        "ting"\n      ],\n      [\n        "Ġp",\n        "an"\n      ],\n      [\n        "al",\n        "ous"\n      ],\n      [\n        "ur",\n        "ed"\n      ],\n      [\n        "Ġbu",\n        "ll"\n      ],\n      [\n        "Ġcr",\n        "ack"\n      ],\n      [\n        "Ġtri",\n        "es"\n      ],\n      [\n        "Ġje",\n        "alous"\n      ],\n      [\n        "Ġmist",\n        "ake"\n      ],\n      [\n        "Ġhon",\n        "est"\n      ],\n      [\n        "Ġple",\n        "ased"\n      ],\n      [\n        "Ġfig",\n        "ure"\n      ],\n      [\n        "A",\n        "re"\n      ],\n      [\n        "B",\n        "illy"\n      ],\n      [\n        "Ġt",\n        "ask"\n      ],\n      [\n        "Ġs",\n        "ighed"\n      ],\n      [\n        "Ġc",\n        "ri"\n      ],\n      [\n        "im",\n        "ed"\n      ],\n      [\n        "Ġdo",\n        "lp"\n      ],\n      [\n        "Ġman",\n        "ag"\n      ],\n      [\n        "as",\n        "ing"\n      ],\n      [\n        "Ġclos",\n        "et"\n      ],\n      [\n        "Ġmar",\n        "ch"\n      ],\n      [\n        "Ġpen",\n        "ny"\n      ],\n      [\n        "Ġca",\n        "bin"\n      ],\n      [\n        "Ġhappen",\n        "ing"\n      ],\n      [\n        "Ġdist",\n        "ance"\n      ],\n      [\n        "Ġinc",\n        "red"\n      ],\n      [\n        "Ġcomfort",\n        "able"\n      ],\n      [\n        "Ġsplas",\n        "hed"\n      ],\n      [\n        "s",\n        "el"\n      ],\n      [\n        "in",\n        "ess"\n      ],\n      [\n        "Ġso",\n        "ap"\n      ],\n      [\n        "Ġlo",\n        "g"\n      ],\n      [\n        "Ġsp",\n        "end"\n      ],\n      [\n        "Ġcl",\n        "ass"\n      ],\n      [\n        "Ġbr",\n        "idge"\n      ],\n      [\n        "Ġjo",\n        "g"\n      ],\n      [\n        "Ġdr",\n        "ank"\n      ],\n      [\n        "Ġdepend",\n        "able"\n      ],\n      [\n        "sel",\n        "ves"\n      ],\n      [\n        "Ġ",\n        "Z"\n      ],\n      [\n        "re",\n        "en"\n      ],\n      [\n        "Ġc",\n        "her"\n      ],\n      [\n        "il",\n        "t"\n      ],\n      [\n        "at",\n        "or"\n      ],\n      [\n        "Ġr",\n        "ing"\n      ],\n      [\n        "Ġwor",\n        "king"\n      ],\n      [\n        "Åĵ",\n        "That"\n      ],\n      [\n        "Ġgl",\n        "ue"\n      ],\n      [\n        "Ġstrong",\n        "er"\n      ],\n      [\n        "Ġmess",\n        "y"\n      ],\n      [\n        "ĠMr",\n        "s"\n      ],\n      [\n        "Ġleg",\n        "s"\n      ],\n      [\n        "illi",\n        "ant"\n      ],\n      [\n        "d",\n        "ge"\n      ],\n      [\n        "e",\n        "orge"\n      ],\n      [\n        "t",\n        "i"\n      ],\n      [\n        "Ġw",\n        "is"\n      ],\n      [\n        "in",\n        "ation"\n      ],\n      [\n        "Ġl",\n        "ying"\n      ],\n      [\n        "Ġd",\n        "est"\n      ],\n      [\n        "Ġr",\n        "ough"\n      ],\n      [\n        "Ġkn",\n        "ife"\n      ],\n      [\n        "Ġbr",\n        "illiant"\n      ],\n      [\n        "est",\n        "ed"\n      ],\n      [\n        "Ġfr",\n        "own"\n      ],\n      [\n        "Ġreal",\n        "ised"\n      ],\n      [\n        "Ġtow",\n        "el"\n      ],\n      [\n        "ĠR",\n        "ed"\n      ],\n      [\n        "Ġac",\n        "cept"\n      ],\n      [\n        "Ġmight",\n        "y"\n      ],\n      [\n        "Ġeag",\n        "er"\n      ],\n      [\n        "Ġmir",\n        "ror"\n      ],\n      [\n        "Ġter",\n        "ri"\n      ],\n      [\n        "W",\n        "ell"\n      ],\n      [\n        "j",\n        "ect"\n      ],\n      [\n        "Ġc",\n        "ries"\n      ],\n      [\n        "oo",\n        "f"\n      ],\n      [\n        "ke",\n        "ep"\n      ],\n      [\n        "Ġcat",\n        "erp"\n      ],\n      [\n        "Ġcr",\n        "own"\n      ],\n      [\n        "Ġfing",\n        "er"\n      ],\n      [\n        "Ġed",\n        "ge"\n      ],\n      [\n        "Ġcaterp",\n        "ill"\n      ],\n      [\n        "S",\n        "orry"\n      ],\n      [\n        "l",\n        "and"\n      ],\n      [\n        "Ġs",\n        "ight"\n      ],\n      [\n        "Ġwa",\n        "gon"\n      ],\n      [\n        "Ġp",\n        "ay"\n      ],\n      [\n        "Ġwe",\n        "althy"\n      ],\n      [\n        "Ġget",\n        "s"\n      ],\n      [\n        "Ġch",\n        "oose"\n      ],\n      [\n        "Ġsqu",\n        "are"\n      ],\n      [\n        "Ġcra",\n        "w"\n      ],\n      [\n        "Ġsaf",\n        "ely"\n      ],\n      [\n        "M",\n        "e"\n      ],\n      [\n        "Ġf",\n        "ake"\n      ],\n      [\n        "Ġbo",\n        "dy"\n      ],\n      [\n        "ie",\n        "f"\n      ],\n      [\n        "Åĵ",\n        "It"\n      ],\n      [\n        "Ġdisco",\n        "vered"\n      ],\n      [\n        "Ġcaterpill",\n        "ar"\n      ],\n      [\n        "Ġcraw",\n        "led"\n      ],\n      [\n        "in",\n        "a"\n      ],\n      [\n        "Ġb",\n        "itter"\n      ],\n      [\n        "Ġwa",\n        "nd"\n      ],\n      [\n        "Ġg",\n        "ate"\n      ],\n      [\n        "Ġr",\n        "are"\n      ],\n      [\n        "Ġon",\n        "es"\n      ],\n      [\n        "ĠA",\n        "lice"\n      ],\n      [\n        "Ġbr",\n        "ush"\n      ],\n      [\n        "ĠE",\n        "nd"\n      ],\n      [\n        "au",\n        "l"\n      ],\n      [\n        "Ġsmall",\n        "er"\n      ],\n      [\n        "umb",\n        "ers"\n      ],\n      [\n        "Ġkiss",\n        "ed"\n      ],\n      [\n        "Ġsuccess",\n        "ful"\n      ],\n      [\n        "keep",\n        "er"\n      ],\n      [\n        "ĠI",\n        "f"\n      ],\n      [\n        "Ġno",\n        "d"\n      ],\n      [\n        "op",\n        "h"\n      ],\n      [\n        "Ġexcited",\n        "ly"\n      ],\n      [\n        "Ġch",\n        "ange"\n      ],\n      [\n        "Ġpe",\n        "e"\n      ],\n      [\n        "Ġpick",\n        "ing"\n      ],\n      [\n        "Ġneed",\n        "s"\n      ],\n      [\n        "Ġclean",\n        "ed"\n      ],\n      [\n        "Ġpre",\n        "p"\n      ],\n      [\n        "ee",\n        "z"\n      ],\n      [\n        "ĠBr",\n        "own"\n      ],\n      [\n        "Ġincred",\n        "ible"\n      ],\n      [\n        "A",\n        "my"\n      ],\n      [\n        "J",\n        "en"\n      ],\n      [\n        "a",\n        "w"\n      ],\n      [\n        "Ġ",\n        "er"\n      ],\n      [\n        "nd",\n        "pa"\n      ],\n      [\n        "at",\n        "ely"\n      ],\n      [\n        "Ġan",\n        "c"\n      ],\n      [\n        "Ġta",\n        "pe"\n      ],\n      [\n        "ct",\n        "us"\n      ],\n      [\n        "Ġfam",\n        "ous"\n      ],\n      [\n        "Ġhurt",\n        "s"\n      ],\n      [\n        "Ġmix",\n        "ed"\n      ],\n      [\n        "Ġhar",\n        "m"\n      ],\n      [\n        "Ġpuzz",\n        "le"\n      ],\n      [\n        "a",\n        "by"\n      ],\n      [\n        "h",\n        "ip"\n      ],\n      [\n        "k",\n        "in"\n      ],\n      [\n        "Ġs",\n        "ugg"\n      ],\n      [\n        "Ġhe",\n        "al"\n      ],\n      [\n        "Ġst",\n        "ones"\n      ],\n      [\n        "Ġso",\n        "ld"\n      ],\n      [\n        "Ġfor",\n        "k"\n      ],\n      [\n        "Ġexc",\n        "la"\n      ],\n      [\n        "Åĵ",\n        "What"\n      ],\n      [\n        "Ġor",\n        "d"\n      ],\n      [\n        "Ġwo",\n        "lf"\n      ],\n      [\n        "Ġey",\n        "e"\n      ],\n      [\n        "Ġz",\n        "ipp"\n      ],\n      [\n        "Ġprin",\n        "ce"\n      ],\n      [\n        "Ġdes",\n        "k"\n      ],\n      [\n        "Ġpack",\n        "ed"\n      ],\n      [\n        "ĠTweet",\n        "ie"\n      ],\n      [\n        "Ġbelong",\n        "ed"\n      ],\n      [\n        "ĠLe",\n        "o"\n      ],\n      [\n        "Ġanc",\n        "ient"\n      ],\n      [\n        "Ġb",\n        "ur"\n      ],\n      [\n        "Ġn",\n        "umbers"\n      ],\n      [\n        "im",\n        "p"\n      ],\n      [\n        "Ġe",\n        "ars"\n      ],\n      [\n        "Ġon",\n        "ce"\n      ],\n      [\n        "Ġse",\n        "ed"\n      ],\n      [\n        "Ġbr",\n        "anc"\n      ],\n      [\n        "Ġmo",\n        "d"\n      ],\n      [\n        "Ġte",\n        "am"\n      ],\n      [\n        "Ġprin",\n        "t"\n      ],\n      [\n        "umb",\n        "led"\n      ],\n      [\n        "Ġca",\n        "ctus"\n      ],\n      [\n        "Ġwhe",\n        "el"\n      ],\n      [\n        "Ġexcla",\n        "imed"\n      ],\n      [\n        "Ġbranc",\n        "hes"\n      ],\n      [\n        "i",\n        "que"\n      ],\n      [\n        "Ġ",\n        "OK"\n      ],\n      [\n        "Ġt",\n        "ie"\n      ],\n      [\n        "Ġt",\n        "ank"\n      ],\n      [\n        "re",\n        "ady"\n      ],\n      [\n        "at",\n        "s"\n      ],\n      [\n        "or",\n        "y"\n      ],\n      [\n        "Ġso",\n        "ar"\n      ],\n      [\n        "!\\"",\n        "."\n      ],\n      [\n        "Ġal",\n        "ready"\n      ],\n      [\n        "Ġro",\n        "de"\n      ],\n      [\n        "Ġwor",\n        "d"\n      ],\n      [\n        "Ġla",\n        "w"\n      ],\n      [\n        "Ġpe",\n        "bble"\n      ],\n      [\n        "Åĵ",\n        "Yes"\n      ],\n      [\n        "Ġgl",\n        "ass"\n      ],\n      [\n        "Ġun",\n        "ique"\n      ],\n      [\n        "Ġuse",\n        "ful"\n      ],\n      [\n        "Ġdr",\n        "op"\n      ],\n      [\n        "Ġwall",\n        "s"\n      ],\n      [\n        "Ġpract",\n        "ice"\n      ],\n      [\n        "Ġdelight",\n        "ed"\n      ],\n      [\n        "Ġimpre",\n        "ssed"\n      ],\n      [\n        "ÅĵLet",\n        "Ã¢"\n      ],\n      [\n        "Ġmanag",\n        "ed"\n      ],\n      [\n        "f",\n        "ic"\n      ],\n      [\n        "Ġt",\n        "iger"\n      ],\n      [\n        "Ġa",\n        "dded"\n      ],\n      [\n        "Ġf",\n        "ool"\n      ],\n      [\n        "Ġd",\n        "ish"\n      ],\n      [\n        "Ġn",\n        "umber"\n      ],\n      [\n        "ad",\n        "i"\n      ],\n      [\n        "ĠB",\n        "l"\n      ],\n      [\n        "um",\n        "sy"\n      ],\n      [\n        "ĠD",\n        "ucky"\n      ],\n      [\n        "Ġball",\n        "s"\n      ],\n      [\n        "ob",\n        "y"\n      ],\n      [\n        "ber",\n        "ry"\n      ],\n      [\n        "ĠBob",\n        "o"\n      ],\n      [\n        "Ġdolp",\n        "hin"\n      ],\n      [\n        "c",\n        "il"\n      ],\n      [\n        "h",\n        "a"\n      ],\n      [\n        "Ġa",\n        "w"\n      ],\n      [\n        "ed",\n        "ient"\n      ],\n      [\n        "Ġto",\n        "ug"\n      ],\n      [\n        "Ġb",\n        "ake"\n      ],\n      [\n        "Ġc",\n        "one"\n      ],\n      [\n        "an",\n        "ger"\n      ],\n      [\n        "ĠB",\n        "et"\n      ],\n      [\n        "ĠB",\n        "obby"\n      ],\n      [\n        "art",\n        "h"\n      ],\n      [\n        "Ġco",\n        "ver"\n      ],\n      [\n        "Ġkind",\n        "s"\n      ],\n      [\n        "Ġkind",\n        "ness"\n      ],\n      [\n        "Ġbus",\n        "hes"\n      ],\n      [\n        "Ġpen",\n        "cil"\n      ],\n      [\n        "Ġbott",\n        "om"\n      ],\n      [\n        "Ġfruit",\n        "s"\n      ],\n      [\n        "Ġgrand",\n        "pa"\n      ],\n      [\n        "Ġwra",\n        "pped"\n      ],\n      [\n        "Ġmed",\n        "ic"\n      ],\n      [\n        "Ġstra",\n        "w"\n      ],\n      [\n        "Ġboss",\n        "y"\n      ],\n      [\n        "v",\n        "ice"\n      ],\n      [\n        "Ġs",\n        "ug"\n      ],\n      [\n        "Ġh",\n        "unt"\n      ],\n      [\n        "Ġboy",\n        "s"\n      ],\n      [\n        "ag",\n        "es"\n      ],\n      [\n        "Ġdec",\n        "ide"\n      ],\n      [\n        "Ġab",\n        "ove"\n      ],\n      [\n        "Ġexpl",\n        "ored"\n      ],\n      [\n        "Ġadvent",\n        "ur"\n      ],\n      [\n        "Ġenjoy",\n        "ing"\n      ],\n      [\n        "Ġdif",\n        "fic"\n      ],\n      [\n        "Ġpain",\n        "ted"\n      ],\n      [\n        "Ġde",\n        "ad"\n      ],\n      [\n        "Ġgi",\n        "ving"\n      ],\n      [\n        "Ġtunn",\n        "el"\n      ],\n      [\n        "Ġmedic",\n        "ine"\n      ],\n      [\n        "u",\n        "able"\n      ],\n      [\n        "Ġt",\n        "ent"\n      ],\n      [\n        "Ġg",\n        "oose"\n      ],\n      [\n        "at",\n        "es"\n      ],\n      [\n        "Ġe",\n        "ld"\n      ],\n      [\n        "el",\n        "p"\n      ],\n      [\n        "ĠB",\n        "u"\n      ],\n      [\n        "Ġgra",\n        "ce"\n      ],\n      [\n        "Ġen",\n        "vel"\n      ],\n      [\n        "ip",\n        "e"\n      ],\n      [\n        "Ġob",\n        "edient"\n      ],\n      [\n        "Ġremind",\n        "ed"\n      ],\n      [\n        "Ġflash",\n        "light"\n      ],\n      [\n        "Ġfool",\n        "ish"\n      ],\n      [\n        "Ġadventur",\n        "ous"\n      ],\n      [\n        "g",\n        "u"\n      ],\n      [\n        "o",\n        "e"\n      ],\n      [\n        "ot",\n        "e"\n      ],\n      [\n        "Ġse",\n        "ll"\n      ],\n      [\n        "Ġpo",\n        "st"\n      ],\n      [\n        "Ġfi",\n        "er"\n      ],\n      [\n        "ee",\n        "ze"\n      ],\n      [\n        "Ġgu",\n        "ess"\n      ],\n      [\n        "ol",\n        "og"\n      ],\n      [\n        "Ġsmo",\n        "ke"\n      ],\n      [\n        "Ġboun",\n        "ce"\n      ],\n      [\n        "Ġquest",\n        "ions"\n      ],\n      [\n        "Ġval",\n        "uable"\n      ],\n      [\n        "Ġterri",\n        "ble"\n      ],\n      [\n        "Ġdiffic",\n        "ult"\n      ],\n      [\n        "w",\n        "ards"\n      ],\n      [\n        "Ġ",\n        "icy"\n      ],\n      [\n        "Ġb",\n        "orrow"\n      ],\n      [\n        "Ġf",\n        "is"\n      ],\n      [\n        "Ġc",\n        "am"\n      ],\n      [\n        "at",\n        "ure"\n      ],\n      [\n        "Ġch",\n        "ar"\n      ],\n      [\n        "Ġtr",\n        "ump"\n      ],\n      [\n        "ia",\n        "ble"\n      ],\n      [\n        "Ġgr",\n        "own"\n      ],\n      [\n        "ĠC",\n        "l"\n      ],\n      [\n        "Ġtruck",\n        "s"\n      ],\n      [\n        "Ġbright",\n        "ly"\n      ],\n      [\n        "Ġsound",\n        "ed"\n      ],\n      [\n        "Ġcra",\n        "wl"\n      ],\n      [\n        "Ġrel",\n        "ax"\n      ],\n      [\n        "Ġrel",\n        "iable"\n      ],\n      [\n        "ĠTo",\n        "day"\n      ],\n      [\n        "ol",\n        "a"\n      ],\n      [\n        "Ġap",\n        "olog"\n      ],\n      [\n        "Ġang",\n        "el"\n      ],\n      [\n        "Ġtoug",\n        "h"\n      ],\n      [\n        "Ġsug",\n        "ar"\n      ],\n      [\n        "Ġfier",\n        "ce"\n      ],\n      [\n        "u",\n        "nd"\n      ],\n      [\n        "y",\n        "al"\n      ],\n      [\n        "Ġd",\n        "izzy"\n      ],\n      [\n        "an",\n        "ged"\n      ],\n      [\n        "Ġme",\n        "al"\n      ],\n      [\n        "Ġco",\n        "ins"\n      ],\n      [\n        "Ġsk",\n        "ip"\n      ],\n      [\n        "Ġfr",\n        "idge"\n      ],\n      [\n        "Ġrec",\n        "ord"\n      ],\n      [\n        "Ġbanan",\n        "a"\n      ],\n      [\n        "Ġcirc",\n        "les"\n      ],\n      [\n        "H",\n        "elp"\n      ],\n      [\n        "Ġn",\n        "orm"\n      ],\n      [\n        "Ġth",\n        "ose"\n      ],\n      [\n        "ra",\n        "g"\n      ],\n      [\n        "Ġex",\n        "t"\n      ],\n      [\n        "Ġno",\n        "isy"\n      ],\n      [\n        "Ġsc",\n        "are"\n      ],\n      [\n        "ĠL",\n        "ittle"\n      ],\n      [\n        "Ġstr",\n        "anger"\n      ],\n      [\n        "Ġoff",\n        "ice"\n      ],\n      [\n        "Ġwr",\n        "ote"\n      ],\n      [\n        "Ġfav",\n        "our"\n      ],\n      [\n        "Ġlea",\n        "ving"\n      ],\n      [\n        "Ġlight",\n        "s"\n      ],\n      [\n        "Ġcheer",\n        "ful"\n      ],\n      [\n        "R",\n        "e"\n      ],\n      [\n        "Ġ",\n        "No"\n      ],\n      [\n        "he",\n        "ad"\n      ],\n      [\n        "ar",\n        "oo"\n      ],\n      [\n        "Ġshe",\n        "ep"\n      ],\n      [\n        "Ġso",\n        "ck"\n      ],\n      [\n        "ra",\n        "ge"\n      ],\n      [\n        "Ġot",\n        "ter"\n      ],\n      [\n        "ap",\n        "a"\n      ],\n      [\n        "Ġmo",\n        "v"\n      ],\n      [\n        "Ġgr",\n        "umpy"\n      ],\n      [\n        "Ġdream",\n        "ing"\n      ],\n      [\n        "ang",\n        "aroo"\n      ],\n      [\n        "Ġcray",\n        "on"\n      ],\n      [\n        "Ġenvel",\n        "ope"\n      ],\n      [\n        "Ġtrump",\n        "et"\n      ],\n      [\n        "a",\n        "f"\n      ],\n      [\n        "l",\n        "oo"\n      ],\n      [\n        "Ġf",\n        "u"\n      ],\n      [\n        "Ġm",\n        "is"\n      ],\n      [\n        "The",\n        "ir"\n      ],\n      [\n        "Ġk",\n        "angaroo"\n      ],\n      [\n        "Ġlo",\n        "se"\n      ],\n      [\n        "Ġcl",\n        "umsy"\n      ],\n      [\n        "ell",\n        "a"\n      ],\n      [\n        "Ġcar",\n        "p"\n      ],\n      [\n        "Ġch",\n        "ase"\n      ],\n      [\n        "Ġbox",\n        "es"\n      ],\n      [\n        "bo",\n        "dy"\n      ],\n      [\n        "Ġflut",\n        "e"\n      ],\n      [\n        "Ġescap",\n        "e"\n      ],\n      [\n        "Ġtur",\n        "key"\n      ],\n      [\n        "Ġcam",\n        "era"\n      ],\n      [\n        "Ġnorm",\n        "al"\n      ],\n      [\n        "Ġfavour",\n        "ite"\n      ],\n      [\n        "r",\n        "op"\n      ],\n      [\n        "Ġ",\n        "ind"\n      ],\n      [\n        "Ġha",\n        "m"\n      ],\n      [\n        "et",\n        "it"\n      ],\n      [\n        "Ġbe",\n        "et"\n      ],\n      [\n        "Ġr",\n        "at"\n      ],\n      [\n        "Ġsh",\n        "ake"\n      ],\n      [\n        "Ġlo",\n        "yal"\n      ],\n      [\n        "Ġgo",\n        "at"\n      ],\n      [\n        "Ġcl",\n        "ay"\n      ],\n      [\n        "Ġco",\n        "co"\n      ],\n      [\n        "ĠF",\n        "r"\n      ],\n      [\n        "Ġsl",\n        "ipped"\n      ],\n      [\n        "em",\n        "o"\n      ],\n      [\n        "ipp",\n        "o"\n      ],\n      [\n        "Ġstu",\n        "dy"\n      ],\n      [\n        "Ġorgan",\n        "ized"\n      ],\n      [\n        "Ġgas",\n        "ped"\n      ],\n      [\n        "Ġcoco",\n        "a"\n      ],\n      [\n        "A",\n        "lice"\n      ],\n      [\n        "f",\n        "a"\n      ],\n      [\n        "i",\n        "v"\n      ],\n      [\n        "o",\n        "ppy"\n      ],\n      [\n        "Ġf",\n        "ountain"\n      ],\n      [\n        "il",\n        "ty"\n      ],\n      [\n        "Ġtw",\n        "ig"\n      ],\n      [\n        "Ġpick",\n        "s"\n      ],\n      [\n        "Ġad",\n        "vice"\n      ],\n      [\n        "ber",\n        "ries"\n      ],\n      [\n        "Ġbag",\n        "s"\n      ],\n      [\n        "Ġcomp",\n        "etit"\n      ],\n      [\n        "Ġgu",\n        "ilty"\n      ],\n      [\n        "Ġfeather",\n        "s"\n      ],\n      [\n        "Ġer",\n        "aser"\n      ],\n      [\n        "Ġs",\n        "na"\n      ],\n      [\n        "in",\n        "al"\n      ],\n      [\n        "Ġd",\n        "ull"\n      ],\n      [\n        "Ġst",\n        "airs"\n      ],\n      [\n        "Ġsp",\n        "un"\n      ],\n      [\n        "ĠW",\n        "hile"\n      ],\n      [\n        "Ġyour",\n        "s"\n      ],\n      [\n        "Ġbu",\n        "ilt"\n      ],\n      [\n        "Ġus",\n        "ing"\n      ],\n      [\n        "Ġloud",\n        "ly"\n      ],\n      [\n        "ac",\n        "he"\n      ],\n      [\n        "Ġz",\n        "ip"\n      ],\n      [\n        "Ġob",\n        "ject"\n      ],\n      [\n        "Ġbutt",\n        "ons"\n      ],\n      [\n        "Ġstret",\n        "ch"\n      ],\n      [\n        "Ġber",\n        "ries"\n      ],\n      [\n        "Ġsugg",\n        "ested"\n      ],\n      [\n        "S",\n        "ee"\n      ],\n      [\n        "Ġf",\n        "re"\n      ],\n      [\n        "Ġo",\n        "w"\n      ],\n      [\n        "ll",\n        "ie"\n      ],\n      [\n        "ig",\n        "inal"\n      ],\n      [\n        "ent",\n        "ion"\n      ],\n      [\n        "ĠM",\n        "ar"\n      ],\n      [\n        "Ġsun",\n        "fl"\n      ],\n      [\n        "Ġmo",\n        "le"\n      ],\n      [\n        "Ġstr",\n        "ugg"\n      ],\n      [\n        "Ġoff",\n        "ered"\n      ],\n      [\n        "Ġwar",\n        "ned"\n      ],\n      [\n        "Ġset",\n        "t"\n      ],\n      [\n        "Ġdelic",\n        "ate"\n      ],\n      [\n        "Ġsho",\n        "ot"\n      ],\n      [\n        "Ġpa",\n        "le"\n      ],\n      [\n        "Ġthin",\n        "king"\n      ],\n      [\n        "Ġres",\n        "c"\n      ],\n      [\n        "Ġcoo",\n        "king"\n      ],\n      [\n        "Ġeld",\n        "erly"\n      ],\n      [\n        "Ġext",\n        "ra"\n      ],\n      [\n        "Ġm",\n        "ay"\n      ],\n      [\n        "Ġg",\n        "em"\n      ],\n      [\n        "an",\n        "ny"\n      ],\n      [\n        "Ġst",\n        "ri"\n      ],\n      [\n        "Ġhapp",\n        "iness"\n      ],\n      [\n        "ĠM",\n        "y"\n      ],\n      [\n        "Ġis",\n        "land"\n      ],\n      [\n        "op",\n        "us"\n      ],\n      [\n        "Ġch",\n        "ain"\n      ],\n      [\n        "Ġlearn",\n        "ing"\n      ],\n      [\n        "Ġtr",\n        "op"\n      ],\n      [\n        "Ġor",\n        "iginal"\n      ],\n      [\n        "Ġpu",\n        "p"\n      ],\n      [\n        "âĤ¬â",\n        "Ģ"\n      ],\n      [\n        "ct",\n        "opus"\n      ],\n      [\n        "Ġpat",\n        "ch"\n      ],\n      [\n        "Ġdoes",\n        "n"\n      ],\n      [\n        "Ġju",\n        "icy"\n      ],\n      [\n        "Ġdes",\n        "ign"\n      ],\n      [\n        "Ġreg",\n        "ular"\n      ],\n      [\n        "f",\n        "ast"\n      ],\n      [\n        "k",\n        "n"\n      ],\n      [\n        "Ġo",\n        "ctopus"\n      ],\n      [\n        "ĠS",\n        "m"\n      ],\n      [\n        "Ġy",\n        "og"\n      ],\n      [\n        "oo",\n        "p"\n      ],\n      [\n        "ur",\n        "ing"\n      ],\n      [\n        "Ġsay",\n        "ing"\n      ],\n      [\n        "Ġcare",\n        "less"\n      ],\n      [\n        "Ġsol",\n        "ve"\n      ],\n      [\n        "Ġt",\n        "ummy"\n      ],\n      [\n        "Ġt",\n        "imes"\n      ],\n      [\n        "Ġc",\n        "orn"\n      ],\n      [\n        "Ġm",\n        "elt"\n      ],\n      [\n        "Ġp",\n        "ant"\n      ],\n      [\n        "Ġbe",\n        "es"\n      ],\n      [\n        "Ġli",\n        "cked"\n      ],\n      [\n        "Ġj",\n        "ack"\n      ],\n      [\n        "Ġno",\n        "sy"\n      ],\n      [\n        "ell",\n        "ig"\n      ],\n      [\n        "Ġch",\n        "ir"\n      ],\n      [\n        "Ġv",\n        "an"\n      ],\n      [\n        "Ġthought",\n        "ful"\n      ],\n      [\n        "Ġint",\n        "ellig"\n      ],\n      [\n        "Ġshow",\n        "ing"\n      ],\n      [\n        "os",\n        "ie"\n      ],\n      [\n        "Ġmo",\n        "der"\n      ],\n      [\n        "Ġfr",\n        "ight"\n      ],\n      [\n        "ĠGra",\n        "ce"\n      ],\n      [\n        "Ġbreak",\n        "fast"\n      ],\n      [\n        "Ġquest",\n        "ion"\n      ],\n      [\n        "Ġpir",\n        "ate"\n      ],\n      [\n        "Ġpee",\n        "ked"\n      ],\n      [\n        "Ġcarp",\n        "et"\n      ],\n      [\n        "Ġtrop",\n        "hy"\n      ],\n      [\n        "c",\n        "om"\n      ],\n      [\n        "l",\n        "ies"\n      ],\n      [\n        "Ġh",\n        "ippo"\n      ],\n      [\n        "it",\n        "her"\n      ],\n      [\n        "Ġst",\n        "umbled"\n      ],\n      [\n        "Ġmom",\n        "s"\n      ],\n      [\n        "Ġdo",\n        "ve"\n      ],\n      [\n        "Ġj",\n        "elly"\n      ],\n      [\n        "Ġhelp",\n        "s"\n      ],\n      [\n        "Ġold",\n        "er"\n      ],\n      [\n        "Ġknow",\n        "s"\n      ],\n      [\n        "Ġsk",\n        "in"\n      ],\n      [\n        "Ġen",\n        "c"\n      ],\n      [\n        "Ġclean",\n        "ing"\n      ],\n      [\n        "Ġmag",\n        "n"\n      ],\n      [\n        "Ġhat",\n        "s"\n      ],\n      [\n        "Ġatt",\n        "ic"\n      ],\n      [\n        "Ġcelebr",\n        "ate"\n      ],\n      [\n        "Ġser",\n        "ious"\n      ],\n      [\n        "ĠRe",\n        "x"\n      ],\n      [\n        "kn",\n        "own"\n      ],\n      [\n        "Ġmoder",\n        "n"\n      ],\n      [\n        "re",\n        "ss"\n      ],\n      [\n        "Ġf",\n        "ine"\n      ],\n      [\n        "om",\n        "ed"\n      ],\n      [\n        "Ġher",\n        "o"\n      ],\n      [\n        "Ġsh",\n        "one"\n      ],\n      [\n        "Ġre",\n        "fused"\n      ],\n      [\n        "Ġal",\n        "ert"\n      ],\n      [\n        "ous",\n        "es"\n      ],\n      [\n        "ist",\n        "ed"\n      ],\n      [\n        "Ġproud",\n        "ly"\n      ],\n      [\n        "ĠP",\n        "apa"\n      ],\n      [\n        "oom",\n        "y"\n      ],\n      [\n        "Ġneighb",\n        "our"\n      ],\n      [\n        "Ġsal",\n        "ad"\n      ],\n      [\n        "Ġbubble",\n        "s"\n      ],\n      [\n        "Ġchick",\n        "en"\n      ],\n      [\n        "ĠBl",\n        "ue"\n      ],\n      [\n        "Ġintellig",\n        "ent"\n      ],\n      [\n        "t",\n        "ime"\n      ],\n      [\n        "he",\n        "t"\n      ],\n      [\n        "ĠT",\n        "oby"\n      ],\n      [\n        "or",\n        "ant"\n      ],\n      [\n        "Ġr",\n        "ich"\n      ],\n      [\n        "ug",\n        "ged"\n      ],\n      [\n        "Ġle",\n        "op"\n      ],\n      [\n        "Ġle",\n        "ash"\n      ],\n      [\n        "Ġend",\n        "ing"\n      ],\n      [\n        "He",\n        "re"\n      ],\n      [\n        "ĠP",\n        "olly"\n      ],\n      [\n        "ars",\n        "h"\n      ],\n      [\n        "Ġinv",\n        "ited"\n      ],\n      [\n        "Ġcir",\n        "cle"\n      ],\n      [\n        "Ġadm",\n        "ired"\n      ],\n      [\n        "Ġegg",\n        "s"\n      ],\n      [\n        "Ġzipp",\n        "er"\n      ],\n      [\n        "a",\n        "z"\n      ],\n      [\n        "in",\n        "es"\n      ],\n      [\n        "Ġh",\n        "ouses"\n      ],\n      [\n        "Ġm",\n        "ap"\n      ],\n      [\n        "Ġp",\n        "ow"\n      ],\n      [\n        "Ġo",\n        "y"\n      ],\n      [\n        "Ġst",\n        "ream"\n      ],\n      [\n        "Ġso",\n        "fa"\n      ],\n      [\n        "Ġli",\n        "br"\n      ],\n      [\n        "Ġlo",\n        "cked"\n      ],\n      [\n        "ĠA",\n        "l"\n      ],\n      [\n        "Ġle",\n        "an"\n      ],\n      [\n        "Ġsw",\n        "ord"\n      ],\n      [\n        "Ġhard",\n        "er"\n      ],\n      [\n        "ĠN",\n        "emo"\n      ],\n      [\n        "Ġper",\n        "m"\n      ],\n      [\n        "Ġdream",\n        "ed"\n      ],\n      [\n        "ĠGra",\n        "ndpa"\n      ],\n      [\n        "Ġcount",\n        "ed"\n      ],\n      [\n        "Ġpen",\n        "gu"\n      ],\n      [\n        "oph",\n        "ie"\n      ],\n      [\n        "S",\n        "p"\n      ],\n      [\n        "Ġ",\n        "ing"\n      ],\n      [\n        "Ġt",\n        "urt"\n      ],\n      [\n        "Ġp",\n        "h"\n      ],\n      [\n        "Ġg",\n        "um"\n      ],\n      [\n        "Ġg",\n        "ym"\n      ],\n      [\n        "Ġfriends",\n        "hip"\n      ],\n      [\n        "ag",\n        "het"\n      ],\n      [\n        "ist",\n        "ent"\n      ],\n      [\n        "Ġtr",\n        "ack"\n      ],\n      [\n        "Ġhand",\n        "le"\n      ],\n      [\n        "Ġra",\n        "ced"\n      ],\n      [\n        "ĠG",\n        "eorge"\n      ],\n      [\n        "Ġma",\n        "ze"\n      ],\n      [\n        "red",\n        "ient"\n      ],\n      [\n        "Ġpig",\n        "e"\n      ],\n      [\n        "Ġlift",\n        "ed"\n      ],\n      [\n        "Ġpar",\n        "ade"\n      ],\n      [\n        "Ġhur",\n        "ry"\n      ],\n      [\n        "Ġcand",\n        "le"\n      ],\n      [\n        "Ġbeet",\n        "le"\n      ],\n      [\n        "Ġfre",\n        "sh"\n      ],\n      [\n        "ĠSm",\n        "ith"\n      ],\n      [\n        "Ġfright",\n        "ened"\n      ],\n      [\n        "Ġing",\n        "redient"\n      ],\n      [\n        "aghet",\n        "ti"\n      ],\n      [\n        "M",\n        "ary"\n      ],\n      [\n        "e",\n        "xt"\n      ],\n      [\n        "g",\n        "ust"\n      ],\n      [\n        "Ġ",\n        "U"\n      ],\n      [\n        "Ġd",\n        "ig"\n      ],\n      [\n        "Ġwas",\n        "hed"\n      ],\n      [\n        "Ġr",\n        "ushed"\n      ],\n      [\n        "Ġr",\n        "oof"\n      ],\n      [\n        "Ġplay",\n        "ful"\n      ],\n      [\n        "Ġfor",\n        "th"\n      ],\n      [\n        "Ġsp",\n        "aghetti"\n      ],\n      [\n        "Ġch",\n        "anged"\n      ],\n      [\n        "Ġshow",\n        "s"\n      ],\n      [\n        "Ġsor",\n        "ts"\n      ],\n      [\n        "ĠWhen",\n        "ever"\n      ],\n      [\n        "Ġsto",\n        "ve"\n      ],\n      [\n        "Ġun",\n        "l"\n      ],\n      [\n        "Ġru",\n        "bbed"\n      ],\n      [\n        "Ġign",\n        "orant"\n      ],\n      [\n        "Ġpil",\n        "ot"\n      ],\n      [\n        "Ġcri",\n        "cket"\n      ],\n      [\n        "Ġsunfl",\n        "ower"\n      ],\n      [\n        "n",\n        "ow"\n      ],\n      [\n        "u",\n        "d"\n      ],\n      [\n        "y",\n        "ear"\n      ],\n      [\n        "Ġs",\n        "udden"\n      ],\n      [\n        "Ġh",\n        "ook"\n      ],\n      [\n        "er",\n        "able"\n      ],\n      [\n        "Ġp",\n        "ed"\n      ],\n      [\n        "Ġe",\n        "arth"\n      ],\n      [\n        "Ġst",\n        "amp"\n      ],\n      [\n        "Ġli",\n        "z"\n      ],\n      [\n        "Ġbird",\n        "ie"\n      ],\n      [\n        "ĠF",\n        "red"\n      ],\n      [\n        "Ġpe",\n        "pper"\n      ],\n      [\n        "Ġsn",\n        "iff"\n      ],\n      [\n        "Ġar",\n        "row"\n      ],\n      [\n        "Ġloud",\n        "er"\n      ],\n      [\n        "Ġun",\n        "known"\n      ],\n      [\n        "Ġdis",\n        "gust"\n      ],\n      [\n        "Ġba",\n        "ld"\n      ],\n      [\n        "Ġdream",\n        "s"\n      ],\n      [\n        "Ġshap",\n        "e"\n      ],\n      [\n        "Ġspo",\n        "ke"\n      ],\n      [\n        "Ġsearch",\n        "ing"\n      ],\n      [\n        "Ġspe",\n        "ed"\n      ],\n      [\n        "Ġwip",\n        "ed"\n      ],\n      [\n        "Ġharm",\n        "less"\n      ],\n      [\n        "Ġmis",\n        "erable"\n      ],\n      [\n        "Ġingredient",\n        "s"\n      ],\n      [\n        "g",\n        "est"\n      ],\n      [\n        "Ġs",\n        "on"\n      ],\n      [\n        "Ġto",\n        "ast"\n      ],\n      [\n        "Ġb",\n        "in"\n      ],\n      [\n        "Ġb",\n        "ase"\n      ],\n      [\n        "Ġbig",\n        "gest"\n      ],\n      [\n        "Ġsw",\n        "e"\n      ],\n      [\n        "ĠF",\n        "in"\n      ],\n      [\n        "Ġcom",\n        "b"\n      ],\n      [\n        "Ġsn",\n        "ugg"\n      ],\n      [\n        "ĠJoe",\n        "y"\n      ],\n      [\n        "Ġfall",\n        "s"\n      ],\n      [\n        "Ġwhis",\n        "p"\n      ],\n      [\n        "Ġspoil",\n        "ed"\n      ],\n      [\n        "Ġchar",\n        "ming"\n      ],\n      [\n        "Ġmov",\n        "ie"\n      ],\n      [\n        "Ġenc",\n        "ou"\n      ],\n      [\n        "Ġph",\n        "ot"\n      ],\n      [\n        "Ġf",\n        "our"\n      ],\n      [\n        "Ġl",\n        "amp"\n      ],\n      [\n        "Ġc",\n        "udd"\n      ],\n      [\n        "at",\n        "ch"\n      ],\n      [\n        "Ġsh",\n        "ield"\n      ],\n      [\n        "Ġsp",\n        "ace"\n      ],\n      [\n        "Ġsc",\n        "ale"\n      ],\n      [\n        "Ġco",\n        "st"\n      ],\n      [\n        "Ġcouldn",\n        "Ã¢"\n      ],\n      [\n        "Ġway",\n        "s"\n      ],\n      [\n        "app",\n        "ing"\n      ],\n      [\n        "Ġshop",\n        "keeper"\n      ],\n      [\n        "Ġtom",\n        "ato"\n      ],\n      [\n        "Ġdisco",\n        "ver"\n      ],\n      [\n        "Ġhel",\n        "m"\n      ],\n      [\n        "com",\n        "fort"\n      ],\n      [\n        "Ġt",\n        "ire"\n      ],\n      [\n        "Ġa",\n        "unt"\n      ],\n      [\n        "Ġb",\n        "ump"\n      ],\n      [\n        "Ġc",\n        "ush"\n      ],\n      [\n        "Ġm",\n        "ask"\n      ],\n      [\n        "Ġu",\n        "mb"\n      ],\n      [\n        "se",\n        "um"\n      ],\n      [\n        "Ġla",\n        "und"\n      ],\n      [\n        "bb",\n        "orn"\n      ],\n      [\n        "Ġen",\n        "orm"\n      ],\n      [\n        "Ġdon",\n        "Ã¢"\n      ],\n      [\n        "Ġmean",\n        "s"\n      ],\n      [\n        "Ġtell",\n        "ing"\n      ],\n      [\n        "Ġbat",\n        "ter"\n      ],\n      [\n        "Ġgu",\n        "ard"\n      ],\n      [\n        "Ġstu",\n        "bborn"\n      ],\n      [\n        "Ġpers",\n        "istent"\n      ],\n      [\n        "ho",\n        "st"\n      ],\n      [\n        "Ġear",\n        "ly"\n      ],\n      [\n        "Ġmic",\n        "rop"\n      ],\n      [\n        "eez",\n        "ed"\n      ],\n      [\n        "Ġpant",\n        "s"\n      ],\n      [\n        "Ġturt",\n        "le"\n      ],\n      [\n        "Ġwhisp",\n        "ered"\n      ],\n      [\n        "Ġumb",\n        "re"\n      ],\n      [\n        "Ġenorm",\n        "ous"\n      ],\n      [\n        "Ġmicrop",\n        "hone"\n      ],\n      [\n        "Ġc",\n        "ell"\n      ],\n      [\n        "Ġn",\n        "urse"\n      ],\n      [\n        "ck",\n        "ey"\n      ],\n      [\n        "Ġy",\n        "et"\n      ],\n      [\n        "Ġlo",\n        "ad"\n      ],\n      [\n        "ill",\n        "a"\n      ],\n      [\n        "Ġthem",\n        "selves"\n      ],\n      [\n        "Ġro",\n        "ared"\n      ],\n      [\n        "ĠW",\n        "ould"\n      ],\n      [\n        "ĠYou",\n        "r"\n      ],\n      [\n        "Ġmu",\n        "seum"\n      ],\n      [\n        "Ġagain",\n        "st"\n      ],\n      [\n        "Ġho",\n        "ckey"\n      ],\n      [\n        "ĠP",\n        "e"\n      ],\n      [\n        "Ġjo",\n        "ined"\n      ],\n      [\n        "Ġun",\n        "comfort"\n      ],\n      [\n        "Ġsand",\n        "c"\n      ],\n      [\n        "Ġcount",\n        "ing"\n      ],\n      [\n        "Ġpack",\n        "age"\n      ],\n      [\n        "Ġkick",\n        "ed"\n      ],\n      [\n        "Ġnut",\n        "s"\n      ],\n      [\n        "Ġtool",\n        "s"\n      ],\n      [\n        "Ġliz",\n        "ard"\n      ],\n      [\n        "Ġuncomfort",\n        "able"\n      ],\n      [\n        "W",\n        "hile"\n      ],\n      [\n        "b",\n        "led"\n      ],\n      [\n        "c",\n        "ream"\n      ],\n      [\n        "x",\n        "e"\n      ],\n      [\n        "Ġt",\n        "ied"\n      ],\n      [\n        "Ġa",\n        "xe"\n      ],\n      [\n        "Ġf",\n        "lex"\n      ],\n      [\n        "Ġc",\n        "amp"\n      ],\n      [\n        "Ġth",\n        "irst"\n      ],\n      [\n        "Ġth",\n        "ief"\n      ],\n      [\n        "ir",\n        "it"\n      ],\n      [\n        "Ġan",\n        "x"\n      ],\n      [\n        "Ġlo",\n        "ll"\n      ],\n      [\n        "Ġsad",\n        "ly"\n      ],\n      [\n        "Ġen",\n        "v"\n      ],\n      [\n        "Ġtra",\n        "ins"\n      ],\n      [\n        "Ġpri",\n        "ze"\n      ],\n      [\n        "Ġfil",\n        "m"\n      ],\n      [\n        "Ġleop",\n        "ard"\n      ],\n      [\n        "Ġlaund",\n        "ry"\n      ],\n      [\n        "c",\n        "ope"\n      ],\n      [\n        "p",\n        "id"\n      ],\n      [\n        "Ġs",\n        "uit"\n      ],\n      [\n        "Ġw",\n        "itch"\n      ],\n      [\n        "in",\n        "ary"\n      ],\n      [\n        "er",\n        "ry"\n      ],\n      [\n        "ĠS",\n        "andy"\n      ],\n      [\n        "ĠL",\n        "ook"\n      ],\n      [\n        "Ġch",\n        "asing"\n      ],\n      [\n        "Ġv",\n        "i"\n      ],\n      [\n        "ĠD",\n        "ave"\n      ],\n      [\n        "ip",\n        "op"\n      ],\n      [\n        "Ġrest",\n        "aur"\n      ],\n      [\n        "Ġru",\n        "les"\n      ],\n      [\n        "Ġballoon",\n        "s"\n      ],\n      [\n        "Ġgrow",\n        "ing"\n      ],\n      [\n        "gg",\n        "ing"\n      ],\n      [\n        "Ġstu",\n        "pid"\n      ],\n      [\n        "Ġbar",\n        "ber"\n      ],\n      [\n        "Ġcarry",\n        "ing"\n      ],\n      [\n        "Ġimag",\n        "ined"\n      ],\n      [\n        "Ġpige",\n        "on"\n      ],\n      [\n        "Ġthirst",\n        "y"\n      ],\n      [\n        "Ġloll",\n        "ipop"\n      ],\n      [\n        "Ġrestaur",\n        "ant"\n      ],\n      [\n        "Y",\n        "ay"\n      ],\n      [\n        "i",\n        "ence"\n      ],\n      [\n        "he",\n        "art"\n      ],\n      [\n        "Ġa",\n        "head"\n      ],\n      [\n        "Ġwa",\n        "ke"\n      ],\n      [\n        "Ġf",\n        "rag"\n      ],\n      [\n        "Ġl",\n        "ed"\n      ],\n      [\n        "Ġl",\n        "ow"\n      ],\n      [\n        "Ġg",\n        "host"\n      ],\n      [\n        "le",\n        "v"\n      ],\n      [\n        "Ġk",\n        "itty"\n      ],\n      [\n        "Ġre",\n        "p"\n      ],\n      [\n        "Ġnot",\n        "ice"\n      ],\n      [\n        "Ġsp",\n        "illed"\n      ],\n      [\n        "Ġsc",\n        "reen"\n      ],\n      [\n        "Ġch",\n        "ose"\n      ],\n      [\n        "Ġfind",\n        "ing"\n      ],\n      [\n        "Ġcom",\n        "pl"\n      ],\n      [\n        "Ġru",\n        "b"\n      ],\n      [\n        "Ġhun",\n        "g"\n      ],\n      [\n        "Ġcomp",\n        "ass"\n      ],\n      [\n        "Ġspark",\n        "led"\n      ],\n      [\n        "Ġsil",\n        "ver"\n      ],\n      [\n        "Ġflex",\n        "ible"\n      ],\n      [\n        "e",\n        "er"\n      ],\n      [\n        "Ġh",\n        "ive"\n      ],\n      [\n        "Ġc",\n        "ity"\n      ],\n      [\n        "at",\n        "h"\n      ],\n      [\n        "Ġsh",\n        "ocked"\n      ],\n      [\n        "Ġre",\n        "ind"\n      ],\n      [\n        "Ġsp",\n        "irit"\n      ],\n      [\n        "ic",\n        "t"\n      ],\n      [\n        "Ġme",\n        "m"\n      ],\n      [\n        "Ġho",\n        "se"\n      ],\n      [\n        "Ġany",\n        "where"\n      ],\n      [\n        "Ġqu",\n        "ite"\n      ],\n      [\n        "Ġbeaut",\n        "y"\n      ],\n      [\n        "ĠR",\n        "ose"\n      ],\n      [\n        "Ġstop",\n        "s"\n      ],\n      [\n        "Ġbutterf",\n        "lies"\n      ],\n      [\n        "Ġcreat",\n        "e"\n      ],\n      [\n        "Ġlaugh",\n        "s"\n      ],\n      [\n        "Ġcomp",\n        "let"\n      ],\n      [\n        "Ġwra",\n        "p"\n      ],\n      [\n        "Ġind",\n        "epend"\n      ],\n      [\n        "Ġdisgust",\n        "ing"\n      ],\n      [\n        "Ġanx",\n        "ious"\n      ],\n      [\n        "i",\n        "kes"\n      ],\n      [\n        "p",\n        "es"\n      ],\n      [\n        "Ġb",\n        "ent"\n      ],\n      [\n        "at",\n        "ter"\n      ],\n      [\n        "Ġy",\n        "a"\n      ],\n      [\n        "Ġst",\n        "at"\n      ],\n      [\n        "Ġsh",\n        "iver"\n      ],\n      [\n        "um",\n        "mer"\n      ],\n      [\n        "ie",\n        "w"\n      ],\n      [\n        "ĠE",\n        "ventually"\n      ],\n      [\n        "Ġbl",\n        "oom"\n      ],\n      [\n        "Ġv",\n        "iew"\n      ],\n      [\n        "Ġhand",\n        "ed"\n      ],\n      [\n        "Ġnice",\n        "ly"\n      ],\n      [\n        "Ġun",\n        "pack"\n      ],\n      [\n        "Ġlisten",\n        "ing"\n      ],\n      [\n        "Ġtra",\n        "sh"\n      ],\n      [\n        "ĠK",\n        "ate"\n      ],\n      [\n        "Ġmet",\n        "al"\n      ],\n      [\n        "gg",\n        "y"\n      ],\n      [\n        "Ġfold",\n        "er"\n      ],\n      [\n        "Ġtwirl",\n        "ed"\n      ],\n      [\n        "Ġgrace",\n        "ful"\n      ],\n      [\n        "Ġumbre",\n        "lla"\n      ],\n      [\n        "Ġreind",\n        "eer"\n      ],\n      [\n        "Ġthe",\n        "ater"\n      ],\n      [\n        "Ġh",\n        "ay"\n      ],\n      [\n        "Ġp",\n        "ump"\n      ],\n      [\n        "ar",\n        "nt"\n      ],\n      [\n        "ĠS",\n        "now"\n      ],\n      [\n        "im",\n        "ney"\n      ],\n      [\n        "Ġu",\n        "r"\n      ],\n      [\n        "Ġbe",\n        "ak"\n      ],\n      [\n        "Ġle",\n        "arnt"\n      ],\n      [\n        "Ġch",\n        "imney"\n      ],\n      [\n        "uc",\n        "t"\n      ],\n      [\n        "Ġar",\n        "g"\n      ],\n      [\n        "Ġjo",\n        "kes"\n      ],\n      [\n        "Ġdel",\n        "iver"\n      ],\n      [\n        "Ġbeh",\n        "ave"\n      ],\n      [\n        "Ġsweet",\n        "heart"\n      ],\n      [\n        "Ġsleep",\n        "ing"\n      ],\n      [\n        "ation",\n        "s"\n      ],\n      [\n        "Ġread",\n        "ing"\n      ],\n      [\n        "Ġwindow",\n        "s"\n      ],\n      [\n        "Ġadm",\n        "ire"\n      ],\n      [\n        "Ġpour",\n        "ed"\n      ],\n      [\n        "Ġdest",\n        "ro"\n      ],\n      [\n        "Ġfrag",\n        "ile"\n      ],\n      [\n        "g",\n        "n"\n      ],\n      [\n        "n",\n        "oon"\n      ],\n      [\n        "u",\n        "bby"\n      ],\n      [\n        "Ġw",\n        "ire"\n      ],\n      [\n        "in",\n        "ce"\n      ],\n      [\n        "Ġh",\n        "oop"\n      ],\n      [\n        "re",\n        "nder"\n      ],\n      [\n        "Ġn",\n        "ature"\n      ],\n      [\n        "Ġre",\n        "f"\n      ],\n      [\n        "Ġre",\n        "ce"\n      ],\n      [\n        "ĠM",\n        "ike"\n      ],\n      [\n        "Ġfa",\n        "ctor"\n      ],\n      [\n        "Ġmu",\n        "sh"\n      ],\n      [\n        "Ġch",\n        "ubby"\n      ],\n      [\n        "Ġjo",\n        "lly"\n      ],\n      [\n        "Ġafter",\n        "noon"\n      ],\n      [\n        "Ġstick",\n        "er"\n      ],\n      [\n        "Ġatt",\n        "ract"\n      ],\n      [\n        "Ġcou",\n        "rage"\n      ],\n      [\n        "Ġmail",\n        "box"\n      ],\n      [\n        "Ġrid",\n        "ing"\n      ],\n      [\n        "Ġjack",\n        "et"\n      ],\n      [\n        "Sp",\n        "ot"\n      ],\n      [\n        "Ġhelm",\n        "et"\n      ],\n      [\n        "Ġya",\n        "wn"\n      ],\n      [\n        "Ġfactor",\n        "y"\n      ],\n      [\n        "Ġmush",\n        "room"\n      ],\n      [\n        "m",\n        "ed"\n      ],\n      [\n        "s",\n        "er"\n      ],\n      [\n        "z",\n        "zy"\n      ],\n      [\n        "Ġsh",\n        "ort"\n      ],\n      [\n        "Ġwh",\n        "ale"\n      ],\n      [\n        "Ġre",\n        "co"\n      ],\n      [\n        "ĠM",\n        "ark"\n      ],\n      [\n        "ter",\n        "n"\n      ],\n      [\n        "ĠW",\n        "h"\n      ],\n      [\n        "Ġag",\n        "ree"\n      ],\n      [\n        "ro",\n        "be"\n      ],\n      [\n        "uck",\n        "ed"\n      ],\n      [\n        "Ġen",\n        "g"\n      ],\n      [\n        "Åĵ",\n        "You"\n      ],\n      [\n        "ĠAnd",\n        "y"\n      ],\n      [\n        "Ġtow",\n        "ers"\n      ],\n      [\n        "Ġru",\n        "bb"\n      ],\n      [\n        "Ġpot",\n        "ato"\n      ],\n      [\n        "Ġtrou",\n        "bled"\n      ],\n      [\n        "Ġcreature",\n        "s"\n      ],\n      [\n        "Ġsur",\n        "render"\n      ],\n      [\n        "Ġfoot",\n        "ball"\n      ],\n      [\n        "Ġstat",\n        "ue"\n      ],\n      [\n        "L",\n        "ittle"\n      ],\n      [\n        "a",\n        "id"\n      ],\n      [\n        "h",\n        "ood"\n      ],\n      [\n        "o",\n        "ot"\n      ],\n      [\n        "Ġt",\n        "em"\n      ],\n      [\n        "Ġs",\n        "ym"\n      ],\n      [\n        "Ġh",\n        "arsh"\n      ],\n      [\n        "Ġc",\n        "omet"\n      ],\n      [\n        "Ġc",\n        "ards"\n      ],\n      [\n        "Ġd",\n        "ess"\n      ],\n      [\n        "is",\n        "k"\n      ],\n      [\n        "ir",\n        "on"\n      ],\n      [\n        "Ġro",\n        "se"\n      ],\n      [\n        "Ġmu",\n        "le"\n      ],\n      [\n        "Ġtr",\n        "unk"\n      ],\n      [\n        "Ġra",\n        "ven"\n      ],\n      [\n        "Ġclos",\n        "ely"\n      ],\n      [\n        "Ġhair",\n        "y"\n      ],\n      [\n        "lc",\n        "ano"\n      ],\n      [\n        "Ġplac",\n        "ed"\n      ],\n      [\n        "Ġsur",\n        "f"\n      ],\n      [\n        "Ġfing",\n        "ers"\n      ],\n      [\n        "Ġenv",\n        "ious"\n      ],\n      [\n        "Ġcompass",\n        "ion"\n      ],\n      [\n        "Ġindepend",\n        "ent"\n      ],\n      [\n        "c",\n        "u"\n      ],\n      [\n        "Ġs",\n        "end"\n      ],\n      [\n        "Ġb",\n        "et"\n      ],\n      [\n        "an",\n        "ic"\n      ],\n      [\n        "Ġit",\n        "Ã¢"\n      ],\n      [\n        "Ġit",\n        "self"\n      ],\n      [\n        "Ġst",\n        "ared"\n      ],\n      [\n        "Ġst",\n        "ation"\n      ],\n      [\n        "un",\n        "e"\n      ],\n      [\n        "Ġsh",\n        "aking"\n      ],\n      [\n        "ĠA",\n        "re"\n      ],\n      [\n        "ic",\n        "hes"\n      ],\n      [\n        "Ġstart",\n        "s"\n      ],\n      [\n        "Ġmu",\n        "ff"\n      ],\n      [\n        "Ġfl",\n        "our"\n      ],\n      [\n        "ĠE",\n        "lla"\n      ],\n      [\n        "udd",\n        "y"\n      ],\n      [\n        "Ġpe",\n        "ek"\n      ],\n      [\n        "Ġstick",\n        "ers"\n      ],\n      [\n        "Ġcr",\n        "ane"\n      ],\n      [\n        "Ġpoin",\n        "t"\n      ],\n      [\n        "Ġmiss",\n        "ing"\n      ],\n      [\n        "Ġmix",\n        "er"\n      ],\n      [\n        "Ġdist",\n        "ant"\n      ],\n      [\n        "Ġwhe",\n        "at"\n      ],\n      [\n        "Ġpract",\n        "iced"\n      ],\n      [\n        "Ġshell",\n        "s"\n      ],\n      [\n        "Ġdecor",\n        "ate"\n      ],\n      [\n        "Ġfear",\n        "ful"\n      ],\n      [\n        "Ġvi",\n        "ol"\n      ],\n      [\n        "Ġdess",\n        "ert"\n      ],\n      [\n        "p",\n        "ar"\n      ],\n      [\n        "Ġd",\n        "ough"\n      ],\n      [\n        "Ġo",\n        "ak"\n      ],\n      [\n        "Ġst",\n        "ack"\n      ],\n      [\n        "ch",\n        "anic"\n      ],\n      [\n        "Ġre",\n        "pe"\n      ],\n      [\n        "al",\n        "ed"\n      ],\n      [\n        "ec",\n        "ause"\n      ],\n      [\n        "Ġbo",\n        "ots"\n      ],\n      [\n        "Ġse",\n        "par"\n      ],\n      [\n        "Ġall",\n        "owed"\n      ],\n      [\n        "Ġsc",\n        "atter"\n      ],\n      [\n        "Ġme",\n        "chanic"\n      ],\n      [\n        "us",\n        "ed"\n      ],\n      [\n        "Ġgr",\n        "in"\n      ],\n      [\n        "Ġpain",\n        "ter"\n      ],\n      [\n        "Ġsqu",\n        "ash"\n      ],\n      [\n        "Ġtra",\n        "vel"\n      ],\n      [\n        "Ġdraw",\n        "er"\n      ],\n      [\n        "Ġtri",\n        "ck"\n      ],\n      [\n        "bo",\n        "l"\n      ],\n      [\n        "Ġsec",\n        "ond"\n      ],\n      [\n        "ipp",\n        "ing"\n      ],\n      [\n        "Gra",\n        "ndma"\n      ],\n      [\n        "Ġdestro",\n        "y"\n      ],\n      [\n        "Ġreco",\n        "gn"\n      ],\n      [\n        "Ġs",\n        "ummer"\n      ],\n      [\n        "Ġf",\n        "at"\n      ],\n      [\n        "Ġf",\n        "urt"\n      ],\n      [\n        "Ġg",\n        "or"\n      ],\n      [\n        "Ġsa",\n        "uce"\n      ],\n      [\n        "Ġher",\n        "b"\n      ],\n      [\n        "ĠI",\n        "Ã¢"\n      ],\n      [\n        "ĠB",\n        "aby"\n      ],\n      [\n        "Ġsh",\n        "ark"\n      ],\n      [\n        "Ġsh",\n        "adow"\n      ],\n      [\n        "ic",\n        "op"\n      ],\n      [\n        "ĠW",\n        "here"\n      ],\n      [\n        "ĠL",\n        "ola"\n      ],\n      [\n        "Ġsn",\n        "e"\n      ],\n      [\n        "Ġgl",\n        "oomy"\n      ],\n      [\n        "Ġar",\n        "t"\n      ],\n      [\n        "Ġfr",\n        "ame"\n      ],\n      [\n        "ĠR",\n        "o"\n      ],\n      [\n        "Ġpat",\n        "tern"\n      ],\n      [\n        "Ġsandw",\n        "iches"\n      ],\n      [\n        "Ġhel",\n        "icop"\n      ],\n      [\n        "Ġord",\n        "inary"\n      ],\n      [\n        "Ġmod",\n        "est"\n      ],\n      [\n        "Ġcompetit",\n        "ive"\n      ],\n      [\n        "Ġsym",\n        "bol"\n      ],\n      [\n        "Ġviol",\n        "in"\n      ],\n      [\n        "Ġhelicop",\n        "ter"\n      ],\n      [\n        "O",\n        "w"\n      ],\n      [\n        "Ġm",\n        "ill"\n      ],\n      [\n        "Ġm",\n        "ama"\n      ],\n      [\n        "Ġp",\n        "it"\n      ],\n      [\n        "Ġp",\n        "urse"\n      ],\n      [\n        "Ġo",\n        "l"\n      ],\n      [\n        "or",\n        "able"\n      ],\n      [\n        "ĠB",\n        "ella"\n      ],\n      [\n        "Ġsh",\n        "ine"\n      ],\n      [\n        "Ġimp",\n        "at"\n      ],\n      [\n        "Ġimp",\n        "ro"\n      ],\n      [\n        "Ġte",\n        "leph"\n      ],\n      [\n        "ĠR",\n        "osie"\n      ],\n      [\n        "Ġsail",\n        "or"\n      ],\n      [\n        "Ġpass",\n        "port"\n      ],\n      [\n        "ĠRe",\n        "my"\n      ],\n      [\n        "Ġhum",\n        "ble"\n      ],\n      [\n        "Ġoy",\n        "ster"\n      ],\n      [\n        "Ġlibr",\n        "ary"\n      ],\n      [\n        "Ġattract",\n        "ive"\n      ],\n      [\n        "Ġteleph",\n        "one"\n      ],\n      [\n        "G",\n        "o"\n      ],\n      [\n        "c",\n        "i"\n      ],\n      [\n        "f",\n        "fe"\n      ],\n      [\n        "i",\n        "est"\n      ],\n      [\n        "l",\n        "oe"\n      ],\n      [\n        "y",\n        "er"\n      ],\n      [\n        "Ġc",\n        "ur"\n      ],\n      [\n        "Ġn",\n        "ail"\n      ],\n      [\n        "ri",\n        "c"\n      ],\n      [\n        "el",\n        "ery"\n      ],\n      [\n        "Ġk",\n        "ay"\n      ],\n      [\n        "ill",\n        "ie"\n      ],\n      [\n        "ĠA",\n        "lex"\n      ],\n      [\n        "Ġkn",\n        "ock"\n      ],\n      [\n        "Ġcl",\n        "ock"\n      ],\n      [\n        "Ġstart",\n        "ing"\n      ],\n      [\n        "pl",\n        "ay"\n      ],\n      [\n        "bb",\n        "ie"\n      ],\n      [\n        "Ġcat",\n        "s"\n      ],\n      [\n        "os",\n        "h"\n      ],\n      [\n        "Ġad",\n        "orable"\n      ],\n      [\n        "Ġche",\n        "ap"\n      ],\n      [\n        "Ġgl",\n        "ove"\n      ],\n      [\n        "Ġbright",\n        "er"\n      ],\n      [\n        "Ġbrother",\n        "s"\n      ],\n      [\n        "Ġgi",\n        "ven"\n      ],\n      [\n        "Ġvis",\n        "ited"\n      ],\n      [\n        "Ġroll",\n        "ing"\n      ],\n      [\n        "Ġinc",\n        "l"\n      ],\n      [\n        "Ġorgan",\n        "ize"\n      ],\n      [\n        "Ġlemon",\n        "ade"\n      ],\n      [\n        "Ġseed",\n        "s"\n      ],\n      [\n        "ache",\n        "l"\n      ],\n      [\n        "âĤ¬âĢ",\n        "ľ"\n      ],\n      [\n        "Ġcush",\n        "ion"\n      ],\n      [\n        "Ġfurt",\n        "her"\n      ],\n      [\n        "Ġgor",\n        "illa"\n      ],\n      [\n        "Ġimpat",\n        "ient"\n      ],\n      [\n        "ffe",\n        "e"\n      ],\n      [\n        "n",\n        "ight"\n      ],\n      [\n        "s",\n        "hed"\n      ],\n      [\n        "Ġs",\n        "ize"\n      ],\n      [\n        "is",\n        "ion"\n      ],\n      [\n        "an",\n        "s"\n      ],\n      [\n        "ut",\n        "h"\n      ],\n      [\n        "am",\n        "ond"\n      ],\n      [\n        "ke",\n        "let"\n      ],\n      [\n        "Ġhad",\n        "n"\n      ],\n      [\n        "Ġsp",\n        "ell"\n      ],\n      [\n        "ĠJ",\n        "ul"\n      ],\n      [\n        "Ġsc",\n        "rew"\n      ],\n      [\n        "Ġpo",\n        "le"\n      ],\n      [\n        "Ġen",\n        "er"\n      ],\n      [\n        "Ġnow",\n        "here"\n      ],\n      [\n        "Ġvo",\n        "lcano"\n      ],\n      [\n        "Ġdis",\n        "ag"\n      ],\n      [\n        "Ġdis",\n        "play"\n      ],\n      [\n        "Ġforg",\n        "ave"\n      ],\n      [\n        "Ġz",\n        "eb"\n      ],\n      [\n        "Ġpus",\n        "hing"\n      ],\n      [\n        "Ġste",\n        "al"\n      ],\n      [\n        "ĠK",\n        "im"\n      ],\n      [\n        "Ġve",\n        "nd"\n      ],\n      [\n        "Ġtreasure",\n        "s"\n      ],\n      [\n        "Ġent",\n        "h"\n      ],\n      [\n        "Ġspin",\n        "ning"\n      ],\n      [\n        "Ġdi",\n        "amond"\n      ],\n      [\n        "Ġav",\n        "o"\n      ],\n      [\n        "Ġlaw",\n        "yer"\n      ],\n      [\n        "Ġfu",\n        "el"\n      ],\n      [\n        "Ġzeb",\n        "ra"\n      ],\n      [\n        "D",\n        "aisy"\n      ],\n      [\n        "re",\n        "ci"\n      ],\n      [\n        "is",\n        "on"\n      ],\n      [\n        "Ġsa",\n        "us"\n      ],\n      [\n        "ĠS",\n        "ophie"\n      ],\n      [\n        "Ġr",\n        "ice"\n      ],\n      [\n        "al",\n        "k"\n      ],\n      [\n        "ĠW",\n        "ill"\n      ],\n      [\n        "Ġv",\n        "est"\n      ],\n      [\n        "Ġtr",\n        "uth"\n      ],\n      [\n        "Ġclimb",\n        "ing"\n      ],\n      [\n        "Ġapp",\n        "reci"\n      ],\n      [\n        "Ġboat",\n        "s"\n      ],\n      [\n        "Ġfire",\n        "work"\n      ],\n      [\n        "Ġdri",\n        "ving"\n      ],\n      [\n        "Ġboun",\n        "ced"\n      ],\n      [\n        "Ġexam",\n        "ine"\n      ],\n      [\n        "Ġpop",\n        "ular"\n      ],\n      [\n        "Ġfrown",\n        "ed"\n      ],\n      [\n        "ÅĵIt",\n        "Ã¢"\n      ],\n      [\n        "Ġsnugg",\n        "led"\n      ],\n      [\n        "ĠSnow",\n        "y"\n      ],\n      [\n        "Ġenth",\n        "us"\n      ],\n      [\n        "c",\n        "ome"\n      ],\n      [\n        "e",\n        "ath"\n      ],\n      [\n        "r",\n        "ay"\n      ],\n      [\n        "Ġ",\n        "iron"\n      ],\n      [\n        "Ġto",\n        "r"\n      ],\n      [\n        "Ġc",\n        "ree"\n      ],\n      [\n        "Ġc",\n        "ro"\n      ],\n      [\n        "ĠT",\n        "r"\n      ],\n      [\n        "ar",\n        "ian"\n      ],\n      [\n        "Ġha",\n        "lf"\n      ],\n      [\n        "ĠHe",\n        "re"\n      ],\n      [\n        "Ġshe",\n        "l"\n      ],\n      [\n        "ĠB",\n        "uddy"\n      ],\n      [\n        "Ġme",\n        "as"\n      ],\n      [\n        "Ġtra",\n        "p"\n      ],\n      [\n        "Ġob",\n        "ser"\n      ],\n      [\n        "Ġspr",\n        "ing"\n      ],\n      [\n        "Ġcolour",\n        "s"\n      ],\n      [\n        "Ġcard",\n        "board"\n      ],\n      [\n        "Ġimpre",\n        "ss"\n      ],\n      [\n        "Ġstret",\n        "ched"\n      ],\n      [\n        "cy",\n        "cle"\n      ],\n      [\n        "Ġresc",\n        "ue"\n      ],\n      [\n        "G",\n        "ive"\n      ],\n      [\n        "r",\n        "on"\n      ],\n      [\n        "Ġs",\n        "kelet"\n      ],\n      [\n        "in",\n        "king"\n      ],\n      [\n        "Ġb",\n        "ikes"\n      ],\n      [\n        "Ġl",\n        "end"\n      ],\n      [\n        "Ġl",\n        "uck"\n      ],\n      [\n        "Ġd",\n        "ive"\n      ],\n      [\n        "Ġm",\n        "en"\n      ],\n      [\n        "Ġp",\n        "age"\n      ],\n      [\n        "Ġin",\n        "s"\n      ],\n      [\n        "Ġr",\n        "ug"\n      ],\n      [\n        "Ġsh",\n        "ut"\n      ],\n      [\n        "Ġre",\n        "ve"\n      ],\n      [\n        "Ġis",\n        "n"\n      ],\n      [\n        "Ġme",\n        "at"\n      ],\n      [\n        "Ġra",\n        "ft"\n      ],\n      [\n        "Ġbad",\n        "ly"\n      ],\n      [\n        "Ġcup",\n        "board"\n      ],\n      [\n        "Ġce",\n        "re"\n      ],\n      [\n        "Ġslid",\n        "es"\n      ],\n      [\n        "Ġhunt",\n        "er"\n      ],\n      [\n        "Ġmagn",\n        "et"\n      ],\n      [\n        "Ġpow",\n        "der"\n      ],\n      [\n        "e",\n        "f"\n      ],\n      [\n        "f",\n        "ish"\n      ],\n      [\n        "z",\n        "or"\n      ],\n      [\n        "er",\n        "c"\n      ],\n      [\n        "Ġp",\n        "ad"\n      ],\n      [\n        "Ġp",\n        "air"\n      ],\n      [\n        "Ġg",\n        "ro"\n      ],\n      [\n        "Ġy",\n        "ach"\n      ],\n      [\n        "Ġgo",\n        "al"\n      ],\n      [\n        "ur",\n        "ch"\n      ],\n      [\n        "Ġsome",\n        "where"\n      ],\n      [\n        "Ġco",\n        "ffee"\n      ],\n      [\n        "Ġsun",\n        "set"\n      ],\n      [\n        "Ġra",\n        "zor"\n      ],\n      [\n        "Ġde",\n        "af"\n      ],\n      [\n        "Ġsu",\n        "b"\n      ],\n      [\n        "Ġcoll",\n        "ar"\n      ],\n      [\n        "Ġwis",\n        "hes"\n      ],\n      [\n        "Ġham",\n        "mer"\n      ],\n      [\n        "ĠMar",\n        "ia"\n      ],\n      [\n        "Ġbase",\n        "ball"\n      ],\n      [\n        "Ġcost",\n        "um"\n      ],\n      [\n        "Ġeng",\n        "ine"\n      ],\n      [\n        "Ġvend",\n        "or"\n      ],\n      [\n        "Ġyach",\n        "t"\n      ],\n      [\n        "B",\n        "ecause"\n      ],\n      [\n        "c",\n        "c"\n      ],\n      [\n        "d",\n        "om"\n      ],\n      [\n        "l",\n        "b"\n      ],\n      [\n        "o",\n        "cer"\n      ],\n      [\n        "Ġd",\n        "ug"\n      ],\n      [\n        "an",\n        "y"\n      ],\n      [\n        "Ġr",\n        "hin"\n      ],\n      [\n        "ad",\n        "o"\n      ],\n      [\n        "Ġwe",\n        "ep"\n      ],\n      [\n        "Ġre",\n        "ef"\n      ],\n      [\n        "ind",\n        "y"\n      ],\n      [\n        "Ġnam",\n        "es"\n      ],\n      [\n        "Ġhelp",\n        "less"\n      ],\n      [\n        "Ġsc",\n        "old"\n      ],\n      [\n        "Ġla",\n        "b"\n      ],\n      [\n        "Ġput",\n        "ting"\n      ],\n      [\n        "Ġany",\n        "way"\n      ],\n      [\n        "Ġbu",\n        "lb"\n      ],\n      [\n        "ect",\n        "ed"\n      ],\n      [\n        "ast",\n        "les"\n      ],\n      [\n        "Ġshould",\n        "n"\n      ],\n      [\n        "Ġopen",\n        "s"\n      ],\n      [\n        "Ġexp",\n        "er"\n      ],\n      [\n        "ail",\n        "able"\n      ],\n      [\n        "Ġrespect",\n        "ful"\n      ],\n      [\n        "Ġsupp",\n        "ort"\n      ],\n      [\n        "arl",\n        "ie"\n      ],\n      [\n        "Ġprep",\n        "are"\n      ],\n      [\n        "Jen",\n        "ny"\n      ],\n      [\n        "Ġcree",\n        "k"\n      ],\n      [\n        "ocer",\n        "os"\n      ],\n      [\n        "Ġrhin",\n        "oceros"\n      ],\n      [\n        "c",\n        "orn"\n      ],\n      [\n        "c",\n        "col"\n      ],\n      [\n        "g",\n        "s"\n      ],\n      [\n        "g",\n        "en"\n      ],\n      [\n        "r",\n        "ise"\n      ],\n      [\n        "Ġt",\n        "en"\n      ],\n      [\n        "Ġt",\n        "ut"\n      ],\n      [\n        "Ġw",\n        "ife"\n      ],\n      [\n        "Ġp",\n        "un"\n      ],\n      [\n        "Ġbe",\n        "ep"\n      ],\n      [\n        "st",\n        "airs"\n      ],\n      [\n        "Ġwe",\n        "ather"\n      ],\n      [\n        "Ġre",\n        "ward"\n      ],\n      [\n        "Ġj",\n        "et"\n      ],\n      [\n        "ill",\n        "s"\n      ],\n      [\n        "Ġex",\n        "erc"\n      ],\n      [\n        "âĤ¬",\n        "."\n      ],\n      [\n        "Ġcl",\n        "apping"\n      ],\n      [\n        "Ġsw",\n        "amp"\n      ],\n      [\n        "Ġv",\n        "ide"\n      ],\n      [\n        "Ġv",\n        "ine"\n      ],\n      [\n        "Ġdidn",\n        "Ã¢"\n      ],\n      [\n        "ĠEvery",\n        "thing"\n      ],\n      [\n        "Ġneed",\n        "le"\n      ],\n      [\n        "ĠG",\n        "ive"\n      ],\n      [\n        "Ġhop",\n        "ing"\n      ],\n      [\n        "Ġpap",\n        "ers"\n      ],\n      [\n        "Ġmar",\n        "ched"\n      ],\n      [\n        "Ġres",\n        "ist"\n      ],\n      [\n        "Ġband",\n        "age"\n      ]\n    ]\n  }\n}', 'tests/test_learning_metrics.py': "import unittest\nimport torch\nfrom src.gpt import GPTConfig,TinyGPT\nfrom src.learning_metrics import (build_arm,gradient_norms,snapshot_experts,decay_removed_updates,\n    fixed_hidden_probes,function_diversity,ablate_to_initial)\nfrom src.routing_lab import set_policy\n\nclass LearningMetricTests(unittest.TestCase):\n    def setUp(self):\n        torch.manual_seed(17);torch.set_num_threads(2)\n        self.parent=TinyGPT(GPTConfig(vocab_size=13,context=8,layers=1,width=16,heads=4,intermediate=32)).upcycle()\n        self.cfg={'seed':17,'copies':3,'noise_fraction':.01}\n        self.x=torch.randint(13,(3,8));self.y=torch.randint(13,(3,8))\n\n    def test_family_selection_preserves_parent_and_noise_breaks_router_rows(self):\n        with torch.no_grad():\n            self.parent.blocks[0].ffn.experts[0].down.weight.add_(.02)\n            self.parent.blocks[0].ffn.experts[1].gate.weight.mul_(.8)\n        m=build_arm(self.parent,'family_bias',self.cfg).eval()\n        torch.testing.assert_close(m(self.x)[0],self.parent.eval()(self.x)[0],atol=1e-6,rtol=1e-5)\n        n=build_arm(self.parent,'noise',self.cfg)\n        self.assertFalse(torch.equal(n.blocks[0].ffn.router.weight[0],n.blocks[0].ffn.router.weight[1]))\n        torch.testing.assert_close(n.blocks[0].ffn.experts[0].gate.weight,n.blocks[0].ffn.experts[1].gate.weight)\n\n    def test_decay_only_is_not_counted_as_learned_update(self):\n        m=build_arm(self.parent,'hard',self.cfg)\n        opt=torch.optim.AdamW(m.parameters(),lr=.01,weight_decay=.1)\n        for p in m.parameters():p.grad=torch.zeros_like(p)\n        before=snapshot_experts(m);opt.step()\n        self.assertEqual(float(decay_removed_updates(m,before,.01,.1).max()),0.)\n        opt.zero_grad(set_to_none=True)\n        self.assertEqual(float(gradient_norms(m).max()),0.)\n\n    def test_reset_ablation_zero_for_initial_experts_and_diversity_zero(self):\n        m=build_arm(self.parent,'hard',self.cfg).eval()\n        probes=fixed_hidden_probes(self.parent,self.x,'cpu')\n        self.assertTrue(all(abs(v)<1e-8 for row in function_diversity(m,probes) for v in row))\n        states={k:v.clone() for k,v in m.state_dict().items()}\n        result=ablate_to_initial(m,self.parent,(self.x,self.y),'cpu')\n        self.assertTrue(all(abs(v)<1e-6 for row in result['expert_reset_ce_delta'] for v in row))\n        for k,v in m.state_dict().items():torch.testing.assert_close(v,states[k])\n\n    def test_bias_routes_tied_clones_without_random_sampling(self):\n        m=build_arm(self.parent,'bias',self.cfg).train()\n        with torch.no_grad():m.blocks[0].ffn.router.weight.zero_()\n        seen=torch.zeros(12)\n        set_policy(m,0.,bias_rate=.01)\n        for _ in range(12):\n            _,aux=m(self.x);seen+=aux[0]['counts'].detach()\n        self.assertTrue(bool((seen>0).all()))\n"}
for name,content in payload.items():
    path=Path(name);path.parent.mkdir(parents=True,exist_ok=True);path.write_text(content)
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],check=True)


## Reuse the trained parent

Use the `parent.pt` from the completed original clone experiment. The cell searches existing `/content` files and accepts only the pinned SHA-256. If none survives, upload that checkpoint from the prior evidence archive (or the project's provided `artifacts/parent.pt`). The model is not pretrained again.


In [ ]:
manifest=json.loads(Path("data_specs/parent_manifest.json").read_text())
expected=manifest["parent_sha256"]
parent_path=Path("artifacts/parent.pt")
parent_path.parent.mkdir(exist_ok=True)
found=None
for candidate in Path("/content").glob("**/parent.pt"):
    if hashlib.sha256(candidate.read_bytes()).hexdigest()==expected:
        found=candidate;break
if found is None:
    from google.colab import files
    uploaded=files.upload()
    for name,content in uploaded.items():
        if hashlib.sha256(content).hexdigest()==expected:
            parent_path.write_bytes(content);found=parent_path;break
assert found is not None,"Upload the original clone study's parent.pt; no arbitrary checkpoint is accepted."
if found.resolve()!=parent_path.resolve():shutil.copy2(found,parent_path)
assert hashlib.sha256(parent_path.read_bytes()).hexdigest()==expected
print("Verified trained parent:",parent_path)


## Prepare the identical data/token IDs

Pinned TinyStories revision, 12,000 train / 1,200 validation / 1,200 test stories; reuse the archived 4,096-vocabulary tokenizer. Validate revision, tokenizer checksum and token counts before training. Train/validation/test token arrays are regenerated, and their file hash is recorded.


In [ ]:
from scripts.run_logged import run_logged
from datetime import datetime,timezone
stamp=datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
if not Path("data/tinystories_lab.npz").exists():
    run_logged([sys.executable,"-u","scripts/prepare_tinystories_lab.py",
        "--revision","f54c09fd23315a6f9c86f9dc80f725de7d8f9c64",
        "--tokenizer","data_specs/tokenizer.json"],"work/logs/data-"+stamp+".log")
print(Path("data/tinystories_lab/manifest.json").read_text())
print(Path("configs/t4.json").read_text())


## Train and audit learning

All five arms have 12,030,336 total parameters / 3,182,976 selected-parameter convention, top-2 and batch 4×128. Each sees 384,000 additional targets, with no auxiliary loss so expert gradients are CE-only. Persistent bias updates affect selection only; mixing weights use unbiased logits. Family-aware routing selects two different parents and then one child each, preventing sibling crowding at initialization.

The original 50-update traffic window is retained. Hard calibration uses 64 sequences; held-out reset ablations use a fixed four-sequence validation probe. Per-step gradient telemetry and periodic expert diagnostics add overhead; these are not kernel-speed benchmarks.


In [ ]:
run_dir="results/hard-routing-"+stamp
try:
    run_logged([sys.executable,"-u","-m","src.improve_routing",
        "--config","configs/t4.json","--device","cuda","--output",run_dir],
        "work/logs/train-"+stamp+".log")
    run_logged([sys.executable,"-u","scripts/report_learning.py","--results",run_dir],
        "work/logs/report-"+stamp+".log")
finally:
    # Preserve logs/partial records even if an arm fails.
    archive=shutil.make_archive("/content/MoE_Routing_Followup-"+stamp,"zip",root_dir=root)
    print("Evidence archive:",archive)


In [ ]:
from IPython.display import display,Markdown,Image
report=Path(run_dir,"REPORT.md")
if report.exists():
    display(Markdown(report.read_text()))
    display(Image(filename=str(Path(run_dir,"routing_curves.png"))))
    for arm in json.loads(Path(run_dir,"summary.json").read_text()):
        display(Image(filename=str(Path(run_dir,"experts-"+arm["arm"]+".png"))))
else:
    print("Preserve the archive and inspect the training log before modifying settings.")


## Download before disconnecting

The archive includes source, config, pinned parent/tokenizer, full metrics, figures and per-arm optimizer/RNG checkpoints. Keep failed runs as evidence; do not restart without downloading it.


In [ ]:
from google.colab import files
files.download(archive)
